# Channel Analytics — v9

End-to-end pipeline for the H&D India channel: SAP data → cleaned Sales / OB / Backlog → master-data upkeep → Retail, HS, Partner and SSP reports.

**How to run**

1. Run **0.1 Setup** once per machine.
2. Run **0.2 Run mode** and answer the prompt:
   - `1` → **use cached data** (fast; no SharePoint downloads, no changes to input/master files).
   - `0` → **check SharePoint for new data** (pulls the latest SAP broadcast, refreshes the pre-processed files and the master data, then saves a fresh cache).
3. `Run All` (or run section by section). Every report is a function called through `run_report(...)`, so one failing report doesn't stop the rest — the **Run summary** cell at the end lists what succeeded, what was skipped and what failed.

To switch a report off, set it to `False` in `REPORTS` (cell **0.3 Configuration**).

| Section | What it does |
|---|---|
| 0 | Setup, run mode, configuration, SharePoint helpers, report style, shared business helpers |
| 1 | Data: SAP broadcast → raw files → pre-processed yearly files → `df_Sales_final`, `df_OB_final`, `df_backlog_final` |
| 2 | Master data: Partner Master + Material Master updates, then one load of every master file |
| 3 | Retail reports |
| 4 | HS (Retail + Project) reports |
| 5 | Partner Analysis, New GEO and Insights |
| 6 | SSP performance |
| 7 | Advanced analytics & ad-hoc tools |
| 8 | Archive (superseded code, kept for reference — never executed) |

All Excel reports share one house style (cell **0.5 Report style**): one font, one header colour, the same number formats (₹ `#,##0`, ₹ Mn `#,##0.0`, `0.0%`, `dd-mmm-yyyy`), banded rows, frozen headers, filters and bold totals rows.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 0.1 SETUP — run once per machine (installs anything missing)
# ════════════════════════════════════════════════════════════════════
import importlib
import subprocess
import sys

PACKAGES = {
    # pip name       import name
    "pandas":        "pandas",
    "numpy":         "numpy",
    "openpyxl":      "openpyxl",
    "xlsxwriter":    "xlsxwriter",
    "pyarrow":       "pyarrow",
    "msal":          "msal",
    "requests":      "requests",
    "python-dotenv": "dotenv",
    "scikit-learn":  "sklearn",
    "scipy":         "scipy",
}

for pip_name, import_name in PACKAGES.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"  ⏳ installing {pip_name} ...", end=" ", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "--quiet"])
        print("done")

print("✅ All packages ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 0.2 RUN MODE — cached data or check SharePoint for new data?
#
#   1 = True  → use cached data, NO refresh: Sales and OB are read from the
#               SharePoint "parquet_cache" folders (Pre-processed Sales Data /
#               Pre-processed OB Data); Backlog from its source folder (it has
#               no cache). No SAP broadcast, no conversion, no uploads, no
#               master-data updates — only the reports are written.
#   0 = False → check SharePoint for new data: process the latest SAP
#               broadcast, refresh the pre-processed yearly files, update
#               Partner/Material master data, then save a new cache.
#
# For unattended runs set the environment variable CHANNEL_USE_CACHE=0/1
# and the prompt is skipped.
# ════════════════════════════════════════════════════════════════════
import os


def ask_use_cache():
    preset = os.environ.get("CHANNEL_USE_CACHE")
    if preset in ("0", "1"):
        print(f"CHANNEL_USE_CACHE={preset} (set in environment) — prompt skipped.")
        return preset == "1"
    while True:
        answer = input("Use data from cache? (1 = True: use cache, 0 = False: check for new data): ").strip()
        if answer in ("0", "1"):
            return answer == "1"
        print("  Please type 1 or 0.")


USE_CACHE = ask_use_cache()
print("▶ Run mode:", "USE CACHE (no data refresh)" if USE_CACHE else "CHECK FOR NEW DATA (full refresh)")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 0.3 CONFIGURATION — the only cell you should normally need to edit
# ════════════════════════════════════════════════════════════════════
import gc
import io
import json
import math
import os
import re
import sys
import time
import traceback
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

pd.set_option("display.max_columns", None)

# --- Secrets: read ONLY from a local .env file / environment -----------
# The .env (never committed) must define SP_TENANT_ID, SP_CLIENT_ID and
# SP_CLIENT_SECRET. If yours lives somewhere else, put its full path here:
ENV_FILE_PATH = ""          # e.g. r"C:\Users\<you>\OneDrive\Channel Analytics\.env"

import getpass
import glob

SECRET_KEYS = ("SP_TENANT_ID", "SP_CLIENT_ID", "SP_CLIENT_SECRET")


def notebook_dir():
    """Folder of this notebook file (VS Code and Jupyter both expose it),
    falling back to the kernel's working folder."""
    path = globals().get("__vsc_ipynb_file__") or os.environ.get("JPY_SESSION_NAME", "")
    return os.path.dirname(os.path.abspath(path)) if path and os.path.exists(path) else os.getcwd()


NOTEBOOK_DIR = notebook_dir()
if os.getcwd() != NOTEBOOK_DIR:
    os.chdir(NOTEBOOK_DIR)          # relative paths (./local_cache, ../Final Reports) resolve next to the notebook
print(f"Notebook folder: {NOTEBOOK_DIR}")

# --- Run log: timestamps on screen + a log file per run -------------------
# Every print() in this notebook goes through log(): it shows
#   HH:MM:SS +mm:ss │ message      (clock time, time since the run started)
# and is also written to logs/Channel_Analytics_<date>_<time>.log.
import builtins
import logging

RUN_START = time.time()
LOG_DIR = os.path.join(NOTEBOOK_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, f"Channel_Analytics_{datetime.now():%Y-%m-%d_%H%M}.log")


class _RunFormatter(logging.Formatter):
    def format(self, record):
        elapsed = int(record.created - RUN_START)
        prefix = f"{datetime.fromtimestamp(record.created):%H:%M:%S} +{elapsed // 60:02d}:{elapsed % 60:02d} │ "
        lines = record.getMessage().splitlines() or [""]
        return "\n".join([prefix + lines[0]] + [" " * len(prefix) + line for line in lines[1:]])


logger = logging.getLogger("channel_analytics")
logger.setLevel(logging.INFO)
logger.propagate = False
for _h in list(logger.handlers):                 # re-running the cell must not double every line
    logger.removeHandler(_h)
    _h.close()
_screen = logging.StreamHandler(sys.stdout)
_screen.setFormatter(_RunFormatter())
_file = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file.setFormatter(_RunFormatter())
logger.addHandler(_screen)
logger.addHandler(_file)


def log(message="", level=None):
    text = str(message)
    if level is None:
        level = logging.ERROR if "❌" in text else logging.WARNING if "⚠️" in text else logging.INFO
    for h in logger.handlers:                    # keep the screen in step with the notebook's own output
        h.flush()
    logger.log(level, text.lstrip("\n"))


def print(*args, sep=" ", end="\n", file=None, flush=False):   # noqa: A001 — routes this notebook's prints to log()
    if file is not None or end != "\n":
        builtins.print(*args, sep=sep, end=end, file=file, flush=flush)
    else:
        log(sep.join(str(a) for a in args))


print(f"📝 Log file: {LOG_FILE}")


def _env_candidates():
    """Where to look for .env, in order: the path above, CHANNEL_ENV_FILE,
    the notebook's folder (and its parents), the kernel's working folder,
    then home / Documents / OneDrive folders."""
    home = os.path.expanduser("~")
    found = [ENV_FILE_PATH, os.environ.get("CHANNEL_ENV_FILE", "")]
    folder = NOTEBOOK_DIR
    while True:                                    # notebook folder and its parents
        found.append(os.path.join(folder, ".env"))
        parent = os.path.dirname(folder)
        if parent == folder:
            break
        folder = parent
    found += [find_dotenv(usecwd=True), os.path.join(home, ".env"), os.path.join(home, "Documents", ".env")]
    found += glob.glob(os.path.join(home, "OneDrive*", ".env")) + glob.glob(os.path.join(home, "OneDrive*", "*", ".env"))
    return [p for p in dict.fromkeys(found) if p and os.path.isfile(p)]


ENV_FILE = None
for _path in _env_candidates():
    load_dotenv(_path, override=False)
    if all(os.environ.get(k) for k in SECRET_KEYS):
        ENV_FILE = _path
        print(f"Loaded secrets from: {ENV_FILE}")
        break

missing = [k for k in SECRET_KEYS if not os.environ.get(k)]
if missing:
    print(f"⚠️ No .env with {missing} found.\n   Notebook folder: {NOTEBOOK_DIR}\n   Working folder:  {os.getcwd()}\n"
          f"   Also checked: parent folders, {os.path.expanduser('~')}, Documents, OneDrive.\n"
          f"   Files tried: {_env_candidates() or 'none found'}\n"
          f"   Set ENV_FILE_PATH above to fix this permanently.")
    print("   For now, enter them here (input is hidden and not saved):")
    for k in missing:
        os.environ[k] = getpass.getpass(f"   {k}: ").strip()
    missing = [k for k in SECRET_KEYS if not os.environ.get(k)]
    if missing:
        raise EnvironmentError(f"Missing {missing}. Add them to a .env file and set ENV_FILE_PATH in this cell.")

# --- Local folders -------------------------------------------------------
OUTPUT_DIR = os.environ.get("REPORT_OUTPUT_DIR", os.path.expanduser("~/Downloads"))
LOCAL_PARTNER_REPORTS_RETAIL = "../Final Reports/Retail Reports/Partner Reports"
LOCAL_PARTNER_REPORTS_HS = "../Final Reports/HS Reports/Partner Reports"
CACHE_DIR_SALES_RAW = "./local_cache"          # per-file cache of raw Sales downloads
CACHE_DIR_OB_RAW = "./local_cache_ob"          # per-file cache of raw OB downloads
for _d in (OUTPUT_DIR, LOCAL_PARTNER_REPORTS_RETAIL, LOCAL_PARTNER_REPORTS_HS,
           CACHE_DIR_SALES_RAW, CACHE_DIR_OB_RAW):
    os.makedirs(_d, exist_ok=True)

# --- Date anchors (every report uses these) -----------------------------
TODAY = datetime.now()
YESTERDAY = TODAY - timedelta(days=1)             # YTD cut-off
TARGET_YEAR = YESTERDAY.year                      # "current" year of the reports
PREV_YEAR = TARGET_YEAR - 1


def same_day_in_year(dt, year):
    """Same calendar day in another year; 29-Feb falls back to 28-Feb."""
    try:
        return dt.replace(year=year)
    except ValueError:
        return dt.replace(year=year, day=28)


LY_YESTERDAY = same_day_in_year(YESTERDAY, PREV_YEAR)   # last-year YTD cut-off
MONTH_NAME = YESTERDAY.strftime("%B")                   # e.g. "October"
MONTH_ABBR = YESTERDAY.strftime("%b")                   # e.g. "Oct"
HISTORY_YEARS = list(range(2022, TARGET_YEAR + 1))      # years kept in pre-processed data
DASHBOARD_YEAR_LABELS = {TARGET_YEAR: "Year", TARGET_YEAR - 1: "Year-1", TARGET_YEAR - 2: "Year-2"}

# --- SharePoint locations -------------------------------------------------
SITE_MAIN = "schneiderelectric.sharepoint.com:/sites/HDIndiaTeamGroup"
SITE_BROADCAST = "schneiderelectric.sharepoint.com:/sites/HDIndiaTeamGroup-BroadcastReports"
ROOT = "Test - haripriya"
INPUTS = f"{ROOT}/Data Downloads - Input Files"
MASTER = f"{INPUTS}/Master data"

PATHS = {
    # SAP broadcast mailbox (zip per e-mail). The folder is per month; both
    # "EmailMessages_9_2026" and "EmailMessages_09_2026" spellings are tried.
    "broadcast_folders": list(dict.fromkeys([
        f"Apps/Microsoft Teams Mailhook/Shared Documents/EmailMessages_{TODAY.month}_{TODAY.year}",
        f"Apps/Microsoft Teams Mailhook/Shared Documents/EmailMessages_{TODAY.month:02d}_{TODAY.year}",
    ])),
    "raw_sales": f"{INPUTS}/Primary/Sales Data/{TARGET_YEAR}",
    "raw_ob": f"{INPUTS}/Primary/OB Data/{TARGET_YEAR}",
    "raw_backlog": f"{INPUTS}/Primary/Backlog Data/Final backlog",
    "pre_sales": f"{INPUTS}/Primary/Sales Data/Pre-processed Sales Data",
    "pre_ob": f"{INPUTS}/Primary/OB Data/Pre-processed OB Data",
    "dict_retail": f"{MASTER}/Data Dictionary.xlsx",
    "dict_project": f"{MASTER}/Data Dictionary Retail + Project.xlsx",
    "material_master": f"{MASTER}/Material Master Data.xlsx",
    "material_backups": f"{MASTER}/Backups",
    "targets_master": f"{MASTER}/Master Targets File.xlsx",
    "targets_qtod": f"{MASTER}/Targets File/QTOD and ATOD file.xlsx",
    "dashboard_hd": f"{ROOT}/Dashboard Input Files/H&D Channel",
    "retail_reports": f"{ROOT}/Final Reports/Retail Reports",
    "retail_ytd": f"{ROOT}/Final Reports/Retail Reports/YTD Reports",
    "hs_reports": f"{ROOT}/Final Reports/HS Reports",
    "hs_ytd": f"{ROOT}/Final Reports/HS Reports/YTD Reports",
    "final_reports": f"{ROOT}/Final Reports",
    "ssp_reports": f"{ROOT}/Final Reports/SSP Reports",
    "ssp_individual": f"{ROOT}/Final Reports/Individual SSPs",
}

# --- Business rules (single source of truth) ----------------------------
RETAIL_PRICING_L2 = ["ED1", "ED2", "RE3"]
RETAIL_PRICING_L3 = ["ERW", "EKR", "R36", "ZD1"]
RETAIL_HELIOS_CODES = [
    "HDCFD", "HDCWD", "HDCOS", "HDSPE", "HDCLD", "HDCLN", "HDCLS", "HDFDA", "HDFDB",
    "HDFDS", "HDLSD", "HDPNL", "HSCOS", "HSCWD", "HSSPE", "PPCFD", "PREVC",
]
# These variants are kept exactly as each report used them before, so the
# reorganisation doesn't move any numbers. Align them here if they should match.
PARTNER_ANALYSIS_HELIOS_CODES = [c for c in RETAIL_HELIOS_CODES if c != "PREVC"]   # Partner Analysis / New GEO
HS_REPORT_HELIOS_CODES = [c for c in RETAIL_HELIOS_CODES if c != "PPCFD"]          # HS MTD / YTD / COS-SPE / Wiser
HS_LIFETIME_HELIOS_CODES = ["HDCWD", "HDCOS", "HDSPE", "HSCOS", "HSCWD", "HSSPE", "PREVC"]

RETAIL_REBATES = {2026: 0.17, 2025: 0.122, 2024: 0.145, 2023: 0.14, 2022: 0.13}
RETAIL_REBATE_DEFAULT = 0.17
PROJECT_REBATES = {2026: 0.051, 2025: 0.045, 2024: 0.045, 2023: 0.045, 2022: 0.045}
PROJECT_REBATE_DEFAULT = 0.045

METRO_CITIES = ["Bangalore", "Chennai", "Hyderabad", "Secunderabad", "Kolkata", "Mumbai", "Navi Mumbai",
                "Thane", "Pune", "Delhi", "Gurugram", "Faridabad", "Noida", "Ghaziabad", "Pune- PCMC"]
T1_CITIES = ["Ahmedabad", "Bhubaneswar", "Chandigarh", "Coimbatore", "Ernakulam", "Jaipur", "Kozhikode",
             "Lucknow", "Thiruvananthapuram", "Trivandrum", "Vijayawada", "Visakhapatnam"]

# Partners that must always count as "Old", whatever their sales history says.
MANUAL_OLD_PARTNER_IDS = {"500073231"}

# Old Sold-to codes re-issued under a new code ("Partner Code Changes" tab of
# the Data Dictionary) are merged into the new code right after loading, so
# every report sees one partner with one history.
APPLY_PARTNER_CODE_REMAP = True

NEW_PARTNER_TARGETS_BY_ZONE = pd.DataFrame({
    "Zone": ["East", "North", "South", "West1", "West2"],
    "Partner Count Target": [30, 50, 50, 20, 30],
    "Gr.Sales Target": [50, 130, 130, 60, 60],
}).set_index("Zone")

# SSP (sub-stockist) → branch and annual target (Rs Mn). One row per branch.
SSP_MAP = [
    {"Region": "E2", "SSP": "Bihar Allied", "Branch": "JKD", "Annual_Target_Mn": 74.2},
    {"Region": "N", "SSP": "Balaji Sales", "Branch": "UP-2", "Annual_Target_Mn": 95.7},
    {"Region": "N", "SSP": "Balaji Sales", "Branch": "NOIDA-GHZ", "Annual_Target_Mn": 95.7},
    {"Region": "N", "SSP": "Smart Lights", "Branch": "UP-1", "Annual_Target_Mn": 64.5},
    {"Region": "S", "SSP": "Katson Kerala", "Branch": "KER", "Annual_Target_Mn": 729.5},
    {"Region": "W2", "SSP": "Amogh Sales", "Branch": "RAJ", "Annual_Target_Mn": 190.1},
]

# Weekly Pavilion slab file for the optional Pavilion sheets (None = skip).
PAVILION_FILE_PATH = None

# --- Which reports to produce (False = skip) ----------------------------
REPORTS = {
    "retail_dashboard_input": True,
    "retail_mtd": True,
    "retail_ytd": True,
    "retail_partner_lifetime": True,
    "retail_partner_ly_ytd": True,
    "retail_wiser_lifetime": True,
    "retail_product_mix": True,
    "retail_target_vs_achievement": True,
    "hs_dashboard_input": True,
    "hs_mtd": True,
    "hs_ytd": True,
    "hs_ytd_orders": True,           # HS YTD per sales order × material (no dates)
    "hs_cos_spe_ytd": True,
    "hs_wiser_ytd": True,
    "hs_product_mix": True,
    "hs_partner_lifetime": True,
    "hs_wiser_lifetime": True,
    "partner_analysis": True,        # Partner Analysis + New GEO + Insights Summary
    "ssp_performance": True,
    "hs_branch_range_ytd": True,     # HS Gross Sales by Branch × Range, YTD vs LY YTD
    "target_split": True,             # 10: last year's sales contribution + organised targets file
    # Ad-hoc tools — off by default, switch on when needed
    "mumbai_report": False,
    "reference_sku_lookup": False,
    "activity_wise_sales": False,
}
STOP_ON_ERROR = False   # True = stop at the first failing report

print(f"✅ Config loaded | YTD cut-off {YESTERDAY:%d-%b-%Y} (vs {LY_YESTERDAY:%d-%b-%Y}) | "
      f"target year {TARGET_YEAR} | outputs → {OUTPUT_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 0.4 SHAREPOINT HELPERS — one client for the whole notebook
#
# Replaces the ~20 copies of "msal login → find site → find drive" that
# every cell used to carry. The token is re-requested on every call (msal
# caches it and renews it before expiry), so long runs no longer fail with
# "Lifetime validation failed". Network calls are retried with back-off.
# ════════════════════════════════════════════════════════════════════
from urllib.parse import quote

import msal
import requests

GRAPH = "https://graph.microsoft.com/v1.0"
SIMPLE_UPLOAD_LIMIT = 4 * 1024 * 1024    # Graph's limit for a single PUT
UPLOAD_CHUNK = 5 * 1024 * 1024            # multiple of 320 KiB, as Graph requires


class SharePoint:
    def __init__(self):
        self._app = msal.ConfidentialClientApplication(
            os.environ["SP_CLIENT_ID"],
            authority=f"https://login.microsoftonline.com/{os.environ['SP_TENANT_ID']}",
            client_credential=os.environ["SP_CLIENT_SECRET"],
        )
        self._drives = {}

    # --- plumbing ---------------------------------------------------------
    def headers(self):
        token = self._app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        if "access_token" not in token:
            raise PermissionError(f"SharePoint authentication failed: {token.get('error_description', token)}")
        return {"Authorization": f"Bearer {token['access_token']}"}

    def _request(self, method, url, auth=True, attempts=4, **kwargs):
        extra_headers = kwargs.pop("headers", {})
        for attempt in range(1, attempts + 1):
            try:
                hdrs = {**(self.headers() if auth else {}), **extra_headers}
                res = requests.request(method, url, headers=hdrs, timeout=300, **kwargs)
                if res.status_code in (429, 500, 502, 503, 504) and attempt < attempts:
                    raise requests.ConnectionError(f"HTTP {res.status_code}")
                return res
            except (requests.ConnectionError, requests.Timeout) as e:
                if attempt == attempts:
                    raise
                wait = 3 * 2 ** (attempt - 1)
                print(f"   ⚠️ {method} failed ({e}); retrying in {wait}s ...")
                time.sleep(wait)

    def drive_id(self, site=SITE_MAIN):
        if site not in self._drives:
            site_id = self._request("GET", f"{GRAPH}/sites/{site}").json()["id"]
            drives = self._request("GET", f"{GRAPH}/sites/{site_id}/drives").json()["value"]
            self._drives[site] = next(d["id"] for d in drives if d["name"] == "Documents")
        return self._drives[site]

    def _item_url(self, path, site):
        return f"{GRAPH}/drives/{self.drive_id(site)}/root:/{quote(path)}"

    # --- reading ----------------------------------------------------------
    def list_folder(self, folder, site=SITE_MAIN):
        """Items in a folder (each has name, lastModifiedDateTime, @microsoft.graph.downloadUrl)."""
        res = self._request("GET", f"{self._item_url(folder, site)}:/children").json()
        if "value" not in res:
            raise FileNotFoundError(f"SharePoint folder not found: {folder} ({res.get('error', {}).get('message', res)})")
        return res["value"]

    def exists(self, path, site=SITE_MAIN):
        return self._request("GET", self._item_url(path, site)).status_code == 200

    def download_item(self, item):
        return self._request("GET", item["@microsoft.graph.downloadUrl"], auth=False).content

    def download(self, path, site=SITE_MAIN):
        meta = self._request("GET", self._item_url(path, site)).json()
        if "@microsoft.graph.downloadUrl" not in meta:
            raise FileNotFoundError(f"SharePoint file not found: {path}")
        return io.BytesIO(self.download_item(meta))

    def read_excel(self, path, site=SITE_MAIN, **kwargs):
        return pd.read_excel(self.download(path, site), **kwargs)

    # --- writing ----------------------------------------------------------
    def upload(self, path, data, site=SITE_MAIN):
        """Create/replace a file. Uses an upload session for files over 4 MB."""
        if isinstance(data, io.BytesIO):
            data = data.getvalue()
        if len(data) <= SIMPLE_UPLOAD_LIMIT:
            res = self._request("PUT", f"{self._item_url(path, site)}:/content", data=data)
        else:
            session = self._request(
                "POST", f"{self._item_url(path, site)}:/createUploadSession",
                json={"item": {"@microsoft.graph.conflictBehavior": "replace"}},
            ).json()
            if "uploadUrl" not in session:
                raise IOError(f"Could not open upload session for {path}: {session}")
            res = None
            for start in range(0, len(data), UPLOAD_CHUNK):
                chunk = data[start:start + UPLOAD_CHUNK]
                res = self._request("PUT", session["uploadUrl"], auth=False, data=chunk, headers={
                    "Content-Length": str(len(chunk)),
                    "Content-Range": f"bytes {start}-{start + len(chunk) - 1}/{len(data)}",
                })
        if res is None or res.status_code not in (200, 201):
            hint = " — the file is open/locked in SharePoint, close it and re-run" if res is not None and res.status_code == 423 else ""
            raise IOError(f"Upload failed for {path}: HTTP {getattr(res, 'status_code', '?')}{hint}")
        print(f"   ⬆️  {path.split('/')[-1]} → SharePoint ({len(data) / 1e6:.1f} MB)")
        return res

    def ensure_folder(self, folder, site=SITE_MAIN):
        parent, _, name = folder.rpartition("/")
        url = f"{self._item_url(parent, site)}:/children" if parent else f"{GRAPH}/drives/{self.drive_id(site)}/root/children"
        self._request("POST", url, json={"name": name, "folder": {}, "@microsoft.graph.conflictBehavior": "fail"})


sp = SharePoint()


# Compatibility wrappers so older-style code (fetch_sp_file(drive_id, headers, path))
# keeps working unchanged in the ad-hoc tools.
def get_graph_headers():
    return sp.headers()


def get_drive_id(headers=None, site_path=SITE_MAIN):
    return sp.drive_id(site_path)


def fetch_sp_file(drive_id, headers, file_path):
    return sp.download(file_path)


print("✅ SharePoint client ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 0.5 REPORT STYLE — one look for every Excel output
#
#   THEME            colours, font and number formats (change them here once)
#   base_formats()   the named xlsxwriter formats every report uses
#   report_writer()  pandas ExcelWriter pre-set with the house font/date format
#   write_table()    a DataFrame as a styled sheet: typed number formats,
#                    sized columns, banded rows, frozen header, filter, totals
#   kpi_cards()      headline "stat card" blocks
#   apply_3tier_cf() ≥100% green / 80–99% amber / <80% red
#   save_report()    build a workbook in memory → SharePoint and/or local disk
#
# Data sheets that other files or Power BI read keep the header on row 1
# (no title rows), so nothing downstream breaks.
# ════════════════════════════════════════════════════════════════════
from contextlib import contextmanager

THEME = {
    "font": "Calibri",
    "font_size": 10,
    "header_bg": "#1B7A3E",      # column headers (white text)
    "band_bg": "#215967",        # group/section bands (white text)
    "accent": "#3DCD58",         # Schneider green — tab colour, KPI highlights
    "title": "#1B7A3E",
    "subtitle": "#595959",
    "zebra": "#F2F9F4",
    "total_bg": "#DAEFDF",
    "kpi_bg": "#EAF6EF",
    "border": "#BFBFBF",
    "good": ("#C6EFCE", "#276221"),
    "warn": ("#FFEB9C", "#9C5700"),
    "bad": ("#FFC7CE", "#9C0006"),
    # distinct, consistent colours for multi-group tables and tabs
    "series": ["#215967", "#1B7A3E", "#2E75B6", "#7030A0", "#C55A11", "#BF8F00", "#546E7A"],
    "tab_summary": "#3DCD58",
    "tab_detail": "#215967",
    "tab_tracker": "#C55A11",
    "tab_data": "#A6A6A6",
}
NUM_FMT = {
    "inr": "#,##0",
    "mn": "#,##0.0",
    "int": "#,##0",
    "pct": "0.0%",
    "date": "dd-mmm-yyyy",
    "id": "0",
    "plain": "0",
}


def base_formats(workbook):
    """Every named format used by the reports. Keys from the older per-report
    format dicts are kept as aliases so all report code shares this one set."""
    T, N = THEME, NUM_FMT
    base = {"font_name": T["font"], "font_size": T["font_size"], "valign": "vcenter"}
    cell = {**base, "border": 1, "border_color": T["border"]}

    def fmt(**kw):
        return workbook.add_format({**cell, **kw})

    hdr = {"bold": True, "bg_color": T["header_bg"], "font_color": "#FFFFFF", "align": "center", "text_wrap": True}
    band = {"bold": True, "bg_color": T["band_bg"], "font_color": "#FFFFFF", "align": "center"}
    tot = {"bold": True, "bg_color": T["total_bg"], "top": 2}
    zebra = {"bg_color": T["zebra"]}
    f = {
        "title": workbook.add_format({**base, "bold": True, "font_size": 14, "font_color": T["title"]}),
        "subtitle": workbook.add_format({**base, "bold": True, "font_size": 11, "font_color": T["band_bg"]}),
        "note": workbook.add_format({**base, "italic": True, "font_size": 9, "font_color": T["subtitle"]}),
        "hdr": fmt(**hdr),
        "band": fmt(**band),
        "text": fmt(align="left"),
        "text_alt": fmt(align="left", **zebra),
        "inr": fmt(num_format=N["inr"], align="right"),
        "inr_alt": fmt(num_format=N["inr"], align="right", **zebra),
        "mn": fmt(num_format=N["mn"], align="right"),
        "int": fmt(num_format=N["int"], align="center"),
        "pct": fmt(num_format=N["pct"], align="center"),
        "pct_alt": fmt(num_format=N["pct"], align="center", **zebra),
        "id": fmt(num_format=N["id"], align="left"),
        "id_alt": fmt(num_format=N["id"], align="left", **zebra),
        "date": fmt(num_format=N["date"], align="center"),
        "date_alt": fmt(num_format=N["date"], align="center", **zebra),
        "tot_text": fmt(align="left", **tot),
        "tot_inr": fmt(num_format=N["inr"], align="right", **tot),
        "tot_mn": fmt(num_format=N["mn"], align="right", **tot),
        "tot_int": fmt(num_format=N["int"], align="center", **tot),
        "tot_pct": fmt(num_format=N["pct"], align="center", **tot),
        "kpi_label": fmt(bold=True, bg_color=T["kpi_bg"], align="left", font_size=11),
        "kpi_value": fmt(bold=True, align="center", font_size=11, num_format=N["mn"]),
        "kpi_value_int": fmt(bold=True, align="center", font_size=11, num_format=N["int"]),
        "kpi_pct_hero": fmt(bold=True, bg_color=T["accent"], align="center", font_size=13, num_format=N["pct"]),
        "stat_label": fmt(**hdr),
        "stat_value": fmt(bold=True, bg_color=T["kpi_bg"], align="center", font_size=16),
        "runrate": fmt(num_format=N["inr"], align="right", bold=True, bg_color=T["warn"][0], font_color=T["bad"][1]),
        "cf_green": fmt(bg_color=T["good"][0], font_color=T["good"][1], num_format=N["pct"], align="center"),
        "cf_yellow": fmt(bg_color=T["warn"][0], font_color=T["warn"][1], num_format=N["pct"], align="center"),
        "cf_red": fmt(bg_color=T["bad"][0], font_color=T["bad"][1], num_format=N["pct"], align="center"),
        "zebra_fill": workbook.add_format({"bg_color": T["zebra"]}),
    }
    # Aliases used by the Partner Analysis / SSP code
    f.update({
        "hdr_wrap": f["hdr"], "hdr_alt": f["band"], "hdr_pol": f["hdr"], "hdr_green": f["hdr"], "hdr_dark": f["band"],
        "txt_det": f["text"], "dec": f["mn"], "num_mil": f["mn"],
        "num_det": f["inr"], "num_abs": f["inr"], "num_abs_alt": f["inr_alt"], "policy_num": f["inr"],
        "num_id": f["id"], "num_id_alt": f["id_alt"],
        "tot_dec": f["tot_mn"], "tot_det": f["tot_inr"], "tot_num": f["tot_inr"],
        "policy_runrate": f["runrate"],
    })
    return f


@contextmanager
def report_writer(target):
    """pandas ExcelWriter (xlsxwriter) with the house font and date format.
    `target` is a file path or a BytesIO."""
    with pd.ExcelWriter(
        target, engine="xlsxwriter",
        date_format=NUM_FMT["date"], datetime_format=NUM_FMT["date"],
        engine_kwargs={"options": {"nan_inf_to_errors": True}},
    ) as writer:
        default = writer.book.formats[0]          # workbook default cell format
        default.set_font_name(THEME["font"])
        default.set_font_size(THEME["font_size"])
        writer.fmt = base_formats(writer.book)     # handy: writer.fmt["hdr"] ...
        yield writer


# --- column typing --------------------------------------------------------
_PCT_WORDS = ("%", "pct", "percent", "achievement", "growth", "gwt", "grw", "contr", "rebate", "pace")
_MN_WORDS = ("(mn)", " mn", "mnr", "_mn")
_ID_WORDS = ("id", "code", "party", "order", "header", "item", "gstn", "gstin", "invoice", "po", "pin", "material")
_PLAIN_WORDS = ("year", "month", "_y", "_m", "rank", "remaining months")


def column_kind(name, series):
    """Classify a column → 'date' | 'pct' | 'mn' | 'id' | 'plain' | 'inr' | 'text'."""
    n = str(name).strip().lower()
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    if series.dtype == object and len(series.dropna()) and isinstance(series.dropna().iloc[0], (datetime, pd.Timestamp)):
        return "date"
    if not pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
        return "text"
    words = re.split(r"[\s_/().-]+", n)
    if any(w in n for w in _PCT_WORDS) and "run rate" not in n:
        return "pct"
    if any(n.endswith(w) or w in n for w in _MN_WORDS):
        return "mn"
    is_amount = any(w in n for w in ("value", "sales", "amt", "amount", "qty", "quantity", "target", "total"))
    if not is_amount and (any(w in words for w in _ID_WORDS) or n.endswith("id")):
        return "id"
    if n in _PLAIN_WORDS or any(n == w or n.endswith(w) for w in ("year", "month")):
        return "plain"
    return "inr"


def _col_width(name, series, kind):
    sample = series.dropna().head(500)
    if kind in ("inr", "mn", "int"):
        longest = len(f"{sample.abs().max():,.0f}") + 2 if len(sample) else 6
    elif kind in ("pct", "date", "plain"):
        longest = 11
    else:
        longest = int(sample.astype(str).str.len().quantile(0.95)) if len(sample) else 6
    return max(8, min(45, max(longest, len(str(name)) * 0.9) + 2))


def write_table(writer, df, sheet_name, title=None, subtitle=None, startrow=0, tab=None,
                freeze_cols=0, totals=False, kinds=None, autofilter=True, zebra=True, header_height=30):
    """Write `df` as a house-style table. Returns the row after the table.

    Without a title the header is on row 1 (safe for files other code reads).
    `kinds` overrides the auto-detected column type, e.g. {"Target": "mn"}."""
    f = writer.fmt
    t0 = time.time()
    print(f"      ✎ sheet '{sheet_name}' — {len(df):,} rows × {len(df.columns)} cols")
    df = df.reset_index(drop=True)
    df.columns = [str(c) for c in df.columns]
    hdr_row = startrow + (2 if title else 0) + (1 if subtitle else 0)
    df.to_excel(writer, sheet_name=sheet_name, startrow=hdr_row + 1, index=False, header=False)
    ws = writer.sheets[sheet_name]
    if title:
        ws.write(startrow, 0, title, f["title"])
    if subtitle:
        ws.write(startrow + (1 if title else 0), 0, subtitle, f["note"])

    kinds = {**{c: column_kind(c, df[c]) for c in df.columns}, **(kinds or {})}
    for i, col in enumerate(df.columns):
        ws.write(hdr_row, i, col, f["hdr"])
        k = kinds[col]
        cell_fmt = {"inr": f["inr"], "mn": f["mn"], "int": f["int"], "pct": f["pct"], "id": f["id"],
                    "date": f["date"], "plain": None, "text": f["text"]}[k]
        ws.set_column(i, i, _col_width(col, df[col], k), cell_fmt)
    ws.set_row(hdr_row, header_height)

    first, last = hdr_row + 1, hdr_row + len(df)
    if len(df) and zebra:
        ws.conditional_format(first, 0, last, len(df.columns) - 1,
                              {"type": "formula", "criteria": f"=MOD(ROW()-{first},2)=1", "format": f["zebra_fill"]})
    if autofilter and len(df.columns):
        ws.autofilter(hdr_row, 0, max(last, hdr_row), len(df.columns) - 1)
    ws.freeze_panes(hdr_row + 1, freeze_cols)
    if tab:
        ws.set_tab_color(tab)

    end = last + 1
    if totals and len(df):
        ws.write(end, 0, "Grand Total", f["tot_text"])
        for i, col in enumerate(df.columns[1:], start=1):
            k = kinds[col]
            if k in ("inr", "mn", "int"):
                ws.write(end, i, float(df[col].sum()), f[f"tot_{k}"])
            else:
                ws.write_blank(end, i, None, f["tot_text"])
        end += 1
    if time.time() - t0 > 5:
        print(f"        sheet '{sheet_name}' written in {time.time() - t0:.0f}s")
    return end + 1


def kpi_cards(ws, f, row, cards, per_row=3, span=2, gap=1):
    """Stat cards: list of (label, value). Returns the next free row."""
    for i, (label, value) in enumerate(cards):
        r = row + (i // per_row) * 3
        c = (i % per_row) * (span + gap)
        ws.merge_range(r, c, r, c + span - 1, label, f["stat_label"])
        ws.merge_range(r + 1, c, r + 1, c + span - 1, value, f["stat_value"])
    return row + ((len(cards) - 1) // per_row + 1) * 3


def apply_3tier_cf(ws, row_start, row_end, col_idx, f):
    """≥100% green, 80–99% amber, <80% red — same thresholds on every report."""
    ws.conditional_format(row_start, col_idx, row_end, col_idx, {"type": "cell", "criteria": ">=", "value": 1.0, "format": f["cf_green"]})
    ws.conditional_format(row_start, col_idx, row_end, col_idx, {"type": "cell", "criteria": "between", "minimum": 0.8, "maximum": 0.9999, "format": f["cf_yellow"]})
    ws.conditional_format(row_start, col_idx, row_end, col_idx, {"type": "cell", "criteria": "<", "value": 0.8, "format": f["cf_red"]})


def save_report(build, filename, sp_folder=None, local_dir=None):
    """Run build(writer) on an in-memory workbook, then upload it to
    SharePoint (`sp_folder`) and/or save it in `local_dir`. Returns where it went."""
    print(f"   📄 building {filename}")
    t0 = time.time()
    buf = io.BytesIO()
    with report_writer(buf) as writer:
        build(writer)
    data = buf.getvalue()
    print(f"   📦 {filename} ready ({len(data) / 1e6:.1f} MB, {time.time() - t0:.0f}s) — saving")
    targets = []
    if local_dir:
        os.makedirs(local_dir, exist_ok=True)
        path = os.path.join(local_dir, filename)
        with open(path, "wb") as fh:
            fh.write(data)
        print(f"   💾 {path}")
        targets.append(path)
    if sp_folder:
        sp.upload(f"{sp_folder}/{filename}", data)
        targets.append(f"SharePoint: {sp_folder}/{filename}")
    return "; ".join(targets)


print("✅ Report style ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 0.6 SHARED BUSINESS HELPERS + REPORT RUNNER
# ════════════════════════════════════════════════════════════════════
MONTH_ABBRS = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
               7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
MONTHS_ORDER = ["January", "February", "March", "April", "May", "June",
                "July", "August", "September", "October", "November", "December"]


def clean_id(series):
    """Partner/material codes as clean text: '500073231.0 ' → '500073231'."""
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


def is_retail(df):
    """Retail channel = PricingClassL2 in RETAIL_PRICING_L2 and L3 in RETAIL_PRICING_L3."""
    return df["PricingClassL2"].isin(RETAIL_PRICING_L2) & df["PricingClassL3"].isin(RETAIL_PRICING_L3)


def retail_filter(df, helios_codes=RETAIL_HELIOS_CODES):
    """Retail channel rows for the given Helios codes (copy)."""
    return df[is_retail(df) & df["HeliosCode"].isin(helios_codes)].copy()


def tier_category(city):
    if city in METRO_CITIES:
        return "Metro"
    if city in T1_CITIES:
        return "T1"
    return "T2&T3"


# --- Partner code changes (old Sold-to code → new code) ------------------
CODE_MAP = {}


def load_partner_code_map():
    """'Partner Code Changes' tab of the Data Dictionary → CODE_MAP."""
    global CODE_MAP
    try:
        changes = sp.read_excel(PATHS["dict_retail"], sheet_name="Partner Code Changes")
        CODE_MAP = dict(zip(clean_id(changes["Old Code"]), clean_id(changes["New Code"])))
        print(f"   🔁 {len(CODE_MAP)} partner code change(s) loaded")
    except Exception as e:
        CODE_MAP = {}
        print(f"   ⚠️ 'Partner Code Changes' not loaded ({e}); partner codes are used as-is.")
    return CODE_MAP


def remap_partner_codes(series):
    """Old code → new code; codes not in CODE_MAP pass through (cleaned)."""
    return clean_id(series).replace(CODE_MAP)


def consolidate_master_by_code(df, code_col):
    """Remap a master table's partner codes and keep one row per code —
    preferring the row that was already on the new code (latest details)."""
    if not CODE_MAP or code_col not in df.columns:
        return df
    was_new = clean_id(df[code_col]).isin(set(CODE_MAP.values()))
    out = df.copy()
    out[code_col] = remap_partner_codes(out[code_col])
    out["_was_new"] = was_new.values
    return (out.sort_values("_was_new", ascending=False)
               .drop_duplicates(subset=code_col, keep="first")
               .drop(columns="_was_new"))


# --- Partner report column order -------------------------------------------
# Every partner-wise report leads with:
#   Sold-to code · Name · GST · KRO/RD/PD · Branch · Region · City  → then the rest.
# Freeze panes stop right after Branch (the 5th column).
_PARTNER_LEAD_ALIASES = [
    ("SoldToPartyID", "Sold-to Party", "Partner ID"),
    ("Patner", "Partner", "Partner Name"),
    ("GSTN",),
    ("RD/KRO", "KRO/RD", "KRO/RD/PD", "Partner Type"),
    ("Branch",),
    ("Region", "Zone"),
    ("City",),
]


def partner_front(df):
    """Move the partner identity columns to the front in the standard order."""
    cols = list(df.columns)
    extra_kro = globals().get("kro_col")
    lead = []
    for names in _PARTNER_LEAD_ALIASES:
        if extra_kro and names[0] == "RD/KRO":
            names = names + (extra_kro,)
        hit = next((n for n in names if n in cols and n not in lead), None)
        if hit is not None:
            lead.append(hit)
    return df[lead + [c for c in cols if c not in lead]]


def partner_sheet(df):
    """partner_front + show 'Zone' as 'Region' (display only)."""
    return partner_front(df).rename(columns={"Zone": "Region"})


def partner_freeze(df, default=5):
    """Number of columns to freeze = everything up to and including Branch."""
    cols = [str(c) for c in df.columns]
    return cols.index("Branch") + 1 if "Branch" in cols else default


def monthly_timeline(df, year_col="Y", month_col="M", value_col="Gross Sales", index="SoldToPartyID"):
    """Partner × (year, month) pivot flattened to 2025_Jan … 2025_Dec,
    2025_Q1_Total … 2025_Annual_Total per year (used by the lifetime reports)."""
    pivot = df.pivot_table(index=index, columns=[year_col, month_col], values=value_col, aggfunc="sum", fill_value=0)
    chunks = []
    for yr in sorted(df[year_col].dropna().unique()):
        if yr not in pivot.columns.get_level_values(0):
            continue
        yr_df = pivot[yr].copy()
        for m in range(1, 13):
            if m not in yr_df.columns:
                yr_df[m] = 0
        yr_df = yr_df[list(range(1, 13))]
        for q in range(4):
            yr_df[f"Q{q + 1}_Total"] = yr_df[[3 * q + 1, 3 * q + 2, 3 * q + 3]].sum(axis=1)
        yr_df["Annual_Total"] = yr_df[list(range(1, 13))].sum(axis=1)
        yr_df.columns = [f"{int(yr)}_{MONTH_ABBRS[c]}" if isinstance(c, (int, np.integer)) else f"{int(yr)}_{c}"
                         for c in yr_df.columns]
        chunks.append(yr_df)
    if not chunks:
        return pd.DataFrame(columns=[index])
    return pd.concat(chunks, axis=1).reset_index()


def write_timeline_sheets(writer, sheets, freeze_cols=None):
    """Lifetime-report sheets: metadata columns + monthly timeline, totals highlighted."""
    f = writer.fmt
    for sheet_name, df in sheets.items():
        fc = partner_freeze(df) if freeze_cols is None else freeze_cols
        write_table(writer, df, sheet_name, freeze_cols=min(fc, max(len(df.columns) - 1, 0)), tab=THEME["tab_detail"])
        ws = writer.sheets[sheet_name]
        for i, col in enumerate(df.columns):
            if "Total" in str(col):
                ws.set_column(i, i, 14, f["tot_inr"])


def lookup_merge(df, lookup, label, **kwargs):
    """Left-join a mapping table and make sure no sales row gets duplicated
    (a duplicated key in the lookup would otherwise inflate sales)."""
    out = df.merge(lookup, how="left", **kwargs)
    if len(out) != len(df):
        raise ValueError(f"{label}: join added {len(out) - len(df):,} rows — the lookup has duplicate keys. "
                         f"Check Master_Data_Review → 'Duplicates in Master Data'.")
    return out


# --- Report runner ----------------------------------------------------------
RUN_LOG = []
_TIMINGS_FILE = os.path.join(LOG_DIR, "report_timings.json")   # last run's duration per report
try:
    _LAST_TIMINGS = json.load(open(_TIMINGS_FILE))
except Exception:
    _LAST_TIMINGS = {}
_ALWAYS_RUN = ["master_data_review"] + ([] if USE_CACHE else ["partner_master_update", "material_master_update"])
TOTAL_STEPS = sum(1 for on in REPORTS.values() if on) + len(_ALWAYS_RUN)


def _fmt_secs(s):
    s = int(round(s))
    return f"{s // 60}m {s % 60:02d}s" if s >= 60 else f"{s}s"


def run_report(key, func, *args, **kwargs):
    """Run one report if switched on in REPORTS, with progress logging:
    [n/total] ▶ start (and how long it took last time) → ✅ done / ❌ failed.
    A failing report is logged and the notebook carries on (unless STOP_ON_ERROR)."""
    if not REPORTS.get(key, True):
        print(f"⏭️  {key}: switched off in REPORTS")
        RUN_LOG.append({"Report": key, "Status": "skipped", "Seconds": 0, "Output / error": ""})
        return None
    step = sum(1 for r in RUN_LOG if r["Status"] != "skipped") + 1
    last = f" (last run {_fmt_secs(_LAST_TIMINGS[key])})" if key in _LAST_TIMINGS else ""
    print(f"━━━ [{step}/{TOTAL_STEPS}] ▶ {key}{last}")
    start = time.time()
    try:
        result = func(*args, **kwargs)
        secs = time.time() - start
        out = result if isinstance(result, str) else ""
        RUN_LOG.append({"Report": key, "Status": "ok", "Seconds": round(secs, 1), "Output / error": out})
        _LAST_TIMINGS[key] = round(secs, 1)
        print(f"✅ [{step}/{TOTAL_STEPS}] {key} done in {_fmt_secs(secs)}")
        return result
    except Exception as e:
        secs = time.time() - start
        RUN_LOG.append({"Report": key, "Status": "FAILED", "Seconds": round(secs, 1), "Output / error": f"{type(e).__name__}: {e}"})
        log(f"❌ [{step}/{TOTAL_STEPS}] {key} failed after {_fmt_secs(secs)}: {type(e).__name__}: {e}\n{traceback.format_exc()}")
        if STOP_ON_ERROR:
            raise
        return None
    finally:
        try:
            json.dump(_LAST_TIMINGS, open(_TIMINGS_FILE, "w"), indent=1)
        except Exception:
            pass
        gc.collect()


print("✅ Shared helpers ready")

## 1 · Data
SAP broadcast → raw Excel → pre-processed yearly files → `df_Sales_final`, `df_OB_final`, `df_backlog_final`.
In cache mode (`USE_CACHE = 1`) only cell 1.2 does any work: it loads the snapshot saved by the last full run.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 1.1 DATA PIPELINE FUNCTIONS
#
#   refresh_sap_broadcasts()   newest SAP broadcast zip → raw Excel in the
#                              Primary Sales / OB / Backlog input folders
#   load_raw_folder(kind)      raw Excel files (2 merged header rows) →
#                              one DataFrame; per-file local parquet cache
#   publish_preprocessed()     raw → "Primary_<Kind>_Data_<year>.xlsx"
#   load_preprocessed(kind)    all years of pre-processed data, via a
#                              parquet cache kept next to them on SharePoint
#   load_from_cache()          cache mode: SharePoint parquet_cache for
#                              Sales/OB + Backlog from its source folder
# ════════════════════════════════════════════════════════════════════
import zipfile

import openpyxl
import pyarrow as pa
import pyarrow.parquet as pq


def _arrow_table(data):
    """Parquet bytes → Arrow table (no pandas copy yet)."""
    return pq.read_table(pa.BufferReader(data))


def tables_to_frame(tables):
    """Combine yearly Arrow tables and convert to pandas ONCE, releasing Arrow
    memory column by column. Peak memory is far lower than reading every year
    into pandas and then pd.concat-ing (which briefly holds everything twice)."""
    try:
        combined = pa.concat_tables(tables, promote_options="permissive")
    except (pa.ArrowInvalid, pa.ArrowTypeError, TypeError):
        # columns with different types across years: convert year by year
        frames = []
        while tables:
            frames.append(tables.pop(0).to_pandas())
        return pd.concat(frames, ignore_index=True)
    tables.clear()
    df = combined.to_pandas(self_destruct=True, split_blocks=True)
    del combined
    gc.collect()
    return df

# --- Column rename maps (SAP header text → standard names) -----------------
SALES_RENAME_MAP = {
    'Sales Organization': 'SalesOrgCode', 'Sales Organization_1': 'SalesOrgName',
    'Sales Office': 'SalesOfficeCode', 'Sales Office_1': 'SalesOfficeName',
    'Helios Code': 'HeliosCode', 'Helios Code_1': 'HeliosDescription',
    'Pricing Class L2': 'PricingClassL2', 'Pricing Class L2_1': 'PricingClassL2Description',
    'Pricing Class L3': 'PricingClassL3', 'Pricing Class L3_1': 'PricingClassL3Description',
    'Family': 'ProductFamilyCode', 'Family_1': 'ProductFamilyDescription',
    'Employee responsible': 'EmployeeResponsibleID', 'Employee responsible_1': 'EmployeeResponsibleName',
    'External Agent': 'ExternalAgentID', 'External Agent_1': 'ExternalAgentName',
    'Sales Representative': 'SalesRepresentativeID', 'Sales Representative_1': 'SalesRepresentativeName',
    'Soldto Party': 'SoldToPartyID', 'Soldto Party_1': 'SoldToPartyName','Soldto City':'PartnerCity','Region (Soldto)':'PartnerState',
    'Final Customer': 'FinalCustomerID', 'Final Customer_1': 'FinalCustomerName','ShipTo Party':'ShipToPartyID','ShipTo Party_1':'ShipToPartyName',
    'Material':'MaterialCode', 'Material_1':'MaterialDescription',
    'Material Pricing Group': 'MaterialPricingGroupCode', 'Material Pricing Group_1': 'MaterialPricingGroupDescription',
    'Inv. Creation date': 'InvoiceDate','Inv Creation date': 'InvoiceDate', 'Sales Order Header': 'SalesOrderHeader', 'Inv Creation month': 'InvoiceMonth','Inv Creation month_1': 'InvoiceMonthName','UnnamedColumn_31':'InvoiceMonthName','nan':'InvoiceMonthName',
    'Sales Order Item': 'SalesOrderItem', 'Customer PO': 'CustomerPO', 'Sales Group': 'SalesGroup',
    'Billing Quantity': 'BillingQuantity', 'Net Value': 'Gross Sales', 'INR_Currency': 'Gross Sales', 'Billing Quantity 2021': 'BillingQuantity',
    'Net Value 2021_INR': 'Gross Sales', 
    'Billing Quantity 2022': 'BillingQuantity',
    'Net Value 2022_INR': 'Gross Sales', 
    'Billing Quantity 2023': 'BillingQuantity',
    'Net Value 2023_INR': 'Gross Sales',
    'Billing Quantity 2024': 'BillingQuantity',
    'Net Value 2024_INR': 'Gross Sales',
    'Billing Quantity 2025': 'BillingQuantity',
    'Net Value 2025_INR': 'Gross Sales',
    'Billing Quantity 2026' :'BillingQuantity',
    'Net Value 2026_INR': 'Gross Sales',
    'Net Value 2026' : 'Gross Sales',
}

OB_RENAME_MAP = {
    'Sales Organization': 'SalesOrgCode',
    'Sales Organization_1': 'SalesOrgName',
    'Sales Office': 'SalesOfficeCode',
    'Sales Office_1': 'SalesOfficeName',
    'Pricing Class L2': 'PricingClassL2',
    'Pricing Class L2_1': 'PricingClassL2Description',
    'Pricing Class L3': 'PricingClassL3',
    'Pricing Class L3_1': 'PricingClassL3Description',
    'Family': 'ProductFamilyCode',
    'Family_1': 'ProductFamilyDescription',
    'Employee Responsible': 'EmployeeResponsibleID', 
    'Employee responsible':'EmployeeResponsibleID',
    'Employee responsible_1':'EmployeeResponsibleName',
    'Employee Responsible_1': 'EmployeeResponsibleName',
    'External Agent': 'ExternalAgentID',
    'External Agent_1': 'ExternalAgentName',
    'Sales Representative': 'SalesRepresentativeID',
    'Sales Representative_1': 'SalesRepresentativeName',
    'Soldto Party': 'SoldToPartyID',
    'Soldto Party_1': 'SoldToPartyName',
    'SoldTo City':'PartnerCity',
    'Region':'PartnerState',
    'Final Customer': 'FinalCustomerID',
    'ShipTo Party':'ShipToPartyID',
    'ShipTo Party_1':'ShipToPartyName',
    'Final Customer_1': 'FinalCustomerName',
    'Material':'MaterialCode',
    'Material_1':'MaterialDescription',
    'Helios Code' : 'HeliosCode',
    'Helios Code_1' : 'HeliosDescription',
    'Material Pricing Gro' : 'MaterialPricingGroupCode',
    'Material Pricing Gro_1' : 'MaterialPricingGroupDescription',
    'Calendar month': 'OB Month',
    'Sales Order Number': 'SalesOrderHeader',
    'Sales Order Header': 'SalesOrderHeader',
    'INR_Currency': 'Gross Sales',
    'Sales Order Item Number': 'SalesOrderItem',
    'Po Number': 'CustomerPO',
    'Sales Group': 'SalesGroup',
    'Calendar month_1':'OB Month Name',
    'Quantity 2026': 'BillingQuantity', 
    'Billing Quantity 2026': 'BillingQuantity',
    'Net Net Order Value 2026_INR': 'Gross Sales',
    'Net Value 2026_INR': 'Gross Sales',
    'Net Net Order Value': 'Gross Sales',
    'Net Net Order Value 2026':'Gross Sales',
    'Quantity':'BillingQuantity'
}

BACKLOG_RENAME_MAP = {
    'Backlog Version': 'BacklogStatus',
    'Helios code': 'HeliosCode',
    'Helios code (m)': 'HeliosCode',
    'Helios code (m)_1': 'HeliosDescription',
    'Helios code (m) 1': 'HeliosDescription',
    'Family(m)': 'ProductFamilyCode',
    'Family(m)_1': 'ProductFamilyDescription',
    'Family(m) 1': 'ProductFamilyDescription',
    'Employee responsible': 'EmployeeResponsibleID',
    'Employee responsible_1': 'EmployeeResponsibleName',
    'Sales office': 'SalesOfficeCode',
    'Sales office_1': 'SalesOfficeName',
    'Sales office 1': 'SalesOfficeName',
    'Sold-to party': 'SoldToPartyID',
    'Sold-to party_1': 'SoldToPartyName',
    'Sold-to party 1': 'SoldToPartyName',
    'Sales order header': 'SalesOrderHeader',
    'Created on': 'OrderCreationDate',
    'Reference date': 'ReferenceDate',
    'Request delivery date': 'RequestedDate',
    'Material': 'MaterialCode',
    'Material_1': 'MaterialDescription',
    'Material 1': 'MaterialDescription',
    'Sales order item': 'SalesOrderItem',
    'Credit status': 'CreditStatus', 
    'Credit status_1': 'CreditStatusDescription',
    'Credit status 1': 'CreditStatusDescription',
    'Plant': 'PlantCode',   
    'Plant_1': 'PlantName',
    'Plant 1': 'PlantName',
    'Pricing class l1 (m)':'PricingClassL2',
    'Pricing class l1 (m)_1' : 'PricingClassL2Description',
    'Pricing class l1 (m)_2':'PricingClassL3',
    'Pricing class l1 (m)_3': 'PricingClassL3Description',    'Col_22':'PricingClassL3Description',  
    'Pricing class l1 (m) 1' : 'PricingClassL2Description',
    'Pricing class l1 (m) 2':'PricingClassL3',
    'Pricing class l1 (m) 3': 'PricingClassL3Description',  
    'Total / backlog qty': 'BillingQuantity',
    'InrTotal / backlog amt': 'Gross Sales',
    'Inr': 'Gross Sales',
    'Total / Backlog Qty': 'BillingQuantity',
    'Inrtotal / Backlog Amt': 'Gross Sales',
    'Total / backlog amt Inr': 'Gross Sales',
    'Total / backlog amt': 'Gross Sales',
    'Total / backlog qty': 'BillingQuantity',
}

RAW_HEADER_ROWS = [2, 3]   # 0-based rows holding the two-level SAP headers


# --- 1.1a SAP broadcast → raw files --------------------------------------
def _clean_sap_numbers(df, keywords):
    """SAP CSV numbers use European format: '3.000,50' → 3000.50."""
    for col in df.columns:
        if any(k.lower() in col.lower() for k in keywords):
            s = df[col].astype(str).str.replace(r"[^0-9.,-]", "", regex=True)
            s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
            df[col] = pd.to_numeric(s, errors="coerce").fillna(0)
            print(f"      ✔ {col}: total {df[col].sum():,.2f}")
    return df


def _unique_names(columns):
    seen, out = {}, []
    for col in columns:
        name = str(col).replace('"', "").strip()
        seen[name] = seen.get(name, -1) + 1
        out.append(f"{name}_{seen[name]}" if seen[name] else name)
    return out


def _process_broadcast(items, prefix, target_folder, numeric_keys, date_col, label, generic_name=False):
    files = [f for f in items if f["name"].startswith(prefix) and f.get("file")]
    if not files:
        print(f"   ❌ No {label} broadcast ({prefix}*) found")
        return False
    latest = max(files, key=lambda x: x["lastModifiedDateTime"])
    print(f"   🚀 {label}: {latest['name']}")
    with zipfile.ZipFile(io.BytesIO(sp.download_item(latest))) as z:
        csv_name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
        with z.open(csv_name) as fh:
            meta = pd.read_csv(fh, sep=";", nrows=3, header=None, engine="python")
            if len(meta) >= 3:
                print(f"      SAP last data update: {meta.iloc[2, 0]}")
            fh.seek(0)
            df = pd.read_csv(fh, sep=";", skiprows=3, header=None, engine="python")
    df.columns = _unique_names(df.iloc[0].tolist())
    df = _clean_sap_numbers(df.iloc[1:].reset_index(drop=True), numeric_keys)

    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
    latest_date = df[date_col].max()
    if generic_name:
        out_name = f"SEIN Primary {label} data.xlsx"
    else:
        out_name = f"SEIN Primary {label} data {latest_date:%b %Y}.xlsx" if pd.notnull(latest_date) else f"SEIN Primary {label} data Unknown.xlsx"

    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as writer:
        pd.DataFrame([[""], [""], [""]]).to_excel(writer, index=False, header=False)   # keep SAP's 3-row top padding
        df.to_excel(writer, index=False, startrow=3)
    sp.upload(f"{target_folder}/{out_name}", buf.getvalue())
    return True


def refresh_sap_broadcasts():
    items = None
    for folder in PATHS["broadcast_folders"]:
        try:
            items = sp.list_folder(folder, site=SITE_BROADCAST)
            print(f"   📬 Broadcast folder: {folder}")
            break
        except FileNotFoundError:
            continue
    if items is None:
        print(f"   ⚠️ No broadcast folder found ({PATHS['broadcast_folders']}); raw files left as they are.")
        return
    _process_broadcast(items, "YQQBLY", PATHS["raw_sales"], ["Net Value", "Billing Quantity"], "Inv. Creation date", "Sales")
    _process_broadcast(items, "YQQSO", PATHS["raw_ob"], ["Net Order Value", "Quantity"], "Calendar Day", "OB")
    _process_broadcast(items, "YQQCSY", PATHS["raw_backlog"], ["Backlog Qty", "Backlog Amt"], "Created on", "Backlog", generic_name=True)


# --- 1.1b raw Excel → DataFrame ------------------------------------------
def _headers_from_workbook(workbook, header_rows):
    """Column names from SAP's two merged header rows (Sales / OB layout)."""
    sheet = workbook.active
    max_col = sheet.max_column - 1 if sheet.max_column > 0 else 0
    parts = {i: [] for i in range(max_col + 1)}
    merged = {}
    for rng in sheet.merged_cells.ranges:
        min_col, min_row, max_c, max_row = rng.bounds
        if any(r in range(min_row - 1, max_row) for r in header_rows):
            value = sheet.cell(row=min_row, column=min_col).value
            if value:
                merged[(min_row - 1, min_col - 1)] = (str(value).strip(), max_c - min_col + 1)
                for c in range(min_col, max_c + 1):
                    for r in range(min_row, max_row + 1):
                        if (r - 1, c - 1) != (min_row - 1, min_col - 1):
                            merged[(r - 1, c - 1)] = ("MERGED_SUB_PART", 0)
    for r in header_rows:
        for c in range(max_col + 1):
            if merged.get((r, c), ("",))[0] == "MERGED_SUB_PART":
                continue
            value = sheet.cell(row=r + 1, column=c + 1).value
            if (r, c) in merged:
                text, span = merged[(r, c)]
                for i in range(span):
                    if text not in parts[c + i]:
                        parts[c + i].insert(0, text)
            elif value is not None:
                parts[c].append(str(value).strip())
    names = []
    for c in sorted(parts):
        clean = [p for p in parts[c] if p and str(p).strip() not in ("None", "")]
        name = "_".join(clean).replace("(M)", "").replace(".", "").replace("-", "").strip() if clean else ""
        name = re.sub(r"_{2,}", "_", name).strip("_")
        names.append(name or f"UnnamedColumn_{c}")
    seen, unique = set(), []
    for name in names:
        base, n = name, 1
        while name in seen:
            name, n = f"{base}_{n}", n + 1
        seen.add(name)
        unique.append(name)
    return unique


def _headers_backlog(file_stream, header_rows):
    """Column names for the Backlog layout (title-cased to match BACKLOG_RENAME_MAP)."""
    file_stream.seek(0)
    sheet = openpyxl.load_workbook(file_stream, data_only=True).active
    max_col = sheet.max_column - 1 if sheet.max_column > 0 else 0
    parts = {i: [] for i in range(max_col + 1)}
    for rng in sheet.merged_cells.ranges:
        min_col, min_row, max_c, max_row = rng.bounds
        if any(r in range(min_row - 1, max_row) for r in header_rows):
            value = sheet.cell(row=min_row, column=min_col).value
            if value:
                for c in range(min_col, max_c + 1):
                    parts[c - 1].append(str(value).strip())
    for r in header_rows:
        for c in range(max_col + 1):
            value = sheet.cell(row=r + 1, column=c + 1).value
            if value and str(value).strip() not in parts[c]:
                parts[c].append(str(value).strip())
    names, seen = [], set()
    for i in range(max_col + 1):
        name = " ".join(w.capitalize() for w in "_".join(parts[i]).split("_")) or f"Col_{i}"
        base, n = name, 1
        while name in seen:
            name, n = f"{base}_{n}", n + 1
        seen.add(name)
        names.append(name)
    return names


RAW_KINDS = {
    "sales": {"folder": "raw_sales", "cache": CACHE_DIR_SALES_RAW, "rename": SALES_RENAME_MAP, "date": "InvoiceDate", "type": "Sales",
              "str_cols": ["SalesOrgCode", "SalesOfficeCode", "EmployeeResponsibleID", "SalesOrderHeader", "Invoice Header",
                           "SoldToPartyID", "FinalCustomerID", "ShipToPartyID", "MaterialCode", "SalesOrderItem"]},
    "ob": {"folder": "raw_ob", "cache": CACHE_DIR_OB_RAW, "rename": OB_RENAME_MAP, "date": "Calendar Day", "type": "OB",
           "str_cols": ["SalesOrgCode", "SalesOfficeCode", "EmployeeResponsibleID", "SalesOrderHeader", "SoldToPartyID",
                        "FinalCustomerID", "ShipToPartyID", "MaterialCode"]},
}


def _parse_raw_sales_ob(content, spec):
    stream = io.BytesIO(content)
    wb = openpyxl.load_workbook(stream, data_only=True)
    headers = _headers_from_workbook(wb, RAW_HEADER_ROWS)
    wb.close()
    stream.seek(0)
    df = pd.read_excel(stream, header=None, skiprows=max(RAW_HEADER_ROWS) + 1)
    df.columns = headers[:len(df.columns)]
    df.columns = [re.sub(r"\s+(_\d+)", r"\1", str(c)).strip() for c in df.columns]
    df = df.rename(columns=spec["rename"])
    for col in spec["str_cols"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r"\.0$", "", regex=True).replace("nan", "")
    if "SalesOrderItem" in df.columns and spec["type"] == "Sales":
        df["SalesOrderItem"] = df["SalesOrderItem"].str.replace(r"^APAC/", "", regex=True)
    if spec["date"] in df.columns:
        df[spec["date"]] = pd.to_datetime(df[spec["date"]], dayfirst=True, errors="coerce")
        df["Year"] = df[spec["date"]].dt.year.astype(str).replace("nan", "Unknown")
    elif spec["type"] == "OB":
        df["Year"] = str(TARGET_YEAR)
    df["Type"] = spec["type"]
    for col in ("BillingQuantity", "Gross Sales"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df


RAW_CHANGED_YEARS = {"sales": set(), "ob": set()}   # years touched by files read fresh in this run


def load_raw_folder(kind):
    """Sales or OB raw files → DataFrame.

    Only files that changed since the last run are downloaded and parsed from
    Excel. Everything else comes from the per-file parquet cache: the local
    folder first, then the shared copy kept on SharePoint in
    '<raw folder>/parquet_cache' — so a new machine, or a kernel started in a
    different folder, no longer re-reads every month. The years covered by the
    freshly read files are remembered in RAW_CHANGED_YEARS[kind] so only those
    pre-processed yearly files get rewritten."""
    spec = RAW_KINDS[kind]
    raw_folder = PATHS[spec["folder"]]
    sp_cache_folder = f"{raw_folder}/parquet_cache"
    meta_file = os.path.join(spec["cache"], "cache_metadata.json")
    meta = json.load(open(meta_file)) if os.path.exists(meta_file) else {}
    try:
        shared_cache = {f["name"]: f for f in sp.list_folder(sp_cache_folder) if f["name"].endswith(".parquet")}
    except FileNotFoundError:
        shared_cache = {}
    frames, changed, changed_years = [], False, set()
    for item in sp.list_folder(raw_folder):
        name = item["name"]
        if not name.endswith(".xlsx") or name.startswith("~"):
            continue
        cache_name = re.sub(r"[^a-zA-Z0-9_\-]", "_", name) + ".parquet"
        cache_file = os.path.join(spec["cache"], cache_name)
        cached = meta.get(item["id"], {})
        stamp = {"name": name, "lastModifiedDateTime": item["lastModifiedDateTime"]}

        # 1) local cache, unchanged file
        if cached.get("lastModifiedDateTime") == item["lastModifiedDateTime"] and os.path.exists(cache_file):
            print(f"   📦 cache: {name}")
            frames.append(pd.read_parquet(cache_file))
            continue

        # 2) shared SharePoint cache, written after the file last changed
        shared_name = name[:-len(".xlsx")] + ".parquet"          # same naming as the other parquet caches
        shared = shared_cache.get(shared_name) or shared_cache.get(cache_name)
        if shared and pd.to_datetime(shared["lastModifiedDateTime"]) >= pd.to_datetime(item["lastModifiedDateTime"]):
            try:
                data = sp.download_item(shared)
                df = pd.read_parquet(io.BytesIO(data))
                if not {spec["date"], "Gross Sales"} <= set(df.columns):
                    raise ValueError("unexpected columns (not a raw-month cache file)")
                with open(cache_file, "wb") as fh:
                    fh.write(data)
                meta[item["id"]] = stamp
                changed = True
                print(f"   📦 SharePoint cache: {name}")
                frames.append(df)
                continue
            except Exception as e:
                print(f"   ⚠️ shared cache for {name} unreadable ({e}) — reading the Excel instead")

        # 3) new or updated file → download + parse, then refresh both caches
        print(f"   ⬇️  download: {name}")
        try:
            df = _parse_raw_sales_ob(sp.download_item(item), spec)
            df.to_parquet(cache_file, index=False)
            meta[item["id"]] = stamp
            changed = True
            try:
                with open(cache_file, "rb") as fh:
                    sp.upload(f"{sp_cache_folder}/{shared_name}", fh.read())
            except Exception as e:
                print(f"   ⚠️ could not save the shared cache copy of {name}: {e}")
            if spec["date"] in df.columns:
                changed_years |= {int(y) for y in pd.to_datetime(df[spec["date"]], errors="coerce").dt.year.dropna().unique()}
            frames.append(df)
        except Exception as e:
            print(f"   ❌ could not read {name}: {e}")
    if changed:
        with open(meta_file, "w") as fh:
            json.dump(meta, fh, indent=4)
    RAW_CHANGED_YEARS[kind] = changed_years
    if not frames:
        raise FileNotFoundError(f"No readable {kind} files in {raw_folder}")
    df = pd.concat(frames, ignore_index=True)
    print(f"   {kind}: {len(df):,} raw rows" + (f" | new/updated data in year(s) {sorted(changed_years)}" if changed_years else " | no new data"))
    return df


def load_raw_backlog():
    frames = []
    for item in sp.list_folder(PATHS["raw_backlog"]):
        name = item["name"]
        if not name.endswith(".xlsx") or name.startswith("~"):
            continue
        print(f"   ⬇️  backlog: {name}")
        try:
            stream = io.BytesIO(sp.download_item(item))
            cols = _headers_backlog(stream, RAW_HEADER_ROWS)
            stream.seek(0)
            df = pd.read_excel(stream, header=None, skiprows=4)
            df.columns = [str(c).strip() for c in cols[:len(df.columns)]]
            df = df.rename(columns=BACKLOG_RENAME_MAP)
            if "BillingQuantity" in df.columns:
                df["BillingQuantity"] = pd.to_numeric(df["BillingQuantity"].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce").fillna(0)
            if "Gross Sales" in df.columns:
                df["Gross Sales"] = pd.to_numeric(df["Gross Sales"].astype(str).str.replace(r"[^0-9.-]", "", regex=True), errors="coerce").fillna(0)
            frames.append(df)
        except Exception as e:
            print(f"   ❌ could not read {name}: {e}")
    if not frames:
        print("   ⚠️ No backlog files found — backlog will be empty.")
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df["OrderCreationDate"] = pd.to_datetime(df["OrderCreationDate"], errors="coerce")
    df["Year"] = df["OrderCreationDate"].dt.year.astype("Int64")
    df["Type"] = "Backlog"
    print(f"   backlog: {len(df):,} rows")
    return df


# --- 1.1c raw → pre-processed yearly files --------------------------------
EXCEL_MAX_ROWS = 1_000_000


def publish_preprocessed(df, kind, years=None):
    """Write one Excel per year (split in parts above 1M rows) into the pre-processed folder.
    With `years` given, only those years are rewritten (plus any year that has
    no file yet) — unchanged years keep their file and their parquet cache."""
    if kind == "sales":
        date_col, folder, stem = "InvoiceDate", PATHS["pre_sales"], "Primary_Sales_Data"
    else:
        date_col, folder, stem = "Calendar Day", PATHS["pre_ob"], "Primary_OB_Data"
    data = df.copy()
    data[date_col] = pd.to_datetime(data[date_col])
    data = data[data[date_col].dt.year.isin(HISTORY_YEARS)]
    years_col = data[date_col].dt.year
    if kind == "ob":
        data["Year"] = years_col            # OB files carry Year as a number

    todo = sorted(int(y) for y in years_col.unique())
    if years is not None:
        try:
            existing = {f["name"] for f in sp.list_folder(folder)}
        except FileNotFoundError:
            existing = set()
        todo = [y for y in todo if y in years or not any(n.startswith(f"{stem}_{y}") and n.endswith(".xlsx") for n in existing)]
    if not todo:
        print(f"   ✅ no new {kind} data — pre-processed {kind} files left as they are")
        return
    print(f"   ✍️  rewriting pre-processed {kind} for year(s): {todo}")
    for year in todo:
        part = data[years_col == year]
        n_parts = math.ceil(len(part) / EXCEL_MAX_ROWS)
        for i in range(n_parts):
            chunk = part.iloc[i * EXCEL_MAX_ROWS:(i + 1) * EXCEL_MAX_ROWS]
            name = f"{stem}_{int(year)}" + (f"_part{i + 1}" if n_parts > 1 else "") + ".xlsx"
            buf = io.BytesIO()
            with pd.ExcelWriter(buf, engine="xlsxwriter") as writer:
                chunk.to_excel(writer, index=False)
            sp.upload(f"{folder}/{name}", buf.getvalue())


# --- 1.1d pre-processed yearly files → final DataFrames ----------------------
def load_preprocessed(kind):
    """All pre-processed years; each Excel is mirrored as parquet in a
    'parquet_cache' sub-folder on SharePoint and only re-converted when newer."""
    folder = PATHS["pre_sales"] if kind == "sales" else PATHS["pre_ob"]
    prefix = "" if kind == "sales" else "Primary_OB_Data_"
    cache_folder = f"{folder}/parquet_cache"
    excel_files = [f for f in sp.list_folder(folder) if f["name"].endswith(".xlsx") and f["name"].startswith(prefix)]
    try:
        cached = {f["name"]: f for f in sp.list_folder(cache_folder) if f["name"].endswith(".parquet")}
    except FileNotFoundError:
        cached = {}
    frames = []
    for item in excel_files:
        pq_name = item["name"].replace(".xlsx", ".parquet")
        pq = cached.get(pq_name)
        if pq and pd.to_datetime(item["lastModifiedDateTime"]) <= pd.to_datetime(pq["lastModifiedDateTime"]):
            print(f"   📦 {pq_name}")
            frames.append(_arrow_table(sp.download_item(pq)))
            continue
        print(f"   🔄 converting {item['name']}")
        try:
            df = pd.read_excel(io.BytesIO(sp.download_item(item)), engine="openpyxl")
            for col in df.select_dtypes(include=["object", "string"]).columns:
                df[col] = df[col].astype(str)
            buf = io.BytesIO()
            df.to_parquet(buf, index=False, engine="pyarrow")
            del df
            sp.upload(f"{cache_folder}/{pq_name}", buf.getvalue())
            frames.append(_arrow_table(buf.getvalue()))
        except Exception as e:
            print(f"   ❌ {item['name']} skipped: {e}")
    if not frames:
        raise FileNotFoundError(f"No pre-processed {kind} files in {folder}")
    return tables_to_frame(frames)


def load_cached_parquet_only(kind):
    """Cache mode: read ONLY the parquet files in the SharePoint 'parquet_cache'
    folder — no Excel, no conversion, no upload. Pre-processed years that have
    no parquet in the cache are listed so nothing goes missing silently."""
    folder = {"sales": PATHS["pre_sales"], "ob": PATHS["pre_ob"]}[kind]
    prefix = "" if kind == "sales" else "Primary_OB_Data_"
    try:
        cached = [f for f in sp.list_folder(f"{folder}/parquet_cache") if f["name"].endswith(".parquet")]
    except FileNotFoundError:
        cached = []
    if not cached:
        raise FileNotFoundError(f"No cached {kind} data in {folder}/parquet_cache. Run once with 0 (check for new data).")
    for f in cached:
        print(f"   📦 {f['name']}  (cached {f['lastModifiedDateTime'][:10]})")
    excel = {f["name"].replace(".xlsx", ".parquet") for f in sp.list_folder(folder)
             if f["name"].endswith(".xlsx") and f["name"].startswith(prefix)}
    missing = sorted(excel - {f["name"] for f in cached})
    if missing:
        print(f"   ⚠️ No cache for {missing} — these years are NOT included. Run once with 0 to cache them.")
    tables = []
    for f in cached:
        tables.append(_arrow_table(sp.download_item(f)))   # downloaded bytes are freed after each file
    df = tables_to_frame(tables)
    print(f"   {kind}: {len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")
    return df


def load_from_cache():
    """Cache mode (no refresh): Sales/OB from the SharePoint parquet_cache
    folders; Backlog has no cache, so it is read from its source folder."""
    print("   📦 Sales — Pre-processed Sales Data/parquet_cache")
    sales = load_cached_parquet_only("sales")
    print("   📦 OB — Pre-processed OB Data/parquet_cache")
    ob = load_cached_parquet_only("ob")
    print("   📄 Backlog — source folder (no cache for backlog)")
    return {"sales": sales, "ob": ob, "backlog": load_raw_backlog()}


print("✅ Data pipeline functions ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 1.2 LOAD DATA — cache or fresh, depending on USE_CACHE (cell 0.2)
#
# Result (used by every report):
#   df_Sales_final     all pre-processed Sales years
#   df_OB_final        all pre-processed OB years
#   df_backlog_final   current backlog
# ════════════════════════════════════════════════════════════════════
def load_fresh_data():
    print("① SAP broadcast → raw files")
    refresh_sap_broadcasts()
    print("② raw Sales → pre-processed yearly files")
    raw_sales = load_raw_folder("sales")
    publish_preprocessed(raw_sales, "sales", RAW_CHANGED_YEARS["sales"])
    del raw_sales
    print("③ raw OB → pre-processed yearly files")
    raw_ob = load_raw_folder("ob")
    publish_preprocessed(raw_ob, "ob", RAW_CHANGED_YEARS["ob"])
    del raw_ob
    print("④ backlog")
    backlog = load_raw_backlog()
    print("⑤ all pre-processed years")
    return {"sales": load_preprocessed("sales"), "ob": load_preprocessed("ob"), "backlog": backlog}


# Free data left over from an earlier run of this notebook in the same kernel
for _name in ("df_Sales_final", "df_OB_final", "df_backlog_final", "final_combined_backlog_df", "frames",
              "retail_sales", "retail_ob", "retail_backlog", "hs_sales", "hs_ob", "hs_backlog",
              "hs_dash_sales", "hs_dash_ob", "hs_dash_backlog", "df_retail", "df_final", "df_active"):
    globals().pop(_name, None)
gc.collect()

if USE_CACHE:
    # NO REFRESH: no SAP broadcast, no conversion, no uploads, no master updates
    print("🔒 Cache mode — data is NOT refreshed.")
    DATA_SOURCE = "cache (SharePoint parquet_cache) + backlog source"
    frames = load_from_cache()
else:
    DATA_SOURCE = "SharePoint (fresh)"
    frames = load_fresh_data()

df_Sales_final = frames["sales"]
df_OB_final = frames["ob"]
df_backlog_final = frames["backlog"]
del frames

# --- Standard types every report expects ---------------------------------
df_Sales_final["SalesOrderHeader"] = df_Sales_final["SalesOrderHeader"].astype(str)
df_Sales_final["InvoiceDate"] = pd.to_datetime(df_Sales_final["InvoiceDate"]).dt.date
df_OB_final["Calendar Day"] = pd.to_datetime(df_OB_final["Calendar Day"]).dt.date
df_OB_final["SalesOrderHeader"] = df_OB_final["SalesOrderHeader"].astype(str)
if not df_backlog_final.empty:
    df_backlog_final["SalesOrderHeader"] = df_backlog_final["SalesOrderHeader"].astype(str)

# --- One partner, one code (see APPLY_PARTNER_CODE_REMAP) ----------------
if APPLY_PARTNER_CODE_REMAP:
    load_partner_code_map()
    if CODE_MAP:
        for _df in (df_Sales_final, df_OB_final, df_backlog_final):
            if "SoldToPartyID" in _df.columns:
                _df["SoldToPartyID"] = remap_partner_codes(_df["SoldToPartyID"])

final_combined_backlog_df = df_backlog_final   # old name, still used by a few ad-hoc tools
print(f"\n✅ Data ready (source: {DATA_SOURCE}) | Sales {len(df_Sales_final):,} rows | "
      f"OB {len(df_OB_final):,} rows |                                                  AQacklog {len(df_backlog_final):,} rows | "
      f"latest invoice {pd.to_datetime(df_Sales_final['InvoiceDate']).max():%d-%b-%Y}")

## 2 · Master data
Retail datasets (needed by the Material Master update), Partner/Material Master updates (skipped in cache mode), then one load of every master file.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 2.1 RETAIL DATASETS — Retail channel filter + rebate → Net Sales
#
#   retail_sales / retail_ob   last 3 years (Year_Temp = invoice/OB year)
#   retail_backlog             full backlog, no year filter
# Used by the Retail reports and by the Material Master update.
# ════════════════════════════════════════════════════════════════════
DATE_CANDIDATES = ["Created on", "OrderCreationDate", "Calendar Day", "InvoiceDate"]


def build_retail_dataset(df, date_col, filter_by_year=True):
    if df is None or df.empty:
        return pd.DataFrame()
    df.columns = [str(c).strip() for c in df.columns]
    if date_col not in df.columns:
        date_col = next((c for c in DATE_CANDIDATES if c in df.columns), None)
        if date_col is None:
            return pd.DataFrame()
    years = pd.to_datetime(df[date_col], errors="coerce").dt.year.fillna(0).astype(int)
    mask = years.isin(DASHBOARD_YEAR_LABELS) if filter_by_year else pd.Series(True, index=df.index)
    if all(c in df.columns for c in ("PricingClassL2", "PricingClassL3", "HeliosCode")):
        mask &= is_retail(df) & df["HeliosCode"].isin(RETAIL_HELIOS_CODES)
    out = df[mask].copy()
    out["Year_Temp"] = years[mask]
    out["Gross Sales"] = pd.to_numeric(out["Gross Sales"], errors="coerce").fillna(0)
    out["Rebate"] = out["Year_Temp"].map(RETAIL_REBATES).fillna(RETAIL_REBATE_DEFAULT)
    out["Net Sales"] = out["Gross Sales"] * (1 - out["Rebate"])
    return out


retail_sales = build_retail_dataset(df_Sales_final, "InvoiceDate")
retail_ob = build_retail_dataset(df_OB_final, "Calendar Day")
retail_backlog = build_retail_dataset(df_backlog_final, "OrderCreationDate", filter_by_year=False)
print(f"✅ Retail datasets | Sales {len(retail_sales):,} | OB {len(retail_ob):,} | Backlog {len(retail_backlog):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 2.2 PARTNER MASTER UPDATE  (only when checking for new data)
#
# Data Dictionary → "Partner Master Data", from RD/KRO (ERW/EKR) Sales + OB:
#   • always refresh Onboarded Year, RD/KRO, Tier Category
#   • overwrite City/State from sales data, except Mumbai / Pune-PCMC partners
#   • fill Partner name / ASM only where blank
#   • add partners not in the master yet as Active, "New/Revival"
# City names are standardised through the "Alias Mapping" sheet.
# ════════════════════════════════════════════════════════════════════
import openpyxl


def update_partner_master():
    dict_bytes = sp.download(PATHS["dict_retail"]).getvalue()
    book = openpyxl.load_workbook(io.BytesIO(dict_bytes))
    master = pd.read_excel(io.BytesIO(dict_bytes), sheet_name="Partner Master Data", dtype={"Sold-to Party": str})
    geo = pd.read_excel(io.BytesIO(dict_bytes), sheet_name="Geography mapping")
    try:
        alias = pd.read_excel(io.BytesIO(dict_bytes), sheet_name="Alias Mapping")
        alias_map = dict(zip(alias["Old Name"].str.title(), alias["Standard Name"].str.title()))
    except Exception:
        alias_map = {}
        print("   ⚠️ Alias Mapping sheet missing")

    def standardise(series, label):
        raw = series.astype(str).str.strip().str.title().replace("Pune- Pcmc", "Pune- PCMC")
        out = raw.replace(alias_map)
        if (raw != out).sum():
            print(f"   ✨ {(raw != out).sum()} city names standardised in {label}")
        return out

    master["Sold-to Party"] = master["Sold-to Party"].apply(lambda v: "" if pd.isna(v) else re.sub(r"\.0$", "", str(v).strip()))
    master["City"] = standardise(master["City"], "Partner Master")
    geo["City"] = standardise(geo["City"], "Geography mapping")

    rename = {"Calendar Day": "Date", "InvoiceDate": "Date", "SoldToPartyID": "ID", "SoldToPartyName": "Patner",
              "PartnerCity": "City", "PartnerState": "State", "EmployeeResponsibleName": "ASM"}
    source = pd.concat([
        df_OB_final[df_OB_final["PricingClassL3"].isin(["ERW", "EKR"])].rename(columns=rename, errors="ignore"),
        df_Sales_final[df_Sales_final["PricingClassL3"].isin(["ERW", "EKR"])].rename(columns=rename, errors="ignore"),
    ], ignore_index=True)
    source["ID"] = source["ID"].apply(lambda v: "" if pd.isna(v) else re.sub(r"\.0$", "", str(v).strip()))
    source["City"] = standardise(source["City"], "sales data")
    source["RD/KRO"] = source["PricingClassL3"].map({"ERW": "RD", "EKR": "KRO"})
    source["Date"] = pd.to_datetime(source["Date"], errors="coerce")

    first_seen = source.sort_values("Date").drop_duplicates("ID", keep="first").set_index("ID")["Date"]
    onboarded = first_seen.apply(lambda d: "Unknown" if pd.isna(d) else ("2022 & Before" if d.year <= 2022 else str(d.year))).to_dict()
    latest = source.drop_duplicates("ID", keep="last").copy()
    latest["Onboarded Year"] = latest["ID"].map(onboarded)
    latest["Tier Category"] = latest["City"].apply(tier_category)
    candidates = latest.rename(columns={"ID": "Sold-to Party"}).set_index("Sold-to Party")

    for col in ["Onboarded Year", "RD/KRO", "Tier Category", "City", "State", "Patner", "ASM"]:
        if col not in master.columns:
            master[col] = None
        master[col] = master[col].astype(object)   # text values may go into an all-blank column
    known = master["Sold-to Party"].isin(candidates.index)
    excluded_cities = ["Pune- PCMC", "Pune - PCMC", "Mumbai"]
    changes = {}
    for col in ["Onboarded Year", "RD/KRO", "Tier Category"]:
        before = master.loc[known, col].copy()
        master.loc[known, col] = master.loc[known, "Sold-to Party"].map(candidates[col])
        changes[col] = int((before != master.loc[known, col]).sum())
    for col in ["City", "State"]:
        mask = known & ~master["City"].isin(excluded_cities)
        before = master.loc[mask, col].copy()
        master.loc[mask, col] = master.loc[mask, "Sold-to Party"].map(candidates[col])
        changes[col] = int((before != master.loc[mask, col]).sum())
    for col in ["Patner", "ASM"]:
        empty = master[col].isna() | master[col].astype(str).isin(["", "Unknown", "nan"])
        mask = known & empty & ~master["City"].isin(excluded_cities)
        before = master.loc[mask, col].copy()
        master.loc[mask, col] = master.loc[mask, "Sold-to Party"].map(candidates[col])
        changes[col] = int((before != master.loc[mask, col]).sum())

    new = candidates[~candidates.index.isin(set(master["Sold-to Party"]))].reset_index()
    if not new.empty:
        new["Active/Inactive"] = "Active"
        new["Channel Landscape"] = "New/Revival"
        master = pd.concat([master, new[master.columns.intersection(new.columns)]], ignore_index=True)
    master["Tier Category"] = master["City"].apply(tier_category)
    master["Last Updated"] = datetime.now().strftime("%Y-%m-%d %H:%M")

    for sheet, df in (("Partner Master Data", master), ("Geography mapping", geo)):
        if sheet not in book.sheetnames:
            book.create_sheet(sheet)
        ws = book[sheet]
        ws.delete_rows(1, ws.max_row + 1)
        for c, col in enumerate(df.columns, 1):
            ws.cell(row=1, column=c, value=col)
        for r, row in enumerate(df.values.tolist(), 2):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value="" if pd.isna(val) else val)
    out = io.BytesIO()
    book.save(out)
    sp.upload(PATHS["dict_retail"], out.getvalue())
    print(f"   🆕 new partners: {len(new)} | changes: {changes}")
    return f"{len(new)} new partners"


if USE_CACHE:
    print("⏭️  Partner Master update skipped (cache mode — master data left as it is)")
else:
    run_report("partner_master_update", update_partner_master)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 2.3 MATERIAL MASTER UPDATE  (only when checking for new data)
#
# Adds every HS material (all HS Helios codes, Retail AND Project, from Sales,
# OB and backlog) that isn't in the Material Master yet, refreshes descriptions,
# then (re)classifies every material:
#   Category · Helios Code · Family Actual · Classification
# A timestamped backup is written before the master is overwritten.
# ════════════════════════════════════════════════════════════════════
# --- Product lists & classification rules (unchanged) ---

WISER_1_0 = {
    "IH100WFVDP", "IH101ZW2IR", 
    "IH101ZWIFI", "IH11000200", "IH12000020", "IH1200010F", "IH14000100",  "IH12001010"
}

WISER_2_0 = {
    "CS16A1MSW_WE", "CS16A1MSW_GY", "CS16A1MSW_WG",
    "CD4SFR2M_WE", "CD4SFR2M_GY", "CD4SFR2M_WG",
    "CDWCRTC2M_WE", "CDWCRTC2M_GY", "CDWCRTC2M_WG",
    "CDWDIM2M_WE", "CDWDIM2M_GY", "CDWDIM2M_WG",
    "CDWSW2W6A1M_WE", "CDWSW2W6A1M_GY", "CDWSW2W6A1M_WG",
    "CDWAQI2M_WE", "CDWAQI2M_GY",
    "CR2CHRLYPG", "CCIRC1M", "ENMOD1P", "ENMOD3P", "CDWENMOD3P",
    "B1GTEWY", "CDWRLY1CH16A",
    "CSO616A_WE", "NXI-130", "NXI-150", "IH12001010",
}

WISER_SKUS = WISER_1_0 | WISER_2_0

# SAP marks every Wiser material with pricing group M5 "Easy Home" (product family 1LCY1JD1).
# Materials with it count as Wiser even before they are added to the lists above.
WISER_PRICING_GROUPS = {"M5", "EASY HOME"}


def sap_wiser_materials(*frames):
    """Material codes that carry a Wiser pricing group in any of the frames."""
    found = set()
    for df in frames:
        if df is None or df.empty or "MaterialCode" not in df.columns:
            continue
        hit = pd.Series(False, index=df.index)
        for col in ("MaterialPricingGroupCode", "MaterialPricingGroupDescription"):
            if col in df.columns:
                hit |= df[col].astype(str).str.strip().str.upper().isin(WISER_PRICING_GROUPS)
        found |= set(df.loc[hit, "MaterialCode"].astype(str).str.strip().str.upper())
    return found


SAP_WISER_SKUS = sap_wiser_materials(df_Sales_final, df_OB_final, df_backlog_final)
_unlisted = sorted(SAP_WISER_SKUS - WISER_SKUS)
print(f"Wiser in SAP (pricing group M5): {len(SAP_WISER_SKUS)} materials"
      + (f" | ⚠️ {len(_unlisted)} not in WISER_1_0 / WISER_2_0 yet (counted as Wiser anyway): {', '.join(_unlisted)}"
         if _unlisted else ""))


UnicaPurePlus = {
    "UN6FSW1W1M_WE","UN6FSW2W1M_WE","UN6FSW2W2M_WE","UN6FSW1W1M_GY","UN6FSW2W1M_GY","UN6FSW2W2M_GY","UN16FSW1W1M_WE",
    "UN16FSW2W1M_WE","UN16FSW2W2M_WE","UN16FSW1W1M_GY","UN16FSW2W1M_GY","UN16FSW2W2M_GY","UN20FDPSW2M_WE","UN32FDPSW2M_WE",
    "UN20FDPSW2M_GY","UN32FDPSW2M_GY","UN6FB1M_WE","UN6FB2M_WE","UN6FB1M_GY","UN6FB2M_GY","UNFDND1M_WE","UNFMMR1M_WE","UNFDMIL2M_WE",
    "UNFCUR2M_WE","NUPCEKCSW_WE","NUPCEKCSW_WG","UNFDND1M_GY","UNFMMR1M_GY","UNFDMIL2M_GY","UNFCUR2M_GY","NUPCEKCSW_GY",
    "NUPCEKCSW_SZ","NUPC1M_WE","NUPC2M_WE","NUPC3M_WE","NUPC4M_WE","NUPC6M_WE","NUPC8ML_WE","NUPC8MSQ_WE","NUPC9M_WE",
    "NUPC12M_WE","NUPC16M_WE","NUPC18M_WE","NUPC1M_GY","NUPC2M_GY","NUPC3M_GY","NUPC4M_GY","NUPC6M_GY","NUPC8ML_GY",
    "NUPC8MSQ_GY","NUPC9M_GY","NUPC12M_GY","NUPC16M_GY","NUPC18M_GY","NUPC1M_SZ","NUPC2M_SZ","NUPC3M_SZ","NUPC4M_SZ",
    "NUPC6M_SZ","NUPC8ML_SZ","NUPC8MSQ_SZ","NUPC12M_SZ","NUPC16M_SZ","NUPC18M_SZ","NUPC1M_WG","NUPC2M_WG","NUPC3M_WG",
    "NUPC4M_WG","NUPC6M_WG","NUPC8ML_WG","NUPC8MSQ_WG","NUPC12M_WG","NUPC16M_WG","NUPC18M_WG"
}


def normalize_helios(code: str) -> str:
    code = str(code).strip().upper()
    mapping = {"PPCFD": "HDCFD", "HSCWD": "HDCWD", "HSCOS": "HDCOS", "HSSPE": "HDSPE", "PREVC":"PREVC"}
    return mapping.get(code, code)

def get_easy9_ka(code: str) -> str:
    if not isinstance(code, str) or len(code) < 6: return ""
    code_clean = code.strip().upper()
    if code_clean.startswith('EZ9F'):
        ka_char = code_clean[5]
        return " 6kA" if ka_char == '6' else " 10kA" if ka_char == '1' else ""
    return ""

def get_pole_type(code: str, description: str = "") -> str:
    code_clean, desc_clean = str(code).upper(), str(description).upper()
    if any(x in desc_clean for x in ["1P", "1 POLE", "SP", "SINGLE POLE"]): return " SP"
    if any(x in desc_clean for x in ["2P", "3P", "4P", "MP", "DP", "TP", "3 POLE", "4 POLE"]): return " MP"
    if any(x in code_clean for x in ['1P', 'SP']): return " SP"
    if any(x in code_clean for x in ['2P', '3P', '4P', 'MP', 'DP', 'TP']): return " MP"
    if code_clean.startswith(('EZ9S', 'EZ9F', 'EZ9R', 'A9', 'M9')) and len(code_clean) >= 7:
        pole_digit = code_clean[6]
        return " SP" if pole_digit == '1' else " MP" if pole_digit in '234' else ""
    return ""

def get_category_by_description(description: str, code: str) -> str:
    desc, code = str(description).upper(), str(code).upper()
    if any(x in desc for x in ["RCCB", "RCD", "VIGI", "EARTH LEAKAGE"]): return "RCCB"
    if any(x in desc for x in ["ISOLATOR", "SWITCH-DISCONNECTOR", "SWITCH DISCONNECTOR"]): return "Isolator"
    if "SWITCH" in desc or "SOCKET" in desc or "SKT" in desc: return "Switch/Socket"
    if "MCB" in desc or "BREAKER" in desc: return "MCB"
    if any(x in desc for x in ["DISTRIBUTION BOARD", "ENCLOSURE", "DB", "CONSUMER UNIT"]): return "Distribution Board"
    if "SURGE" in desc or " SPD" in desc: return "Surge Arrestor"
    if "METAL BOX" in desc: return "Metal Box"
    if code.startswith(('EZ9R', 'A9R', 'A9N16', 'A9Z', 'A9KR', 'M9R', 'LV')): return "RCCB"
    if code.startswith(('EZ9F', 'M9', '186', '187', 'A9')): return "MCB"
    if code in WISER_SKUS: return "Wiser"
    return "Accessory/Other"

def get_fallback_helios(family: str) -> str:
    family_str = str(family)
    if any(x in family_str for x in ["Vigi", "Easy9", "Acti9", "Multi9", "FD"]): return "HDCFD"
    if any(x in family_str for x in ["AvatarOn", "Unica Pure", "Livia", "Miluz", "Clipsal", "Electra"]): return "HDCWD"
    if family_str in ["Wiser", "LRC"]: return "HDCOS"
    return ""

def categorize_material_code(code: str, category: str, helios_code: str = "") -> str:
    code = str(code).strip().upper()
    if code in WISER_1_0: return "Wiser 1.0"
    if code in WISER_2_0: return "Wiser 2.0"
    if code in SAP_WISER_SKUS: return "Wiser 2.0"
    if code in UnicaPurePlus: return "Unica Pure Plus"
    if code.startswith('E83'): return "AvatarOn"
    if code.startswith('P') and len(code) >= 2 and code[1] != 'K': return "Livia"
    if code.startswith('UN'): return "Unica Pure"
    if code.startswith('NUX'): return "UnicaX"
    if code.startswith('MZ'): return "Miluz Zeta"
    if code.startswith('LV'): return "FD- All others"
    if category == "RCCB" and code.startswith('EZ9'): return "Easy9 RCCB"
    if code.startswith('EZ9E'): return "Easy9 DB"
    if code.startswith('EZ9F'):
        ka = get_easy9_ka(code)
        return "Easy9 6kA" if "6kA" in ka else "Easy9 10kA" if "10kA" in ka else "FD- All others"
    if helios_code == "HDCFD": return "FD- Others"
    if helios_code == "HDCWD": return "WD- Others"
    if helios_code == "HDCOS": return "COS- Others"
    if helios_code == "HDSPE": return "SPE- Others"
    if helios_code == "PREVC": return "PREVC"
    if helios_code == "": return "SPE- Others"
    return "Others"

def get_family_actual_classification(code: str, category: str, description: str, helios_code: str = "") -> str:
    code = str(code).strip().upper()
    if code in WISER_SKUS or code in SAP_WISER_SKUS: return "Wiser"
    
    # 1. High-Priority Infrastructure
    if code.startswith('LV'): return "Vigi"
    if code.startswith('A9'): return "Acti9"
    
    # 2. Easy9 Logic (Keep your detailed DB logic here)
    if code.startswith('EZ9E'):
        sub_prefix = code[4:6]
        mapping = {
            'SN': "Easy9 DB SPN", 'TN': "Easy9 DB TPN", 'VN': "Easy9 DB VTPN",
            'VM': "Easy9 DB VTPN", 'PV': "Easy9 DB Phase Selector",
            'PS': "Easy9 DB Selector Switch", 'ME': "Easy9 Enclosure", 'MS': "Easy9 Metal P and S"
        }
        return mapping.get(sub_prefix, "Easy9 DB")

    if code.startswith('EZ9F'): 
        p_sfx, k_sfx = get_pole_type(code, description), get_easy9_ka(code)
        return f"Easy9 MCB{k_sfx}{p_sfx}"
    if code.startswith('EZ9L'): return "Easy9 Surge Arrestor"
    if code.startswith('EZ9R'): return "Easy9 RCCB"
    if code.startswith('EZ9S'): return "Easy9 Isolator"

    # 3. Specific Prefix Fixes
    if code.startswith(('AAKY', 'AAXY')): return "Opale"
    if code.startswith('AAKX'): return "Livia"
    if code.startswith(('PKE', 'PKF')): return "PratiKa" # Added specific PratiKa
    if code.startswith('MTN'): return "KNX" # For SpaceLogic/KNX items
    
    # 4. Standard Range Logic
    if code in UnicaPurePlus: return "Unica Pure Plus"
    if code.startswith('UN'): return "Unica Pure"
    if code.startswith('NUX'): return "UnicaX"
    if code.startswith(('NP', 'NUX', 'UN')): return "Unica Pure"
    if code.startswith('MZ'): return "Miluz Zeta"
    if code.startswith('ML'): return "Miluz Lara"
    if code.startswith('CPX'): return "ClipsalX"
    if code.startswith('X'): return "Opale"
    if code.startswith('IN'): return "Zencelo"
    if code.startswith(('E83', 'E84')): return "AvatarOn"
    if code.startswith(('AP', 'AS', 'BP', 'BS')) and len(code) <= 5: return "Electra"
    if code.startswith('P'): return "Livia"
    
    # 5. Helios Fallbacks
    helios_map = {
        "HDCFD": "FD- Others", "HDCWD": "WD- Others",
        "HDCOS": "COS- Others", "HDSPE": "SPE- Others", "HSSPE": "SPE-Others", "PREVC": "EVC"
    }
    return helios_map.get(helios_code, "Others")

def hs_material_rows():
    """Material code, description, Helios code and channel of every HS row in Sales, OB and backlog
    (all HS Helios codes, Retail and Project), oldest first so the last row per material is the latest."""
    parts = []
    for df, date_col in ((df_backlog_final, "OrderCreationDate"), (df_OB_final, "Calendar Day"),
                         (df_Sales_final, "InvoiceDate")):
        if df is None or df.empty or "MaterialCode" not in df.columns:
            continue
        d = df[df["HeliosCode"].isin(RETAIL_HELIOS_CODES)] if "HeliosCode" in df.columns else df
        part = d[[c for c in ("MaterialCode", "MaterialDescription", "HeliosCode") if c in d.columns]].copy()
        part["MaterialCode"] = part["MaterialCode"].astype(str).str.strip().str.upper()
        part["Date"] = pd.to_datetime(d[date_col], errors="coerce") if date_col in d.columns else pd.NaT
        part["Retail"] = is_retail(d) if {"PricingClassL2", "PricingClassL3"} <= set(d.columns) else False
        parts.append(part)
    if not parts:
        return pd.DataFrame(columns=["MaterialCode", "MaterialDescription", "HeliosCode", "Date", "Retail"])
    rows = pd.concat(parts, ignore_index=True)
    return rows[~rows["MaterialCode"].isin(["", "NAN", "NONE"])].sort_values("Date", kind="stable", na_position="first")


def update_material_master():
    master = sp.read_excel(PATHS["material_master"])
    master["Material"] = master["Material"].astype(str).str.strip().str.upper()
    was_wiser = set(master.loc[master["Family Actual"] == "Wiser", "Material"]) if "Family Actual" in master else set()

    combined = hs_material_rows()
    helios_by_material = {}
    if not combined.empty:
        latest = combined.drop_duplicates(subset=["MaterialCode"], keep="last")
        new_codes = set(latest["MaterialCode"].unique()) - set(master["Material"].unique())
        if new_codes:
            master = pd.concat([master, pd.DataFrame({"Material": sorted(new_codes)})], ignore_index=True)
            retail_codes = set(combined.loc[combined["Retail"], "MaterialCode"])
            print(f"   🆕 {len(new_codes)} new material(s): {len(new_codes & retail_codes)} sold in Retail, "
                  f"{len(new_codes - retail_codes)} only in Project")
        helios_by_material = dict(zip(latest["MaterialCode"], latest["HeliosCode"]))
        desc_by_material = dict(zip(latest["MaterialCode"], latest["MaterialDescription"]))
        if "Product Description" not in master.columns:
            master["Product Description"] = ""
        master["Product Description"] = master["Material"].map(desc_by_material).fillna(master["Product Description"])

    master["Category"] = master.apply(lambda x: get_category_by_description(x["Product Description"], x["Material"]), axis=1)
    family_guess = master.apply(lambda x: get_family_actual_classification(x["Material"], x["Category"], x["Product Description"]), axis=1)
    # Helios code: latest in the data, else the one already in the master, else a guess from the family
    existing = master["Helios Code"].astype(str).str.strip().replace({"": None, "nan": None, "None": None}) \
        if "Helios Code" in master else pd.Series(None, index=master.index)
    master["Helios Code"] = (master["Material"].map(helios_by_material).fillna(existing)
                             .fillna(family_guess.apply(get_fallback_helios)).apply(normalize_helios))
    master["Family Actual"] = master.apply(lambda x: get_family_actual_classification(x["Material"], x["Category"], x["Product Description"], x["Helios Code"]), axis=1)
    master["Classification"] = master.apply(lambda x: categorize_material_code(x["Material"], x["Category"], x["Helios Code"]), axis=1)
    now_wiser = set(master.loc[master["Family Actual"] == "Wiser", "Material"])
    if now_wiser - was_wiser:
        print(f"   🔌 now Wiser: {', '.join(sorted(now_wiser - was_wiser))}")

    buf = io.BytesIO()
    with report_writer(buf) as writer:
        write_table(writer, master, "Sheet1", tab=THEME["tab_data"])
    data = buf.getvalue()
    sp.ensure_folder(PATHS["material_backups"])
    stem = PATHS["material_master"].split("/")[-1].replace(".xlsx", "")
    sp.upload(f"{PATHS['material_backups']}/{stem}_Backup_{datetime.now():%Y%m%d_%H%M}.xlsx", data)
    sp.upload(PATHS["material_master"], data)
    return f"{len(master):,} materials"



run_report("material_master_update", update_material_master)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 2.4 LOAD MASTER DATA — once, for every report
#
# Read from SharePoint in both modes (read only — in cache mode the Partner /
# Material Master updates above are skipped, so nothing is refreshed).
# Reports get copies via master("name"), so no report can change another
# report's inputs.
# ════════════════════════════════════════════════════════════════════
_MASTER_SOURCES = {
    "partner_retail": ("dict_retail", "Partner Master Data"),
    "geo_retail": ("dict_retail", "Geography mapping"),
    "ssa_map": ("dict_retail", "SSA Mapping"),
    "alias_map": ("dict_retail", "Alias Mapping"),
    "partner_project": ("dict_project", "Partner Master Data"),
    "geo_project": ("dict_project", "Geography mapping"),
    "ns_branches": ("dict_project", "NS Branches"),
    "material": ("material_master", "Sheet1"),
    "partner_targets": ("targets_master", "Partner Targets"),
    "branch_targets": ("targets_master", "Branch Targets"),
    "qtod_atod": ("targets_qtod", "QTOD - ATOD"),
}

_MASTER = {}
_workbooks = {}
for key, (path_key, sheet) in _MASTER_SOURCES.items():
    try:
        if path_key not in _workbooks:
            _workbooks[path_key] = sp.download(PATHS[path_key]).getvalue()
        df = pd.read_excel(io.BytesIO(_workbooks[path_key]), sheet_name=sheet)
        df.columns = [str(c).strip() for c in df.columns]
        _MASTER[key] = df
    except Exception as e:
        print(f"   ⚠️ {key} ({PATHS[path_key].split('/')[-1]} › {sheet}) not loaded: {e}")
        _MASTER[key] = pd.DataFrame()
del _workbooks

# One row per partner code after code changes (see APPLY_PARTNER_CODE_REMAP)
_MASTER_RAW = {k: v.copy() for k, v in _MASTER.items()}   # as in the files — used by the review (cell 9)

if APPLY_PARTNER_CODE_REMAP and CODE_MAP:
    for key, col in (("partner_retail", "Sold-to Party"), ("partner_project", "Sold-to Party"),
                     ("partner_targets", "Sold-to Party"), ("qtod_atod", "CC")):
        _MASTER[key] = consolidate_master_by_code(_MASTER[key], col)

# One row per key in every mapping table. A duplicated key (e.g. a city listed
# twice in Geography mapping, a material twice in Material Master) would copy
# every matching sales row in a join and inflate sales. The first row is kept;
# cell 9 lists all duplicates so they can be removed from the files themselves.
MAPPING_KEYS = {
    "partner_retail": "Sold-to Party", "partner_project": "Sold-to Party",
    "geo_retail": "City", "geo_project": "City", "material": "Material",
    "ns_branches": "Sales Office", "ssa_map": "Branch",
}


def mapping_key(series):
    """How keys are compared: text, trimmed, upper-case, no trailing '.0'."""
    return series.map(lambda v: "" if pd.isna(v) else str(v)).str.strip().str.upper().str.replace(r"\.0$", "", regex=True)


for key, col in MAPPING_KEYS.items():
    df = _MASTER[key]
    if df.empty or col not in df.columns:
        continue
    k = mapping_key(df[col])
    dup = k.duplicated(keep="first") & (k != "")      # blank keys can't match sales — left alone
    if dup.any():
        print(f"   ⚠️ {key}: {int(dup.sum())} duplicate '{col}' row(s) ignored (first kept) — see Master_Data_Review")
        _MASTER[key] = df[~dup.values].reset_index(drop=True)


def master(name):
    """A private copy of one master table."""
    return _MASTER[name].copy()


# --- Derived lookups shared by several reports ---------------------------
def make_partner_metadata(df_p, df_g, df_s):
    """Partner Master + Geography (City→Branch) + SSA Mapping (Branch→SSA)."""
    for df in (df_p, df_g, df_s):
        df.drop(columns=[c for c in df.columns if "Unnamed" in c], inplace=True)
    df_p["Sold-to Party"] = df_p["Sold-to Party"].astype(str).str.split(".").str[0].str.strip()
    df_p["City"] = df_p["City"].astype(str).str.strip().str.upper()
    df_g["City"] = df_g["City"].astype(str).str.strip().str.upper()
    df_g["Branch"] = df_g["Branch"].astype(str).str.strip().str.upper()
    df_s["Branch"] = df_s["Branch"].astype(str).str.strip().str.upper()
    meta = (df_p[["Sold-to Party", "Patner", "GSTN", "ASM", "RD/KRO", "City", "State", "Region", "Tier Category"]]
            .merge(df_g[["City", "Branch"]], on="City", how="left")
            .merge(df_s[["Branch", "SSA"]], on="Branch", how="left")
            .drop_duplicates(subset=["Sold-to Party"], keep="first")
            .rename(columns={"Sold-to Party": "SoldToPartyID"}))
    return meta


def make_partner_map(df_partner, df_geo):
    """Sold-to Party → ASM, Zone, Branch (via City)."""
    pm = (df_partner.merge(df_geo[["City", "Zone", "Branch"]].drop_duplicates("City"), on="City", how="left")
          [["Sold-to Party", "ASM", "Zone", "Branch"]].drop_duplicates("Sold-to Party"))
    pm["Sold-to Party"] = pm["Sold-to Party"].astype(str).str.strip()
    return pm


def make_office_maps():
    """Sales Office → Branch / Region for Project (non-retail) rows (NS Branches)."""
    ns = master("ns_branches")
    ns["Sales office location"] = ns["Sales Office"].astype(str).str.strip().str.upper()
    ns = ns.drop_duplicates("Sales office location").set_index("Sales office location")
    return ns["Branch"].to_dict(), ns["Region"].to_dict()


Partner_Metadata = make_partner_metadata(master("partner_retail"), master("geo_retail"), master("ssa_map"))
Partner_Metadata = Partner_Metadata[[c for c in ["SoldToPartyID", "Patner", "GSTN", "ASM", "RD/KRO", "City", "Branch",
                                                 "Region", "Tier Category", "SSA"] if c in Partner_Metadata.columns]]
Partner_Metadata_HS = make_partner_metadata(master("partner_project"), master("geo_project"), master("ssa_map"))
Partner_Metadata_HS = Partner_Metadata_HS.rename(columns={"State_x": "State", "Tier Category_x": "Tier Category", "Zone": "Region"})
Partner_Metadata_HS = Partner_Metadata_HS[["SoldToPartyID", "Patner", "GSTN", "RD/KRO", "City", "Branch", "Region", "Tier Category"]]

Material_Map = master("material")
Targets_Map = master("qtod_atod")
family_map = Material_Map[["Material", "Family Actual"]].copy()
family_map["Material"] = family_map["Material"].astype(str).str.strip().str.upper()
family_map = family_map.drop_duplicates(subset=["Material"])

# Wiser per SAP pricing group (cell 2.3) also when the master hasn't been updated yet (e.g. cache mode)
_sap_wiser = globals().get("SAP_WISER_SKUS", set())
_known = set(family_map.loc[family_map["Family Actual"] == "Wiser", "Material"])
if _sap_wiser - _known:
    _mm_key = Material_Map["Material"].astype(str).str.strip().str.upper()
    Material_Map.loc[_mm_key.isin(_sap_wiser), "Family Actual"] = "Wiser"
    _absent = sorted(_sap_wiser - set(_mm_key))
    Material_Map = pd.concat([Material_Map, pd.DataFrame({"Material": _absent, "Family Actual": "Wiser"})], ignore_index=True)
    family_map.loc[family_map["Material"].isin(_sap_wiser), "Family Actual"] = "Wiser"
    family_map = pd.concat([family_map, pd.DataFrame({"Material": _absent, "Family Actual": "Wiser"})], ignore_index=True)
    print(f"   🔌 {len(_sap_wiser - _known)} material(s) counted as Wiser from SAP pricing group M5 "
          f"(not Wiser in the Material Master yet)")

Partner_Map_Retail = make_partner_map(master("partner_retail"), master("geo_retail"))
Partner_Map_Project = make_partner_map(master("partner_project"), master("geo_project"))
project_branch_map, project_region_map = make_office_maps()

print(f"✅ Master data | {len(Partner_Metadata):,} retail partners | {len(Partner_Metadata_HS):,} HS partners | "
      f"{len(Material_Map):,} materials | {len(Targets_Map):,} QTOD/ATOD rows")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 2.5 TARGET KEYS — columns that link sales to Channel_Targets_<year>.xlsx
#
# add_target_keys(df) adds, to the Retail and HS dashboard input files:
#   Target Branch   Retail: the partner's branch (Partner Master city → Geography
#                   mapping, as the Retail dashboard); Project: the sales office
#                   branch (NS Branches)
#   Target GTM      Retail / Project
#   Target Month    1st of the invoice month (Sales) or calendar-day month (OB)
#   Target Year · Target Quarter (Q1–Q4)
#   Target Activity WD / FD / COS / SPE / EVC (from the Helios code)
#   Product Group   Unica Pure, Livia, AvatarOn, Miluz Zeta, WD- All others,
#                   Easy9 6kA / 10kA / RCCB / DB, FD- All others, Wiser,
#                   COS- All others, SPE, EVC
#   Partner key     partner code as text
# The same rules split the targets (section 10), so a sale and its target always
# meet. In Tableau, relate each targets sheet on these columns (no calculations).
# Source: tableau/targets/organize_targets.py (this cell is generated from it).
# ════════════════════════════════════════════════════════════════════
import types


def _target_split_tools():
    """organize_targets.py, wrapped so its names stay out of the notebook's globals."""
    import datetime as _dt
    import re
    import numpy as np
    import pandas as pd

    SALES_COLUMNS = ["Year", "Month", "GTM", "Branch", "Region", "HeliosCode", "Family", "Partner", "Gross"]

    ACTIVITY = {"HDCWD": "WD", "HSCWD": "WD", "HDCFD": "FD", "PPCFD": "FD", "HDFDA": "FD", "HDFDB": "FD", "HDFDS": "FD",
                "HDCOS": "COS", "HSCOS": "COS", "HDSPE": "SPE", "HSSPE": "SPE", "PREVC": "EVC"}
    ACTIVITIES = ["WD", "FD", "COS", "SPE", "EVC"]
    # product groups as in the 'Product Targets' sheet, with the activity each belongs to
    GROUPS = [("Unica Pure", "WD"), ("Livia", "WD"), ("AvatarOn", "WD"), ("Miluz Zeta", "WD"), ("WD- All others", "WD"),
              ("Easy9 6kA", "FD"), ("Easy9 10kA", "FD"), ("Easy9 RCCB", "FD"), ("Easy9 DB", "FD"), ("FD- All others", "FD"),
              ("Wiser", "COS"), ("COS- All others", "COS"), ("SPE", "SPE"), ("EVC", "EVC")]
    GROUP_ACTIVITY = dict(GROUPS)
    FOCUS = {"Miluz Zeta": "Miluz Zeta", "Wiser": "Wiser"}      # monthly target column -> product group
    FOCUS_FAMILY = {"Zeta Target": "Miluz Zeta", "Wiser Target": "Wiser"}   # QTOD columns

    # branch spellings in the target files -> the branch the sales data uses
    BRANCH_ALIASES = {"NCR(DEL+ROH)": "NCR", "NCR(NOI+GHAZ)": "NOIDA-GHZ", "PNQ": "PUNE", "NAN": "NASHIK", "TN2": "TN-2",
                      "TN -1": "TN-1", "TN-1 ": "TN-1", "AP ": "AP", "ap": "AP", "UP": "UP-1", "MP": "INDORE"}
    SSP_ALIASES = {"Katson Agency": "Kaston Agency", "Balaji Sales": "Balaji Sales Corp"}
    QUARTERS = {"Q1": (1, 2, 3), "Q2": (4, 5, 6), "Q3": (7, 8, 9), "Q4": (10, 11, 12)}
    MIN_PARTNER_SALES = 1e5          # partners below this base-year sales get their branch's product mix


    def product_group(activity, family):
        f = str(family or "")
        if activity == "WD":
            if f in ("Unica Pure", "Unica Pure Plus"):
                return "Unica Pure"
            return f if f in ("Livia", "AvatarOn", "Miluz Zeta") else "WD- All others"
        if activity == "FD":
            if f.startswith("Easy9 MCB 6kA"):
                return "Easy9 6kA"
            if f.startswith("Easy9 MCB 10kA"):
                return "Easy9 10kA"
            if f == "Easy9 RCCB":
                return "Easy9 RCCB"
            if f.startswith("Easy9 DB") or f in ("Easy9 Enclosure", "Easy9 Metal P and S"):
                return "Easy9 DB"
            return "FD- All others"
        if activity == "COS":
            return "Wiser" if f == "Wiser" else "COS- All others"
        return activity if activity in ("SPE", "EVC") else None


    def canon_branch(b):
        if b is None or (isinstance(b, float) and np.isnan(b)) or str(b).strip() in ("", "0", "nan"):
            return None
        s = str(b).strip()
        return BRANCH_ALIASES.get(s, BRANCH_ALIASES.get(s.upper(), s))


    def clean_code(v):
        return re.sub(r"\.0$", "", str(v).strip()) if pd.notna(v) else ""


    def num(s):
        return pd.to_numeric(s, errors="coerce")


    # ====================================================================== sales basis
    def prepare_sales(sales):
        s = sales.copy()
        s["Activity"] = s["HeliosCode"].astype(str).str.strip().str.upper().map(ACTIVITY)
        s["Group"] = [product_group(a, f) for a, f in zip(s["Activity"], s["Family"])]
        retail = s["GTM"] == "Retail"          # the aliases are Retail spellings; Project uses NS Branches names (NAN, MP ...)
        s.loc[retail, "Branch"] = s.loc[retail, "Branch"].map(canon_branch)
        s["Partner"] = s["Partner"].map(clean_code)
        s["Gross"] = num(s["Gross"]).fillna(0.0)
        return s[s["Group"].notna()]


    def shares(df, keys, col, weight="Gross"):
        """Share of `col` within `keys` (e.g. month within branch)."""
        g = df.groupby(keys + [col])[weight].sum()
        tot = g.groupby(level=list(range(len(keys)))).transform("sum")
        return (g / tot.where(tot > 0)).fillna(0.0)


    def lookup_share(table, key, values, fallback):
        """{value: share} for one key, or the fallback table's shares when the key has no sales."""
        try:
            s = table.loc[key]
            if s.sum() > 0:
                return s.reindex(values, fill_value=0.0)
        except KeyError:
            pass
        return fallback.reindex(values, fill_value=0.0)


    def contribution(sales, base_year):
        """Base-year sales contribution tables (values in rupees and shares)."""
        s = sales[sales["Year"] == base_year]
        out = {}

        def table(part, keys, col, name):
            v = part.groupby(keys + [col])["Gross"].sum().unstack(col, fill_value=0.0)
            if col == "Month":
                v.columns = [_dt.date(2000, int(m), 1).strftime("%b") for m in v.columns]
            tot = v.sum(axis=1)
            sh = v.div(tot.where(tot > 0), axis=0).fillna(0.0)
            sh.columns = [f"{c} %" for c in sh.columns]
            out[name] = pd.concat([v.assign(Total=tot), sh], axis=1).reset_index()

        for g in ("Retail", "Project"):
            part = s[s["GTM"] == g]
            if part.empty:
                continue
            table(part, ["Region", "Branch"], "Month", f"{g} branch x month")
            table(part, ["Region", "Branch"], "Activity", f"{g} branch x activity")
            table(part, ["Region", "Branch"], "Group", f"{g} branch x product")
            table(part, ["Region"], "Group", f"{g} region x product")
        v = s.groupby(["Region", "Branch", "GTM"])["Gross"].sum().unstack("GTM", fill_value=0.0)
        tot = v.sum(axis=1)
        out["Retail vs Project"] = v.assign(Total=tot, **{f"{c} %": v[c] / tot.where(tot > 0) for c in v.columns}).reset_index()
        p = s[s["GTM"] == "Retail"].groupby(["Branch", "Partner", "Group"])["Gross"].sum().unstack("Group", fill_value=0.0)
        out["Retail partner x product"] = p.assign(Total=p.sum(axis=1)).sort_values("Total", ascending=False).reset_index()
        return out


    # ====================================================================== organise
    class Issues(list):
        def add(self, sheet, what, action, rows=""):
            self.append({"Source sheet": sheet, "Problem": what, "What was done": action, "Rows / values": str(rows)[:500]})


    def _mix_rows(total, mix, focus, keys, source_split):
        """Rows of one target split over product groups: `focus` groups as given, the rest by `mix`."""
        rest = total - sum(v for v, _ in focus.values())
        others = mix.drop(list(focus), errors="ignore")
        others = others / others.sum() if others.sum() > 0 else others
        rows = []
        for grp, act in GROUPS:
            if grp in focus:
                v, how = focus[grp]
            else:
                v, how = max(rest, 0.0) * others.get(grp, 0.0), source_split
            if v:
                rows.append((*keys, act, grp, v, "Net", how))
        return rows, rest


    def retail_targets(master, sales, year, issues):
        """Retail yearly targets ('Branch Targets') and monthly targets ('Monthly Targets' = the monthly commit), each as
        given and split over product groups by the branch's base-year mix. The two are independent: months are never
        derived from the yearly target and need not add up to it."""
        base = sales[(sales["Year"] == year - 1) & (sales["GTM"] == "Retail")]
        grp_sh = shares(base, ["Branch"], "Group")
        reg_sh = shares(base, ["Region"], "Group")
        grp_all = base.groupby("Group")["Gross"].sum() / base["Gross"].sum()
        names = [g for g, _ in GROUPS]

        bt = master["Branch Targets"].copy()
        bt["Branch"] = bt["Branch"].map(canon_branch)
        region = dict(zip(bt["Branch"], bt["Zone"]))
        yearly = bt[bt["Year"] == year].groupby("Branch")["Target"].sum(min_count=1).dropna()
        if bt.loc[bt["Year"] == year - 1, "Target"].isna().all() and (bt["Year"] == year - 1).any():
            issues.add("Branch Targets", f"{year - 1} rows have no target", "ignored")

        mt = master["Monthly Targets"].iloc[:, :6].copy()
        mt.columns = ["Zone", "Branch", "Month", "Target", "Miluz Zeta", "Wiser"]
        renamed = sorted({b for b in mt["Branch"].dropna().astype(str).str.strip() if canon_branch(b) != b})
        if renamed:
            issues.add("Monthly Targets", "branch names that differ from the sales data",
                       "renamed: " + ", ".join(f"{b} -> {canon_branch(b)}" for b in renamed))
        side = master["Monthly Targets"].iloc[:, 6:].dropna(how="all")
        if not side.empty:
            issues.add("Monthly Targets", "a second table to the right of the data (e.g. 'Oct WD / FD / Wiser Plan', in lakh)",
                       "not read; keep only the data columns in this sheet", f"{len(side)} rows")
        mt["Branch"] = mt["Branch"].map(canon_branch)
        mt["Month"] = pd.to_datetime(mt["Month"], errors="coerce")
        mt = mt[mt["Month"].dt.year == year]
        for c in ("Target", "Miluz Zeta", "Wiser"):
            mt[c] = num(mt[c])
        mt["m"] = mt["Month"].dt.month
        monthly = mt.groupby(["Branch", "m"])[["Target", "Miluz Zeta", "Wiser"]].sum(min_count=1).dropna(subset=["Target"])
        have = sorted(monthly.index.get_level_values(1).unique())
        if len(have) < 12:
            missing = [_dt.date(year, m, 1).strftime("%b") for m in range(1, 13) if m not in have]
            issues.add("Monthly Targets", f"no monthly targets yet for {', '.join(missing)}",
                       "left empty (monthly targets are never derived from the yearly target) — add them when set")
        for c in ("Miluz Zeta", "Wiser"):
            blank = mt[mt["Target"].notna() & mt[c].isna()]
            if len(blank):
                issues.add("Monthly Targets", f"blank {c} target", "taken as 0",
                           ", ".join(f"{b} {d:%b}" for b, d in zip(blank["Branch"], blank["Month"])))
        for b in sorted(set(monthly.index.get_level_values(0)) - set(region)):
            issues.add("Monthly Targets", f"branch {b} is not in 'Branch Targets'", "region left blank")

        def mix_of(b):
            return lookup_share(grp_sh, b, names, lookup_share(reg_sh, region.get(b), names, grp_all))

        split = "Split by last year's product mix"
        y_rows, m_rows = [], []
        for b, y in yearly.items():
            rows, _ = _mix_rows(y, mix_of(b), {}, (year, region.get(b), b, "Retail"), "Yearly as given; " + split.lower())
            y_rows += rows
        for (b, m), r in monthly.iterrows():
            focus = {grp: (float(r[col]) if pd.notna(r[col]) else 0.0, "Given") for col, grp in FOCUS.items()}
            rows, rest = _mix_rows(r["Target"], mix_of(b), focus, (year, _dt.date(year, m, 1), region.get(b), b, "Retail"),
                                   "Given month; " + split.lower())
            if rest < 0:
                issues.add("Monthly Targets", f"{b} {_dt.date(year, m, 1):%b}: Zeta + Wiser above the branch target",
                           "other product groups 0")
            m_rows += rows
        tm = monthly["Target"].groupby(level=0).sum()
        over = [(b, tm[b], yearly[b]) for b in tm.index if b in yearly and tm[b] > yearly[b]]
        if over:
            issues.add("Monthly Targets", "monthly targets already add up to more than the yearly target",
                       "both kept as given (they are separate targets) — just for information",
                       "; ".join(f"{b}: months {t / 1e7:,.2f} Cr vs yearly {y / 1e7:,.2f} Cr" for b, t, y in over))
        ycols = ["Year", "Region", "Branch", "GTM", "Activity", "Product group", "Target", "Basis", "Source"]
        mcols = ["Year", "Month", "Region", "Branch", "GTM", "Activity", "Product group", "Target", "Basis", "Source"]
        return pd.DataFrame(y_rows, columns=ycols), pd.DataFrame(m_rows, columns=mcols), yearly, tm, region


    def _project_mix(base, b, act):
        sub = base[base["Activity"] == act]
        grp = sub.groupby(["Branch", "Group"])["Gross"].sum()
        mix = grp.loc[b] if b in grp.index.get_level_values(0) and grp.loc[b].sum() > 0 else sub.groupby("Group")["Gross"].sum()
        if mix.sum() <= 0:
            mix = pd.Series({[g for g, a in GROUPS if a == act][-1]: 1.0})
        return mix / mix.sum()


    def project_targets(hs, sales, year, issues):
        """Project yearly targets from the HS file's 'Branch Targets' (by activity), split over product groups."""
        if hs is None or "Branch Targets" not in hs:
            return pd.DataFrame()
        bt = hs["Branch Targets"].copy()
        rr = bt[(bt["Year"] == year) & (bt["Category"] == "Retail")]
        if len(rr) and rr["Target"].nunique() == 1:
            issues.add("HS Branch Targets", f"every Retail branch has the same {year} target ({rr['Target'].iloc[0]:,.0f})",
                       "replaced by the Retail 'Branch Targets' (Retail and HS now share one Retail target)")
        bt = bt[(bt["Year"] == year) & (bt["Category"] == "Project")]
        dup = bt[bt.duplicated(["Branch", "Helios Code"], keep=False)]
        if len(dup):
            issues.add("HS Branch Targets", "two rows for the same branch and Helios code (HDCOS twice per branch)",
                       "added together as COS", f"{dup['Branch'].nunique()} branches")
        bt["Activity"] = bt["Helios Code"].map(ACTIVITY)
        base = sales[(sales["Year"] == year - 1) & (sales["GTM"] == "Project")]
        region = base.groupby("Branch")["Region"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None).to_dict()
        rows = []
        for (b, act), y in bt.groupby(["Branch", "Activity"])["Target"].sum().items():
            for g, sh in _project_mix(base, b, act).items():
                rows.append((year, region.get(b), b, "Project", act, g, y * sh, "Net",
                             "Yearly by activity as given; split by last year's Project product mix"))
        if (~bt["Branch"].isin(region)).any():
            issues.add("HS Branch Targets", "Project branches with no Project sales last year",
                       "product split by all Project sales; region left blank", ", ".join(sorted(set(bt["Branch"]) - set(region))))
        return pd.DataFrame(rows, columns=["Year", "Region", "Branch", "GTM", "Activity", "Product group", "Target", "Basis",
                                           "Source"])


    def project_monthly(com, sales, year, issues, yearly_p=None):
        """Project monthly targets = the HS file's Project commits by activity, split over product groups.
        The HS file's Retail commits are not used: Retail monthly targets come from the Retail 'Monthly Targets' sheet,
        so the Retail and HS dashboards show the same Retail numbers."""
        if com is None or com.empty:
            return pd.DataFrame()
        if (com["GTM"] == "Retail").any():
            issues.add("Activity wise Month Target", "Retail commits in the HS file differ from the Retail 'Monthly Targets'",
                       "not used — Retail monthly targets come from the Retail file for both dashboards",
                       f"{(com['GTM'] == 'Retail').sum()} rows")
        p = com[(com["GTM"] == "Project") & com["Activity"].notna()]
        fd = p[p["Activity"] == "FD"]
        if len(fd):
            issues.add("Activity wise Month Target", "Project FD commits (FD is always Retail; values up to 124 Cr a month)",
                       "not used", f"{len(fd)} rows")
        p = p[p["Activity"] != "FD"]
        if yearly_p is not None and len(yearly_p):
            # a month whose commit equals the whole yearly target is the yearly figure pasted in, not a monthly commit
            y = yearly_p.groupby(["Branch", "Activity"])["Target"].sum()
            tot = p.groupby(["Branch", "Month", "Activity"])["Commit"].transform("sum")
            yv = pd.Series([y.get((b, a), np.nan) for b, a in zip(p["Branch"], p["Activity"])], index=p.index)
            pasted = (yv > 0) & ((tot - yv).abs() <= 0.001 * yv)
            if pasted.any():
                months = sorted(p.loc[pasted, "Month"].dt.strftime("%b").unique())
                issues.add("Activity wise Month Target", f"Project commits for {', '.join(months)} equal the whole yearly target",
                           "not used (the yearly figure pasted in as a month)", f"{p.loc[pasted, 'Branch'].nunique()} branches")
                p = p[~pasted]
        if (com["Commit"].round(0) != com["OB Commit"].round(0)).mean() < 0.01:
            issues.add("Activity wise Month Target", "'OB Commit' is the same as 'Commit' on almost every row",
                       "only 'Commit' kept (as the monthly target)")
        base = sales[(sales["Year"] == year - 1) & (sales["GTM"] == "Project")]
        rows = []
        for (b, m, reg, act), v in p.groupby(["Branch", "Month", "Region", "Activity"], dropna=False)["Commit"].sum().items():
            for g, sh in _project_mix(base, b, act).items():
                rows.append((year, pd.Timestamp(m).date(), reg, b, "Project", act, g, v * sh, "Net",
                             "Given commit by activity; split by last year's Project product mix"))
        return pd.DataFrame(rows, columns=["Year", "Month", "Region", "Branch", "GTM", "Activity", "Product group", "Target",
                                           "Basis", "Source"])


    def commits(hs, year, issues):
        """The HS file's 'Activity wise Month Target' rows for the year (Commit by branch, month, GTM and activity)."""
        if hs is None or "Activity wise Month Target" not in hs:
            return pd.DataFrame()
        c = hs["Activity wise Month Target"].copy()
        c["Month"] = pd.to_datetime(c["Month and year"], errors="coerce")
        c = c[c["Month"].dt.year == year]
        for k in ("Commit", "OB Commit"):
            c[k] = num(c[k])
        c["Activity"] = c["Helios Code"].map(ACTIVITY)
        months = sorted(c.loc[c["Category"] == "Project", "Month"].dt.month.unique())
        if months and len(months) < 12:
            issues.add("Activity wise Month Target", f"Project commits only for {len(months)} months "
                       f"({', '.join(_dt.date(year, m, 1).strftime('%b') for m in months)})", "other months left empty")
        out = c.rename(columns={"Zone": "Region", "Category": "GTM"})
        retail = out["GTM"] == "Retail"
        out.loc[retail, "Branch"] = out.loc[retail, "Branch"].map(canon_branch)
        return out[["Month", "Region", "Branch", "GTM", "Activity", "Commit", "OB Commit"]]


    def partner_targets(qtod, master, sales, year, partner_info, region, issues):
        q = qtod.copy()
        q.columns = [str(c).strip() for c in q.columns]
        q = q[q["Target Year"] == year].copy()
        q["Partner Code"] = q["CC"].map(clean_code)
        qcols = ["Qtr 1 Target", "Qtr 2 Target", "Qtr 3 Target", "Qtr 4 Target"]
        for c in qcols + ["Target 100%", "Target 108%"] + list(FOCUS_FAMILY):
            q[c] = num(q[c])
        text = [c for c in qcols if qtod[c].map(lambda v: isinstance(v, str)).any()]
        if text:
            issues.add("QTOD - ATOD", f"{', '.join(text)} stored as text", "converted to numbers")
        # partner branch: partner master, else the QTOD branch (aliases fixed)
        pt = master.get("Partner Targets")
        pm_branch = {}
        if pt is not None:
            pm_branch = {clean_code(k): canon_branch(v) for k, v in zip(pt["Sold-to Party"], pt["Branch"]) if canon_branch(v)}
        info_branch = {k: canon_branch(v.get("Branch")) for k, v in partner_info.items() if canon_branch(v.get("Branch"))}
        qb = q["Branch"].map(canon_branch)
        q["Branch"] = [info_branch.get(c) or pm_branch.get(c) or b for c, b in zip(q["Partner Code"], qb)]
        diff = (qb != q["Branch"]).sum()
        codes = sorted({str(b) for b in qtod["Branch"].dropna().unique()} - set(region))
        if codes:
            issues.add("QTOD - ATOD", "branch codes not used by the sales data / Branch Targets: " + ", ".join(codes),
                       f"branch taken from the partner master ({diff} partners changed)")
        q["Partner type"] = q["Partner Code"].map(lambda c: partner_info.get(c, {}).get("RD/KRO"))
        if q["RD/ KRO"].isna().all() if "RD/ KRO" in q else True:
            issues.add("QTOD - ATOD", "'RD/ KRO' is empty", "filled from the partner master / sales data")
        q["Region"] = q["Branch"].map(region)
        bad_q = q[(q[qcols].sum(axis=1) - q["Target 100%"]).abs() > 0.01 * q["Target 100%"]]
        if len(bad_q):
            issues.add("QTOD - ATOD", "quarter targets do not add up to 'Target 100%'",
                       "kept as given (mostly partners added during the year)",
                       "; ".join(f"{p} ({c})" for p, c in zip(bad_q["Partner"], bad_q["Partner Code"]))[:500])
        ratio = q["Target 108%"] / q["Target 100%"]
        bad_r = q[(ratio - 1.08).abs() > 0.005]
        if len(bad_r):
            issues.add("QTOD - ATOD", "'Target 108%' is not 108% of 'Target 100%'",
                       "quarter 108% targets = quarter x 1.08; check the yearly 108% figure",
                       "; ".join(f"{p}: {r:.2f}x" for p, r in zip(bad_r["Partner"], ratio[bad_r.index])))
        dn = q[q["Partner"].duplicated(keep=False)]
        if len(dn):
            issues.add("QTOD - ATOD", "same partner name on different partner codes (check for duplicates)", "kept",
                       ", ".join(sorted(dn["Partner"].unique()))[:400])
        rows = []
        base = sales[(sales["Year"] == year - 1) & (sales["GTM"] == "Retail")]
        qsh_b = shares(base.assign(Q=(base["Month"] - 1) // 3 + 1), ["Branch"], "Q")
        for _, r in q.iterrows():
            qt = r[qcols].fillna(0.0).values.astype(float)
            ok = abs(qt.sum() - r["Target 100%"]) <= 0.01 * r["Target 100%"] and qt.sum() > 0
            qs = qt / qt.sum() if ok else lookup_share(qsh_b, r["Branch"], [1, 2, 3, 4], pd.Series(0.25, index=[1, 2, 3, 4])).values
            for i, qn in enumerate(QUARTERS):
                rows.append((year, qn, r["Partner Code"], r["Partner"], r["Branch"], r["Region"], "Retail", r["Partner type"],
                             "All", qt[i], qt[i] * 1.08, "Gross", "Given (108% = quarter x 1.08)"))
                for col, fam in FOCUS_FAMILY.items():
                    v = r[col] if pd.notna(r[col]) else 0.0
                    rows.append((year, qn, r["Partner Code"], r["Partner"], r["Branch"], r["Region"], "Retail", r["Partner type"],
                                 fam, v * qs[i], v * qs[i] * 1.08, "Gross",
                                 "Yearly, by the partner's quarters" if ok else "Yearly, by the branch's quarters last year"))
        cols = ["Year", "Quarter", "Partner Code", "Partner", "Branch", "Region", "GTM", "Partner type", "Family",
                "Target 100%", "Target 108%", "Basis", "Source"]
        return pd.DataFrame(rows, columns=cols), q


    def partner_product_split(q, sales, year):
        base = sales[(sales["Year"] == year - 1) & (sales["GTM"] == "Retail")]
        ps = base.groupby(["Partner", "Group"])["Gross"].sum()
        ptot = ps.groupby(level=0).sum()
        bsh = shares(base, ["Branch"], "Group")
        allsh = base.groupby("Group")["Gross"].sum() / base["Gross"].sum()
        rows = []
        for _, r in q.iterrows():
            t = r["Target 100%"]
            focus = {fam: (r[col] if pd.notna(r[col]) else 0.0) for col, fam in FOCUS_FAMILY.items()}
            code = r["Partner Code"]
            if ptot.get(code, 0) >= MIN_PARTNER_SALES:
                mix, how = ps.loc[code], "Partner's product mix last year"
            else:
                mix, how = lookup_share(bsh, r["Branch"], [g for g, _ in GROUPS], allsh), "Branch's product mix last year (partner below 1 L)"
            mix = mix.drop(list(focus), errors="ignore")
            mix = mix / mix.sum() if mix.sum() > 0 else allsh.drop(list(focus)) / allsh.drop(list(focus)).sum()
            rest = max(t - sum(focus.values()), 0.0)
            for g, _ in GROUPS:
                v, src = (focus[g], "Given (QTOD)") if g in focus else (rest * mix.get(g, 0.0), how)
                if v:
                    rows.append((year, code, r["Partner"], r["Branch"], r["Region"], r["Partner type"], GROUP_ACTIVITY[g], g,
                                 v, "Gross", src))
        return pd.DataFrame(rows, columns=["Year", "Partner Code", "Partner", "Branch", "Region", "Partner type", "Activity",
                                           "Product group", "Target 100%", "Basis", "Source"])


    def ssp_targets(master, year, issues):
        s = master.get("SSA Targets")
        if s is None:
            return pd.DataFrame()
        s = s.copy()
        s["SSP (as given)"] = s["SSP"].astype(str).str.strip()
        s["SSP"] = s["SSP (as given)"].replace(SSP_ALIASES)
        ch = s[s["SSP"] != s["SSP (as given)"]]
        if len(ch):
            issues.add("SSA Targets", "SSP names spelt differently from the sales data",
                       "renamed: " + ", ".join(sorted({f"{a} -> {b}" for a, b in zip(ch['SSP (as given)'], ch['SSP'])})))
        s["Branch (as given)"] = s["Branch"]
        s["Branch"] = s["Branch"].map(canon_branch)
        if "old Target" in s and s["old Target"].notna().any():
            issues.add("SSA Targets", "'old Target' column", "kept as 'Previous target' for reference")
        s = s.rename(columns={"old Target": "Previous target"})
        s["Basis"] = "Net"
        return s[["Year", "SSP", "Branch", "Target", "Previous target", "Basis", "SSP (as given)", "Branch (as given)"]]


    def region_targets(master, issues):
        out = []
        z = master.get("Zone Targets")
        if z is not None:
            out.append(z.rename(columns={"Zone": "Region"}).assign(**{"Product group": "All"}))
        p = master.get("Product Targets")
        if p is not None:
            p = p.iloc[:, :4].copy()
            p.columns = ["Product group", "Region", "Year", "Target"]
            d = p[p.duplicated(["Product group", "Region", "Year"], keep=False)]
            if len(d):
                issues.add("Product Targets", "'East' appears twice per family (East1 and East2 both written as East)",
                           "added together as East", f"{len(d)} rows")
            p = p.groupby(["Year", "Region", "Product group"], as_index=False)["Target"].sum()
            out.append(p)
            if master["Product Targets"].shape[1] > 4:
                issues.add("Product Targets", "a list of family names to the right of the data", "not read")
        if not out:
            return pd.DataFrame()
        r = pd.concat(out, ignore_index=True)[["Year", "Region", "Product group", "Target"]]
        r["Basis"] = "Net"
        return r


    def organize(master, qtod, sales, year, hs=None, partner_info=None):
        """master / qtod / hs: {sheet: DataFrame} of the target files. Returns ({sheet: DataFrame}, issues DataFrame)."""
        issues = Issues()
        sales = prepare_sales(sales)
        partner_info = partner_info or {}
        yearly_r, monthly_r, yearly_given, monthly_given, region = retail_targets(master, sales, year, issues)
        yearly_p = project_targets(hs, sales, year, issues)
        com = commits(hs, year, issues)
        if len(com) and len(yearly_p):
            # the commit sheet writes some Project branches in capitals (MUM, PUNE); use the target sheet's spelling
            names = {b.upper(): b for b in yearly_p["Branch"].unique()}
            p_rows = com["GTM"] == "Project"
            fixed = com.loc[p_rows, "Branch"].map(lambda b: names.get(str(b).upper(), b))
            changed = sorted(set(com.loc[p_rows, "Branch"][fixed != com.loc[p_rows, "Branch"]]))
            if changed:
                issues.add("Activity wise Month Target", "Project branch names in capitals: " + ", ".join(changed),
                           "matched to the Project targets' spelling")
            com.loc[p_rows, "Branch"] = fixed
        monthly_p = project_monthly(com, sales, year, issues, yearly_p)
        yearly = pd.concat([yearly_r, yearly_p], ignore_index=True)
        monthly = pd.concat([monthly_r, monthly_p], ignore_index=True)
        yearly, monthly = yearly[yearly["Target"].round(0) != 0], monthly[monthly["Target"].round(0) != 0]
        pt, q = partner_targets(qtod, master, sales, year, partner_info, region, issues)
        pps = partner_product_split(q, sales, year)
        branches = (pd.concat([yearly[["Branch", "Region", "GTM"]], monthly[["Branch", "Region", "GTM"]]])
                    .drop_duplicates(["Branch", "GTM"]).sort_values(["GTM", "Region", "Branch"]))
        branches["Also written as"] = branches["Branch"].map(
            lambda b: ", ".join(sorted(k for k, v in BRANCH_ALIASES.items() if v == b)))
        tables = {"Branches": branches, "Yearly Targets": yearly, "Monthly Targets": monthly,
                  "Partner Targets": pt, "Partner Product Split": pps, "SSP Targets": ssp_targets(master, year, issues),
                  "Region Targets": region_targets(master, issues)}
        for k in tables:
            for c in ("Target", "Target 100%", "Target 108%"):
                if c in tables[k]:
                    tables[k][c] = tables[k][c].astype(float).round(0)
        given = pd.DataFrame({"Yearly target (given)": yearly_given, "Monthly targets (given, all months)": monthly_given})
        return tables, pd.DataFrame(issues), given


    # ====================================================================== workbook
    def write_targets(path, tables, issues, given, year, base_year):
        from openpyxl import Workbook
        from openpyxl.styles import Alignment, Font, PatternFill
        from openpyxl.utils import get_column_letter
        from openpyxl.worksheet.datavalidation import DataValidation
        from openpyxl.worksheet.table import Table, TableStyleInfo

        green = "1F7A4D"
        wb = Workbook()
        ws = wb.active
        ws.title = "Read me"
        notes = [
            (f"Channel Targets {year}", "Retail, HS (Retail + Project) and Wiser targets in one place, one row per target."),
            ("", ""),
            ("Source", "Built from 'Master Targets File', 'QTOD and ATOD file' and the HS targets file. Every target is kept "
                       f"as given and only split over product groups by {base_year} sales; column 'Source' says which."),
            ("Amounts", "Rupees. 'Basis' says Net or Gross: yearly, monthly and SSP targets are Net (gross = net / 0.83); "
                        "partner targets are Gross."),
            ("Yearly Targets", "Year x branch x GTM x product group (activity alongside). Retail: 'Branch Targets'; Project: "
                               "the HS yearly targets."),
            ("Monthly Targets", "Month x branch x GTM x product group: the monthly commit. Retail: 'Monthly Targets' (Zeta and "
                                "Wiser as given); Project: the HS commits. Separate from the yearly target — the months "
                                "need not add up to it, and months not set yet are empty."),
            ("Partner Targets", "Partner x quarter x family ('All' = the partner's total, as in QTOD)."),
            ("Partner Product Split", "Each partner's yearly target split over product groups by its own "
                                      f"{base_year} mix — for planning, not a given target."),
            ("SSP Targets", "SSP x branch x year, names as in the sales data."),
            ("Region Targets", "Zone and product targets as given (region level)."),
            ("Issues", "Everything found in the source files and what was done about it."),
            ("Check", "Each sheet against the source file; the 'Difference' columns should be 0."),
            ("Editing", "Add rows at the bottom of a table; keep one row per key; use the dropdowns."),
        ]
        ws.column_dimensions["A"].width, ws.column_dimensions["B"].width = 22, 120
        for r, (a, b) in enumerate(notes, 1):
            ws.cell(r, 1, a).font = Font(bold=True, size=14 if r == 1 else 11, color=green if r == 1 else "000000")
            ws.cell(r, 2, b).alignment = Alignment(wrap_text=True, vertical="top")

        fmt = {"Month": "mmm-yy", "Target": "#,##0", "Target 100%": "#,##0", "Target 108%": "#,##0", "Commit": "#,##0",
               "OB Commit": "#,##0", "Previous target": "#,##0"}

        def sheet(name, df, widths=None):
            w = wb.create_sheet(name)
            df = df.copy()
            if "Month" in df:
                df["Month"] = pd.to_datetime(df["Month"]).dt.date
            cols = list(df.columns)
            for j, c in enumerate(cols, 1):
                cell = w.cell(1, j, c)
                cell.font, cell.fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor=green)
                cell.alignment = Alignment(wrap_text=True, vertical="center")
                w.column_dimensions[get_column_letter(j)].width = (widths or {}).get(c, max(10, min(40, len(c) + 4)))
            for row in df.itertuples(index=False):
                w.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row])
            w.row_dimensions[1].height = 32
            w.freeze_panes = "A2"
            n = max(2, len(df) + 1)
            t = Table(displayName=re.sub(r"\W", "", name), ref=f"A1:{get_column_letter(len(cols))}{n}")
            t.tableStyleInfo = TableStyleInfo(name="TableStyleLight9", showRowStripes=True)
            w.add_table(t)
            for j, c in enumerate(cols, 1):
                if c in fmt:
                    for r in range(2, n + 1):
                        w.cell(r, j).number_format = fmt[c]
            return w, cols

        lists = {"GTM": ["Retail", "Project"], "Activity": ACTIVITIES, "Product group": [g for g, _ in GROUPS],
                 "Family": ["All", "Miluz Zeta", "Wiser"], "Basis": ["Net", "Gross"], "Quarter": list(QUARTERS)}

        def dropdowns(w, cols):
            for c, vals in lists.items():
                if c in cols:
                    letter = get_column_letter(cols.index(c) + 1)
                    dv = DataValidation(type="list", formula1='"' + ",".join(vals) + '"', allow_blank=True)
                    dv.add(f"{letter}2:{letter}50000")
                    w.add_data_validation(dv)

        order = ["Branches", "Yearly Targets", "Monthly Targets", "Partner Targets", "Partner Product Split",
                 "SSP Targets", "Region Targets"]
        for name in order:
            df = tables.get(name)
            if df is None or df.empty:
                continue
            w, cols = sheet(name, df, {"Source": 46, "Partner": 30, "Problem": 60})
            dropdowns(w, cols)
        if len(issues):
            w, _ = sheet("Issues", issues, {"Source sheet": 24, "Problem": 70, "What was done": 60, "Rows / values": 80})
            for row in w.iter_rows(min_row=2):
                for c in row:
                    c.alignment = Alignment(wrap_text=True, vertical="top")
        # check: per Retail branch, each sheet against the source file
        yt, mt, pt = tables["Yearly Targets"], tables["Monthly Targets"], tables["Partner Targets"]
        chk = given.copy()
        chk["Yearly Targets sheet"] = yt[yt["GTM"] == "Retail"].groupby("Branch")["Target"].sum()
        chk["Difference (yearly)"] = (chk["Yearly Targets sheet"] - chk["Yearly target (given)"]).round(0)
        chk["Monthly Targets sheet"] = mt[mt["GTM"] == "Retail"].groupby("Branch")["Target"].sum()
        chk["Difference (monthly)"] = (chk["Monthly Targets sheet"] - chk["Monthly targets (given, all months)"]).round(0)
        chk["Months set"] = mt[mt["GTM"] == "Retail"].groupby("Branch")["Month"].nunique()
        chk["Months as % of yearly (info)"] = chk["Monthly Targets sheet"] / chk["Yearly target (given)"]
        chk["Partner targets 100% (gross)"] = pt[pt["Family"] == "All"].groupby("Branch")["Target 100%"].sum()
        chk = chk.reset_index().rename(columns={"index": "Branch"})
        w, cols = sheet("Check", chk, {"Branch": 14})
        for j, c in enumerate(cols, 1):
            f = "0%" if c.startswith("Months as") else ("0" if c == "Months set" else ("#,##0" if c != "Branch" else None))
            if f:
                for rr in range(2, len(chk) + 2):
                    w.cell(rr, j).number_format = f
        wb.move_sheet("Check", offset=-(len(wb.sheetnames) - 2))
        if "Issues" in wb.sheetnames:
            wb.move_sheet("Issues", offset=-(wb.sheetnames.index("Issues") - 2))
        for w in wb.worksheets:
            w.sheet_properties.tabColor = green if w.title in ("Read me", "Check", "Issues") else "8FBC9F"
        wb.save(path)


    def write_contribution(path, tables):
        with pd.ExcelWriter(path, engine="openpyxl") as xw:
            for name, df in tables.items():
                df.to_excel(xw, sheet_name=name[:31], index=False)

    return types.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})


TS = _target_split_tools()


def _retail_target_branch():
    """Partner code → branch: Partner Master city → Geography mapping (as the Retail dashboard and HS datasets)."""
    p = master("partner_retail")
    p["City"] = p["City"].astype(str).str.strip()
    g = master("geo_retail")[["City", "Branch"]].copy()
    g["City"] = g["City"].astype(str).str.strip()
    p = p.merge(g.drop_duplicates("City"), on="City", how="left")
    code = p["Sold-to Party"].astype(str).str.strip().str.upper().str.replace(r"\.0$", "", regex=True)
    return dict(zip(code, p["Branch"]))


RETAIL_TARGET_BRANCH = _retail_target_branch()
TARGET_KEY_KINDS = {"Target Year": "plain"}      # write_table would format a column named "Target …" as rupees
_FAMILY = family_map.drop_duplicates("Material").set_index("Material")["Family Actual"]


def add_target_keys(df, gtm=None):
    """df + the target key columns. gtm="Retail" for Retail-only data (branch from the partner);
    otherwise Retail / Project come from 'Classification' and the branch from 'Branch' (HS datasets)."""
    if df is None or df.empty:
        return df
    out = df.copy()
    partner = out["SoldToPartyID"].astype(str).str.strip().str.upper().str.replace(r"\.0$", "", regex=True)
    out["Target GTM"] = gtm if gtm else out["Classification"]
    branch = partner.map(RETAIL_TARGET_BRANCH) if gtm == "Retail" else out["Branch"]
    retail = out["Target GTM"] == "Retail"
    names = {b: TS.canon_branch(b) for b in branch[retail].dropna().unique()}
    out["Target Branch"] = branch.where(~retail, branch.map(names))
    date_col = next((c for c in ("InvoiceDate", "Calendar Day") if c in out.columns), None)
    d = pd.to_datetime(out[date_col], errors="coerce") if date_col else pd.Series(pd.NaT, index=out.index)
    out["Target Month"] = d.dt.to_period("M").dt.to_timestamp()
    out["Target Year"] = d.dt.year.astype("Int64")
    out["Target Quarter"] = ("Q" + d.dt.quarter.astype("Int64").astype(str)).where(d.notna())
    out["Target Activity"] = out["HeliosCode"].astype(str).str.strip().str.upper().map(TS.ACTIVITY)
    family = out["MaterialCode"].astype(str).str.strip().str.upper().map(_FAMILY)
    pairs = pd.DataFrame({"a": out["Target Activity"], "f": family}).drop_duplicates()
    group = {(a, f): TS.product_group(a, f) for a, f in zip(pairs["a"], pairs["f"])}
    out["Product Group"] = [group[k] for k in zip(out["Target Activity"], family)]
    out["Partner key"] = partner
    missing = out["Target Branch"].isna() & out["Target GTM"].eq("Retail")
    if missing.any():
        gross = pd.to_numeric(out.loc[missing, "Gross Sales"], errors="coerce").sum() if "Gross Sales" in out else 0
        print(f"   ⚠️ {int(missing.sum()):,} Retail rows ({gross / 1e5:,.1f} L) have no Target Branch — "
              f"partner not in the Partner Master or city not in the Geography mapping")
    return out


print(f"✅ Target keys ready | {len(RETAIL_TARGET_BRANCH):,} Retail partners mapped to a branch")

## 3 · Retail reports

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 3.1 RETAIL — dashboard input, MTD and YTD data files
# ════════════════════════════════════════════════════════════════════
def enrich(df_clean, partner_map, drop_existing=(), extra_drop=()):
    """Add ASM / Zone / Branch (partner map) and Family Actual (material master)."""
    if df_clean is None or df_clean.empty:
        return pd.DataFrame()
    df = df_clean.drop(columns=[c for c in drop_existing if c in df_clean.columns])
    df["SoldToPartyID"] = df["SoldToPartyID"].astype(str).str.strip()
    df = lookup_merge(df, partner_map, "partner map", left_on="SoldToPartyID", right_on="Sold-to Party")
    df["MaterialCode"] = df["MaterialCode"].astype(str).str.strip().str.upper()
    df = lookup_merge(df, family_map, "material master", left_on="MaterialCode", right_on="Material")
    drop = ["Sold-to Party", "Material", "Year_Temp", "Year_Export", *extra_drop]
    return df.drop(columns=[c for c in drop if c in df.columns])


def report_retail_dashboard_input():
    """Power BI input: Sales/OB per year (Year, Year-1, Year-2) + Backlog."""
    def build(writer):
        # + Target Branch / GTM / Month / Year / Quarter / Activity, Product Group, Partner key (cell 2.5)
        for label, df in (("Sales", add_target_keys(retail_sales, "Retail")), ("OB", add_target_keys(retail_ob, "Retail"))):
            for year, year_label in DASHBOARD_YEAR_LABELS.items():
                part = df[df["Year_Temp"] == year]
                if not part.empty:
                    write_table(writer, part.drop(columns=["Year_Temp"]), f"{label}_{year_label}", tab=THEME["tab_data"], kinds=TARGET_KEY_KINDS)
        if not retail_backlog.empty:
            backlog = add_target_keys(retail_backlog, "Retail").drop(columns=["Year_Temp"])
            if len(backlog) > EXCEL_MAX_ROWS:
                for p in range(math.ceil(len(backlog) / EXCEL_MAX_ROWS)):
                    write_table(writer, backlog.iloc[p * EXCEL_MAX_ROWS:(p + 1) * EXCEL_MAX_ROWS], f"Backlog_p{p + 1}", tab=THEME["tab_data"], kinds=TARGET_KEY_KINDS)
            else:
                write_table(writer, backlog, "Backlog_Data", tab=THEME["tab_data"], kinds=TARGET_KEY_KINDS)
    return save_report(build, "Retail_Sales_OB_Backlog_Combined.xlsx", sp_folder=PATHS["dashboard_hd"])


def report_retail_mtd():
    """Current month (month of the YTD cut-off) Sales + OB, and full backlog."""
    def month_slice(df, date_col):
        if df.empty:
            return df
        d = pd.to_datetime(df[date_col])
        return df[(d.dt.year == TARGET_YEAR) & (d.dt.month == YESTERDAY.month)]

    sales = enrich(month_slice(retail_sales, "InvoiceDate"), Partner_Map_Retail, extra_drop=["Type"])
    ob = enrich(month_slice(retail_ob, "Calendar Day"), Partner_Map_Retail, extra_drop=["Type"])
    ob = ob.rename(columns={"Gross Sales": "Gross OB", "Net Sales": "Net OB"})
    backlog = enrich(retail_backlog, Partner_Map_Retail, extra_drop=["Type"])

    def build(writer):
        for df, sheet in ((sales, f"Sales_MTD_{MONTH_NAME}"), (ob, f"OB_MTD_{MONTH_NAME}"), (backlog, "Backlog_Lifetime")):
            if not df.empty:
                write_table(writer, df, sheet, tab=THEME["tab_data"])
    return save_report(build, f"MTD_Retail_Sales_OB_Backlog_data_{MONTH_NAME}_{TARGET_YEAR}.xlsx", sp_folder=PATHS["retail_reports"])


retail_ytd = {}   # filled by report_retail_ytd(), reused by the Partner Analysis


def report_retail_ytd():
    """Target-year Sales + OB and the full backlog, with ASM / Zone / Branch."""
    dupes = ("Branch", "Zone", "ASM")
    retail_ytd["sales"] = enrich(retail_sales[retail_sales["Year_Temp"] == TARGET_YEAR], Partner_Map_Retail, dupes, ["Type"])
    retail_ytd["ob"] = enrich(retail_ob[retail_ob["Year_Temp"] == TARGET_YEAR], Partner_Map_Retail, dupes, ["Type"])
    retail_ytd["backlog"] = enrich(retail_backlog, Partner_Map_Retail, dupes, ["Type"])

    def build(writer):
        for key, sheet in (("sales", f"Sales_{TARGET_YEAR}"), ("ob", f"OB_{TARGET_YEAR}"), ("backlog", "Backlog_Overall")):
            if not retail_ytd[key].empty:
                write_table(writer, retail_ytd[key], sheet, tab=THEME["tab_data"])
    return save_report(build, f"Retail_Sales_OB_Backlog_data_{TARGET_YEAR}.xlsx", sp_folder=PATHS["retail_ytd"])




In [ ]:
run_report("retail_dashboard_input", report_retail_dashboard_input)
run_report("retail_mtd", report_retail_mtd)
run_report("retail_ytd", report_retail_ytd)

# Free memory: the Retail datasets are not needed after this cell
# (the Partner Analysis only uses retail_ytd's OB and backlog).
del retail_sales, retail_ob, retail_backlog
retail_ytd.pop("sales", None)
gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 3.2 RETAIL — partner-wise lifetime, last-year YTD, Wiser and product mix
#     (saved locally in LOCAL_PARTNER_REPORTS_RETAIL)
# ════════════════════════════════════════════════════════════════════
def _id_text(series):
    return series.astype(str).str.split(".").str[0].str.strip()


def partner_timeline(df_source, mapping, is_ob=False, start=None, end=None, family=None,
                     helios_codes=RETAIL_HELIOS_CODES):
    """Retail rows → partner × month timeline with Q/annual totals + partner metadata.
    `family` keeps only materials of that Family Actual (e.g. "Wiser");
    `start`/`end` limit the period (by invoice date, or OB year/month)."""
    df = df_source
    if family:
        fam = Material_Map.loc[Material_Map["Family Actual"] == family, ["Material", "Family Actual"]]
        df = df.merge(fam, left_on="MaterialCode", right_on="Material", how="inner")
    df = retail_filter(df, helios_codes)
    df["SoldToPartyID"] = _id_text(df["SoldToPartyID"])
    if is_ob:
        df["Y"], df["M"] = df["Year"], df["OB Month"]
        if start is not None:
            s, e = pd.Timestamp(start), pd.Timestamp(end)
            df = df[((df["Y"] > s.year) | ((df["Y"] == s.year) & (df["M"] >= s.month)))
                    & ((df["Y"] < e.year) | ((df["Y"] == e.year) & (df["M"] <= e.month)))]
    else:
        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
        if start is not None:
            df = df[(df["InvoiceDate"] >= pd.Timestamp(start)) & (df["InvoiceDate"] <= pd.Timestamp(end))]
        df["Y"], df["M"] = df["InvoiceDate"].dt.year, df["InvoiceDate"].dt.month
    if start is None:
        df = df[df["Y"] >= 2022]
    timeline = monthly_timeline(df)
    meta = mapping.copy()
    meta["SoldToPartyID"] = _id_text(meta["SoldToPartyID"])
    return partner_front(meta.drop_duplicates(subset=["SoldToPartyID"]).merge(timeline, on="SoldToPartyID", how="right"))


def product_mix(df_source, mapping, year, helios_codes, retail_only=True):
    """Partner × (Helios code | Family Actual) sales for one year, biggest partners first."""
    df = df_source[df_source["HeliosCode"].isin(helios_codes)]
    if retail_only:
        df = df[is_retail(df)]
    df = df.copy()
    if "InvoiceDate" in df.columns:
        df["Y"] = pd.to_datetime(df["InvoiceDate"], errors="coerce").dt.year
    else:
        df["Y"] = df["Year"]
    df = df[df["Y"] == year]
    if df.empty:
        return pd.DataFrame()
    df["SoldToPartyID"] = _id_text(df["SoldToPartyID"])
    meta = mapping.copy()
    meta["SoldToPartyID"] = _id_text(meta["SoldToPartyID"])
    df = df.drop(columns=[c for c in ("Material", "Family Actual") if c in df.columns])
    df = df.merge(Material_Map[["Material", "Family Actual"]], left_on="MaterialCode", right_on="Material", how="left")
    meta_cols = [c for c in ["SoldToPartyID", "Patner", "GSTN", "RD/KRO", "City", "Tier Category"] if c in meta.columns]
    df = df.drop(columns=[c for c in meta_cols if c in df.columns and c != "SoldToPartyID"])
    df = df.merge(meta[meta_cols], on="SoldToPartyID", how="left")
    index_cols = [c for c in ["SoldToPartyID", "Patner", "GSTN", "RD/KRO", "City", "Branch", "Region", "Tier Category"] if c in df.columns]
    pivot = df.pivot_table(index=index_cols, columns=["HeliosCode", "Family Actual"], values="Gross Sales", aggfunc="sum", fill_value=0)
    pivot["Partner_Grand_Total"] = pivot.sum(axis=1)
    flat = pivot.sort_values("Partner_Grand_Total", ascending=False).reset_index()
    flat.columns = [" | ".join(str(p) for p in c if str(p).strip()) if isinstance(c, tuple) else str(c) for c in flat.columns]
    return partner_front(flat)


def report_retail_partner_lifetime():
    sheets = {"Lifetime_Sales": partner_timeline(df_Sales_final, Partner_Metadata),
              "Lifetime_OB": partner_timeline(df_OB_final, Partner_Metadata, is_ob=True)}
    return save_report(lambda w: write_timeline_sheets(w, sheets), "Partner_Lifetime_Performance_Final.xlsx",
                       local_dir=LOCAL_PARTNER_REPORTS_RETAIL)


def report_retail_partner_ly_ytd():
    """Last year Jan-1 → same day last year (like-for-like with this year's YTD)."""
    start, end = f"{PREV_YEAR}-01-01", LY_YESTERDAY.strftime("%Y-%m-%d")
    sheets = {"Lifetime_Sales": partner_timeline(df_Sales_final, Partner_Metadata, start=start, end=end),
              "Lifetime_OB": partner_timeline(df_OB_final, Partner_Metadata, is_ob=True, start=start, end=end)}
    sheets = {k: v for k, v in sheets.items() if not v.empty}
    return save_report(lambda w: write_timeline_sheets(w, sheets), f"Partner_Performance_{PREV_YEAR} YTD.xlsx",
                       local_dir=LOCAL_PARTNER_REPORTS_RETAIL)


def report_retail_wiser_lifetime():
    sheets = {"Lifetime_Sales": partner_timeline(df_Sales_final, Partner_Metadata, family="Wiser"),
              "Lifetime_OB": partner_timeline(df_OB_final, Partner_Metadata, is_ob=True, family="Wiser")}
    return save_report(lambda w: write_timeline_sheets(w, sheets), "Retail_Wiser_Lifetime_Performance_Partnerwise.xlsx",
                       local_dir=LOCAL_PARTNER_REPORTS_RETAIL)


def report_retail_product_mix():
    mix = product_mix(df_Sales_final, Partner_Metadata, TARGET_YEAR, RETAIL_HELIOS_CODES, retail_only=True)
    if mix.empty:
        return f"no {TARGET_YEAR} retail sales"

    def build(writer):
        write_table(writer, mix, "Product_Mix", freeze_cols=partner_freeze(mix), tab=THEME["tab_summary"])
        writer.sheets["Product_Mix"].set_column(len(mix.columns) - 1, len(mix.columns) - 1, 18, writer.fmt["tot_inr"])
    return save_report(build, f"Retail_Partnerwise_Product_Mix_{TARGET_YEAR}.xlsx", local_dir=LOCAL_PARTNER_REPORTS_RETAIL)


run_report("retail_partner_lifetime", report_retail_partner_lifetime)
run_report("retail_partner_ly_ytd", report_retail_partner_ly_ytd)
run_report("retail_wiser_lifetime", report_retail_wiser_lifetime)
run_report("retail_product_mix", report_retail_product_mix)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 3.3 RETAIL — Target vs Achievement, partner-wise (QTOD / ATOD targets)
#
# Annual + quarterly target vs actual (overall, Miluz Zeta, Wiser), the
# Partner Policy Tracker on QTOD/ATOD targets, and lifetime sheets.
# Logic unchanged from the previous version; colours/fonts now come from
# the house style. Saved in LOCAL_PARTNER_REPORTS_RETAIL.
# ════════════════════════════════════════════════════════════════════
import xlsxwriter


def apply_retail_filter(df):
    return retail_filter(df, RETAIL_HELIOS_CODES)


def report_retail_target_vs_achievement():
    # ═══════════════════════════════════════════════════════════════════
    # 1. AGGREGATION ENGINE
    # ═══════════════════════════════════════════════════════════════════
    def get_master_sales_pivot(df_sales, df_material):
        df = apply_retail_filter(df_sales)
        df['MaterialCode']  = df['MaterialCode'].astype(str).str.split('.').str[0].str.strip()
        # Local copy only -- remapped for this partner-wise pivot, raw df_sales untouched.
        df['SoldToPartyID'] = remap_partner_codes(df['SoldToPartyID'].astype(str).str.split('.').str[0].str.strip())

        df_mat = df_material[['Material', 'Family Actual']].copy()
        df_mat['Material'] = df_mat['Material'].astype(str).str.split('.').str[0].str.strip()

        df = df.merge(df_mat, left_on='MaterialCode', right_on='Material', how='left')
        df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
        df_2026 = df[df['InvoiceDate'].dt.year == 2026].copy()
        df_2026['M'] = df_2026['InvoiceDate'].dt.month

        all_piv  = df_2026.pivot_table(index='SoldToPartyID', columns='M', values='Gross Sales', aggfunc='sum', fill_value=0)
        zeta_piv = df_2026[df_2026['Family Actual'] == 'Miluz Zeta'].pivot_table(index='SoldToPartyID', columns='M', values='Gross Sales', aggfunc='sum', fill_value=0)
        wise_piv = df_2026[df_2026['Family Actual'] == 'Wiser'].pivot_table(index='SoldToPartyID', columns='M', values='Gross Sales', aggfunc='sum', fill_value=0)

        return all_piv, zeta_piv, wise_piv


    # ═══════════════════════════════════════════════════════════════════
    # 2. GENERATE TRACKER SHEETS
    # ═══════════════════════════════════════════════════════════════════
    def generate_master_workbooks(df_targets, df_meta, all_piv, z_piv, w_piv):
        sheets_map = {}

        df_base = df_meta[['SoldToPartyID', 'Patner', 'GSTN', 'RD/KRO', 'Branch', 'Region', 'City', 'ASM', 'Tier Category']].copy()

        df_t = df_targets.copy()
        df_t['SoldToPartyID'] = df_t['CC'].astype(str).str.split('.').str[0].str.strip()
        df_t.columns = df_t.columns.str.strip()   # remove trailing spaces like 'Zeta Target '
        df_t = df_t[['SoldToPartyID', 'Target 100%',
                     'Qtr 1 Target', 'Qtr 2 Target', 'Qtr 3 Target', 'Qtr 4 Target',
                     'Zeta Target', 'Wiser Target']]

        df_full = df_base.merge(df_t, on='SoldToPartyID', how='left').fillna(0)

        def get_sum(piv, cc, months):
            if cc not in piv.index:
                return 0.0
            return float(piv.loc[cc, [m for m in months if m in piv.columns]].sum())

        # ── New/Revival flag ──────────────────────────────────────────
        # Build from raw filtered sales: zero sales in both 2024 & 2025, but sales in 2026
        def build_new_revival_set(df_sales_raw, df_material):
            df = apply_retail_filter(df_sales_raw).copy()
            # Local copy only -- remapped for this partner-wise classification.
            df['SoldToPartyID'] = remap_partner_codes(df['SoldToPartyID'].astype(str).str.split('.').str[0].str.strip())
            df['InvoiceDate']   = pd.to_datetime(df['InvoiceDate'], errors='coerce')
            df['Y']             = df['InvoiceDate'].dt.year

            def yr_sales(yr):
                sub = df[df['Y'] == yr].groupby('SoldToPartyID')['Gross Sales'].sum()
                return sub

            sales_2024 = yr_sales(2024)
            sales_2025 = yr_sales(2025)
            sales_2026 = yr_sales(2026)

            all_ids = set(df['SoldToPartyID'].unique())
            revival = set()
            for pid in all_ids:
                s24 = sales_2024.get(pid, 0)
                s25 = sales_2025.get(pid, 0)
                s26 = sales_2026.get(pid, 0)
                if s24 == 0 and s25 == 0 and s26 > 0:
                    revival.add(pid)
            return revival

        new_revival_set = build_new_revival_set(df_Sales_final, Material_Map)

        def tag_new_revival(pid):
            return 'New/Revival' if pid in new_revival_set else 'Existing'

        # ── Annual Tab ────────────────────────────────────────────────
        ann = df_full[['SoldToPartyID', 'Patner', 'GSTN', 'RD/KRO', 'Branch', 'Region', 'City', 'ASM', 'Tier Category', 'Target 100%']]\
                  .rename(columns={'Target 100%': 'Annual Target'}).copy()
        # Insert New/Revival right after Tier Category (before Annual Target)
        ann.insert(ann.columns.get_loc('Annual Target'), 'New/Revival',
                   ann['SoldToPartyID'].apply(tag_new_revival))

        # 108% stretch target
        ann['108% Target']    = ann['Annual Target'] * 1.08

        # Annual Sales
        ann['Annual Sales']   = ann['SoldToPartyID'].apply(lambda x: get_sum(all_piv, x, range(1, 13)))

        # Ach % on 100% target
        ann['Ach %']          = np.where(ann['Annual Target'] > 0,
                                         ann['Annual Sales'] / ann['Annual Target'], 0)

        # Quarter-wise sales breakdown
        quarters = {'Q1': [1,2,3], 'Q2': [4,5,6], 'Q3': [7,8,9], 'Q4': [10,11,12]}
        for q_name, months in quarters.items():
            ann[f'{q_name} Sales'] = ann['SoldToPartyID'].apply(lambda x: get_sum(all_piv, x, months))

        # Zeta annual section  (Zeta Target * 4 = annual target)
        ann['Zeta Annual Target'] = df_full['Zeta Target'] * 4
        ann['Zeta Annual Sales']  = ann['SoldToPartyID'].apply(lambda x: get_sum(z_piv, x, range(1, 13)))
        ann['Zeta Ach %']         = np.where(ann['Zeta Annual Target'] > 0,
                                              ann['Zeta Annual Sales'] / ann['Zeta Annual Target'], 0)

        # Wiser annual section  (Wiser Target * 4 = annual target)
        ann['Wiser Annual Target'] = df_full['Wiser Target'] * 4
        ann['Wiser Annual Sales']  = ann['SoldToPartyID'].apply(lambda x: get_sum(w_piv, x, range(1, 13)))
        ann['Wiser Ach %']         = np.where(ann['Wiser Annual Target'] > 0,
                                               ann['Wiser Annual Sales'] / ann['Wiser Annual Target'], 0)

        sheets_map['Annual Target vs Ach'] = ann

        # ── Quarterly Tabs ────────────────────────────────────────────
        for q_name, months in quarters.items():
            q_num = q_name[-1]
            q_df  = df_full[['SoldToPartyID', 'Patner', 'GSTN', 'RD/KRO', 'Branch', 'Region', 'City', 'ASM', 'Tier Category']].copy()
            # Insert New/Revival right after Tier Category
            q_df.insert(q_df.columns.get_loc('Tier Category') + 1, 'New/Revival',
                        q_df['SoldToPartyID'].apply(tag_new_revival))

            q_df['Qtr Target'] = df_full[f'Qtr {q_num} Target']
            q_df['Qtr Sales']  = df_full['SoldToPartyID'].apply(lambda x: get_sum(all_piv, x, months))
            q_df['Ach %']      = np.where(q_df['Qtr Target'] > 0,
                                          q_df['Qtr Sales'] / q_df['Qtr Target'], 0)
            q_df['Balance']    = q_df['Qtr Target'] - q_df['Qtr Sales']

            for m in months:
                m_name = pd.Timestamp(2026, m, 1).strftime('%b')
                q_df[f'{m_name} Actual'] = df_full['SoldToPartyID'].apply(
                    lambda x, _m=m: get_sum(all_piv, x, [_m]))
                q_df[f'{m_name} Ach %']  = np.where(
                    q_df['Qtr Target'] > 0,
                    q_df[f'{m_name} Actual'] / q_df['Qtr Target'], 0)

            q_df['Zeta Target']  = df_full['Zeta Target']
            q_df['Zeta Actual']  = df_full['SoldToPartyID'].apply(lambda x: get_sum(z_piv, x, months))
            q_df['Zeta Ach %']   = np.where(q_df['Zeta Target'] > 0,
                                             q_df['Zeta Actual'] / q_df['Zeta Target'], 0)

            q_df['Wiser Target'] = df_full['Wiser Target']
            q_df['Wiser Actual'] = df_full['SoldToPartyID'].apply(lambda x: get_sum(w_piv, x, months))
            q_df['Wiser Ach %']  = np.where(q_df['Wiser Target'] > 0,
                                             q_df['Wiser Actual'] / q_df['Wiser Target'], 0)

            sheets_map[f'QTOD Balance {q_name}'] = q_df

        return {k: partner_front(v) for k, v in sheets_map.items()}


    # ═══════════════════════════════════════════════════════════════════
    # 3. LIFETIME REPORT
    # ═══════════════════════════════════════════════════════════════════
    def generate_lifetime_report(df_source, df_mapping, is_ob=False):
        df_clean = apply_retail_filter(df_source)   # retail filter applied to both Sales and OB
        df_clean['SoldToPartyID'] = df_clean['SoldToPartyID'].astype(str).str.split('.').str[0].str.strip()

        if is_ob:
            df_clean['Y'] = df_clean['Year']
            df_clean['M'] = df_clean['OB Month']
        else:
            df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')
            df_clean['Y'] = df_clean['InvoiceDate'].dt.year
            df_clean['M'] = df_clean['InvoiceDate'].dt.month

        df_clean = df_clean[df_clean['Y'] >= 2022].copy()

        pivot = df_clean.pivot_table(
            index='SoldToPartyID', columns=['Y', 'M'],
            values='Gross Sales', aggfunc='sum', fill_value=0)

        month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                       7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
        years  = sorted(df_clean['Y'].unique())
        chunks = []

        for yr in years:
            if yr not in pivot.columns.get_level_values(0):
                continue
            yr_df = pivot[yr].copy()
            for m in range(1, 13):
                if m not in yr_df.columns:
                    yr_df[m] = 0
            yr_df = yr_df[range(1, 13)]
            yr_df['Q1_Total']     = yr_df[[1,2,3]].sum(axis=1)
            yr_df['Q2_Total']     = yr_df[[4,5,6]].sum(axis=1)
            yr_df['Q3_Total']     = yr_df[[7,8,9]].sum(axis=1)
            yr_df['Q4_Total']     = yr_df[[10,11,12]].sum(axis=1)
            yr_df['Annual_Total'] = yr_df[range(1,13)].sum(axis=1)
            yr_df.columns = [
                f"{int(yr)}_{month_names[c]}" if isinstance(c, int) else f"{int(yr)}_{c}"
                for c in yr_df.columns]
            chunks.append(yr_df)

        financial_timeline = pd.concat(chunks, axis=1).reset_index()
        mapping_clean = df_mapping[['SoldToPartyID','Patner','GSTN','RD/KRO','Branch','Region','City','ASM','Tier Category']].copy()
        mapping_clean['SoldToPartyID'] = mapping_clean['SoldToPartyID'].astype(str).str.split('.').str[0].str.strip()
        mapping_clean = mapping_clean.drop_duplicates(subset=['SoldToPartyID'])
        result = partner_front(mapping_clean.merge(financial_timeline, on='SoldToPartyID', how='right'))
        # Add New/Revival — computed outside, passed via a lookup on the returned df
        return result


    # ═══════════════════════════════════════════════════════════════════
    # 4. RUN PIPELINE
    # ═══════════════════════════════════════════════════════════════════
    all_sales_piv, zeta_sales_piv, wiser_sales_piv = get_master_sales_pivot(df_Sales_final, Material_Map)
    tracker_data         = generate_master_workbooks(Targets_Map, Partner_Metadata, all_sales_piv, zeta_sales_piv, wiser_sales_piv)

    # ═══════════════════════════════════════════════════════════════════
    # 4b. PARTNER_POLICY_TRACKER — new sheet added to the same workbook.
    #     - Targets come from QTOD/ATOD (Targets_Map), same as every other
    #       tab in this file — NOT the separate "Master Targets File.xlsx"
    #       used by the old standalone Partner_Analysis_2026_PTP_test3 script.
    #     - Partner code remapping: an "old" Sold-to Party code and the
    #       "new" code it was reissued as (per the Data Dictionary's
    #       "Partner Code Changes" tab) are treated as the SAME partner —
    #       their sales history is merged before any year-over-year
    #       classification, so a partner doesn't look newly onboarded just
    #       because their code changed.
    #     This remap is now applied GLOBALLY (see the GLOBAL PARTNER CODE
    #     REMAP cell earlier in the notebook) -- df_Sales_final, Targets_Map,
    #     and Partner_Metadata are already consolidated by the time this cell
    #     runs, so every tab in this workbook (Annual/Quarterly and this
    #     sheet) agrees on one row per remapped partner.
    # ═══════════════════════════════════════════════════════════════════
    def build_partner_policy_tracker(df_sales_final, df_targets_raw, df_meta):
        # NOTE: no local remap here anymore -- df_Sales_final, Targets_Map, and
        # Partner_Metadata are already globally remapped + consolidated by the
        # GLOBAL PARTNER CODE REMAP cell earlier in this notebook (old code ==
        # new code applied once, upstream, for every report). This function
        # just consumes already-consolidated IDs.
        PT_TARGET_YEAR, PT_PREV_YEAR = 2026, 2025
        pt_today = pd.Timestamp.now()
        pt_yesterday = pt_today - pd.Timedelta(days=1)
        pt_rem_months = 12 - (pt_yesterday.month - 1)

        # ── Retail sales for 3-year history + Helios split ─────────────
        df_retail = apply_retail_filter(df_sales_final).copy()
        # Local copy only -- remapped for this partner-wise sheet, raw df_sales_final untouched.
        df_retail["SoldToPartyID"] = remap_partner_codes(df_retail["SoldToPartyID"].astype(str).str.split(".").str[0].str.strip())
        df_retail["InvoiceDate"] = pd.to_datetime(df_retail["InvoiceDate"], errors="coerce")
        df_retail["Year"] = df_retail["InvoiceDate"].dt.year

        sales_matrix = df_retail.groupby(["SoldToPartyID", "Year"])["Gross Sales"].sum().unstack(fill_value=0)
        s23 = sales_matrix.get(PT_PREV_YEAR - 2, pd.Series(0, index=sales_matrix.index))
        s24 = sales_matrix.get(PT_PREV_YEAR - 1, pd.Series(0, index=sales_matrix.index))
        s25 = sales_matrix.get(PT_PREV_YEAR, pd.Series(0, index=sales_matrix.index))
        s26 = sales_matrix.get(PT_TARGET_YEAR, pd.Series(0, index=sales_matrix.index))

        first_sale_prev_year = (
            df_retail.loc[df_retail["Year"] == PT_PREV_YEAR]
            .groupby("SoldToPartyID")["InvoiceDate"].min()
        )

        # ── Meta + QTOD/ATOD targets (already remapped/consolidated upstream) ──
        meta_r = df_meta.rename(columns={"Patner": "Partner"}) if "Patner" in df_meta.columns else df_meta.copy()
        meta_cols = [c for c in ["SoldToPartyID", "Partner", "GSTN", "RD/KRO", "Branch", "Region", "City", "ASM", "Tier Category"] if c in meta_r.columns]
        meta_r = meta_r[meta_cols].drop_duplicates("SoldToPartyID")

        tgt = df_targets_raw.copy()
        tgt["SoldToPartyID"] = tgt["CC"].astype(str).str.split(".").str[0].str.strip()
        tgt.columns = tgt.columns.str.strip()
        tgt = tgt[["SoldToPartyID", "Target 100%"]].rename(columns={"Target 100%": "2026 Assigned Target (QTOD/ATOD)"})
        tgt = tgt.groupby("SoldToPartyID", as_index=False)["2026 Assigned Target (QTOD/ATOD)"].sum()

        tracker = meta_r.merge(tgt, on="SoldToPartyID", how="outer")
        tracker["2026 Assigned Target (QTOD/ATOD)"] = tracker["2026 Assigned Target (QTOD/ATOD)"].fillna(0)
        tracker = tracker.set_index("SoldToPartyID")

        s23r = s23.reindex(tracker.index).fillna(0)
        s24r = s24.reindex(tracker.index).fillna(0)
        s25r = s25.reindex(tracker.index).fillna(0)
        s26r = s26.reindex(tracker.index).fillna(0)
        first_sale_r = first_sale_prev_year.reindex(tracker.index)

        is_new_target_year = (s26r > 0) & (s25r == 0) & (s24r == 0)
        is_new_prev_year = (s25r > 0) & (s24r == 0) & (s23r == 0)
        is_q4_prev_year = is_new_prev_year & first_sale_r.dt.month.isin([10, 11, 12])
        is_5l_plus = s25r >= 500000

        tracker["Policy Category"] = np.select(
            [is_new_target_year, is_new_prev_year & is_q4_prev_year,
             is_new_prev_year & ~is_q4_prev_year & is_5l_plus,
             is_new_prev_year & ~is_q4_prev_year & ~is_5l_plus],
            [f"New Partner {PT_TARGET_YEAR}", f"Q4'{PT_PREV_YEAR} Onboarded",
             f">5L {PT_PREV_YEAR} Onboarded", f"<5L {PT_PREV_YEAR} Onboarded"],
            default="Old Partner",
        )
        is_new_bucket = is_new_target_year | is_q4_prev_year | (is_new_prev_year & ~is_q4_prev_year & ~is_5l_plus)
        is_handholding_bucket = is_new_prev_year & ~is_q4_prev_year & is_5l_plus
        tracker["Old/New/Handholding"] = np.select(
            [is_new_bucket, is_handholding_bucket], ["New Partner", "Handholding Partner"], default="Old Partner"
        )

        # ── Manual overrides ────────────────────────────────────────────
        # Forced to Old Partner regardless of the computed rule above -- this
        # sheet carries one classification per partner (not one per year), so
        # forcing it here applies for both 2025 and 2026 at once.
        override_mask = tracker.index.isin(MANUAL_OLD_PARTNER_IDS)
        tracker.loc[override_mask, "Policy Category"] = "Old Partner"
        tracker.loc[override_mask, "Old/New/Handholding"] = "Old Partner"

        tracker[f"{PT_PREV_YEAR} FY Actual"] = s25r
        tracker[f"{PT_TARGET_YEAR} YTD Actual"] = s26r
        tracker["Req. Monthly Run Rate"] = np.where(
            tracker["2026 Assigned Target (QTOD/ATOD)"] > 0,
            (tracker["2026 Assigned Target (QTOD/ATOD)"] - tracker[f"{PT_TARGET_YEAR} YTD Actual"]).clip(lower=0) / pt_rem_months,
            0,
        )
        tracker["Achievement %"] = np.where(
            tracker["2026 Assigned Target (QTOD/ATOD)"] > 0,
            tracker[f"{PT_TARGET_YEAR} YTD Actual"] / tracker["2026 Assigned Target (QTOD/ATOD)"], 0,
        )

        # ── Per-partner Helios code x year split (compact display list) ──
        HELIOS_DISPLAY_CODES = ["HSCOS", "HSCWD", "HSSPE", "PPCFD"]
        hp = df_retail[df_retail["Year"].isin([PT_PREV_YEAR, PT_TARGET_YEAR]) & df_retail["HeliosCode"].isin(HELIOS_DISPLAY_CODES)]
        helios_split = (hp.groupby(["SoldToPartyID", "Year", "HeliosCode"])["Gross Sales"]
                          .sum().unstack(["Year", "HeliosCode"]).fillna(0))
        helios_split.columns = [f"{yr}_{code}" for yr, code in helios_split.columns]
        tracker = tracker.join(helios_split, how="left")
        helios_cols = [c for c in tracker.columns if c.split("_")[0] in {str(PT_PREV_YEAR), str(PT_TARGET_YEAR)} and any(c.endswith(h) for h in HELIOS_DISPLAY_CODES)]
        tracker[helios_cols] = tracker[helios_cols].fillna(0)

        tracker = tracker.sort_values(f"{PT_TARGET_YEAR} YTD Actual", ascending=False).reset_index()
        ordered_cols = (["SoldToPartyID"] + [c for c in meta_cols if c != "SoldToPartyID"]
                        + ["Policy Category", "Old/New/Handholding",
                           f"{PT_PREV_YEAR} FY Actual", f"{PT_TARGET_YEAR} YTD Actual",
                           "2026 Assigned Target (QTOD/ATOD)", "Achievement %", "Req. Monthly Run Rate"]
                        + helios_cols)
        return tracker[[c for c in ordered_cols if c in tracker.columns]]


    policy_tracker_df = build_partner_policy_tracker(df_Sales_final, Targets_Map, Partner_Metadata)
    print(f"     ✅ Partner_Policy_Tracker: {len(policy_tracker_df)} partners (code-remapped, QTOD/ATOD targets)")
    lifetime_sales_final = generate_lifetime_report(df_Sales_final, Partner_Metadata, is_ob=False)
    lifetime_ob_final    = generate_lifetime_report(df_OB_final,    Partner_Metadata, is_ob=True)

    # Build New/Revival set (reuse the same logic, same filter)
    _df_s = apply_retail_filter(df_Sales_final).copy()
    _df_s['SoldToPartyID'] = _df_s['SoldToPartyID'].astype(str).str.split('.').str[0].str.strip()
    _df_s['Y'] = pd.to_datetime(_df_s['InvoiceDate'], errors='coerce').dt.year
    _yr_grp = _df_s.groupby(['SoldToPartyID','Y'])['Gross Sales'].sum().unstack(fill_value=0)
    def _is_revival(pid):
        if pid not in _yr_grp.index: return 'Existing'
        row = _yr_grp.loc[pid]
        s24 = row.get(2024, 0); s25 = row.get(2025, 0); s26 = row.get(2026, 0)
        return 'New/Revival' if (s24 == 0 and s25 == 0 and s26 > 0) else 'Existing'

    # Insert New/Revival into Lifetime DFs after Tier Category
    for _ldf in [lifetime_sales_final, lifetime_ob_final]:
        if 'Tier Category' in _ldf.columns:
            _pos = _ldf.columns.get_loc('Tier Category') + 1
            _ldf.insert(_pos, 'New/Revival', _ldf['SoldToPartyID'].apply(_is_revival))


    # ═══════════════════════════════════════════════════════════════════
    # 5. EXPORT — Professional Formatting
    # ═══════════════════════════════════════════════════════════════════

    file_path = os.path.join(LOCAL_PARTNER_REPORTS_RETAIL, "Target vs Achievement - Partnerwise Report.xlsx")

    with pd.ExcelWriter(
        file_path,
        engine="xlsxwriter",
        engine_kwargs={"options": {"nan_inf_to_errors": True}},
    ) as writer:

        wb = writer.book

        # ── Colour palette ────────────────────────────────────────────
        # House palette (cell 0.5) — product groups keep their own band colour
        CLR = {
            'hdr_blue':   THEME['header_bg'],
            'hdr_teal':   THEME['band_bg'],
            'hdr_zeta':   THEME['series'][3],
            'hdr_wiser':  THEME['series'][5],
            'hdr_slate':  THEME['series'][6],
            'stripe':     THEME['zebra'],
            'tot_bg':     THEME['total_bg'],
            'tot_border': THEME['border'],
            'border':     THEME['border'],
            'red_bg':     THEME['bad'][0],    'red_fg':    THEME['bad'][1],
            'yellow_bg':  THEME['warn'][0],   'yellow_fg': THEME['warn'][1],
            'green_bg':   THEME['good'][0],   'green_fg':  THEME['good'][1],
            'amber_bg':   '#FFD966',          'amber_fg':  '#7F4F00',
            'title_fg':   THEME['title'],
            'sub_fg':     THEME['subtitle'],
        }

        BASE = {'font_name': THEME['font'], 'font_size': THEME['font_size']}

        # ── Format builders ───────────────────────────────────────────
        def hdr(bg):
            return wb.add_format({**BASE, 'bold': True,
                'bg_color': bg, 'font_color': '#FFFFFF',
                'border': 1, 'border_color': '#FFFFFF',
                'align': 'center', 'valign': 'vcenter', 'text_wrap': True})

        def mk_num(alt=False, bold=False, bg=None):
            f = {**BASE, 'num_format': '#,##0', 'align': 'right', 'valign': 'vcenter',
                 'border': 1, 'border_color': CLR['border']}
            if alt: f['bg_color'] = CLR['stripe']
            if bold: f['bold'] = True
            if bg:   f['bg_color'] = bg
            return wb.add_format(f)

        def mk_pct(alt=False, bold=False, bg=None):
            f = {**BASE, 'num_format': '0.0%', 'align': 'center', 'valign': 'vcenter',
                 'border': 1, 'border_color': CLR['border']}
            if alt: f['bg_color'] = CLR['stripe']
            if bold: f['bold'] = True
            if bg:   f['bg_color'] = bg
            return wb.add_format(f)

        def mk_txt(alt=False):
            f = {**BASE, 'align': 'left', 'valign': 'vcenter',
                 'border': 1, 'border_color': CLR['border']}
            if alt: f['bg_color'] = CLR['stripe']
            return wb.add_format(f)

        def mk_tot(pct=False, lbl=False):
            f = {**BASE, 'bold': True, 'bg_color': CLR['tot_bg'],
                 'top': 2, 'bottom': 2, 'border_color': CLR['tot_border'],
                 'valign': 'vcenter'}
            if pct: f.update({'num_format': '0.0%', 'align': 'center'})
            elif lbl: f.update({'align': 'left'})
            else: f.update({'num_format': '#,##0', 'align': 'right'})
            return wb.add_format(f)

        def mk_cf(bg, fg, pct=True):
            f = {**BASE, 'bold': True, 'bg_color': bg, 'font_color': fg,
                 'border': 1, 'border_color': CLR['border']}
            f['num_format'] = '0.0%' if pct else '#,##0'
            f['align'] = 'center' if pct else 'right'
            return wb.add_format(f)

        # ── Pre-build format table ────────────────────────────────────
        FMT = {
            # Headers
            'hdr_blue':   hdr(CLR['hdr_blue']),
            'hdr_teal':   hdr(CLR['hdr_teal']),
            'hdr_zeta':   hdr(CLR['hdr_zeta']),
            'hdr_wiser':  hdr(CLR['hdr_wiser']),
            'hdr_slate':  hdr(CLR['hdr_slate']),
            'hdr_108':    hdr(THEME['series'][4]),           # 108% Target column – orange
            # Data
            'num':        mk_num(),
            'num_alt':    mk_num(alt=True),
            'pct':        mk_pct(),
            'pct_alt':    mk_pct(alt=True),
            'txt':        mk_txt(),
            'txt_alt':    mk_txt(alt=True),
            # Totals
            'tot_num':    mk_tot(),
            'tot_pct':    mk_tot(pct=True),
            'tot_lbl':    mk_tot(lbl=True),
            # Conditional formats (3-tier Ach %)
            'cf_green':   mk_cf(CLR['green_bg'],  CLR['green_fg'],  pct=True),
            'cf_yellow':  mk_cf(CLR['yellow_bg'], CLR['yellow_fg'], pct=True),
            'cf_red':     mk_cf(CLR['red_bg'],    CLR['red_fg'],    pct=True),
            # Balance amber (numeric)
            'cf_amber':   mk_cf(CLR['amber_bg'],  CLR['amber_fg'],  pct=False),
            # Lifetime Total columns
            'lt_tot':     mk_num(bold=True, bg=THEME['total_bg']),
            'lt_tot_alt': mk_num(bold=True, bg=THEME['kpi_bg']),
            # New/Revival highlight
            'nr':         wb.add_format({**BASE, 'align': 'center', 'valign': 'vcenter',
                              'bg_color': THEME['total_bg'], 'font_color': THEME['good'][1], 'bold': True,
                              'border': 1, 'border_color': CLR['border']}),
            'nr_alt':     wb.add_format({**BASE, 'align': 'center', 'valign': 'vcenter',
                              'bg_color': THEME['kpi_bg'], 'font_color': THEME['good'][1], 'bold': True,
                              'border': 1, 'border_color': CLR['border']}),
            # Title / sub
            'title':      wb.add_format({'bold': True, 'font_name': THEME['font'], 'font_size': 14,
                              'font_color': CLR['title_fg'], 'valign': 'vcenter'}),
            'sub':        wb.add_format({**BASE, 'font_color': CLR['sub_fg'], 'italic': True}),
        }

        # ── Helper: column header colour ─────────────────────────────
        def col_hdr_fmt(col):
            if col == '108% Target':       return FMT['hdr_108']
            if 'Zeta' in col:              return FMT['hdr_zeta']
            if 'Wiser' in col:             return FMT['hdr_wiser']
            if 'Actual' in col:            return FMT['hdr_teal']
            return FMT['hdr_blue']

        # ── Static column widths ──────────────────────────────────────
        COL_W = {
            'SoldToPartyID': 14, 'GSTN': 20, 'Patner': 22, 'RD/KRO': 10,
            'Region': 10, 'Branch': 10, 'City': 14, 'Tier Category': 10,'ASM': 13, 'New/Revival': 13,
            'Annual Target': 15, '108% Target': 14, 'Annual Sales': 15,
            'Ach %': 10,
            'Q1 Sales': 12, 'Q2 Sales': 12, 'Q3 Sales': 12, 'Q4 Sales': 12,
            'Zeta Annual Target': 16, 'Zeta Annual Sales': 15, 'Zeta Ach %': 10,
            'Wiser Annual Target': 16,'Wiser Annual Sales': 15,'Wiser Ach %': 10,
            'Qtr Target': 14, 'Qtr Sales': 14, 'Balance': 13,
            'Zeta Target': 13, 'Zeta Actual': 13,
            'Wiser Target': 13,'Wiser Actual': 13,
        }

        ID_COLS = {'SoldToPartyID', 'GSTN', 'Patner', 'RD/KRO', 'Region', 'Branch', 'City', 'Tier Category','ASM', 'New/Revival'}

        MONTH_ACH_COLS   = {f'{m} Ach %' for m in
                            ['Jan','Feb','Mar','Apr','May','Jun',
                             'Jul','Aug','Sep','Oct','Nov','Dec']}
        PRODUCT_ACH_COLS = {'Zeta Ach %', 'Wiser Ach %', 'Zeta Annual Ach %', 'Wiser Annual Ach %'}

        DATA_ROW = 4   # 0-indexed: row0=title, row1=sub, row2=totals, row3=header

        # ── 3-tier conditional format helper ─────────────────────────
        def apply_3tier_cf(ws, row_start, row_end, col_idx):
            ws.conditional_format(row_start, col_idx, row_end, col_idx,
                {'type': 'cell', 'criteria': '>=',      'value': 1.00, 'format': FMT['cf_green']})
            ws.conditional_format(row_start, col_idx, row_end, col_idx,
                {'type': 'cell', 'criteria': 'between', 'minimum': 0.80, 'maximum': 0.9999,
                 'format': FMT['cf_yellow']})
            ws.conditional_format(row_start, col_idx, row_end, col_idx,
                {'type': 'cell', 'criteria': '<',       'value': 0.80, 'format': FMT['cf_red']})

        # ═════════════════════════════════════════════════════════════
        # ── Write Partner_Policy_Tracker sheet into the same workbook ──
        pt_ws = wb.add_worksheet("Partner_Policy_Tracker")
        pt_ws.set_tab_color(THEME["tab_tracker"])
        pt_cols = list(policy_tracker_df.columns)
        pct_cols = {"Achievement %"}
        num_cols = {c for c in pt_cols if c not in pct_cols and c not in
                    {"SoldToPartyID", "Partner", "GSTN", "RD/KRO", "Region", "Branch", "City", "ASM",
                     "Tier Category", "Policy Category", "Old/New/Handholding"}}
        for idx, col in enumerate(pt_cols):
            pt_ws.write(0, idx, col, FMT['hdr_blue'])
            pt_ws.set_column(idx, idx, max(12, min(len(col) + 4, 30)))
        for row_i, row in enumerate(policy_tracker_df.itertuples(index=False)):
            excel_row = 1 + row_i
            alt = (row_i % 2 == 1)
            for col_i, col in enumerate(pt_cols):
                val = row[col_i]
                if col in pct_cols:
                    pt_ws.write(excel_row, col_i, val, FMT['pct_alt'] if alt else FMT['pct'])
                elif col in num_cols:
                    pt_ws.write(excel_row, col_i, val, FMT['num_alt'] if alt else FMT['num'])
                else:
                    pt_ws.write(excel_row, col_i, "" if pd.isna(val) else val, FMT['txt_alt'] if alt else FMT['txt'])
        if "Achievement %" in pt_cols and len(policy_tracker_df):
            ach_idx = pt_cols.index("Achievement %")
            apply_3tier_cf(pt_ws, 1, len(policy_tracker_df), ach_idx)
        pt_ws.freeze_panes(1, partner_freeze(policy_tracker_df))


        # TRACKER SHEETS
        # ═════════════════════════════════════════════════════════════
        for sheet_name, df in tracker_data.items():
            n_rows = len(df)
            cols   = list(df.columns)
            n_cols = len(cols)
            is_annual = 'Annual' in sheet_name

            # Guard: deduplicate column names (suffixes _1, _2 ... if any clash)
            if df.columns.duplicated().any():
                seen = {}
                new_cols = []
                for c in df.columns:
                    if c in seen:
                        seen[c] += 1
                        new_cols.append(f"{c}_{seen[c]}")
                    else:
                        seen[c] = 0
                        new_cols.append(c)
                df.columns = new_cols

            # Clean NaN/Inf BEFORE writing so cells contain real values, not NaN
            df = df.replace([np.inf, -np.inf], 0).fillna(0)

            df.to_excel(writer, sheet_name=sheet_name,
                        startrow=DATA_ROW, index=False, header=False)
            ws = writer.sheets[sheet_name]

            ws.freeze_panes(DATA_ROW, partner_freeze(df))   # freeze through Branch
            ws.set_row(0, 28)
            ws.set_row(1, 14)
            ws.set_row(2, 20)
            ws.set_row(3, 30)

            ws.merge_range(0, 0, 0, min(n_cols - 1, 16),
                           f"  {sheet_name}  –  Partner Performance Dashboard", FMT['title'])
            ws.write(1, 0,
                     "Schneider Electric  |  HS Retail  |  FY 2026  |  All figures in ₹",
                     FMT['sub'])

            cf_end = DATA_ROW + n_rows - 1

            # Track sales/target column indices for weighted Ach% in grand total
            # For annual sheet: Annual Sales / Annual Target
            # For QTOD sheets:  Qtr Sales / Qtr Target
            sales_col_idx  = None
            target_col_idx = None
            ach_col_indices = []          # all overall Ach % column indices (not monthly/product)

            # Track each product's sales + target col for weighted product Ach% totals
            product_pairs = {}            # { ach_col_idx: (sales_col_idx, target_col_idx) }

            for idx, col in enumerate(cols):
                is_pct         = 'Ach %' in col
                is_monthly_ach = col in MONTH_ACH_COLS
                is_product_ach = col in {'Zeta Ach %', 'Wiser Ach %', 'Zeta Annual Ach %', 'Wiser Annual Ach %'}
                is_qtr_ach     = (col == 'Ach %')
                is_ann_ach     = is_annual and is_qtr_ach   # same col name 'Ach %' in annual tab
                is_balance     = (col == 'Balance')
                is_text        = (df.iloc[:, idx].dtype == object) or (col in ID_COLS)

                # Track column positions
                if col in ('Annual Sales', 'Qtr Sales'):
                    sales_col_idx = idx
                if col in ('Annual Target', 'Qtr Target'):
                    target_col_idx = idx
                if col == 'Ach %':
                    ach_col_indices.append(idx)

                # Product pairs for weighted total
                if col == 'Zeta Ach %' and 'Annual' in sheet_name:
                    product_pairs[idx] = ('Zeta Annual Sales', 'Zeta Annual Target')
                elif col == 'Zeta Ach %':
                    product_pairs[idx] = ('Zeta Actual',  'Zeta Target')
                if col == 'Wiser Ach %' and 'Annual' in sheet_name:
                    product_pairs[idx] = ('Wiser Annual Sales', 'Wiser Annual Target')
                elif col == 'Wiser Ach %':
                    product_pairs[idx] = ('Wiser Actual', 'Wiser Target')

                ws.set_column(idx, idx, COL_W.get(col, 10 if is_pct else 13))
                ws.write(3, idx, col, col_hdr_fmt(col))

                # ── Grand Total row ───────────────────────────────────
                # DATA_ROW=4 (0-indexed) → Excel 1-based first data row = DATA_ROW+1 = 5
                cl        = xlsxwriter.utility.xl_col_to_name(idx)
                first_row = DATA_ROW + 1          # Excel 1-based row of first data cell
                last_row  = DATA_ROW + n_rows     # Excel 1-based row of last data cell
                rng = f"{cl}{first_row}:{cl}{last_row}"

                if idx == 0:
                    ws.write(2, idx, 'GRAND TOTAL', FMT['tot_lbl'])
                elif is_text:
                    ws.write(2, idx, '', FMT['tot_lbl'])
                elif is_qtr_ach or is_product_ach:
                    pass   # written after loop with correct weighted formula
                elif is_monthly_ach:
                    ws.write_formula(2, idx, f"=IFERROR(SUBTOTAL(1,{rng}),0)", FMT['tot_pct'])
                else:
                    ws.write_formula(2, idx, f"=SUBTOTAL(9,{rng})", FMT['tot_num'])

                # ── Conditional formats ───────────────────────────────
                if is_qtr_ach:
                    apply_3tier_cf(ws, DATA_ROW, cf_end, idx)
                elif is_monthly_ach:
                    ws.conditional_format(DATA_ROW, idx, cf_end, idx,
                        {'type': 'cell', 'criteria': '>=', 'value': 0.20, 'format': FMT['cf_green']})
                    ws.conditional_format(DATA_ROW, idx, cf_end, idx,
                        {'type': 'cell', 'criteria': '<',  'value': 0.20, 'format': FMT['cf_red']})
                elif is_product_ach:
                    apply_3tier_cf(ws, DATA_ROW, cf_end, idx)
                elif is_balance:
                    ws.conditional_format(DATA_ROW, idx, cf_end, idx,
                        {'type': 'cell', 'criteria': '>', 'value': 0, 'format': FMT['cf_amber']})

            # ── Weighted Grand Total: overall Ach % = SUBTOTAL(sales)/SUBTOTAL(target) ──
            first_row = DATA_ROW + 1
            last_row  = DATA_ROW + n_rows
            if sales_col_idx is not None and target_col_idx is not None:
                s_cl  = xlsxwriter.utility.xl_col_to_name(sales_col_idx)
                t_cl  = xlsxwriter.utility.xl_col_to_name(target_col_idx)
                s_rng = f"{s_cl}{first_row}:{s_cl}{last_row}"
                t_rng = f"{t_cl}{first_row}:{t_cl}{last_row}"
                for ach_idx in ach_col_indices:
                    ws.write_formula(2, ach_idx,
                        f"=IFERROR(SUBTOTAL(9,{s_rng})/SUBTOTAL(9,{t_rng}),0)", FMT['tot_pct'])

            # ── Weighted Grand Total: product Ach % = SUBTOTAL(actual)/SUBTOTAL(target) ──
            for ach_idx, (sales_col, target_col) in product_pairs.items():
                if sales_col in cols and target_col in cols:
                    s_i   = cols.index(sales_col)
                    t_i   = cols.index(target_col)
                    s_cl  = xlsxwriter.utility.xl_col_to_name(s_i)
                    t_cl  = xlsxwriter.utility.xl_col_to_name(t_i)
                    s_rng = f"{s_cl}{first_row}:{s_cl}{last_row}"
                    t_rng = f"{t_cl}{first_row}:{t_cl}{last_row}"
                    ws.write_formula(2, ach_idx,
                        f"=IFERROR(SUBTOTAL(9,{s_rng})/SUBTOTAL(9,{t_rng}),0)", FMT['tot_pct'])

            # ── Alternating row data writes (re-apply per-cell formatting) ──
            for row_i, row in enumerate(df.itertuples(index=False)):
                excel_row = DATA_ROW + row_i
                ws.set_row(excel_row, 16)
                alt = (row_i % 2 == 1)
                for col_i, col in enumerate(cols):
                    val     = row[col_i]          # positional — safe with duplicate col names
                    is_pct  = 'Ach %' in col
                    is_text = (df.iloc[:, col_i].dtype == object) or (col in ID_COLS)
                    if col == 'New/Revival':
                        ws.write(excel_row, col_i, val, FMT['nr_alt'] if alt else FMT['nr'])
                    elif is_pct:
                        ws.write(excel_row, col_i, val, FMT['pct_alt'] if alt else FMT['pct'])
                    elif is_text:
                        ws.write(excel_row, col_i, val, FMT['txt_alt'] if alt else FMT['txt'])
                    else:
                        ws.write(excel_row, col_i, val, FMT['num_alt'] if alt else FMT['num'])

            # ── Tab colours ───────────────────────────────────────────
            tab_colours = {
                'Annual': THEME['series'][0], 'Q1': THEME['series'][1],
                'Q2': THEME['series'][2], 'Q3': THEME['series'][3], 'Q4': THEME['series'][4]
            }
            for key, colour in tab_colours.items():
                if key in sheet_name:
                    ws.set_tab_color(colour)
                    break

        # ═════════════════════════════════════════════════════════════
        # LIFETIME SHEETS
        # ═════════════════════════════════════════════════════════════
        LIFETIME_SHEETS = [
            ('Lifetime_Sales', lifetime_sales_final, THEME['series'][0]),
            ('Lifetime_OB',    lifetime_ob_final,    THEME['series'][3]),
        ]

        for sheet_key, ldf, tab_colour in LIFETIME_SHEETS:
            ldf = ldf.replace([np.inf, -np.inf], 0).fillna(0)
            for _c in ID_COLS:
                if _c in ldf.columns:
                    ldf[_c] = ldf[_c].replace(0, '')

            n_rows = len(ldf)
            cols   = list(ldf.columns)
            n_cols = len(cols)

            ldf.to_excel(writer, sheet_name=sheet_key,
                         startrow=DATA_ROW, index=False, header=False)
            ws = writer.sheets[sheet_key]
            ws.set_tab_color(tab_colour)
            ws.freeze_panes(DATA_ROW, partner_freeze(ldf))   # freeze through Branch

            ws.set_row(0, 28)
            ws.set_row(1, 14)
            ws.set_row(2, 20)
            ws.set_row(3, 30)

            label = ('Lifetime Sales Performance' if 'Sales' in sheet_key
                     else 'Lifetime Order Book Performance')
            ws.merge_range(0, 0, 0, min(n_cols - 1, 16),
                           f"  {label}  –  Partner History", FMT['title'])
            ws.write(1, 0,
                     "Schneider Electric  |  HS Retail  |  FY 2022–2026  |  All figures in ₹",
                     FMT['sub'])
            ws.write(2, 0, 'GRAND TOTAL', FMT['tot_lbl'])

            for idx, col in enumerate(cols):
                is_total = 'Total' in str(col)
                is_text  = (ldf.iloc[:, idx].dtype == object) or (col in ID_COLS)

                if col in ID_COLS:
                    ws.set_column(idx, idx, COL_W.get(col, 14))
                elif is_total:
                    ws.set_column(idx, idx, 14)
                else:
                    ws.set_column(idx, idx, 11)

                if 'Annual' in str(col):   h = FMT['hdr_blue']
                elif 'Q' in str(col) and 'Total' in str(col): h = FMT['hdr_teal']
                elif col in ID_COLS:       h = FMT['hdr_blue']
                else:                      h = FMT['hdr_slate']
                ws.write(3, idx, col, h)

                cl        = xlsxwriter.utility.xl_col_to_name(idx)
                first_row = DATA_ROW + 1
                last_row  = DATA_ROW + n_rows
                rng = f"{cl}{first_row}:{cl}{last_row}"
                if idx == 0:
                    ws.write(2, idx, 'GRAND TOTAL', FMT['tot_lbl'])
                elif is_text:
                    ws.write(2, idx, '', FMT['tot_lbl'])
                else:
                    ws.write_formula(2, idx, f"=SUBTOTAL(9,{rng})", FMT['tot_num'])

            for row_i, row in enumerate(ldf.itertuples(index=False)):
                excel_row = DATA_ROW + row_i
                ws.set_row(excel_row, 16)
                alt = (row_i % 2 == 1)
                for col_i, col in enumerate(cols):
                    val      = row[col_i]
                    is_total = 'Total' in str(col)
                    is_text  = (ldf.iloc[:, col_i].dtype == object) or (col in ID_COLS)
                    if col == 'New/Revival':
                        ws.write(excel_row, col_i, val, FMT['nr_alt'] if alt else FMT['nr'])
                    elif is_text:
                        ws.write(excel_row, col_i, val, FMT['txt_alt'] if alt else FMT['txt'])
                    elif is_total:
                        ws.write(excel_row, col_i, val, FMT['lt_tot_alt'] if alt else FMT['lt_tot'])
                    else:
                        ws.write(excel_row, col_i, val, FMT['num_alt'] if alt else FMT['num'])

    print("✅ Pipeline complete. Target vs Achievement - Partnerwise Report generated.")
    return file_path


run_report("retail_target_vs_achievement", report_retail_target_vs_achievement)

## 4 · HS (Retail + Project) reports

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 4.1 HS (RETAIL + PROJECT) DATASETS
#
# Every HS row is classified Retail (retail L2/L3) or Project (the rest);
# rebate depends on that class and the year. Branch comes from the partner
# (Retail) or from the Sales Office via "NS Branches" (Project).
#
#   hs_dash_*   dashboard input: all retail Helios codes, FD codes kept only
#               on retail rows, Branch + Region       (was "HS Overall" cell)
#   hs_*        HS reports: retail Helios codes except PPCFD, Branch only
# These two variants are deliberately kept as they were — align them here.
# ════════════════════════════════════════════════════════════════════
def _retail_partner_lookup(partner_df, branch_col):
    p = partner_df.copy()
    p["Sold-to Party"] = p["Sold-to Party"].astype(str).str.strip().str.upper()
    return p.drop_duplicates(subset=["Sold-to Party"]).set_index("Sold-to Party")


def rebate_rate(year, classification):
    """Vectorised rebate: retail vs project table by year."""
    retail = year.map(RETAIL_REBATES).fillna(RETAIL_REBATE_DEFAULT)
    project = year.map(PROJECT_REBATES).fillna(PROJECT_REBATE_DEFAULT)
    return np.where(classification == "Retail", retail, project)


def build_hs_dataset(df, date_col, helios_codes, retail_branch_map, retail_region_map=None,
                     filter_by_year=True, drop_non_retail_fd=False):
    if df is None or df.empty:
        return pd.DataFrame()
    df.columns = [str(c).strip() for c in df.columns]
    date_col = next((c for c in DATE_CANDIDATES if c in df.columns), date_col)
    years = pd.to_datetime(df[date_col], errors="coerce").dt.year.fillna(0).astype(int)
    mask = years.isin(DASHBOARD_YEAR_LABELS) if filter_by_year else pd.Series(True, index=df.index)
    if "HeliosCode" in df.columns:
        mask &= df["HeliosCode"].isin(helios_codes)
    out = df[mask].copy()
    out["Year_Temp"] = years[mask]
    if drop_non_retail_fd and {"HeliosCode", "PricingClassL3"} <= set(out.columns):
        out = out[~(out["HeliosCode"].isin(["PPCFD", "HDCFD"]) & ~out["PricingClassL3"].isin(RETAIL_PRICING_L3))].copy()
    has_l2l3 = {"PricingClassL2", "PricingClassL3"} <= set(out.columns)
    cls = pd.Series(np.where(is_retail(out), "Retail", "Project") if has_l2l3 else "Project", index=out.index)
    out["Classification"] = cls          # Retail / Project — carried into every HS output
    out["Gross Sales"] = pd.to_numeric(out["Gross Sales"], errors="coerce").fillna(0)
    out["Net Sales"] = out["Gross Sales"] * (1 - rebate_rate(out["Year_Temp"], cls))

    out["Branch"] = None
    if retail_region_map is not None:
        out["Region"] = None
    retail_rows = cls == "Retail"
    if retail_rows.any() and "SoldToPartyID" in out.columns:
        ids = out.loc[retail_rows, "SoldToPartyID"].astype(str).str.strip().str.upper()
        out.loc[retail_rows, "Branch"] = ids.map(retail_branch_map)
        if retail_region_map is not None:
            out.loc[retail_rows, "Region"] = ids.map(retail_region_map)
    if (~retail_rows).any() and "SalesOfficeCode" in out.columns:
        offices = out.loc[~retail_rows, "SalesOfficeCode"].astype(str).str.strip().str.upper()
        out.loc[~retail_rows, "Branch"] = offices.map(project_branch_map)
        if retail_region_map is not None:
            out.loc[~retail_rows, "Region"] = offices.map(project_region_map)
    return out


# Dashboard variant: partner Branch via City (retail Data Dictionary), Region from Partner Master
_p = master("partner_retail")
_p["City"] = _p["City"].astype(str).str.strip()
_g = master("geo_retail")[["City", "Branch"]].copy()
_g["City"] = _g["City"].astype(str).str.strip()
_dash_lookup = _retail_partner_lookup(_p.merge(_g.drop_duplicates("City"), on="City", how="left"), "Branch")
_dash_args = dict(helios_codes=RETAIL_HELIOS_CODES, retail_branch_map=_dash_lookup["Branch"].to_dict(),
                  retail_region_map=_dash_lookup["Region"].to_dict(), drop_non_retail_fd=True)
hs_dash_sales = build_hs_dataset(df_Sales_final, "InvoiceDate", **_dash_args)
hs_dash_ob = build_hs_dataset(df_OB_final, "Calendar Day", **_dash_args)
hs_dash_backlog = build_hs_dataset(df_backlog_final, "OrderCreationDate", filter_by_year=False, **_dash_args)

# Report variant: partner Branch from the Retail + Project dictionary's "Branch_dummy"
_rp_lookup = _retail_partner_lookup(master("partner_project"), "Branch_dummy")
_rep_args = dict(helios_codes=HS_REPORT_HELIOS_CODES, retail_branch_map=_rp_lookup["Branch_dummy"].to_dict())
hs_sales = build_hs_dataset(df_Sales_final, "InvoiceDate", **_rep_args)
hs_ob = build_hs_dataset(df_OB_final, "Calendar Day", **_rep_args)
hs_backlog = build_hs_dataset(df_backlog_final, "OrderCreationDate", filter_by_year=False, **_rep_args)
del _p, _g

print(f"✅ HS datasets | dashboard: Sales {len(hs_dash_sales):,} OB {len(hs_dash_ob):,} | "
      f"reports: Sales {len(hs_sales):,} OB {len(hs_ob):,} Backlog {len(hs_backlog):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 4.2 HS — dashboard input, MTD, YTD, YTD order level, COS/SPE and Wiser data files
# ════════════════════════════════════════════════════════════════════
def hs_enrich(df_clean, period="ytd", row_filter=None):
    """HS rows for a period → + ASM / Zone / Branch (partner map, Retail) or
    Branch / Region (Sales Office, Project), Family Actual, rebate & Net Sales."""
    if df_clean is None or df_clean.empty:
        return pd.DataFrame()
    df = df_clean.copy()
    if period == "mtd":
        d = pd.to_datetime(df[next(c for c in DATE_CANDIDATES if c in df.columns)])
        df = df[(d.dt.year == TARGET_YEAR) & (d.dt.month == YESTERDAY.month)]
    elif period == "ytd":
        df = df[df["Year_Temp"] == TARGET_YEAR]
    if df.empty:
        return pd.DataFrame()
    cls = pd.Series(np.where(is_retail(df), "Retail", "Project"), index=df.index) \
        if {"PricingClassL2", "PricingClassL3"} <= set(df.columns) else pd.Series("Project", index=df.index)
    df = df.drop(columns=[c for c in ("Branch", "Zone", "ASM", "Region") if c in df.columns])
    df["SoldToPartyID"] = df["SoldToPartyID"].astype(str).str.strip()
    df = lookup_merge(df, Partner_Map_Project, "partner map", left_on="SoldToPartyID", right_on="Sold-to Party")
    df["MaterialCode"] = df["MaterialCode"].astype(str).str.strip().str.upper()
    df = lookup_merge(df, family_map, "material master", left_on="MaterialCode", right_on="Material")
    cls = cls.values
    project = cls == "Project"
    df["Region"] = None
    if project.any() and "SalesOfficeCode" in df.columns:
        offices = df.loc[project, "SalesOfficeCode"].astype(str).str.strip().str.upper()
        df.loc[project, "Branch"] = offices.map(project_branch_map)
        df.loc[project, "Region"] = offices.map(project_region_map)
    if row_filter is not None:
        keep = row_filter(df).values
        df, cls = df[keep], cls[keep]
        if df.empty:
            return pd.DataFrame()
    df["Rebate"] = rebate_rate(df["Year_Temp"], pd.Series(cls, index=df.index))
    df["Net Sales"] = df["Gross Sales"] * (1 - df["Rebate"])
    return df.drop(columns=[c for c in ("Sold-to Party", "Material", "Year_Temp", "Year_Export") if c in df.columns])


def _hs_ytd_file(filename, folder, row_filter=None):
    sheets = {f"HD_Sales_{TARGET_YEAR}": hs_enrich(hs_sales, "ytd", row_filter),
              f"HD_OB_{TARGET_YEAR}": hs_enrich(hs_ob, "ytd", row_filter),
              "HD_Backlog_Overall": hs_enrich(hs_backlog, "all", row_filter)}

    def build(writer):
        for sheet, df in sheets.items():
            if not df.empty:
                write_table(writer, df, sheet, tab=THEME["tab_data"])
    return save_report(build, filename, sp_folder=folder)


def report_hs_dashboard_input():
    def build(writer):
        # + Target Branch / GTM / Month / Year / Quarter / Activity, Product Group, Partner key (cell 2.5)
        for label, df in (("Sales", add_target_keys(hs_dash_sales)), ("OB", add_target_keys(hs_dash_ob))):
            for year, year_label in DASHBOARD_YEAR_LABELS.items():
                part = df[df["Year_Temp"] == year]
                if not part.empty:
                    write_table(writer, part.drop(columns=["Year_Temp"]), f"{label}_{year_label}", tab=THEME["tab_data"], kinds=TARGET_KEY_KINDS)
        if not hs_dash_backlog.empty:
            write_table(writer, add_target_keys(hs_dash_backlog).drop(columns=["Year_Temp"]), "Backlog_Data", tab=THEME["tab_data"], kinds=TARGET_KEY_KINDS)
    return save_report(build, "H&D_Overall_Sales_OB_Backlog_Combined_ppcfd removed.xlsx", sp_folder=PATHS["dashboard_hd"])


def report_hs_mtd():
    sheets = {"MTD_Sales": hs_enrich(hs_sales, "mtd"), "MTD_OB": hs_enrich(hs_ob, "mtd"),
              "Full_Backlog": hs_enrich(hs_backlog, "all")}

    def build(writer):
        for sheet, df in sheets.items():
            if not df.empty:
                write_table(writer, df, sheet, tab=THEME["tab_data"])
    return save_report(build, f"HS_MTD_Sales_OB_Backlog_{MONTH_NAME}_{TARGET_YEAR}.xlsx", sp_folder=PATHS["hs_reports"])


def report_hs_ytd():
    return _hs_ytd_file(f"HD_Overall_OB_Sales_Backlog_Data_{TARGET_YEAR}.xlsx", PATHS["hs_ytd"])


def report_hs_cos_spe_ytd():
    return _hs_ytd_file(f"COS_SPE_OB_Sales_Backlog_Data_{TARGET_YEAR}.xlsx", PATHS["final_reports"],
                        lambda d: d["HeliosCode"].astype(str).str.strip().str.upper().isin(["HSCOS", "HSSPE"]))


def report_hs_wiser_ytd():
    return _hs_ytd_file(f"Wiser_OB_Sales_Backlog_Data_{TARGET_YEAR}.xlsx", PATHS["hs_ytd"],
                        lambda d: d["Family Actual"].astype(str).str.strip().str.title() == "Wiser")


# --- HS YTD on order level: one row per sales order × material, no dates ---
ORDER_COLUMNS = {
    "SalesOrderHeader": "Sales Order", "SoldToPartyID": "Customer Code", "SoldToPartyName": "Customer",
    "MaterialCode": "Material", "MaterialDescription": "Material Description", "HeliosCode": "Helios Code",
    "Family Actual": "Family", "PricingClassL2": "Pricing Class L2", "PricingClassL2Description": "Pricing Class L2 Name",
    "PricingClassL3": "Pricing Class L3", "PricingClassL3Description": "Pricing Class L3 Name",
    "Classification": "Classification", "Branch": "Branch", "Region": "Region",
}
ORDER_VALUES = {"BillingQuantity": "Quantity", "Gross Sales": "Gross Sales", "Net Sales": "Net Sales"}
HS_ORDER_REBATE = 0.135        # this report only: Net = Gross × (1 − 13.5%), Retail and Project alike


def hs_order_level(df):
    """hs_enrich rows summed per order × material (invoice / calendar dates and items dropped).
    Region: Sales Office region (Project rows), else the partner's zone (Retail rows)."""
    if df.empty:
        return df
    df = df.copy()
    if "Zone" in df.columns:
        df["Region"] = df["Region"].fillna(df["Zone"])
    for col in ("SalesOrderHeader", "SoldToPartyID"):
        df[col] = clean_id(df[col])
    keys = [c for c in ORDER_COLUMNS if c in df.columns]
    vals = [c for c in ORDER_VALUES if c in df.columns]
    df[keys] = df[keys].astype(str).apply(lambda s: s.str.strip()).replace({"": "Unmapped", "nan": "Unmapped", "None": "Unmapped"})
    df[vals] = df[vals].apply(pd.to_numeric, errors="coerce").fillna(0)
    out = df.groupby(keys, as_index=False)[vals].sum()
    out["Net Sales"] = out["Gross Sales"] * (1 - HS_ORDER_REBATE)
    out = out.sort_values([c for c in ("Region", "Branch", "SoldToPartyName", "SalesOrderHeader", "MaterialCode") if c in keys])
    return out.rename(columns={**ORDER_COLUMNS, **ORDER_VALUES})


def report_hs_ytd_orders():
    sheets = {f"Sales_{TARGET_YEAR}": hs_order_level(hs_enrich(hs_sales, "ytd")),
              f"OB_{TARGET_YEAR}": hs_order_level(hs_enrich(hs_ob, "ytd")),
              "Backlog_Overall": hs_order_level(hs_enrich(hs_backlog, "all"))}

    def build(writer):
        for sheet, df in sheets.items():
            if not df.empty:
                write_table(writer, df, sheet, tab=THEME["tab_data"], freeze_cols=3, totals=True)
    return save_report(build, f"HS_YTD_Order_Level_{TARGET_YEAR}.xlsx", sp_folder=PATHS["hs_ytd"])




In [ ]:
run_report("hs_dashboard_input", report_hs_dashboard_input)
run_report("hs_mtd", report_hs_mtd)
run_report("hs_ytd", report_hs_ytd)
run_report("hs_ytd_orders", report_hs_ytd_orders)
run_report("hs_cos_spe_ytd", report_hs_cos_spe_ytd)
run_report("hs_wiser_ytd", report_hs_wiser_ytd)

# Free memory: the HS datasets are not needed after this cell
del hs_dash_sales, hs_dash_ob, hs_dash_backlog, hs_sales, hs_ob, hs_backlog
gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 4.3 HS — partner-wise product mix and lifetime (Overall / Retail / Project)
#     (saved locally in LOCAL_PARTNER_REPORTS_HS)
# ════════════════════════════════════════════════════════════════════
FILTER_MODES = ["overall", "retail", "project"]


def _meta(mapping):
    m = mapping.copy()
    m["SoldToPartyID"] = _id_text(m["SoldToPartyID"])
    return m.drop_duplicates(subset=["SoldToPartyID"])


def hs_lifetime(df_source, filter_mode, is_ob=False, helios_codes=None, family=None):
    """Lifetime timeline for HS rows. 'retail' and 'project' are exact complements,
    so overall = retail + project. Each row gets metadata from the table that
    applies to it: Partner_Metadata (retail rows) or Partner_Metadata_HS (project)."""
    df = df_source
    if family:
        fam = Material_Map.loc[Material_Map["Family Actual"] == family, ["Material", "Family Actual"]]
        df = df.merge(fam, left_on="MaterialCode", right_on="Material", how="inner")
    if helios_codes:
        df = df[df["HeliosCode"].isin(helios_codes)]
    retail_rows = is_retail(df)
    df = {"overall": df, "retail": df[retail_rows], "project": df[~retail_rows]}[filter_mode].copy()
    df["SoldToPartyID"] = _id_text(df["SoldToPartyID"])
    if is_ob:
        df["Y"], df["M"] = df["Year"], df["OB Month"]
    else:
        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
        df["Y"], df["M"] = df["InvoiceDate"].dt.year, df["InvoiceDate"].dt.month
    df = df[df["Y"] >= 2022]
    meta_r, meta_h = _meta(Partner_Metadata), _meta(Partner_Metadata_HS)
    meta_cols = [c for c in meta_r.columns if c != "SoldToPartyID"]
    if df.empty:
        return pd.concat([meta_r, meta_h]).drop_duplicates(subset=["SoldToPartyID"]).iloc[0:0]
    r = is_retail(df)
    with_meta = pd.concat([df[r].merge(meta_r, on="SoldToPartyID", how="left"),
                           df[~r].merge(meta_h, on="SoldToPartyID", how="left")], ignore_index=True)
    timeline = monthly_timeline(with_meta)
    partner_meta = with_meta.groupby("SoldToPartyID")[[c for c in meta_cols if c in with_meta.columns]].first().reset_index()
    return partner_front(partner_meta.merge(timeline, on="SoldToPartyID", how="right"))


def _three_way(helios_codes=None, family=None):
    sheets = {}
    for mode in FILTER_MODES:
        sheets[f"Sales_{mode.capitalize()}"] = hs_lifetime(df_Sales_final, mode, False, helios_codes, family)
        sheets[f"OB_{mode.capitalize()}"] = hs_lifetime(df_OB_final, mode, True, helios_codes, family)
    return sheets


def report_hs_product_mix():
    mix = product_mix(df_Sales_final, Partner_Metadata_HS, TARGET_YEAR, RETAIL_HELIOS_CODES, retail_only=False)
    if mix.empty:
        return f"no {TARGET_YEAR} HS sales"

    def build(writer):
        write_table(writer, mix, "Product_Mix", freeze_cols=partner_freeze(mix), tab=THEME["tab_summary"])
        writer.sheets["Product_Mix"].set_column(len(mix.columns) - 1, len(mix.columns) - 1, 18, writer.fmt["tot_inr"])
    return save_report(build, f"Overall HS_Partnerwise_Product_Mix_{TARGET_YEAR}.xlsx", local_dir=LOCAL_PARTNER_REPORTS_HS)


def report_hs_partner_lifetime():
    sheets = _three_way(helios_codes=HS_LIFETIME_HELIOS_CODES)
    return save_report(lambda w: write_timeline_sheets(w, sheets), "HS_Partnerwise_Lifetime_Performance_Final.xlsx",
                       local_dir=LOCAL_PARTNER_REPORTS_HS)


def report_hs_wiser_lifetime():
    sheets = _three_way(family="Wiser")
    return save_report(lambda w: write_timeline_sheets(w, sheets), "Wiser_Lifetime_Performance_Partnerwise.xlsx",
                       local_dir=LOCAL_PARTNER_REPORTS_HS)


run_report("hs_product_mix", report_hs_product_mix)
run_report("hs_partner_lifetime", report_hs_partner_lifetime)
run_report("hs_wiser_lifetime", report_hs_wiser_lifetime)

## 5 · Partner Analysis, New GEO & Insights

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 5.1 PARTNER ANALYSIS — shared preparation
#
# Retail sales (Partner Analysis Helios set) + Partner Master + Geography,
# partner segment (New/Revival vs Old), new geographies and the "Unmapped"
# zone. Everything in 5.2–5.4 reads the globals prepared here.
#
#   New/Revival in year Y = sales in Y, none in Y-1 and Y-2
#   (MANUAL_OLD_PARTNER_IDS are always Old)
# Note: segment logic is written for 2026 vs 2025 (column names Stat_2026 /
# Stat_2025), exactly as before.
# ════════════════════════════════════════════════════════════════════
yesterday = YESTERDAY                 # names the Partner Analysis code uses
last_year_yesterday = LY_YESTERDAY
last_y_ytd_date = LY_YESTERDAY
UNMAPPED = "Unmapped"


def unmapped_last(index_values):
    vals = [v for v in index_values if v != UNMAPPED]
    return vals + ([UNMAPPED] if UNMAPPED in index_values else [])


def prepare_partner_analysis():
    global Partner_Master, df_targets, df_region_target, df_material_master, tier_col, kro_col
    global df_retail, sales_matrix, partner_status, df_final, df_active, new_geo_list, new_geo_list_25

    Partner_Master = master("partner_retail").merge(master("geo_retail").drop_duplicates("City"), on="City", how="left")
    Partner_Master["Sold-to Party"] = Partner_Master["Sold-to Party"].astype(str).str.strip()
    df_targets = master("partner_targets")
    df_region_target = master("branch_targets")
    df_material_master = Material_Map.copy()

    # City key for New Geo only: spelling variants folded via "Alias Mapping"
    aliases = master("alias_map")
    city_alias = dict(zip(aliases["Old Name"].astype(str).str.strip().str.title(),
                          aliases["Standard Name"].astype(str).str.strip().str.title())) if not aliases.empty else {}
    Partner_Master["Geo_City"] = Partner_Master["City"].astype(str).str.strip().str.title().replace(city_alias)
    Partner_Master.loc[Partner_Master["City"].isna(), "Geo_City"] = np.nan

    tier_col = next((c for c in Partner_Master.columns if "tier" in c.lower()), "Tier Category")
    Partner_Master["Tier_Bucket"] = Partner_Master[tier_col].map({"Metro": "Metro & T1", "T1": "Metro & T1", "T2&T3": "T2 & T3"}).fillna("Other")
    kro_col = next((c for c in Partner_Master.columns if "partner type" in c.lower() or "kro" in c.lower()), "Partner Type")

    sales = df_Sales_final.copy()
    sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"])
    df_retail = retail_filter(sales, PARTNER_ANALYSIS_HELIOS_CODES)
    df_retail["Year"] = df_retail["InvoiceDate"].dt.year
    df_retail["Month"] = pd.Categorical(df_retail["InvoiceDate"].dt.month_name(), categories=MONTHS_ORDER, ordered=True)
    df_retail["SoldToPartyID"] = df_retail["SoldToPartyID"].astype(str).str.strip()

    sales_matrix = df_retail.groupby(["SoldToPartyID", "Year"])["Gross Sales"].sum().unstack(fill_value=0)
    s = lambda y: sales_matrix.get(y, 0)
    partner_status = pd.DataFrame({
        "Stat_2026": np.where((s(2026) > 0) & (s(2025) == 0) & (s(2024) == 0), "New/Revival", "Old"),
        "Stat_2025": np.where((s(2025) > 0) & (s(2024) == 0) & (s(2023) == 0), "New/Revival", "Old"),
    }, index=sales_matrix.index)
    partner_status.loc[partner_status.index.intersection(MANUAL_OLD_PARTNER_IDS), ["Stat_2026", "Stat_2025"]] = "Old"

    df_final = lookup_merge(df_retail, Partner_Master, "Partner Master", left_on="SoldToPartyID", right_on="Sold-to Party")
    df_final = df_final.merge(partner_status, left_on="SoldToPartyID", right_index=True, how="left")

    # Partners without a Zone (city not in Geography mapping) stay visible
    no_zone = df_final["Zone"].isna()
    if no_zone.any():
        missing = df_final.loc[no_zone, ["SoldToPartyID", "City"]].drop_duplicates("SoldToPartyID")
        print(f"   ⚠️ {len(missing)} partner(s) have no Zone → '{UNMAPPED}'. Add to 'Geography mapping': "
              f"{sorted(missing['City'].dropna().unique().tolist())}")
    df_final["Zone"] = df_final["Zone"].fillna(UNMAPPED)

    geo = df_final.groupby(["Geo_City", "Year"])["Gross Sales"].sum().unstack(fill_value=0)
    g = lambda y: geo.get(y, 0)
    new_geo_list = geo[(g(2026) > 0) & (g(2025) == 0) & (g(2024) == 0)].index.tolist()
    new_geo_list_25 = geo[(g(2025) > 0) & (g(2024) == 0) & (g(2023) == 0)].index.tolist()

    df_active = df_final[(df_final["Gross Sales"] != 0) & df_final["Gross Sales"].notna()]
    print(f"   Partner Analysis prepared: {df_retail['SoldToPartyID'].nunique():,} retail partners, "
          f"{len(new_geo_list)} new cities")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 5.2 PARTNER ANALYSIS — table builders
#
#   aggregate_periods / write_kpi_table   FY'25 · YTD'25 · YTD'26 tables
#   write_new_partner_dashboard           New Partner snapshot block
#   generate_policy_handholding_tracker   Policy category, targets, run rate
#   generate_zone_target_tracker          Zone target vs actual
#   write_geo_expansion_sheet / write_sheet1   New GEO workbook sheets
# ════════════════════════════════════════════════════════════════════
# 6. GENERIC PERIOD AGGREGATION (used by Summary tab + New Partner tab)
# =====================================================================
def aggregate_periods(group_col, is_segment=False, new_only=False, with_avg=False):
    s25_full = df_active[df_active["Year"] == 2025]
    s25_ytd = df_active[(df_active["Year"] == 2025) & (df_active["InvoiceDate"] <= last_year_yesterday)]
    s26_ytd = df_active[(df_active["Year"] == 2026) & (df_active["InvoiceDate"] <= yesterday)]

    if is_segment:
        group_25, group_26 = "Stat_2025", "Stat_2026"
    else:
        group_25, group_26 = group_col, group_col
        if new_only:
            s25_full = s25_full[s25_full["Stat_2025"] == "New/Revival"]
            s25_ytd = s25_ytd[s25_ytd["Stat_2025"] == "New/Revival"]
            s26_ytd = s26_ytd[s26_ytd["Stat_2026"] == "New/Revival"]

    def agg(df, gcol):
        return df.groupby(gcol).agg(P=("SoldToPartyID", "nunique"), S=("Gross Sales", lambda x: x.sum() / 1e6))

    a25f, a25y, a26y = agg(s25_full, group_25), agg(s25_ytd, group_25), agg(s26_ytd, group_26)
    a25f.columns = ["P25F", "S25F"]
    a25y.columns = ["P25Y", "S25Y"]
    a26y.columns = ["P26Y", "S26Y"]

    res = pd.concat([a25f, a25y, a26y], axis=1).fillna(0)
    res.columns = ["P25F", "S25F", "P25Y", "S25Y", "P26Y", "S26Y"]
    return res.loc[unmapped_last(sorted(res.index, key=str))]


def write_kpi_table(ws, wb, f, start_row, res, title, label, with_avg=True):
    ws.write(start_row, 0, title, f["title"] if with_avg else f["subtitle"])
    hdr_row = start_row + (2 if with_avg else 1)
    sub_row = hdr_row + 1

    if with_avg:
        ws.merge_range(hdr_row, 1, hdr_row, 3, "FY 2025 (Full Year)", f["hdr"])
        ws.merge_range(hdr_row, 4, hdr_row, 6, f"2025 YTD ({last_year_yesterday.strftime('%d-%b')})", f["hdr"])
        ws.merge_range(hdr_row, 7, hdr_row, 9, f"2026 YTD ({yesterday.strftime('%d-%b')})", f["hdr"])
        ws.merge_range(hdr_row, 10, hdr_row, 11, "Performance", f["hdr"])
        sub_headers = [label, "Partners", "Sales Mnr", "Avg Sales",
                       "Partners", "Sales Mnr", "Avg Sales",
                       "Partners", "Sales Mnr", "Avg Sales",
                       "Contr %", "Growth %"]
    else:
        ws.merge_range(hdr_row, 1, hdr_row, 2, "FY 2025", f["hdr"])
        ws.merge_range(hdr_row, 3, hdr_row, 4, f"2025 YTD ({last_year_yesterday.strftime('%d-%b')})", f["hdr"])
        ws.merge_range(hdr_row, 5, hdr_row, 7, f"2026 YTD ({yesterday.strftime('%d-%b')})", f["hdr"])
        ws.merge_range(hdr_row, 8, hdr_row, 9, "Performance", f["hdr"])
        sub_headers = [label, "Partners", "Sales", "Partners", "Sales",
                       "Partners", "Sales", "Avg Sales", "Contr %", "Growth %"]

    for c, h in enumerate(sub_headers):
        ws.write(sub_row, c, h, f["hdr"])

    total_s26 = res["S26Y"].sum() if with_avg else res["S26"].sum() if "S26" in res.columns else res["S26Y"].sum()
    r = sub_row + 1
    for idx, row in res.iterrows():
        s25y = row["S25Y"]
        s26y = row["S26Y"]
        growth = (s26y - s25y) / s25y if s25y > 0 else 0
        ws.write(r, 0, idx, f["txt_det"])
        if with_avg:
            ws.write(r, 1, row["P25F"], f["int"]); ws.write(r, 2, row["S25F"], f["dec"])
            ws.write(r, 3, row["S25F"] / row["P25F"] if row["P25F"] > 0 else 0, f["dec"])
            ws.write(r, 4, row["P25Y"], f["int"]); ws.write(r, 5, row["S25Y"], f["dec"])
            ws.write(r, 6, row["S25Y"] / row["P25Y"] if row["P25Y"] > 0 else 0, f["dec"])
            ws.write(r, 7, row["P26Y"], f["int"]); ws.write(r, 8, row["S26Y"], f["dec"])
            ws.write(r, 9, s26y / row["P26Y"] if row["P26Y"] > 0 else 0, f["dec"])
            ws.write(r, 10, s26y / total_s26 if total_s26 > 0 else 0, f["pct"])
            ws.write(r, 11, growth, f["pct"])
        else:
            ws.write(r, 1, row["P25F"], f["int"]); ws.write(r, 2, row["S25F"], f["dec"])
            ws.write(r, 3, row["P25Y"], f["int"]); ws.write(r, 4, row["S25Y"], f["dec"])
            ws.write(r, 5, row["P26Y"], f["int"]); ws.write(r, 6, row["S26Y"], f["dec"])
            ws.write(r, 7, s26y / row["P26Y"] if row["P26Y"] > 0 else 0, f["dec"])
            ws.write(r, 8, s26y / total_s26 if total_s26 > 0 else 0, f["pct"])
            ws.write(r, 9, growth, f["pct"])
        r += 1

    sums = res.sum()
    ws.write(r, 0, "Grand Total", f["tot_int"])
    if with_avg:
        ws.write(r, 1, sums["P25F"], f["tot_int"]); ws.write(r, 2, sums["S25F"], f["tot_dec"])
        ws.write(r, 3, sums["S25F"] / sums["P25F"] if sums["P25F"] > 0 else 0, f["tot_dec"])
        ws.write(r, 4, sums["P25Y"], f["tot_int"]); ws.write(r, 5, sums["S25Y"], f["tot_dec"])
        ws.write(r, 6, sums["S25Y"] / sums["P25Y"] if sums["P25Y"] > 0 else 0, f["tot_dec"])
        ws.write(r, 7, sums["P26Y"], f["tot_int"]); ws.write(r, 8, sums["S26Y"], f["tot_dec"])
        ws.write(r, 9, sums["S26Y"] / sums["P26Y"] if sums["P26Y"] > 0 else 0, f["tot_dec"])
        ws.write(r, 10, 1.0, f["tot_pct"])
        ws.write(r, 11, (sums["S26Y"] - sums["S25Y"]) / sums["S25Y"] if sums["S25Y"] > 0 else 0, f["tot_pct"])
    else:
        ws.write(r, 1, sums["P25F"], f["tot_int"]); ws.write(r, 2, sums["S25F"], f["tot_dec"])
        ws.write(r, 3, sums["P25Y"], f["tot_int"]); ws.write(r, 4, sums["S25Y"], f["tot_dec"])
        ws.write(r, 5, sums["P26Y"], f["tot_int"]); ws.write(r, 6, sums["S26Y"], f["tot_dec"])
        ws.write(r, 7, sums["S26Y"] / sums["P26Y"] if sums["P26Y"] > 0 else 0, f["tot_dec"])
        ws.write(r, 8, 1.0, f["tot_pct"])
        ws.write(r, 9, (sums["S26Y"] - sums["S25Y"]) / sums["S25Y"] if sums["S25Y"] > 0 else 0, f["tot_pct"])

    return r + 3


# =====================================================================
# 7. NEW PARTNER DASHBOARD  (top of New_Partner_Summary sheet)
# =====================================================================
def write_new_partner_dashboard(ws, wb, f, start_row=0):
    df_new_26_ytd = df_active[
        (df_active["Year"] == 2026) & (df_active["InvoiceDate"] <= yesterday) & (df_active["Stat_2026"] == "New/Revival")
    ]
    total_new_partners = df_new_26_ytd["SoldToPartyID"].nunique()
    total_new_sales = df_new_26_ytd["Gross Sales"].sum() / 1e6

    kro_split = df_new_26_ytd.groupby(kro_col).agg(
        P=("SoldToPartyID", "nunique"), S=("Gross Sales", lambda x: x.sum() / 1e6)
    )

    r = start_row
    ws.merge_range(r, 0, r, 3, f"New Partner Snapshot — YTD {yesterday.strftime('%d-%b-%Y')}", f["title"])
    r += 1
    ws.write(r, 0, "", f["hdr"])
    ws.write(r, 1, "New Partner's App.", f["hdr"])
    ws.write(r, 2, "Sales (Mn)", f["hdr"])
    r += 1
    ws.write(r, 0, "New Partners YTD", f["kpi_label"])
    ws.write(r, 1, total_new_partners, f["kpi_value_int"])
    ws.write(r, 2, total_new_sales, f["kpi_value"])
    r += 1
    for label in ["RD", "KRO"]:
        row = kro_split.loc[label] if label in kro_split.index else pd.Series({"P": 0, "S": 0})
        ws.write(r, 0, label, f["txt_det"])
        ws.write(r, 1, int(row["P"]), f["int"])
        ws.write(r, 2, row["S"], f["dec"])
        r += 1
    r += 1

    bu_target_mn = df_region_target.loc[df_region_target["Year"] == TARGET_YEAR, "Target"].sum() / 1e6
    total_ytd_sales_mn = df_active[
        (df_active["Year"] == 2026) & (df_active["InvoiceDate"] <= yesterday)
    ]["Gross Sales"].sum() / 1e6
    new_pct = total_new_sales / total_ytd_sales_mn if total_ytd_sales_mn else 0

    ws.write(r, 0, "Gross Total BU Target", f["kpi_label"]); ws.write(r, 1, bu_target_mn, f["kpi_value"]); r += 1
    ws.write(r, 0, f"Total YTD {yesterday.strftime('%b')} Sales", f["kpi_label"]); ws.write(r, 1, total_ytd_sales_mn, f["kpi_value"]); r += 1
    # Stat_2026 definition (first billed in 2026). The Geo Expansion sheet's
    # figure is broader (adds Q4'25 + <5L'25 onboarded), so the labels differ.
    ws.write(r, 0, "New Partner's 2026 Sales (New in 2026)", f["kpi_label"]); ws.write(r, 1, total_new_sales, f["kpi_value"])
    ws.merge_range(r, 2, r, 3, "New Partner's Sales %", f["hdr"]); r += 1
    ws.merge_range(r, 2, r, 3, new_pct, f["kpi_pct_hero"])
    r += 3

    ws.write(r, 0, "New Partner Performance by Region — Target vs Actual", f["title"])
    r += 1
    hdr_r = r
    ws.write(hdr_r, 0, "Region", f["hdr"])
    ws.write(hdr_r, 1, "Partner Count Target", f["hdr"])
    ws.write(hdr_r, 2, "Gr.Sales Target", f["hdr"])
    ws.merge_range(hdr_r, 3, hdr_r, 5, f"2026 YTD ({yesterday.strftime('%d-%b')})", f["hdr"])
    ws.merge_range(hdr_r, 6, hdr_r, 8, f"2025 YTD ({last_year_yesterday.strftime('%d-%b')})", f["hdr"])
    ws.merge_range(hdr_r, 9, hdr_r, 10, "Performance", f["hdr"])
    r += 1
    sub_r = r
    sub_headers = ["", "", "", "Partners", "Sales Mnr", "Avg Sales",
                   "Partners", "Sales Mnr", "Avg Sales", "% of Ach.", "Gwt %"]
    for c, h in enumerate(sub_headers):
        if h:
            ws.write(sub_r, c, h, f["hdr"])
    r += 1

    region_new = aggregate_periods("Zone", new_only=True)
    partner_target_by_zone = NEW_PARTNER_TARGETS_BY_ZONE["Partner Count Target"]
    sales_target_by_zone = NEW_PARTNER_TARGETS_BY_ZONE["Gr.Sales Target"]

    all_zones = unmapped_last(sorted(set(partner_target_by_zone.index) | set(region_new.index)))

    for zone in all_zones:
        row = region_new.loc[zone] if zone in region_new.index else pd.Series({"P25Y": 0, "S25Y": 0, "P26Y": 0, "S26Y": 0})
        p_target = int(partner_target_by_zone.get(zone, 0))
        s_target = sales_target_by_zone.get(zone, 0)
        s26 = row["S26Y"]
        s25 = row["S25Y"]
        pct_ach = s26 / s_target if s_target else 0
        gwt = (s26 - s25) / s25 if s25 else 0
        ws.write(r, 0, zone, f["txt_det"])
        ws.write(r, 1, p_target, f["int"])
        ws.write(r, 2, s_target, f["dec"])
        ws.write(r, 3, int(row["P26Y"]), f["int"])
        ws.write(r, 4, s26, f["dec"])
        ws.write(r, 5, s26 / row["P26Y"] if row["P26Y"] else 0, f["dec"])
        ws.write(r, 6, int(row["P25Y"]), f["int"])
        ws.write(r, 7, s25, f["dec"])
        ws.write(r, 8, s25 / row["P25Y"] if row["P25Y"] else 0, f["dec"])
        ws.write(r, 9, pct_ach, f["pct"])
        ws.write(r, 10, gwt, f["pct"])
        r += 1

    tot_p_target = int(partner_target_by_zone.sum())
    tot_s_target = sales_target_by_zone.sum()
    tot_s26 = region_new["S26Y"].sum()
    tot_s25 = region_new["S25Y"].sum()
    ws.write(r, 0, "Grand Total", f["tot_int"])
    ws.write(r, 1, tot_p_target, f["tot_int"])
    ws.write(r, 2, tot_s_target, f["tot_dec"])
    ws.write(r, 3, int(region_new["P26Y"].sum()), f["tot_int"])
    ws.write(r, 4, tot_s26, f["tot_dec"])
    ws.write(r, 5, tot_s26 / region_new["P26Y"].sum() if region_new["P26Y"].sum() else 0, f["tot_dec"])
    ws.write(r, 6, int(region_new["P25Y"].sum()), f["tot_int"])
    ws.write(r, 7, tot_s25, f["tot_dec"])
    ws.write(r, 8, tot_s25 / region_new["P25Y"].sum() if region_new["P25Y"].sum() else 0, f["tot_dec"])
    ws.write(r, 9, tot_s26 / tot_s_target if tot_s_target else 0, f["tot_pct"])
    ws.write(r, 10, (tot_s26 - tot_s25) / tot_s25 if tot_s25 else 0, f["tot_pct"])
    r += 4

    r = write_helios_region_pivot(ws, wb, f, r, new_partners_only=True,
                                   title="Product-wise Break-up — NEW PARTNERS ONLY")
    r += 2

    r = write_new_geo_table(ws, wb, f, r)

    return r + 2


def write_new_geo_table(ws, wb, f, start_row):
    ws.write(start_row, 0, "New Geo Expansion Analysis (Active 2026, Nil in 24/25)", f["subtitle"])
    hdr_r = start_row + 1
    headers_geo = ["City", "Partners (2026)", f"Sales Mnr ({yesterday.strftime('%d-%b')})"]
    for c, h in enumerate(headers_geo):
        ws.write(hdr_r, c, h, f["hdr"])

    geo_data = df_final[
        (df_final["Geo_City"].isin(new_geo_list)) & (df_final["Year"] == TARGET_YEAR) & (df_final["InvoiceDate"] <= yesterday)
    ]
    geo_sum = geo_data.groupby("Geo_City").agg(P=("SoldToPartyID", "nunique"), S=("Gross Sales", lambda x: x.sum() / 1e6))

    r = hdr_r + 1
    for idx, row in geo_sum.iterrows():
        ws.write(r, 0, idx, f["txt_det"])
        ws.write(r, 1, int(row["P"]), f["int"])
        ws.write(r, 2, row["S"], f["dec"])
        r += 1

    ws.write(r, 0, "Grand Total", f["tot_int"])
    ws.write(r, 1, int(geo_sum["P"].sum()), f["tot_int"])
    ws.write(r, 2, geo_sum["S"].sum(), f["tot_dec"])
    return r + 1


def write_helios_region_pivot(ws, wb, f, start_row, new_partners_only=False, title=None, new_partner_ids=None):
    df_src = df_retail.merge(
        df_material_master[["Material", "Family Actual"]],
        left_on="MaterialCode", right_on="Material", how="left",
    )
    zone_map = df_final[["SoldToPartyID", "Zone", "Stat_2026"]].drop_duplicates("SoldToPartyID")
    df_src = df_src.merge(zone_map, on="SoldToPartyID", how="left")

    df_src["Helios_Display"] = np.where(
        (df_src["HeliosCode"] == "HSCOS") & (df_src["Family Actual"].astype(str).str.upper() == "WISER"),
        "Wiser",
        np.where(df_src["HeliosCode"] == "HSCOS", "HSCOS Others", df_src["HeliosCode"]),
    )

    df_yr = df_src[df_src["Year"] == TARGET_YEAR]
    if new_partners_only:
        if new_partner_ids is not None:
            df_yr = df_yr[df_yr["SoldToPartyID"].astype(str).isin(new_partner_ids)]
        else:
            df_yr = df_yr[df_yr["Stat_2026"] == "New/Revival"]

    if df_yr.empty:
        ws.write(start_row, 0, (title or "Product-wise Break-up") + " — no data", f["title"])
        return start_row + 2

    pivot = df_yr.pivot_table(index="Zone", columns="Helios_Display", values="Gross Sales", aggfunc="sum", fill_value=0) / 1e6
    pivot = pivot.loc[unmapped_last(list(pivot.index))]
    pivot["Grand Total"] = pivot.sum(axis=1)
    pivot.loc["Grand Total"] = pivot.sum()

    ws.write(start_row, 0, title or "Product-wise Break-up", f["title"])
    hdr_r = start_row + 1
    ws.write(hdr_r, 0, "Region", f["hdr"])
    for c, col in enumerate(pivot.columns, start=1):
        ws.write(hdr_r, c, col, f["hdr"])

    r = hdr_r + 1
    for idx, row in pivot.iterrows():
        row_fmt = f["tot_int"] if idx == "Grand Total" else f["txt_det"]
        val_fmt = f["tot_dec"] if idx == "Grand Total" else f["dec"]
        ws.write(r, 0, idx, row_fmt)
        for c, val in enumerate(row, start=1):
            ws.write(r, c, val, val_fmt)
        r += 1

    return r


# =====================================================================
# 8. DETAIL SHEET HELPER (pivot by partner, unchanged logic)
# =====================================================================
def _fill_ids(df, kro=False):
    """pivot_table silently drops rows whose index key is blank, so blank
    GSTN (and partner type, when it is newly an index key) become '' first."""
    out = df.copy()
    for c in ["GSTN"] + ([kro_col] if kro else []):
        if c in out.columns:
            out[c] = out[c].astype(object).fillna("")
    return out


def write_detail(writer, wb, f, df_filtered, sheet_name, target_year):
    df_yr = df_filtered[df_filtered["Year"] == target_year]
    if df_yr.empty:
        print(f"No data for {sheet_name} in {target_year}, sheet skipped.")
        return

    pivot = _fill_ids(df_yr).pivot_table(
        index=["SoldToPartyID", "Patner", "GSTN", "RD/KRO", "Branch", "Zone", "City", "Tier_Bucket", "ASM"],
        columns="Month", values="Gross Sales", aggfunc="sum", fill_value=0,
    )
    pivot["Grand Total"] = pivot.sum(axis=1)
    pivot = partner_sheet(pivot.sort_values("Grand Total", ascending=False).reset_index())
    pivot.to_excel(writer, sheet_name=sheet_name, index=False)

    ws = writer.sheets[sheet_name]
    for c, col in enumerate(pivot.columns):
        ws.write(0, c, col, f["hdr"])
        ws.set_column(c, c, 18, f["num_det"] if c > 8 else f["txt_det"])
    ws.freeze_panes(1, partner_freeze(pivot))


# =====================================================================
# 9. POLICY / HANDHOLDING TRACKER (vectorised categorisation)
# =====================================================================
def generate_policy_handholding_tracker(writer, wb, f):
    df_retail["SoldToPartyID"] = df_retail["SoldToPartyID"].astype(str).str.strip()
    df_targets["Sold-to Party"] = df_targets["Sold-to Party"].astype(str).str.strip()

    target_col = [c for c in df_targets.columns if "2026" in str(c)][0]
    custom_targets = df_targets.set_index("Sold-to Party")[target_col].to_dict()
    rem_months = 12 - (yesterday.month - 1)

    sm = df_retail.groupby(["SoldToPartyID", "Year"])["Gross Sales"].sum().unstack(fill_value=0)
    sm.index = sm.index.astype(str)
    s25_val = sm[2025] if 2025 in sm.columns else pd.Series(0, index=sm.index)
    s24_val = sm[2024] if 2024 in sm.columns else pd.Series(0, index=sm.index)
    s23_val = sm[2023] if 2023 in sm.columns else pd.Series(0, index=sm.index)

    first_sale_2025 = (
        df_retail.loc[df_retail["Year"] == 2025]
        .groupby("SoldToPartyID")["InvoiceDate"]
        .min()
    )
    first_sale_2025.index = first_sale_2025.index.astype(str)

    tracker = df_final.groupby("SoldToPartyID").agg(
        Patner=("Patner", "first"), GSTN=("GSTN", "first"), Zone=("Zone", "first"), Branch=("Branch", "first"),
        City=("City", "first"), ASM=("ASM", "first"), Stat_2026=("Stat_2026", "first"),
        **{kro_col: (kro_col, "first")},
    )
    tracker.index = tracker.index.astype(str)

    s25 = s25_val.reindex(tracker.index).fillna(0)
    s24 = s24_val.reindex(tracker.index).fillna(0)
    s23 = s23_val.reindex(tracker.index).fillna(0)
    first_sale = first_sale_2025.reindex(tracker.index)

    stat_26_str = tracker["Stat_2026"].astype(str).str.lower().str.strip()
    is_new_2026 = stat_26_str.str.contains("new") | stat_26_str.str.contains("revival")
    is_new_25 = (s25 > 0) & (s24 == 0) & (s23 == 0)
    is_q4_2025 = is_new_25 & first_sale.dt.month.isin([10, 11, 12])
    is_5l_plus = s25 >= 500000

    tracker["Policy Category"] = np.select(
        [
            is_new_2026,
            is_new_25 & is_q4_2025,
            is_new_25 & ~is_q4_2025 & is_5l_plus,
            is_new_25 & ~is_q4_2025 & ~is_5l_plus,
        ],
        [
            "New Partner 2026",
            "Q4'2025 Onboarded",
            ">5L 2025 Onboarded",
            "<5L 2025 Onboarded",
        ],
        default="Old Partner",
    )

    is_new_partner_bucket = is_new_2026 | is_q4_2025 | (is_new_25 & ~is_q4_2025 & ~is_5l_plus)
    is_handholding_bucket = is_new_25 & ~is_q4_2025 & is_5l_plus

    tracker["Old/New/Handholding"] = np.select(
        [is_new_partner_bucket, is_handholding_bucket],
        ["New Partner", "Handholding partner"],
        default="Old partner",
    )

    override_mask = tracker.index.isin(MANUAL_OLD_PARTNER_IDS)
    tracker.loc[override_mask, "Policy Category"] = "Old Partner"
    tracker.loc[override_mask, "Old/New/Handholding"] = "Old partner"

    perf_26 = (
        df_retail[(df_retail["Year"] == 2026) & (df_retail["InvoiceDate"] <= yesterday)]
        .groupby("SoldToPartyID")
        .agg(YTD_2026=("Gross Sales", "sum"))
    )
    perf_25_ytd = (
        df_retail[(df_retail["Year"] == 2025) & (df_retail["InvoiceDate"] <= last_year_yesterday)]
        .groupby("SoldToPartyID")
        .agg(YTD_2025=("Gross Sales", "sum"))
    )
    tracker = tracker.join(s25.rename("Sales_2025_FY")).join(perf_25_ytd).join(perf_26)
    tracker[["Sales_2025_FY", "YTD_2025", "YTD_2026"]] = tracker[["Sales_2025_FY", "YTD_2025", "YTD_2026"]].fillna(0)

    is_flat_target = tracker["Policy Category"].isin(
        ["New Partner 2026", "Q4'2025 Onboarded", "<5L 2025 Onboarded"]
    )
    custom_target_series = tracker.index.to_series().astype(str).map(custom_targets).fillna(0)
    tracker["2026 Assigned Target"] = np.where(is_flat_target, 2_500_000, custom_target_series)
    tracker["Remaining Months"] = rem_months
    tracker["Req. Monthly Run Rate"] = np.where(
        tracker["2026 Assigned Target"] > 0,
        (tracker["2026 Assigned Target"] - tracker["YTD_2026"]).clip(lower=0) / rem_months,
        0,
    )

    tracker["Achievement %"] = np.where(
        tracker["2026 Assigned Target"] > 0,
        tracker["YTD_2026"] / tracker["2026 Assigned Target"],
        0,
    )

    tracker = tracker.sort_values(by="YTD_2026", ascending=False).reset_index()
    write_policy_tracker_sheet(writer, wb, f, tracker)
    return tracker


def write_policy_tracker_sheet(writer, wb, f, tracker):
    # Separate from the calculation so the same tracker can be written into
    # both the Partner Analysis workbook and the New GEO workbook.
    final_cols = [
        "SoldToPartyID", "Patner", "GSTN", kro_col, "Branch", "Zone", "City", "ASM",
        "Policy Category", "Old/New/Handholding",
        "Sales_2025_FY", "YTD_2025", "YTD_2026", "2026 Assigned Target", "Achievement %",
        "Remaining Months", "Req. Monthly Run Rate",
    ]
    display_df = tracker[final_cols].rename(columns={
        "SoldToPartyID": "Partner ID", "Patner": "Partner Name", "Zone": "Region",
        "Sales_2025_FY": "2025 FY Actual", "YTD_2025": "2025 YTD Actual", "YTD_2026": "2026 YTD Actual",
    })

    display_df.to_excel(writer, sheet_name="Partner_Policy_Tracker", index=False)
    ws = writer.sheets["Partner_Policy_Tracker"]

    for c, val in enumerate(display_df.columns):
        ws.write(0, c, val, f["hdr_pol"])

    n_rows = len(display_df)
    last_row = n_rows + 1

    txt_border = f["txt_det"]

    ws.set_column("A:A", 14, f["policy_num"])
    ws.set_column("B:B", 30, txt_border)
    ws.set_column("C:C", 20, txt_border)   # GSTN
    ws.set_column("D:D", 14, txt_border)   # KRO/RD/PD
    ws.set_column("E:E", 16, txt_border)   # Branch
    ws.set_column("F:F", 16, txt_border)   # Region
    ws.set_column("G:G", 18, txt_border)   # City
    ws.set_column("H:H", 16, txt_border)   # ASM
    ws.set_column("I:I", 22, txt_border)   # Policy Category
    ws.set_column("J:J", 20, txt_border)   # Old/New/Handholding
    ws.set_column("K:N", 18, f["policy_num"])
    ws.set_column("O:O", 15, f["pct"])
    ws.set_column("P:P", 16, f["int"])
    ws.set_column("Q:Q", 22, f["policy_runrate"])
    ws.freeze_panes(1, partner_freeze(display_df))
    ws.set_tab_color(THEME["tab_tracker"])

    policy_range = f"C2:C{last_row}"
    onh_range = f"D2:D{last_row}"

    policy_colors = {
        "New Partner 2026": "#C6EFCE",
        "Q4'2025 Onboarded": "#D4EEBD",
        ">5L 2025 Onboarded": "#BDD7EE",
        "<5L 2025 Onboarded": "#EEDABD",
        "Old Partner": "#D9D9D9",
    }
    for val, color in policy_colors.items():
        ws.conditional_format(policy_range, {
            "type": "text", "criteria": "containing", "value": val,
            "format": wb.add_format({"bg_color": color, "border": 1, "border_color": THEME["border"], "font_name": THEME["font"], "font_size": THEME["font_size"]}),
        })

    onh_colors = {
        "New Partner": "#C6EFCE",
        "Handholding partner": "#BDD7EE",
        "Old partner": "#D9D9D9",
    }
    for val, color in onh_colors.items():
        ws.conditional_format(onh_range, {
            "type": "text", "criteria": "containing", "value": val,
            "format": wb.add_format({"bg_color": color, "border": 1, "border_color": THEME["border"], "font_name": THEME["font"], "font_size": THEME["font_size"]}),
        })

    ws.conditional_format(f"N2:N{last_row}", {
        "type": "3_color_scale",
        "min_color": "#FFFFFF",
        "mid_color": THEME["warn"][0],
        "max_color": THEME["bad"][0],
    })

    ws.conditional_format(f"L2:L{last_row}", {
        "type": "3_color_scale",
        "min_color": THEME["bad"][0],
        "mid_color": THEME["warn"][0],
        "max_color": THEME["good"][0],
    })

    ws.conditional_format(f"J2:J{last_row}", {
        "type": "data_bar",
        "bar_color": THEME["accent"],
    })


# =====================================================================
# 10. ZONE TARGET TRACKER (all partners, with formulas)
# =====================================================================
def generate_zone_target_tracker(writer, wb, f):
    df_temp = df_retail.copy()
    if "Zone" not in df_temp.columns:
        df_temp = df_temp.merge(df_final[["SoldToPartyID", "Zone"]].drop_duplicates(), on="SoldToPartyID", how="left")

    perf_26 = df_temp[(df_temp["Year"] == 2026) & (df_temp["InvoiceDate"] <= yesterday)].groupby("Zone").agg(YTD_26=("Gross Sales", "sum")).reset_index()
    perf_25 = df_temp[(df_temp["Year"] == 2025) & (df_temp["InvoiceDate"] <= last_y_ytd_date)].groupby("Zone").agg(YTD_25=("Gross Sales", "sum")).reset_index()
    target_26 = df_region_target[df_region_target["Year"] == 2026].groupby("Zone").agg(Target=("Target", "sum")).reset_index()

    summary = pd.merge(target_26, perf_26, on="Zone", how="outer")
    summary = pd.merge(summary, perf_25, on="Zone", how="outer").fillna(0)
    summary["Growth %"] = (summary["YTD_26"] - summary["YTD_25"]) / summary["YTD_25"].replace(0, np.nan)
    summary["Achievement %"] = summary["YTD_26"] / summary["Target"].replace(0, 1)
    summary["Gap"] = summary["Target"] - summary["YTD_26"]
    summary = summary.sort_values("Target", ascending=False)

    cols = ["Zone", "Target", "YTD_26", "Gap", "Achievement %", "YTD_25", "Growth %"]
    display_df = summary[cols].rename(columns={
        "YTD_26": f"2026 YTD ({yesterday.strftime('%d-%b')})",
        "YTD_25": f"2025 YTD ({last_y_ytd_date.strftime('%d-%b')})",
    })

    sheet_name = "Zone_Target_Tracker"
    display_df.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]
    for c, val in enumerate(display_df.columns):
        ws.write(0, c, val, f["hdr_alt"])
    ws.set_column("A:A", 25)
    ws.set_column("B:D", 18, f["policy_num"])
    ws.set_column("E:E", 15, f["pct"])
    ws.set_column("F:F", 18, f["policy_num"])
    ws.set_column("G:G", 15, f["pct"])

    last_row = len(display_df) + 1
    tot_fmt = f["tot_inr"]
    tot_pct_fmt = f["tot_pct"]
    ws.write(last_row, 0, "Grand Total", tot_fmt)
    ws.write_formula(last_row, 1, f"=SUM(B2:B{last_row})", tot_fmt)
    ws.write_formula(last_row, 2, f"=SUM(C2:C{last_row})", tot_fmt)
    ws.write_formula(last_row, 3, f"=B{last_row + 1}-C{last_row + 1}", tot_fmt)
    ws.write_formula(last_row, 4, f"=C{last_row + 1}/B{last_row + 1}", tot_pct_fmt)
    ws.write_formula(last_row, 5, f"=SUM(F2:F{last_row})", tot_fmt)
    ws.write_formula(last_row, 6, f"=(C{last_row + 1}-F{last_row + 1})/F{last_row + 1}", tot_pct_fmt)
    ws.freeze_panes(1, 1)
    ws.set_tab_color(THEME["tab_detail"])
    return summary


# =====================================================================
# 10b. Sheet1 + Geo Expansion (for the separate New_GEO report)
# =====================================================================
# ">5 lacs" really meant 5-10 lacs, and "less than 5 lacs" mixed partners who
# billed <5L in 2025 with partners who billed nothing in 2025 (all the
# "New Partner 2026" ones) -- those now get their own bucket.
SALES_BUCKET_LABELS = [">10 lacs", "5-10 lacs", "<5 lacs", "Nil in FY2025"]


def sales_bucket(sales_2025_fy):
    return np.select(
        [sales_2025_fy >= 1_000_000, sales_2025_fy >= 500_000, sales_2025_fy > 0],
        [SALES_BUCKET_LABELS[0], SALES_BUCKET_LABELS[1], SALES_BUCKET_LABELS[2]],
        default=SALES_BUCKET_LABELS[3],
    )


def write_sheet1(workbook, fmt, region_new):
    ws = workbook.add_worksheet("Sheet1")
    ws.write(0, 0, "New Partner Performance by Region — Target vs Actual", fmt["title"])
    ws.merge_range(1, 1, 1, 3, f"2026 YTD ({yesterday.strftime('%d-%b')})", fmt["hdr"])
    ws.merge_range(1, 4, 1, 6, f"2025 YTD ({last_year_yesterday.strftime('%d-%b')})", fmt["hdr"])
    ws.write(1, 7, "", fmt["hdr"])
    headers = ["Region", "Partners", "Sales Mnr", "Avg Sales", "Partners", "Sales Mnr", "Avg Sales", "Gwt %"]
    for c, h in enumerate(headers):
        ws.write(2, c, h, fmt["hdr"])

    r = 3
    for zone, row in region_new.iterrows():
        gwt = (row["S26Y"] - row["S25Y"]) / row["S25Y"] if row["S25Y"] else 0
        ws.write(r, 0, zone, fmt["txt_det"])
        ws.write(r, 1, int(row["P26Y"]), fmt["int"])
        ws.write(r, 2, row["S26Y"], fmt["dec"])
        ws.write(r, 3, row["S26Y"] / row["P26Y"] if row["P26Y"] else 0, fmt["dec"])
        ws.write(r, 4, int(row["P25Y"]), fmt["int"])
        ws.write(r, 5, row["S25Y"], fmt["dec"])
        ws.write(r, 6, row["S25Y"] / row["P25Y"] if row["P25Y"] else 0, fmt["dec"])
        ws.write(r, 7, gwt, fmt["pct"])
        r += 1

    sums = region_new.sum()
    gwt_tot = (sums["S26Y"] - sums["S25Y"]) / sums["S25Y"] if sums["S25Y"] else 0
    ws.write(r, 0, "Grand Total", fmt["tot_int"])
    ws.write(r, 1, int(sums["P26Y"]), fmt["tot_int"])
    ws.write(r, 2, sums["S26Y"], fmt["tot_dec"])
    ws.write(r, 3, sums["S26Y"] / sums["P26Y"] if sums["P26Y"] else 0, fmt["tot_dec"])
    ws.write(r, 4, int(sums["P25Y"]), fmt["tot_int"])
    ws.write(r, 5, sums["S25Y"], fmt["tot_dec"])
    ws.write(r, 6, sums["S25Y"] / sums["P25Y"] if sums["P25Y"] else 0, fmt["tot_dec"])
    ws.write(r, 7, gwt_tot, fmt["tot_pct"])
    ws.set_column("A:A", 14)
    ws.set_column("B:H", 15)


def write_geo_expansion_sheet(workbook, fmt, tracker, kro_col_name):
    """
    'Geo Expansion' sheet: policy-tracker-driven "New Partner" population
    (broader than Stat_2026), with Region/RD-KRO breakdowns and an FY2025
    sales-bucket split.

    Table order matches the reference file: Snapshot -> Policy rollup ->
    BU Target box -> Sales-bucket split -> Handholding region table ->
    New Partners region table -> New Geo Expansion Analysis -> RD/KRO
    breakdown -> Product breakup.

    SKIPPED per confirmed scope: the SSP sheet (no Branch/Distributor(SSA)
    target+sales source loaded anywhere), the two stray manual-note values
    from the reference file, and the 'Chls cont.'/'<5 lacs' columns on the
    Handholding region table (no traceable definition -- ask before adding).
    """
    ws = workbook.add_worksheet("Geo Expansion")
    ws.set_tab_color(THEME["tab_summary"])

    NEW_PARTNER_SUBCATS = ["<5L 2025 Onboarded", "New Partner 2026", "Q4'2025 Onboarded"]
    HANDHOLDING_CAT = ">5L 2025 Onboarded"

    is_new_partner_tracker = tracker["Old/New/Handholding"] == "New Partner"
    is_handholding_tracker = tracker["Old/New/Handholding"] == "Handholding partner"

    total_new_sales_ytd26 = tracker.loc[is_new_partner_tracker, "YTD_2026"].sum() / 1e6
    bu_target_mn = df_region_target.loc[df_region_target["Year"] == TARGET_YEAR, "Target"].sum() / 1e6
    total_ytd_sales_mn = df_final[
        (df_final["Year"] == 2026) & (df_final["InvoiceDate"] <= yesterday)
    ]["Gross Sales"].sum() / 1e6

    # --- A/B) Snapshot ------------------------------------------------
    r = 0
    ws.merge_range(r, 0, r, 3, f"New Partner Snapshot — YTD {yesterday.strftime('%b-%Y')}", fmt["title"]); r += 2
    ws.write(r, 0, "Description", fmt["hdr"])
    # FIXED: the "YTD'25" column used to show FY2025 (full-year) sales.
    # FY'25 and like-for-like YTD'25 are now separate columns.
    ws.write(r, 1, "No.of Partner", fmt["hdr"])
    ws.write(r, 2, "FY'25 (Mn)", fmt["hdr"])
    ws.write(r, 3, f"YTD {last_year_yesterday.strftime('%b').upper()}'25 (Mn)", fmt["hdr"])
    ws.write(r, 4, f"YTD {yesterday.strftime('%b').upper()}'26 (Mn)", fmt["hdr"])
    r += 1

    def write_cat_row(label, mask, bold=False):
        nonlocal r
        n = int(mask.sum())
        s25_fy = tracker.loc[mask, "Sales_2025_FY"].sum() / 1e6
        s25_ytd = tracker.loc[mask, "YTD_2025"].sum() / 1e6
        s26 = tracker.loc[mask, "YTD_2026"].sum() / 1e6
        f_int = fmt["tot_int"] if bold else fmt["int"]
        f_txt = fmt["tot_int"] if bold else fmt["txt_det"]
        f_num = fmt["tot_dec"] if bold else fmt["dec"]
        ws.write(r, 0, label, f_txt)
        ws.write(r, 1, n, f_int)
        ws.write(r, 2, s25_fy, f_num)
        ws.write(r, 3, s25_ytd, f_num)
        ws.write(r, 4, s26, f_num)
        r += 1

    write_cat_row("Handholding partner", is_handholding_tracker, bold=True)
    write_cat_row(HANDHOLDING_CAT, tracker["Policy Category"] == HANDHOLDING_CAT)
    r += 1
    write_cat_row("New Partner", is_new_partner_tracker, bold=True)
    for cat in NEW_PARTNER_SUBCATS:
        write_cat_row(cat, tracker["Policy Category"] == cat)
    r += 1
    write_cat_row("Grand Total", is_new_partner_tracker | is_handholding_tracker, bold=True)
    r += 2

    ws.write(r, 0, "Gross Total BU Target", fmt["kpi_label"]); ws.write(r, 1, bu_target_mn, fmt["kpi_value"]); r += 1
    ws.write(r, 0, f"Total YTD {yesterday.strftime('%b')} Sales", fmt["kpi_label"]); ws.write(r, 1, total_ytd_sales_mn, fmt["kpi_value"]); r += 1
    ws.write(r, 0, "New Partner's 2026 Sales (New'26 + Q4'25 + <5L'25)", fmt["kpi_label"]); ws.write(r, 1, total_new_sales_ytd26, fmt["kpi_value"])
    ws.write(r, 2, "New Partner's Sales %", fmt["hdr"]); r += 1
    ws.write(r, 2, total_new_sales_ytd26 / total_ytd_sales_mn if total_ytd_sales_mn else 0, fmt["kpi_pct_hero"])
    r += 3

    # --- C) Partners by FY2025 Sales Bucket ---------------------------
    ws.write(r, 0, "Partners by FY2025 Sales Bucket (Handholding + New Partner)", fmt["subtitle"]); r += 1
    # Split by Policy Category so it's visible who sits in each bucket.
    bucketed = tracker.loc[is_new_partner_tracker | is_handholding_tracker].copy()
    bucketed["Bucket"] = sales_bucket(bucketed["Sales_2025_FY"])
    bucket_cats = [HANDHOLDING_CAT] + NEW_PARTNER_SUBCATS
    bucket_tbl = (pd.crosstab(bucketed["Bucket"], bucketed["Policy Category"])
                  .reindex(index=SALES_BUCKET_LABELS, columns=bucket_cats, fill_value=0))
    bucket_hdrs = ["Bucket", "Partner Count", "Handholding (>5L 2025 Onboarded)"] + NEW_PARTNER_SUBCATS
    for c, h in enumerate(bucket_hdrs):
        ws.write(r, c, h, fmt["hdr_wrap"])
    ws.set_row(r, 30)
    r += 1
    for label, row in bucket_tbl.iterrows():
        ws.write(r, 0, label, fmt["txt_det"])
        ws.write(r, 1, int(row.sum()), fmt["int"])
        for c, cat in enumerate(bucket_cats, start=2):
            ws.write(r, c, int(row[cat]), fmt["int"])
        r += 1
    ws.write(r, 0, "Grand Total", fmt["tot_int"])
    ws.write(r, 1, int(bucket_tbl.values.sum()), fmt["tot_int"])
    for c, cat in enumerate(bucket_cats, start=2):
        ws.write(r, c, int(bucket_tbl[cat].sum()), fmt["tot_int"])
    r += 3

    # --- D) Handholding partner region breakdown -----------------------
    # FIXED: "YTD'25" was FY2025 sales, so Grw % compared a full year with a
    # part year. Growth is now YTD'26 vs YTD'25 (same cut-off date); FY'25 is
    # shown separately. Column 2 is a partner COUNT, so it's labelled as one.
    # 'Chls cont.' and '<5 lacs' columns from the reference are still left
    # out -- no traceable definition for either.
    ws.write(r, 0, "Handholding Partner — Region Breakdown", fmt["title"]); r += 1
    hh_tracker = tracker.loc[is_handholding_tracker]
    headers_hh = ["Region", "Partner Count", "FY'25 (Mn)",
                  f"YTD {last_year_yesterday.strftime('%b').upper()}'25 (Mn)",
                  f"YTD {yesterday.strftime('%b').upper()}'26 (Mn)", "YTD Grw %"]
    for c, h in enumerate(headers_hh):
        ws.write(r, c, h, fmt["hdr"])
    r += 1

    def write_hh_row(label, sub, bold=False):
        s25_fy = sub["Sales_2025_FY"].sum() / 1e6
        s25_ytd = sub["YTD_2025"].sum() / 1e6
        s26 = sub["YTD_2026"].sum() / 1e6
        gwt = (s26 - s25_ytd) / s25_ytd if s25_ytd else 0
        ws.write(r, 0, label, fmt["tot_int"] if bold else fmt["txt_det"])
        ws.write(r, 1, len(sub), fmt["tot_int"] if bold else fmt["int"])
        ws.write(r, 2, s25_fy, fmt["tot_dec"] if bold else fmt["dec"])
        ws.write(r, 3, s25_ytd, fmt["tot_dec"] if bold else fmt["dec"])
        ws.write(r, 4, s26, fmt["tot_dec"] if bold else fmt["dec"])
        ws.write(r, 5, gwt, fmt["tot_pct"] if bold else fmt["pct"])

    for zone in unmapped_last(sorted(hh_tracker["Zone"].dropna().unique())):
        write_hh_row(zone, hh_tracker[hh_tracker["Zone"] == zone])
        r += 1
    write_hh_row("Grand Total", hh_tracker, bold=True)
    r += 3

    # --- E) New Partner Performance by Region — Target vs Actual (extended) --
    ws.write(r, 0, "New Partner Performance by Region — Target vs Actual", fmt["title"]); r += 1
    np_tracker = tracker.loc[is_new_partner_tracker]
    # region_new gives the confirmed "New Partners 2025/2026" headcount
    # columns (Stat_2025/Stat_2026 New/Revival partner counts by zone, the
    # same numbers already used on the Snapshot dashboard).
    region_new = aggregate_periods("Zone", new_only=True)

    # Sales (Mn) columns for each sub-category, then a partner-COUNT column
    # for each of the same 3 sub-categories -- same count-per-category logic
    # already used in the RD/KRO breakdown table below, just grouped by
    # region here instead.
    sales_headers = [f"{cat} (Mn)" for cat in NEW_PARTNER_SUBCATS]
    count_headers = [f"{cat} (Count)" for cat in NEW_PARTNER_SUBCATS]
    headers_np = (["Region", "Partner Count Target", "Gr.Sales Target"] + sales_headers
                  + ["Total (Mn)"] + count_headers + ["New Partners 2025", "New Partners 2026"])
    for c, h in enumerate(headers_np):
        ws.write(r, c, h, fmt["hdr_wrap"])
    ws.set_row(r, 30)
    r += 1
    all_zones = unmapped_last(sorted(set(NEW_PARTNER_TARGETS_BY_ZONE.index) | set(np_tracker["Zone"].dropna().unique())))
    for zone in all_zones:
        sub = np_tracker[np_tracker["Zone"] == zone]
        ws.write(r, 0, zone, fmt["txt_det"])
        ws.write(r, 1, int(NEW_PARTNER_TARGETS_BY_ZONE["Partner Count Target"].get(zone, 0)), fmt["int"])
        ws.write(r, 2, NEW_PARTNER_TARGETS_BY_ZONE["Gr.Sales Target"].get(zone, 0), fmt["dec"])
        row_total = 0
        for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=3):
            val = sub.loc[sub["Policy Category"] == cat, "YTD_2026"].sum() / 1e6
            ws.write(r, c, val, fmt["dec"])
            row_total += val
        ws.write(r, 6, row_total, fmt["dec"])
        for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=7):
            cnt = int((sub["Policy Category"] == cat).sum())
            ws.write(r, c, cnt, fmt["int"])
        zrow = region_new.loc[zone] if zone in region_new.index else pd.Series({"P25Y": 0, "P26Y": 0})
        ws.write(r, 10, int(zrow["P25Y"]), fmt["int"])
        ws.write(r, 11, int(zrow["P26Y"]), fmt["int"])
        r += 1
    tot_target_p = int(NEW_PARTNER_TARGETS_BY_ZONE["Partner Count Target"].sum())
    tot_target_s = NEW_PARTNER_TARGETS_BY_ZONE["Gr.Sales Target"].sum()
    ws.write(r, 0, "Grand Total", fmt["tot_int"])
    ws.write(r, 1, tot_target_p, fmt["tot_int"])
    ws.write(r, 2, tot_target_s, fmt["tot_dec"])
    grand_total = 0
    for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=3):
        val = np_tracker.loc[np_tracker["Policy Category"] == cat, "YTD_2026"].sum() / 1e6
        ws.write(r, c, val, fmt["tot_dec"])
        grand_total += val
    ws.write(r, 6, grand_total, fmt["tot_dec"])
    for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=7):
        cnt = int((np_tracker["Policy Category"] == cat).sum())
        ws.write(r, c, cnt, fmt["tot_int"])
    ws.write(r, 10, int(region_new["P25Y"].sum()), fmt["tot_int"])
    ws.write(r, 11, int(region_new["P26Y"].sum()), fmt["tot_int"])
    r += 3

    # --- F) New Geo Expansion Analysis ---------------------------------
    # MOVED: this now runs right after the New Partners region table, to
    # match the reference file's table order (it previously ran last).
    r = write_new_geo_table(ws, workbook, fmt, r)
    r += 2

    # --- G) RD/KRO breakdown of the same 3 sub-categories --------------
    # FIXED: sales totals were in raw rupees, and there was no per-category
    # sales breakdown (only counts) -- now matches the region table's
    # shape: a count column AND a sales-Mn column per sub-category.
    ws.write(r, 0, "New Partners by RD/KRO", fmt["subtitle"]); r += 1
    kro_tracker = tracker.loc[is_new_partner_tracker].copy()
    count_headers_kro = [f"{cat} (Count)" for cat in NEW_PARTNER_SUBCATS]
    sales_headers_kro = [f"{cat} (Mn)" for cat in NEW_PARTNER_SUBCATS]
    headers_kro = (["RD/KRO"] + count_headers_kro + sales_headers_kro + [
        "FY '25 (Mn)",
        f"YTD {last_year_yesterday.strftime('%b').upper()} '25 (Mn)",
        f"YTD {yesterday.strftime('%b').upper()} '26 (Mn)",
    ])
    for c, h in enumerate(headers_kro):
        ws.write(r, c, h, fmt["hdr_wrap"])
    ws.set_row(r, 30)
    r += 1
    kro_map = df_final[["SoldToPartyID", kro_col_name]].drop_duplicates("SoldToPartyID")
    kro_map["SoldToPartyID"] = kro_map["SoldToPartyID"].astype(str)
    if kro_col_name not in kro_tracker.columns:      # the tracker now carries it already
        kro_tracker = kro_tracker.merge(kro_map, on="SoldToPartyID", how="left")
    for kro_val, sub in kro_tracker.groupby(kro_col_name):
        ws.write(r, 0, kro_val, fmt["txt_det"])
        for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=1):
            n = int((sub["Policy Category"] == cat).sum())
            ws.write(r, c, n, fmt["int"])
        for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=4):
            val = sub.loc[sub["Policy Category"] == cat, "YTD_2026"].sum() / 1e6
            ws.write(r, c, val, fmt["dec"])
        ws.write(r, 7, sub["Sales_2025_FY"].sum() / 1e6, fmt["dec"])
        ws.write(r, 8, sub["YTD_2025"].sum() / 1e6, fmt["dec"])
        ws.write(r, 9, sub["YTD_2026"].sum() / 1e6, fmt["dec"])
        r += 1
    ws.write(r, 0, "Grand Total", fmt["tot_int"])
    for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=1):
        ws.write(r, c, int((kro_tracker["Policy Category"] == cat).sum()), fmt["tot_int"])
    for c, cat in enumerate(NEW_PARTNER_SUBCATS, start=4):
        val = kro_tracker.loc[kro_tracker["Policy Category"] == cat, "YTD_2026"].sum() / 1e6
        ws.write(r, c, val, fmt["tot_dec"])
    ws.write(r, 7, kro_tracker["Sales_2025_FY"].sum() / 1e6, fmt["tot_dec"])
    ws.write(r, 8, kro_tracker["YTD_2025"].sum() / 1e6, fmt["tot_dec"])
    ws.write(r, 9, kro_tracker["YTD_2026"].sum() / 1e6, fmt["tot_dec"])
    r += 3

    # --- H) Product-wise Break-up, New Partner (tracker) population ---
    new_partner_ids = set(tracker.loc[is_new_partner_tracker, "SoldToPartyID"].astype(str))
    r = write_helios_region_pivot(ws, workbook, fmt, r, new_partners_only=True,
                                   title="Product-wise Break-up — NEW PARTNERS (Policy Tracker)",
                                   new_partner_ids=new_partner_ids)

    ws.set_column("A:A", 26)
    ws.set_column("B:L", 16)
    return r

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 5.3 PARTNER ANALYSIS — extended sheets and the Insights Summary
#
#   SSP_Performance · Partner_Master_Detail · Customer_Retention ·
#   Scheme_Achievement · Cross_Sell_Opportunity · (optional) Pavilion sheets
#   + a separate Partner_Insights_Summary workbook for the team call.
# ════════════════════════════════════════════════════════════════════
BRANCH_TO_SSP = {row["Branch"]: row["SSP"] for row in SSP_MAP}
BRANCH_TO_SSP_REGION = {row["Branch"]: row["Region"] for row in SSP_MAP}
PAVILION_RAW_SHEET_NAME = None   # auto-detected ("Pavilion as on dd-mm-yyyy")
df_ob_26, df_backlog = pd.DataFrame(), pd.DataFrame()


def load_open_orders():
    """Retail OB + backlog for the partner detail sheet: from the Retail YTD
    report run above, or from its file on SharePoint if that report was off."""
    global df_ob_26, df_backlog
    if retail_ytd:
        df_ob_26, df_backlog = retail_ytd.get("ob", pd.DataFrame()).copy(), retail_ytd.get("backlog", pd.DataFrame()).copy()
    else:
        try:
            book = pd.read_excel(sp.download(f"{PATHS['retail_ytd']}/Retail_Sales_OB_Backlog_data_{TARGET_YEAR}.xlsx"), sheet_name=None)
            df_ob_26, df_backlog = book.get(f"OB_{TARGET_YEAR}", pd.DataFrame()), book.get("Backlog_Overall", pd.DataFrame())
        except Exception as e:
            print(f"   ⚠️ OB/Backlog not available ({e}); those columns are left out of Partner_Master_Detail.")
    for d in (df_ob_26, df_backlog):
        if "SoldToPartyID" in d.columns:
            d["SoldToPartyID"] = d["SoldToPartyID"].astype(str).str.strip()


# =====================================================================
# 10b. SSP PERFORMANCE — folded in from your Overall_SSP_Retail_Performance
#      script, now written as a sheet in THIS workbook instead of a
#      separate file uploaded to SharePoint.
# =====================================================================
def write_ssp_performance_sheet(writer, wb, f):
    """
    Region / SSP / Branch dashboard: Target vs Actual, monthly trend, YoY.
    Uses df_retail (already loaded) filtered by Branch per SSP_MAP — see the
    assumption noted at the top of the script about Sales source.
    """
    ws = wb.add_worksheet("SSP_Performance")
    ws.set_tab_color(THEME["tab_detail"])
    ws.hide_gridlines(2)
    ws.write("A1", "SCHNEIDER ELECTRIC — SSP RETAIL PERFORMANCE", f["title"])
    ws.write("A2", f"Reporting Period: 01-Jan-{TARGET_YEAR} to {yesterday.strftime('%d-%b-%Y')}")
    ws.write("A3", "*All values in Rs Mn unless noted")

    month_cols = [datetime(TARGET_YEAR, m, 1).strftime("%b").upper() for m in range(1, yesterday.month + 1)]
    curr_q_start_month = ((yesterday.month - 1) // 3) * 3 + 1

    col_headers = (["Region", "SSP", "Branch", "Target", "108%", "115%", "Qtr Target", "Bal. Task"]
                   + month_cols + ["Qtr Sales", f"{TARGET_YEAR} YTD", "% Ach", f"{PREV_YEAR} YTD", "% Grw"])
    hdr_row = 12
    for c, h in enumerate(col_headers):
        ws.write(hdr_row, c, h, f["hdr"])

    row = hdr_row + 1
    t_y_curr = t_qt = t_annt = t_y_prev = t_q_sales = t_q_sales_prev = 0.0

    for ssp in SSP_MAP:
        branch = ssp["Branch"]
        m_sales = [
            df_final[(df_final["Branch"] == branch) & (df_final["Year"] == TARGET_YEAR)
                     & (df_final["InvoiceDate"].dt.month == m)]["Gross Sales"].sum() / 1e6
            for m in range(1, yesterday.month + 1)
        ]
        y_curr, ann_t = sum(m_sales), ssp["Annual_Target_Mn"]
        q_t = ann_t * 0.25
        q_sales = sum(m_sales[curr_q_start_month - 1:])

        y_prev_mask = ((df_final["Branch"] == branch) & (df_final["Year"] == PREV_YEAR)
                       & (df_final["InvoiceDate"] <= last_year_yesterday))
        y_prev_ytd = df_final[y_prev_mask]["Gross Sales"].sum() / 1e6

        q_sales_prev = sum(
            df_final[(df_final["Branch"] == branch) & (df_final["Year"] == PREV_YEAR)
                     & (df_final["InvoiceDate"].dt.month == m)]["Gross Sales"].sum() / 1e6
            for m in range(curr_q_start_month, yesterday.month + 1)
        )

        t_y_curr += y_curr; t_qt += q_t; t_annt += ann_t; t_y_prev += y_prev_ytd
        t_q_sales += q_sales; t_q_sales_prev += q_sales_prev

        data_row = (
            [ssp["Region"], ssp["SSP"], branch, ann_t, ann_t * 1.08, ann_t * 1.15, q_t, q_t - q_sales]
            + m_sales
            + [q_sales, y_curr, y_curr / ann_t if ann_t else 0, y_prev_ytd, (y_curr / y_prev_ytd - 1) if y_prev_ytd else 0]
        )
        for c, val in enumerate(data_row):
            is_pct = col_headers[c] in ("% Ach", "% Grw")
            fmt = f["pct"] if is_pct else (f["dec"] if isinstance(val, (int, float)) else f["txt_det"])
            ws.write(row, c, val, fmt)
        row += 1

    ws.merge_range("C6:D6", "Qtr Target", f["hdr"]); ws.merge_range("C7:D7", t_qt, f["dec"])
    ws.merge_range("E6:F6", "Qtr Sales", f["hdr"]); ws.merge_range("E7:F7", t_q_sales, f["dec"])
    ws.merge_range("G6:H6", "Qtr Ach%", f["hdr"]); ws.merge_range("G7:H7", t_q_sales / t_qt if t_qt else 0, f["pct"])
    ws.merge_range("I6:J6", "QoQ Grw%", f["hdr"]); ws.merge_range("I7:J7", (t_q_sales / t_q_sales_prev - 1) if t_q_sales_prev else 0, f["pct"])

    ws.merge_range("C9:D9", "Year Target", f["hdr"]); ws.merge_range("C10:D10", t_annt, f["dec"])
    ws.merge_range("E9:F9", "YTD Sales", f["hdr"]); ws.merge_range("E10:F10", t_y_curr, f["dec"])
    ws.merge_range("G9:H9", "Year Ach%", f["hdr"]); ws.merge_range("G10:H10", t_y_curr / t_annt if t_annt else 0, f["pct"])
    ws.merge_range("I9:J9", "YTD Grw%", f["hdr"]); ws.merge_range("I10:J10", (t_y_curr / t_y_prev - 1) if t_y_prev else 0, f["pct"])

    ws.set_column("A:B", 16)
    ws.set_column("C:Z", 12)
    ws.freeze_panes(hdr_row + 1, 3)


# =====================================================================
# 10c. CONSOLIDATED PARTNER DETAIL — replaces the 4 separate detail sheets
#      (New_Partner_Detail_2026 / New_Partner_Detail_2025_YTD /
#      Geo_Expansion_Detail_2026 / Geo_Expansion_Detail_2025) and the old
#      MoM_2026 sheet with ONE sheet + filter columns, matching the "Raw
#      Sales / Partner Performance" flat-sheet style from your SSP script.
#      Use the sheet's AutoFilter (already applied) on New_Partner_2026,
#      New_Geo_2026, etc. instead of navigating between sheets.
# =====================================================================
def write_partner_master_detail(writer, wb, f):
    pivot = _fill_ids(df_final[df_final["Year"] == TARGET_YEAR]).pivot_table(
        index=["SoldToPartyID", "Patner", "GSTN", "ASM", "Branch", "Zone", "City", "Geo_City", "Tier_Bucket", kro_col],
        columns="Month", values="Gross Sales", aggfunc="sum", fill_value=0,
    )
    pivot[f"{TARGET_YEAR} Total"] = pivot.sum(axis=1).round(0)
    df_out = pivot.reset_index()

    df_out = df_out.merge(partner_status[["Stat_2026", "Stat_2025"]], left_on="SoldToPartyID", right_index=True, how="left")
    df_out.rename(columns={"Stat_2026": "Segment_2026", "Stat_2025": "Segment_2025"}, inplace=True)

    df_out["New_Partner_2026"] = np.where(df_out["Segment_2026"] == "New/Revival", "Yes", "No")
    df_out["New_Partner_2025"] = np.where(df_out["Segment_2025"] == "New/Revival", "Yes", "No")
    df_out["New_Geo_2026"] = np.where(df_out["Geo_City"].isin(new_geo_list), "Yes", "No")
    df_out["New_Geo_2025"] = np.where(df_out["Geo_City"].isin(new_geo_list_25), "Yes", "No")

    df_out["SSP"] = df_out["Branch"].map(BRANCH_TO_SSP).fillna("Unmapped")
    df_out["SSP_Region"] = df_out["Branch"].map(BRANCH_TO_SSP_REGION).fillna("")

    if not df_backlog.empty and "SoldToPartyID" in df_backlog.columns and "Gross Sales" in df_backlog.columns:
        backlog_by_partner = df_backlog.groupby("SoldToPartyID")["Gross Sales"].sum().rename("Open Backlog")
        df_out = df_out.merge(backlog_by_partner, left_on="SoldToPartyID", right_index=True, how="left")
        df_out["Open Backlog"] = df_out["Open Backlog"].fillna(0)

    if not df_ob_26.empty and "SoldToPartyID" in df_ob_26.columns and "Gross Sales" in df_ob_26.columns:
        ob_by_partner = df_ob_26.groupby("SoldToPartyID")["Gross Sales"].sum().rename("Open Order Book")
        df_out = df_out.merge(ob_by_partner, left_on="SoldToPartyID", right_index=True, how="left")
        df_out["Open Order Book"] = df_out["Open Order Book"].fillna(0)

    front_cols = ["SoldToPartyID", "Patner", "GSTN", "Segment_2026", "Segment_2025",
                  "New_Partner_2026", "New_Partner_2025", "New_Geo_2026", "New_Geo_2025",
                  "SSP", "SSP_Region", "ASM", "Branch", "Zone", "City", "Tier_Bucket", kro_col]
    tail_cols = [c for c in ("Open Backlog", "Open Order Book") if c in df_out.columns]
    month_and_total_cols = [c for c in df_out.columns if c not in front_cols and c not in tail_cols and c != "Geo_City"]
    df_out = df_out[front_cols + month_and_total_cols + tail_cols]

    df_w = partner_sheet(df_out)          # display copy; df_out is returned unchanged
    df_w.to_excel(writer, sheet_name="Partner_Master_Detail", index=False)
    ws = writer.sheets["Partner_Master_Detail"]
    flag_or_text_cols = {"New_Partner_2026", "New_Partner_2025", "New_Geo_2026", "New_Geo_2025",
                          "Segment_2026", "Segment_2025", "SSP", "SSP_Region"}
    for c, col in enumerate(df_w.columns):
        ws.write(0, c, col, f["hdr"])
        if col in (f"{TARGET_YEAR} Total", "Open Backlog", "Open Order Book"):
            ws.set_column(c, c, 16, f["tot_det"])
        elif col in flag_or_text_cols:
            ws.set_column(c, c, 16, f["txt_det"])
        elif col in MONTHS_ORDER:
            ws.set_column(c, c, 12, f["num_det"])
        else:
            ws.set_column(c, c, 20, f["txt_det"])

    ws.autofilter(0, 0, len(df_w), len(df_w.columns) - 1)
    ws.freeze_panes(1, partner_freeze(df_w))
    ws.set_tab_color(THEME["tab_tracker"])
    return df_out


# =====================================================================
# 10d. INSIGHTS SUMMARY — a SEPARATE, slim workbook for the Sales /
#      Strategy / Offer team call. Reuses everything already computed
#      above (df_final, df_retail, aggregate_periods, SSP_MAP, etc.) so
#      there's no re-fetch — this just re-packages it into a
#      presentation-ready shape.
# =====================================================================
INSIGHTS_OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"Partner_Insights_Summary_{TARGET_YEAR}.xlsx")


def generate_insights_summary_workbook():
    """
    Writes a standalone Insights_Summary_<...>.xlsx with three sheets:
      - Exec_Summary: headline stat cards (the numbers you'd read out loud)
      - Talking_Points: each insight + why it matters + which team owns the follow-up
      - Supporting_Data: the numeric backup, condensed to one screen per topic
    This is deliberately NOT a raw data dump — it's built to be presented from
    directly, or used to refresh the sample PPT deck with live numbers.
    """
    with report_writer(INSIGHTS_OUTPUT_PATH) as iw:
        iwb = iw.book
        ifmt = iw.fmt
        stat_fmt = ifmt["stat_value"]
        label_fmt = ifmt["stat_label"]
        note_fmt = ifmt["note"]

        # --- Headline numbers, computed once and reused across sheets ---
        df_new_ytd = df_active[(df_active["Year"] == TARGET_YEAR) & (df_active["InvoiceDate"] <= yesterday) & (df_active["Stat_2026"] == "New/Revival")]
        new_partners_n = df_new_ytd["SoldToPartyID"].nunique()
        new_sales_mn = df_new_ytd["Gross Sales"].sum() / 1e6
        total_ytd_mn = df_active[(df_active["Year"] == TARGET_YEAR) & (df_active["InvoiceDate"] <= yesterday)]["Gross Sales"].sum() / 1e6
        new_pct = new_sales_mn / total_ytd_mn if total_ytd_mn else 0
        kro_split = df_new_ytd.groupby(kro_col).agg(P=("SoldToPartyID", "nunique"), S=("Gross Sales", lambda x: x.sum() / 1e6))
        rd_p, rd_s = (int(kro_split.loc["RD", "P"]), kro_split.loc["RD", "S"]) if "RD" in kro_split.index else (0, 0)
        kro_p, kro_s = (int(kro_split.loc["KRO", "P"]), kro_split.loc["KRO", "S"]) if "KRO" in kro_split.index else (0, 0)
        rd_avg = rd_s / rd_p if rd_p else 0
        kro_avg = kro_s / kro_p if kro_p else 0

        region_new = aggregate_periods("Zone", new_only=True)
        region_new["Ach%"] = region_new["S26Y"] / (df_region_target[df_region_target["Year"] == TARGET_YEAR].groupby("Zone")["Target"].sum() / 1e6)
        best_region = region_new["Ach%"].idxmax() if not region_new.empty else "N/A"
        worst_region = region_new["Ach%"].idxmin() if not region_new.empty else "N/A"

        helios_new = write_helios_region_pivot  # reused below, not called here
        df_helios_src = df_retail.merge(df_material_master[["Material", "Family Actual"]], left_on="MaterialCode", right_on="Material", how="left")
        zone_map = df_final[["SoldToPartyID", "Zone", "Stat_2026"]].drop_duplicates("SoldToPartyID")
        df_helios_src = df_helios_src.merge(zone_map, on="SoldToPartyID", how="left")
        df_helios_src["Helios_Display"] = np.where(
            (df_helios_src["HeliosCode"] == "HSCOS") & (df_helios_src["Family Actual"].astype(str).str.upper() == "WISER"),
            "Wiser", np.where(df_helios_src["HeliosCode"] == "HSCOS", "HSCOS Others", df_helios_src["HeliosCode"]),
        )
        new_product_mix = df_helios_src[(df_helios_src["Year"] == TARGET_YEAR) & (df_helios_src["Stat_2026"] == "New/Revival")].groupby("Helios_Display")["Gross Sales"].sum() / 1e6
        top_product = new_product_mix.idxmax() if not new_product_mix.empty else "N/A"

        geo_data = df_final[(df_final["Geo_City"].isin(new_geo_list)) & (df_final["Year"] == TARGET_YEAR) & (df_final["InvoiceDate"] <= yesterday)]
        geo_sales = geo_data.groupby("Geo_City")["Gross Sales"].sum() / 1e6
        top_new_city = geo_sales.idxmax() if not geo_sales.empty else "N/A"

        # --- Sheet 1: Exec Summary (stat cards) ---
        ws1 = iwb.add_worksheet("Exec_Summary")
        ws1.set_tab_color(THEME["tab_summary"])
        ws1.write(0, 0, f"Retail Partner Performance — Insights Summary ({yesterday.strftime('%d-%b-%Y')} YTD)", ifmt["title"])
        cards = [
            ("New Partners YTD", f"{new_partners_n:,}"),
            ("New Partner Sales", f"₹{new_sales_mn:,.1f} Mn"),
            ("% of Total BU Sales", f"{new_pct:.1%}"),
            ("RD : KRO Sales Split", f"₹{rd_s:,.1f}M : ₹{kro_s:,.1f}M"),
            ("Avg Sales / RD Partner", f"₹{rd_avg:,.2f} Mn"),
            ("Avg Sales / KRO Partner", f"₹{kro_avg:,.2f} Mn"),
            ("Best-performing Region", f"{best_region} ({region_new.loc[best_region, 'Ach%']:.0%} of target)" if best_region != "N/A" else "N/A"),
            ("Lagging Region", f"{worst_region} ({region_new.loc[worst_region, 'Ach%']:.0%} of target)" if worst_region != "N/A" else "N/A"),
            ("Top Product — New Partners", top_product),
            ("Top New Geography", top_new_city),
        ]
        for i, (label, value) in enumerate(cards):
            r, c = 2 + (i // 3) * 3, (i % 3) * 3
            ws1.merge_range(r, c, r, c + 1, label, label_fmt)
            ws1.merge_range(r + 1, c, r + 1, c + 1, value, stat_fmt)
        ws1.set_column("A:I", 16)
        ws1.write(2 + ((len(cards) - 1) // 3) * 3 + 3, 0, f"Source: Partner_Analysis_{TARGET_YEAR}.xlsx, same run.", note_fmt)

        # --- Sheet 2: Talking Points ---
        ws2 = iwb.add_worksheet("Talking_Points")
        ws2.set_tab_color(THEME["tab_detail"])
        headers = ["#", "Insight", "Key Number", "Why It Matters", "Primary Owner"]
        for c, h in enumerate(headers):
            ws2.write(0, c, h, ifmt["hdr"])

        talking_points = [
            ("New partner contribution is still small relative to BU target",
             f"{new_pct:.1%} of YTD sales from {new_partners_n} new partners",
             "Growth engine is working but hasn't scaled — worth discussing whether new-partner acquisition pace needs to increase to hit the annual target.",
             "Sales + Strategy"),
            ("KRO partners dominate by count, RD partners dominate by value per partner",
             f"RD avg ₹{rd_avg:.2f}Mn/partner vs KRO avg ₹{kro_avg:.2f}Mn/partner",
             "If RD partners consistently outperform per-partner, channel mix strategy (where to invest onboarding effort) may need rebalancing.",
             "Strategy"),
            (f"{worst_region} is the biggest regional gap to target",
             f"{worst_region}: {region_new.loc[worst_region, 'Ach%']:.0%} of target achieved" if worst_region != "N/A" else "N/A",
             "Identifies where sales leadership should focus incremental push — either a resourcing issue or a target-setting issue worth investigating.",
             "Sales"),
            (f"{best_region} is significantly ahead of target",
             f"{best_region}: {region_new.loc[best_region, 'Ach%']:.0%} of target achieved" if best_region != "N/A" else "N/A",
             "Worth understanding what's working there — could be a playbook to replicate in lagging regions.",
             "Sales + Strategy"),
            (f"{top_product} leads new-partner product mix",
             f"₹{new_product_mix.max():.1f} Mn from {top_product}" if not new_product_mix.empty else "N/A",
             "Tells the Offer team which focus-product incentives are actually landing with new partners, and which (e.g. Wiser) may be under-promoted.",
             "Offer"),
            (f"{top_new_city} is the strongest new-geography entry",
             f"Top new city by YTD sales" if top_new_city != "N/A" else "N/A",
             "A candidate case study for the geo-expansion playbook — worth understanding what made this city work.",
             "Strategy"),
        ]
        for i, (insight, num, why, owner) in enumerate(talking_points, start=1):
            r = i
            ws2.write(r, 0, i, ifmt["int"])
            ws2.write(r, 1, insight, ifmt["txt_det"])
            ws2.write(r, 2, num, ifmt["txt_det"])
            ws2.write(r, 3, why, ifmt["txt_det"])
            ws2.write(r, 4, owner, ifmt["txt_det"])
        ws2.set_column("A:A", 4)
        ws2.set_column("B:B", 38)
        ws2.set_column("C:C", 28)
        ws2.set_column("D:D", 48)
        ws2.set_column("E:E", 16)
        ws2.freeze_panes(1, 0)

        # --- Sheet 3: Supporting Data (condensed backup tables) ---
        ws3 = iwb.add_worksheet("Supporting_Data")
        ws3.set_tab_color(THEME["series"][5])
        nr = write_kpi_table(ws3, iwb, ifmt, 0, region_new[["P25F", "S25F", "P25Y", "S25Y", "P26Y", "S26Y"]], "New Partners — Regional Performance", "Region")
        ws3.write(nr, 0, "New Partner Product Mix (Rs Mn)", ifmt["subtitle"])
        r = nr + 1
        ws3.write(r, 0, "Product", ifmt["hdr"]); ws3.write(r, 1, "Sales (Mn)", ifmt["hdr"])
        for i, (prod, val) in enumerate(new_product_mix.sort_values(ascending=False).items(), start=1):
            ws3.write(r + i, 0, prod, ifmt["txt_det"])
            ws3.write(r + i, 1, val, ifmt["dec"])
        ws3.set_column("A:A", 30)
        ws3.set_column("B:L", 14)

    print(f"Insights Summary saved to: {INSIGHTS_OUTPUT_PATH}")


# =====================================================================
# 10d. CUSTOMER RETENTION
# =====================================================================
def write_customer_retention_sheet(writer, wb, f):
    """
    Of partners active in year Y, how many are still active (any Gross Sales)
    in year Y+1? Computed straight off sales_matrix (already built above).
    Retained / Churned / Reactivated, by Tier and by Zone.
    """
    ws = wb.add_worksheet("Customer_Retention")
    ws.set_tab_color(THEME["series"][4])

    partner_zone = df_final[["SoldToPartyID", "Zone", "Tier_Bucket"]].drop_duplicates("SoldToPartyID").set_index("SoldToPartyID")
    sm = sales_matrix.copy()
    sm.index = sm.index.astype(str)

    def retention_block(prior_year, curr_year):
        prior_active = sm.get(prior_year, pd.Series(dtype=float)) > 0
        curr_active = sm.get(curr_year, pd.Series(dtype=float)) > 0
        prior_ids = set(sm.index[prior_active])
        curr_ids = set(sm.index[curr_active])
        retained = prior_ids & curr_ids
        churned = prior_ids - curr_ids
        reactivated = curr_ids - prior_ids
        return prior_ids, curr_ids, retained, churned, reactivated

    prior_ids, curr_ids, retained, churned, reactivated = retention_block(PREV_YEAR, TARGET_YEAR)
    retention_rate = len(retained) / len(prior_ids) if prior_ids else 0

    ws.write(0, 0, f"Customer Retention — {PREV_YEAR} Active Base into {TARGET_YEAR}", f["title"])
    stat_fmt = f["stat_value"]
    lbl_fmt = f["stat_label"]
    cards = [
        (f"{PREV_YEAR} Active Partners", f"{len(prior_ids):,}"),
        ("Retained", f"{len(retained):,}"),
        ("Churned (lost)", f"{len(churned):,}"),
        ("Reactivated (new to 2026 base)", f"{len(reactivated):,}"),
        ("Retention Rate", f"{retention_rate:.1%}"),
    ]
    for i, (label, val) in enumerate(cards):
        c = i * 2
        ws.merge_range(2, c, 2, c + 1, label, lbl_fmt)
        ws.merge_range(3, c, 3, c + 1, val, stat_fmt)
    ws.set_column("A:J", 16)

    # Breakdown by Zone and Tier for the churned group — where retention is weakest
    churned_df = partner_zone.reindex(list(churned)).reset_index().rename(columns={"index": "SoldToPartyID"})
    for col_name, sheet_row in [("Zone", 6), ("Tier_Bucket", 6 + 9)]:
        ws.write(sheet_row, 0, f"Churned Partners by {col_name}", f["subtitle"])
        ws.write(sheet_row + 1, 0, col_name, f["hdr"]); ws.write(sheet_row + 1, 1, "Churned Count", f["hdr"])
        counts = churned_df[col_name].value_counts()
        for i, (k, v) in enumerate(counts.items(), start=1):
            ws.write(sheet_row + 1 + i, 0, k, f["txt_det"])
            ws.write(sheet_row + 1 + i, 1, int(v), f["int"])

    ws.write(24, 0, f"Note: 'Churned' = had Gross Sales > 0 in {PREV_YEAR}, zero in {TARGET_YEAR} YTD "
                     f"(not full-year comparable — {TARGET_YEAR} is still in progress). Revisit after year close.",
             f["note"])
    return {"retention_rate": retention_rate, "retained": len(retained), "churned": len(churned)}


# =====================================================================
# 10e. ACTIVE PARTNER STRENGTH & YEARLY SCHEME ACHIEVEMENT
# =====================================================================
def write_scheme_achievement_sheet(writer, wb, f, policy_tracker_df):
    """
    - Active partner count trend across years (strength of the base)
    - How many partners are tracking to hit 100% / 108% of their Annual
      Growth Incentive target (per Commercial Policy 2026, p.5) based on
      the Policy Tracker's run-rate math
    """
    ws = wb.add_worksheet("Scheme_Achievement")
    ws.set_tab_color(THEME["tab_tracker"])
    ws.write(0, 0, "Active Partner Strength & Yearly Scheme (Annual Growth Incentive) Achievement", f["title"])

    # Active partner strength trend
    sm = sales_matrix.copy()
    years_present = sorted([y for y in sm.columns if isinstance(y, int)])
    ws.write(2, 0, "Active Partner Count by Year", f["subtitle"])
    ws.write(3, 0, "Year", f["hdr"]); ws.write(3, 1, "Active Partners", f["hdr"])
    for i, yr in enumerate(years_present, start=1):
        ws.write(3 + i, 0, yr, f["txt_det"])
        ws.write(3 + i, 1, int((sm[yr] > 0).sum()), f["int"])

    # Scheme achievement: pace-to-target using elapsed-year fraction
    days_elapsed = (yesterday - datetime(TARGET_YEAR, 1, 1)).days + 1
    year_fraction = days_elapsed / 365
    ptd = policy_tracker_df.copy()
    ptd["Pace_to_100pct"] = np.where(
        ptd["2026 Assigned Target"] > 0,
        ptd["YTD_2026"] / (ptd["2026 Assigned Target"] * year_fraction),
        np.nan,
    )
    on_track_100 = (ptd["Pace_to_100pct"] >= 1.0).sum()
    on_track_108 = (ptd["Pace_to_100pct"] >= 1.08).sum()
    behind_pace = (ptd["Pace_to_100pct"] < 1.0).sum()
    total_targeted = ptd["2026 Assigned Target"].gt(0).sum()

    r0 = 4 + len(years_present) + 2
    ws.write(r0, 0, f"Annual Growth Incentive Pace (as of {yesterday.strftime('%d-%b-%Y')}, {year_fraction:.0%} of year elapsed)", f["subtitle"])
    stat_fmt = f["stat_value"]
    lbl_fmt = f["stat_label"]
    cards = [
        ("Partners with a Target", f"{total_targeted:,}"),
        ("On Pace for \u2265100%", f"{on_track_100:,}"),
        ("On Pace for \u2265108%", f"{on_track_108:,}"),
        ("Behind Pace (<100%)", f"{behind_pace:,}"),
    ]
    for i, (label, val) in enumerate(cards):
        c = i * 2
        ws.merge_range(r0 + 1, c, r0 + 1, c + 1, label, lbl_fmt)
        ws.merge_range(r0 + 2, c, r0 + 2, c + 1, val, stat_fmt)

    ws.write(r0 + 4, 0, "Note: pace-to-target is a straight-line estimate (YTD actual vs. target x % of year elapsed) — "
                         "it does not account for seasonality, so treat it as a directional flag, not a final payout call.",
             f["note"])
    ws.set_column("A:H", 16)


# =====================================================================
# 10f. CROSS-SELL OPPORTUNITY
# =====================================================================
def write_cross_sell_sheet(writer, wb, f):
    """
    For each active 2026 partner, which product families (Helios display
    codes) they have ZERO sales in. Ranked by the partner's total 2026
    sales, so the biggest partners with the biggest gaps surface first —
    those are the highest-value cross-sell targets.
    """
    df_src = df_retail.merge(df_material_master[["Material", "Family Actual"]], left_on="MaterialCode", right_on="Material", how="left")
    df_src["Helios_Display"] = np.where(
        (df_src["HeliosCode"] == "HSCOS") & (df_src["Family Actual"].astype(str).str.upper() == "WISER"),
        "Wiser", np.where(df_src["HeliosCode"] == "HSCOS", "HSCOS Others", df_src["HeliosCode"]),
    )
    df_2026 = df_src[df_src["Year"] == TARGET_YEAR]
    pivot = df_2026.pivot_table(index="SoldToPartyID", columns="Helios_Display", values="Gross Sales", aggfunc="sum", fill_value=0)
    product_cols = list(pivot.columns)
    pivot["Total Sales"] = pivot.sum(axis=1)
    pivot = pivot.merge(Partner_Master[["Sold-to Party", "Patner", "GSTN", "Zone", "Branch", "City", kro_col]].drop_duplicates("Sold-to Party"),
                         left_index=True, right_on="Sold-to Party", how="left")

    rows = []
    for _, row in pivot.iterrows():
        missing = [p for p in product_cols if row[p] == 0]
        if missing and row["Total Sales"] > 0:
            rows.append({
                "SoldToPartyID": row["Sold-to Party"], "Partner Name": row["Patner"],
                "GSTN": row["GSTN"], kro_col: row[kro_col], "Branch": row["Branch"],
                "Region": row["Zone"], "City": row["City"],
                "Total 2026 Sales (Rs)": row["Total Sales"],
                "Products Currently Bought": ", ".join([p for p in product_cols if row[p] > 0]) or "None",
                "Missing Product Families": ", ".join(missing),
                "Missing Count": len(missing),
            })
    gap_df = pd.DataFrame(rows).sort_values("Total 2026 Sales (Rs)", ascending=False)

    gap_df.to_excel(writer, sheet_name="Cross_Sell_Opportunity", index=False)
    ws = writer.sheets["Cross_Sell_Opportunity"]
    for c, col in enumerate(gap_df.columns):
        ws.write(0, c, col, f["hdr"])
        width = 32 if col in ("Products Currently Bought", "Missing Product Families") else 18
        ws.set_column(c, c, width, f["num_det"] if col == "Total 2026 Sales (Rs)" else f["txt_det"])
    ws.autofilter(0, 0, len(gap_df), len(gap_df.columns) - 1)
    ws.freeze_panes(1, partner_freeze(gap_df))
    ws.set_tab_color(THEME["series"][1])
    return gap_df


# =====================================================================
# 10g. PAVILION PERFORMANCE — separate source, keyed by GSTN (see note at
#      PAVILION_FILE_PATH above). Built entirely from the raw partner-level
#      sheet in that workbook rather than parsing the free-form dashboard
#      blocks, since those mix merged cells and side-by-side region tables
#      that don't parse reliably.
# =====================================================================
def load_pavilion_raw():
    xls = pd.ExcelFile(PAVILION_FILE_PATH)
    raw_sheet = PAVILION_RAW_SHEET_NAME or next((s for s in xls.sheet_names if s.strip().lower().startswith("pavilion as on")), None)
    if raw_sheet is None:
        raise ValueError(f"Could not find a 'Pavilion as on ...' sheet in {PAVILION_FILE_PATH}. "
                          f"Sheets found: {xls.sheet_names}")
    df = pd.read_excel(PAVILION_FILE_PATH, sheet_name=raw_sheet, header=2)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(how="all")
    for c in ("KRO/RO", "Retailer App T&C Acceptance", "2026 Classification", "Zone/Region", "Branch"):
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
    for c in ("Growth %", "Minimum Target Ach %", "Wiser Slab Ach %"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def write_pavilion_performance_sheets(writer, wb, f):
    if not PAVILION_FILE_PATH:
        return
    try:
        df_pav = load_pavilion_raw()
    except Exception as e:
        print(f"WARNING: could not load Pavilion data ({e}). Skipping Pavilion sheets.")
        return

    # --- Overview ---
    ws = wb.add_worksheet("Pavilion_Overview")
    ws.set_tab_color(THEME["series"][2])
    ws.write(0, 0, "Pavilion Performance Overview", f["title"])
    total_pav = len(df_pav)
    slab_counts = df_pav["2026 Classification"].value_counts()
    kro_ro_counts = df_pav["KRO/RO"].value_counts()
    apptc_counts = df_pav["Retailer App T&C Acceptance"].value_counts()
    apptc_rate = apptc_counts.get("Accepted", 0) / total_pav if total_pav else 0
    wiser_achievers = (df_pav["Wiser Slab Ach %"] >= 1).sum() if "Wiser Slab Ach %" in df_pav.columns else 0
    target_achievers = (df_pav["Minimum Target Ach %"] >= 1).sum() if "Minimum Target Ach %" in df_pav.columns else 0
    avg_growth = df_pav["Growth %"][df_pav["Growth %"].abs() < 5].mean() if "Growth %" in df_pav.columns else np.nan  # exclude extreme outlier growth (e.g. from-zero base)

    stat_fmt = f["stat_value"]
    lbl_fmt = f["stat_label"]
    cards = [
        ("Total Active Pavilions", f"{total_pav:,}"),
        ("KRO / RO Split", f"{kro_ro_counts.get('KRO', 0):,} / {kro_ro_counts.get('RO', 0):,}"),
        ("Retailer App T&C Accepted", f"{apptc_rate:.1%}"),
        ("Hit Min. Target (\u2265100%)", f"{target_achievers:,} ({target_achievers/total_pav:.0%})"),
        ("Hit Wiser Slab (\u2265100%)", f"{wiser_achievers:,} ({wiser_achievers/total_pav:.0%})"),
        ("Avg YoY Growth (ex-outliers)", f"{avg_growth:.1%}" if pd.notna(avg_growth) else "N/A"),
    ]
    for i, (label, val) in enumerate(cards):
        r, c = 2 + (i // 3) * 3, (i % 3) * 4
        ws.merge_range(r, c, r, c + 2, label, lbl_fmt)
        ws.merge_range(r + 1, c, r + 1, c + 2, val, stat_fmt)
    ws.write(2 + 1 * 3 + 3, 0, "Slab Distribution (2026 Classification)", f["subtitle"])
    r = 2 + 1 * 3 + 4
    ws.write(r, 0, "Slab", f["hdr"]); ws.write(r, 1, "Count", f["hdr"])
    for i, (k, v) in enumerate(slab_counts.items(), start=1):
        ws.write(r + i, 0, k, f["txt_det"]); ws.write(r + i, 1, int(v), f["int"])
    ws.set_column("A:L", 16)

    # --- Region/Branch rollup ---
    ws2 = wb.add_worksheet("Pavilion_Region_Branch")
    ws2.set_tab_color(THEME["series"][2])
    ws2.write(0, 0, "Pavilion Performance by Zone / Branch", f["title"])
    df_pav["App_TC_Accepted_Flag"] = (df_pav["Retailer App T&C Acceptance"] == "Accepted").astype(int)
    df_pav["Wiser_Achiever_Flag"] = (df_pav.get("Wiser Slab Ach %", 0) >= 1).astype(int)
    df_pav["Target_Achiever_Flag"] = (df_pav.get("Minimum Target Ach %", 0) >= 1).astype(int)
    rollup = df_pav.groupby(["Zone/Region", "Branch"]).agg(
        Pavilions=("Name of RO/KRO", "count"),
        Avg_Growth_Pct=("Growth %", lambda x: x[x.abs() < 5].mean()),
        Target_Achievers=("Target_Achiever_Flag", "sum"),
        Wiser_Achievers=("Wiser_Achiever_Flag", "sum"),
        App_TC_Accept_Rate=("App_TC_Accepted_Flag", "mean"),
    ).reset_index().sort_values("Pavilions", ascending=False)
    rollup.to_excel(writer, sheet_name="Pavilion_Region_Branch", startrow=2, index=False)
    ws2r = writer.sheets["Pavilion_Region_Branch"]
    for c, col in enumerate(rollup.columns):
        ws2r.write(2, c, col, f["hdr"])
        fmt = f["pct"] if "Rate" in col or "Pct" in col else (f["int"] if col in ("Pavilions", "Target_Achievers", "Wiser_Achievers") else f["txt_det"])
        ws2r.set_column(c, c, 18, fmt)
    ws2r.freeze_panes(3, 2)

    # --- Raw, flagged, filterable ---
    df_pav_out = df_pav.drop(columns=["App_TC_Accepted_Flag"], errors="ignore")
    df_pav_out.to_excel(writer, sheet_name="Pavilion_Raw", index=False)
    ws3 = writer.sheets["Pavilion_Raw"]
    for c, col in enumerate(df_pav_out.columns):
        ws3.write(0, c, col, f["hdr"])
        ws3.set_column(c, c, 16, f["txt_det"])
    ws3.autofilter(0, 0, len(df_pav_out), len(df_pav_out.columns) - 1)
    ws3.freeze_panes(1, 2)
    ws3.set_tab_color(THEME["tab_data"])

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 5.4 PARTNER ANALYSIS — build the workbooks
#
#   Partner_Analysis_<year>.xlsx          summary, new partners, MoM, lifetime,
#                                         detail, policy & zone trackers,
#                                         product mix + extended sheets
#   New_GEO_<year>_YTD_<Mon>.xlsx         new-partner / geo-expansion pack
#   Partner_Insights_Summary_<year>.xlsx  talking points for the team call
# (all in OUTPUT_DIR)
# ════════════════════════════════════════════════════════════════════
PARTNER_ANALYSIS_PATH = os.path.join(OUTPUT_DIR, f"Partner_Analysis_{TARGET_YEAR}.xlsx")
GEO_OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"New_GEO_{TARGET_YEAR}_YTD_{MONTH_ABBR}.xlsx")


def _new_partner_summary_sheet(wb, fmt):
    ws = wb.add_worksheet("New_Partner_Summary")
    ws.set_tab_color(THEME["tab_summary"])
    r = write_new_partner_dashboard(ws, wb, fmt, start_row=0)
    r = write_kpi_table(ws, wb, fmt, r, aggregate_periods(None, is_segment=True), "Partner Segment Analysis (All Partners)", "Segment")
    r = write_kpi_table(ws, wb, fmt, r, aggregate_periods("Zone", new_only=True), "NEW PARTNERS: Regional Performance", "Region")
    r = write_kpi_table(ws, wb, fmt, r, aggregate_periods("Tier_Bucket", new_only=True), "NEW PARTNERS: Tier Analysis", "Tier Category")
    write_kpi_table(ws, wb, fmt, r, aggregate_periods(kro_col, new_only=True), "NEW PARTNERS: Partner Type Split", "Type")
    ws.set_column("A:A", 36)
    ws.set_column("B:L", 12)


def _total_column(writer, sheet, df, col):
    i = list(df.columns).index(col)
    writer.sheets[sheet].set_column(i, i, 16, writer.fmt["tot_inr"])


def report_partner_analysis():
    prepare_partner_analysis()
    load_open_orders()
    tracker = None

    with report_writer(PARTNER_ANALYSIS_PATH) as writer:
        wb, fmt = writer.book, writer.fmt

        ws_sum = wb.add_worksheet("Summary")
        ws_sum.set_tab_color(THEME["tab_summary"])
        r = write_kpi_table(ws_sum, wb, fmt, 0, aggregate_periods("Zone"), "Regional Partner Performance", "Region", with_avg=False)
        r = write_kpi_table(ws_sum, wb, fmt, r, aggregate_periods("Tier_Bucket"), "Tier Category Analysis", "Tier Category", with_avg=False)
        r = write_kpi_table(ws_sum, wb, fmt, r, aggregate_periods(kro_col), "Partner Type Split", "Type", with_avg=False)
        r = write_kpi_table(ws_sum, wb, fmt, r, aggregate_periods(None, is_segment=True), "Partner Segment Analysis", "Segment", with_avg=False)
        write_new_geo_table(ws_sum, wb, fmt, r)
        ws_sum.set_column("A:A", 35)
        ws_sum.set_column("B:J", 16)

        _new_partner_summary_sheet(wb, fmt)

        mom = _fill_ids(df_final[df_final["Year"] == TARGET_YEAR], kro=True).pivot_table(
            index=["SoldToPartyID", "Patner", "GSTN", kro_col, "Branch", "Zone", "City", "Stat_2026", "ASM", "Tier_Bucket"],
            columns="Month", values="Gross Sales", aggfunc="sum", fill_value=0, observed=True)
        mom["Partner Total"] = mom.sum(axis=1).round(0)
        mom = partner_sheet(mom.reset_index().rename(columns={"Stat_2026": "Segment"}))
        write_table(writer, mom, f"MoM_{TARGET_YEAR}", freeze_cols=partner_freeze(mom), tab=THEME["tab_detail"])
        _total_column(writer, f"MoM_{TARGET_YEAR}", mom, "Partner Total")

        life = df_final.groupby(["SoldToPartyID", "Year"])["Gross Sales"].sum().unstack(fill_value=0)
        life["Lifetime Total"] = life.sum(axis=1).round(0)
        crit = ["Sold-to Party", "Patner", "GSTN", kro_col, "Branch", "Zone", "City", "ASM", "Tier_Bucket"]
        life_df = (Partner_Master[crit].merge(partner_status[["Stat_2026"]], left_on="Sold-to Party", right_index=True, how="left")
                   .merge(life.reset_index(), left_on="Sold-to Party", right_on="SoldToPartyID", how="left")
                   .drop(columns=["SoldToPartyID"], errors="ignore").rename(columns={"Stat_2026": "Segment"}))
        life_df = partner_sheet(life_df)
        write_table(writer, life_df, "Lifetime_Sales", freeze_cols=partner_freeze(life_df), tab=THEME["tab_detail"])
        _total_column(writer, "Lifetime_Sales", life_df, "Lifetime Total")

        write_detail(writer, wb, fmt, df_final[df_final["Stat_2026"] == "New/Revival"], "New_Partner_Detail_2026", 2026)
        ly_new = (df_final["Stat_2025"] == "New/Revival") & (df_final["Year"] == 2025) & (df_final["InvoiceDate"] <= last_year_yesterday)
        write_detail(writer, wb, fmt, df_final[ly_new].copy(), "New_Partner_Detail_2025_YTD", 2025)
        write_detail(writer, wb, fmt, df_final[df_final["Geo_City"].isin(new_geo_list)], "Geo_Expansion_Detail_2026", 2026)
        write_detail(writer, wb, fmt, df_final[df_final["Geo_City"].isin(new_geo_list_25)], "Geo_Expansion_Detail_2025", 2025)

        tracker = generate_policy_handholding_tracker(writer, wb, fmt)
        generate_zone_target_tracker(writer, wb, fmt)

        ws_helios = wb.add_worksheet("Helios_Region_Analysis")
        ws_helios.set_tab_color(THEME["series"][5])
        write_helios_region_pivot(ws_helios, wb, fmt, 0, new_partners_only=False, title="Product-wise Break-up — All Partners")
        ws_helios.set_column("A:A", 25)
        ws_helios.set_column("B:H", 14)

        # Extended sheets (5.3)
        write_partner_master_detail(writer, wb, fmt)
        write_ssp_performance_sheet(writer, wb, fmt)
        write_customer_retention_sheet(writer, wb, fmt)
        write_scheme_achievement_sheet(writer, wb, fmt, tracker)
        write_cross_sell_sheet(writer, wb, fmt)
        write_pavilion_performance_sheets(writer, wb, fmt)
    print(f"   💾 {PARTNER_ANALYSIS_PATH}")

    with report_writer(GEO_OUTPUT_PATH) as geo_writer:
        gwb, gfmt = geo_writer.book, geo_writer.fmt
        _new_partner_summary_sheet(gwb, gfmt)
        write_geo_expansion_sheet(gwb, gfmt, tracker, kro_col)
        write_policy_tracker_sheet(geo_writer, gwb, gfmt, tracker)
        write_sheet1(gwb, gfmt, aggregate_periods("Zone", new_only=True))
    print(f"   💾 {GEO_OUTPUT_PATH}")

    generate_insights_summary_workbook()
    return f"{PARTNER_ANALYSIS_PATH}; {GEO_OUTPUT_PATH}; {INSIGHTS_OUTPUT_PATH}"


run_report("partner_analysis", report_partner_analysis)

## 6 · SSP performance

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 6.1 SSP PERFORMANCE — overall + one file per SSP (SharePoint)
#
# Reads this year's and last year's Retail YTD files (3.1), targets from
# QTOD/ATOD, SSP → branch mapping from SSP_MAP (cell 0.3).
# Sheets: Performance · Partner Performance · Raw Sales / OB / Backlog.
# ════════════════════════════════════════════════════════════════════
def report_ssp_performance():
    import xlsxwriter

    TARGET_YEAR_SSP = TARGET_YEAR
    yesterday = YESTERDAY
    curr_year, curr_month = yesterday.year, yesterday.month
    curr_q_start_month = ((curr_month - 1) // 3) * 3 + 1
    REPORT_PERIOD = f"Reporting Period: 01-Jan-{curr_year} to {yesterday.strftime('%d-%b-%Y')}"
    month_names = [datetime(curr_year, m, 1).strftime('%b').upper() for m in range(1, curr_month + 1)]
    FINAL_REPORTS_PATH = PATHS["retail_ytd"]
    OVERALL_REPORTS_PATH = PATHS["ssp_reports"]
    INDIVIDUAL_PATH = PATHS["ssp_individual"]
    OVERALL_FILENAME = f"Overall_SSP_Retail_Performance_{yesterday.strftime('%d-%b-%Y')}.xlsx"
    ID_LIKE_KEYWORDS = ["order", "id", "code", "party", "material", "reference", "doc", "invoice"]
    PCT_LIKE_KEYWORDS = ["rebate", "percent", "pct", "%"]

    # --- 3. DATA LOADING ---
    print("📡 Fetching source data...")
    sales_26_dict = pd.read_excel(sp.download(f"{FINAL_REPORTS_PATH}/Retail_Sales_OB_Backlog_data_{curr_year}.xlsx"), sheet_name=None)
    sales_25_dict = pd.read_excel(sp.download(f"{FINAL_REPORTS_PATH}/Retail_Sales_OB_Backlog_data_{curr_year-1}.xlsx"), sheet_name=None)

    df_pt = master("qtod_atod")
    df_sales_26 = sales_26_dict.get(f'Sales_{curr_year}')
    df_sales_25 = sales_25_dict.get(f'Sales_{curr_year-1}')
    df_ob_26 = sales_26_dict.get(f'OB_{curr_year}')
    df_backlog = sales_26_dict.get('Backlog_Overall')

    d_col_s26 = next(c for c in ['InvoiceDate', 'Invoice Date'] if c in df_sales_26.columns)
    d_col_s25 = next(c for c in ['InvoiceDate', 'Invoice Date'] if c in df_sales_25.columns)
    df_sales_26[d_col_s26] = pd.to_datetime(df_sales_26[d_col_s26])
    df_sales_25[d_col_s25] = pd.to_datetime(df_sales_25[d_col_s25])

    ssp_map = [{'R': s['Region'], 'S': s['SSP'], 'B': [s['Branch']], 'T': s['Annual_Target_Mn']} for s in SSP_MAP]

    # --- 4. GLOBAL PARTNER ATTRIBUTES (no sales figures here -- see note below) ---
    # Targets now come from "QTOD and ATOD file.xlsx" / sheet "QTOD - ATOD",
    # keyed by 'CC' (the Sold-to Party code) rather than the old Master Targets
    # File's 'Sold-to Party' column, and scoped to the current year via
    # 'Target Year' rather than a column literally named '2026'. 'Target 100%'
    # is the annual target (the old file's 'Tgt_Abs'/'2026 Target' equivalent) --
    # 'Target 108%' is already a derived stretch figure in this file, left
    # unused here since Partner Performance still computes its own Ach% off
    # the 100% target. Partner Name is no longer sourced from this file at
    # all -- see section 6.1, which takes it straight from the sales data's
    # SoldToPartyName instead, so a partner always shows the name used on
    # their own invoices rather than falling back to "Not in Partner Master".
    df_pt = df_pt[df_pt['Target Year'] == TARGET_YEAR_SSP].copy()
    df_pt['Tgt_Abs'] = pd.to_numeric(df_pt['Target 100%'].astype(str).str.replace(',', '').replace('-', '0'), errors='coerce').fillna(0)
    df_pt['Tgt_108'] = pd.to_numeric(df_pt['Target 108%'].astype(str).str.replace(',', '').replace('-', '0'), errors='coerce').fillna(0)
    df_pt['CC'] = df_pt['CC'].astype(str)

    df_partner_attrs = df_pt[['CC', 'Branch', 'Tgt_Abs', 'Tgt_108']].copy()
    df_partner_attrs.rename(columns={'CC': 'Sold-to Party', 'Tgt_Abs': '2026 Target', 'Tgt_108': '108% Target'}, inplace=True)


    # =====================================================================
    # 6. EXCEL GENERATION
    # =====================================================================
    def generate_formatted_report(target_ssps, filename, folder_path):
        output = io.BytesIO()
        workbook = xlsxwriter.Workbook(output, {'nan_inf_to_errors': True})
        workbook.formats[0].set_font_name(THEME['font'])
        workbook.formats[0].set_font_size(THEME['font_size'])
        f = base_formats(workbook)

        # ---- 6.1 Scope this report to its SSP(s)' branches ----
        target_list = ssp_map if target_ssps == "ALL" else [s for s in ssp_map if s['S'] == target_ssps]
        all_br = []
        for ssp in target_list:
            br = [ssp['B']] if isinstance(ssp['B'], str) else ssp['B']
            all_br.extend(br)
        report_title_suffix = "ALL SSPs — Overall" if target_ssps == "ALL" else target_ssps

        branch_sales_26 = df_sales_26[df_sales_26['Branch'].isin(all_br)].copy()
        branch_sales_25 = df_sales_25[df_sales_25['Branch'].isin(all_br)]
        branch_sales_26['SoldToPartyID'] = branch_sales_26['SoldToPartyID'].astype(str)

        # Partner Performance is now a straight pivot of df_sales_26 itself,
        # not the Partner Master with sales merged on afterward. Every
        # Sold-to Party on this tab comes directly from branch_sales_26, so
        # YTD Sales here sums to EXACTLY the same total as the Raw Sales tab
        # by construction -- there's no longer a gap to reconcile, so the old
        # "Other Unallocated Sales" row is gone entirely rather than hidden.
        # Branch, ASM, AND Partner Name are all taken from the sales data
        # itself (the branch a partner actually transacted under, and the
        # exact name on their own invoices) rather than any master file --
        # only Target still comes from df_pt/the QTOD-ATOD file, since the
        # sales data doesn't carry that.
        df_p_final = (
            branch_sales_26.groupby('SoldToPartyID')
            .agg(Branch=('Branch', 'first'), ASM=('ASM', 'first'), SoldToPartyName=('SoldToPartyName', 'first'))
            .reset_index()
            .rename(columns={'SoldToPartyName': 'Partner Name'})
        )

        target_map = df_partner_attrs[['Sold-to Party', '2026 Target', '108% Target']].drop_duplicates('Sold-to Party').copy()
        target_map['Sold-to Party'] = target_map['Sold-to Party'].astype(str)
        df_p_final = df_p_final.merge(target_map, left_on='SoldToPartyID', right_on='Sold-to Party', how='left')
        # A Sold-to Party with real sales but no entry in the QTOD-ATOD file
        # still shows up here -- labeled plainly rather than silently dropped
        # or folded into a reconciliation row.
        df_p_final['Sold-to Party'] = df_p_final['Sold-to Party'].fillna(df_p_final['SoldToPartyID'])
        df_p_final['Partner Name'] = df_p_final['Partner Name'].fillna(df_p_final['SoldToPartyID'])
        # GST / KRO-RD-PD / Region / City come from the Partner Metadata
        _pm = Partner_Metadata[["SoldToPartyID", "GSTN", "RD/KRO", "Region", "City"]].copy()
        _pm["SoldToPartyID"] = _pm["SoldToPartyID"].astype(str).str.split(".").str[0].str.strip()
        df_p_final = df_p_final.merge(_pm.drop_duplicates("SoldToPartyID"), on="SoldToPartyID", how="left")
        for _c in ("GSTN", "RD/KRO", "Region", "City"):
            df_p_final[_c] = df_p_final[_c].fillna("")
        df_p_final['2026 Target'] = df_p_final['2026 Target'].fillna(0)
        df_p_final['108% Target'] = df_p_final['108% Target'].fillna(0)
        df_p_final['ASM'] = df_p_final['ASM'].fillna('Unassigned')

        y26_scoped = (
            branch_sales_26[branch_sales_26[d_col_s26] <= yesterday]
            .groupby('SoldToPartyID')['Gross Sales'].sum().rename('YTD Sales')
        )
        df_p_final = df_p_final.merge(y26_scoped, on='SoldToPartyID', how='left').fillna({'YTD Sales': 0})

        for m in range(1, curr_month + 1):
            m_name = datetime(curr_year, m, 1).strftime('%b').upper()
            m_scoped = (
                branch_sales_26[branch_sales_26[d_col_s26].dt.month == m]
                .groupby('SoldToPartyID')['Gross Sales'].sum().rename(m_name)
            )
            df_p_final = df_p_final.merge(m_scoped, on='SoldToPartyID', how='left')
        df_p_final[month_names] = df_p_final[month_names].fillna(0)

        df_p_final['Ach%'] = (df_p_final['YTD Sales'] / df_p_final['2026 Target']).replace([float('inf'), -float('inf')], 0).fillna(0)
        df_p_final = df_p_final.sort_values(by='YTD Sales', ascending=False).reset_index(drop=True)


        # =================================================================
        # 6.2 PERFORMANCE SHEET (SSP x Branch summary -- unchanged logic,
        # formatting tidied to match the rest of the workbook)
        # =================================================================
        ws1 = workbook.add_worksheet('Performance')
        ws1.hide_gridlines(2)
        ws1.write('A1', 'SCHNEIDER ELECTRIC — SSP RETAIL PERFORMANCE', f['title'])
        ws1.write('A2', f"{report_title_suffix}  |  {REPORT_PERIOD}", f['note'])
        ws1.write('A3', "All values in Mn INR (Million Rupees) unless stated otherwise", f['note'])

        row_idx = 13
        t_y26, t_qt, t_annt, t_q_sales = 0, 0, 0, 0
        ach_col_idx = None

        for ssp in target_list:
            br = [ssp['B']] if isinstance(ssp['B'], str) else ssp['B']
            m_sales = [df_sales_26[(df_sales_26['Branch'].isin(br)) & (df_sales_26[d_col_s26].dt.month == m)]['Gross Sales'].sum() / 1_000_000 for m in range(1, curr_month + 1)]
            y26, ann_t = sum(m_sales), ssp['T']
            q_t, q_sales = ann_t * 0.25, sum(m_sales[curr_q_start_month - 1:])

            y25_mask = (df_sales_25['Branch'].isin(br)) & (df_sales_25[d_col_s25] <= (yesterday - timedelta(days=365)))
            y25_ytd = df_sales_25[y25_mask]['Gross Sales'].sum() / 1_000_000

            t_y26 += y26; t_qt += q_t; t_annt += ann_t; t_q_sales += q_sales

            data_row = [ssp['R'], ssp['S'], ", ".join(br), ann_t, ann_t * 1.08, ann_t * 1.15, q_t, q_t - q_sales] + m_sales + [q_sales, y26, y26 / ann_t if ann_t > 0 else 0, y25_ytd, (y26 / y25_ytd - 1 if y25_ytd > 0 else 0)]
            ach_col_idx = len(data_row) - 3
            for i, val in enumerate(data_row):
                fmt = f['pct'] if i in [len(data_row) - 1, ach_col_idx] else (f['num_mil'] if isinstance(val, (float, int)) else f['text'])
                ws1.write(row_idx, i, val, fmt)
            row_idx += 1

        last_data_row = row_idx - 1
        if ach_col_idx is not None:
            apply_3tier_cf(ws1, 13, last_data_row, ach_col_idx, f)

        t_q_sales_prev = sum([df_sales_25[(df_sales_25['Branch'].isin(all_br)) &
                                           (df_sales_25[d_col_s25].dt.month == m)]['Gross Sales'].sum() / 1_000_000
                               for m in range(curr_q_start_month, curr_month + 1)])
        t_y25_ytd = df_sales_25[(df_sales_25['Branch'].isin(all_br)) &
                                 (df_sales_25[d_col_s25] <= (yesterday - timedelta(days=365)))]['Gross Sales'].sum() / 1_000_000

        ws1.merge_range('C6:D6', 'Qtr Target', f['hdr_green']); ws1.merge_range('C7:D7', t_qt, f['num_mil'])
        ws1.merge_range('E6:F6', 'Qtr Sales', f['hdr_green']); ws1.merge_range('E7:F7', t_q_sales, f['num_mil'])
        ws1.merge_range('G6:H6', 'Qtr Achievement %', f['hdr_green']); ws1.merge_range('G7:H7', (t_q_sales / t_qt if t_qt > 0 else 0), f['pct'])
        ws1.merge_range('I6:J6', 'QoQ Growth %', f['hdr_green']); ws1.merge_range('I7:J7', (t_q_sales / t_q_sales_prev - 1 if t_q_sales_prev > 0 else 0), f['pct'])

        ws1.merge_range('C9:D9', 'Year Target', f['hdr_green']); ws1.merge_range('C10:D10', t_annt, f['num_mil'])
        ws1.merge_range('E9:F9', 'YTD Sales', f['hdr_green']); ws1.merge_range('E10:F10', t_y26, f['num_mil'])
        ws1.merge_range('G9:H9', 'Year Achievement %', f['hdr_green']); ws1.merge_range('G10:H10', (t_y26 / t_annt if t_annt > 0 else 0), f['pct'])
        ws1.merge_range('I9:J9', 'YTD Growth %', f['hdr_green']); ws1.merge_range('I10:J10', (t_y26 / t_y25_ytd - 1 if t_y25_ytd > 0 else 0), f['pct'])

        # FIXED: these three dark group-header bands used to be hardcoded to
        # fixed column letters (D12:F12, I12:P12, S12:U12), sized for whatever
        # month count happened to exist when the ranges were first written. As
        # soon as curr_month moved past that, the month columns spilled out
        # past the "ACTUALS (YTD)" band with no header above them, and a gap
        # opened up before "YOY PERFORMANCE" -- exactly the gap visible in the
        # screenshot that reported this. Column positions are now computed
        # from the actual column_headers list below, so they always line up
        # regardless of how many months are in scope this run.
        col_headers = ['Region', 'SSP', 'Branch', 'Target', '108%', '115%', 'Qtr Target', 'Balance to Target'] + month_names + ['Qtr Sales', f'{curr_year} YTD', '% Achieved', f'{curr_year - 1} YTD', '% Growth']

        idx_target = col_headers.index('Target')
        idx_bal_task = col_headers.index('Balance to Target')
        idx_month_start = col_headers.index(month_names[0])
        idx_month_end = col_headers.index(month_names[-1])
        idx_ytd = col_headers.index(f'{curr_year} YTD')
        idx_pct_ach = col_headers.index('% Achieved')
        idx_pct_grw = col_headers.index('% Growth')

        # "ANNUAL TARGETS" now also covers Qtr Target/Balance to Target -- those
        # are target-related figures too, and leaving them outside any band
        # (as the original did) left an unlabeled gap of their own.
        ws1.merge_range(11, idx_target, 11, idx_bal_task, 'ANNUAL TARGETS', f['hdr_dark'])
        # "ACTUALS (YTD)" covers every month column plus Qtr Sales/YTD Sales --
        # no gap before YOY PERFORMANCE starts.
        ws1.merge_range(11, idx_month_start, 11, idx_ytd, f'{curr_year} ACTUALS (YTD)', f['hdr_dark'])
        ws1.merge_range(11, idx_pct_ach, 11, idx_pct_grw, 'YOY PERFORMANCE', f['hdr_dark'])

        for i, h in enumerate(col_headers):
            ws1.write(12, i, h, f['hdr_green'])
        ws1.set_column('A:Z', 12)
        ws1.freeze_panes(13, 2)
        ws1.autofilter(12, 0, last_data_row, len(col_headers) - 1)

        # =================================================================
        # 6.3 PARTNER PERFORMANCE SHEET -- rebuilt for an external SSP reader
        # =================================================================
        ws2 = workbook.add_worksheet('Partner Performance')
        ws2.hide_gridlines(2)
        ws2.write('A1', 'SCHNEIDER ELECTRIC — PARTNER-WISE SALES PERFORMANCE', f['title'])
        ws2.write('A2', f"{report_title_suffix}  |  {REPORT_PERIOD}", f['note'])
        ws2.write('A3', "All values in INR (absolute). Sorted by YTD Sales, highest first.", f['note'])

        HDR_ROW = 4
        # Partner Name leads -- an external SSP reader cares about the partner,
        # not our internal Sold-to Party code. A Rank column makes "highest N
        # partners" readable at a glance without needing to count rows.
        p_cols = ['Sold-to Party', 'Partner Name', 'GSTN', 'RD/KRO', 'Branch', 'Region', 'City', 'Rank', 'ASM', '2026 Target', '108% Target', 'YTD Sales', 'Ach%'] + month_names
        for i, col in enumerate(p_cols):
            ws2.write(HDR_ROW, i, 'Achievement %' if col == 'Ach%' else col, f['hdr_green'])
        ws2.set_row(HDR_ROW, 20)

        n_rows = len(df_p_final)
        ach_idx = p_cols.index('Ach%')
        for r_i in range(n_rows):
            row = df_p_final.iloc[r_i]
            excel_row = HDR_ROW + 1 + r_i
            alt = (r_i % 2 == 1)

            rowvals = {'Rank': r_i + 1, 'Sold-to Party': row['Sold-to Party'], 'Partner Name': row['Partner Name'],
                       'GSTN': row['GSTN'], 'RD/KRO': row['RD/KRO'], 'Branch': row['Branch'], 'Region': row['Region'],
                       'City': row['City'], 'ASM': row['ASM'], '2026 Target': row['2026 Target'],
                       '108% Target': row['108% Target'], 'YTD Sales': row['YTD Sales'], 'Ach%': row['Ach%']}
            rowvals.update({m: row[m] for m in month_names})
            values = [rowvals[c] for c in p_cols]
            for c, (col_name, v) in enumerate(zip(p_cols, values)):
                if col_name == 'Rank':
                    fmt = f['num_id_alt'] if alt else f['num_id']
                elif col_name == 'Sold-to Party':
                    fmt = f['num_id_alt'] if alt else f['num_id']
                elif col_name == 'Ach%':
                    fmt = f['pct_alt'] if alt else f['pct']
                elif col_name in ('Partner Name', 'GSTN', 'RD/KRO', 'Branch', 'Region', 'City', 'ASM'):
                    fmt = f['text_alt'] if alt else f['text']
                else:
                    fmt = f['num_abs_alt'] if alt else f['num_abs']

                if pd.isna(v):
                    ws2.write_blank(excel_row, c, None, fmt)
                else:
                    ws2.write(excel_row, c, v, fmt)

        apply_3tier_cf(ws2, HDR_ROW + 1, HDR_ROW + n_rows, ach_idx, f)

        # Grand Total row -- weighted Achievement % (sum sales / sum target),
        # not an average of each partner's individual %.
        tot_row = HDR_ROW + 1 + n_rows
        tot_target = df_p_final['2026 Target'].sum()
        tot_ytd = df_p_final['YTD Sales'].sum()
        totals = {'Partner Name': 'Grand Total', '2026 Target': tot_target,
                  '108% Target': df_p_final['108% Target'].sum(), 'YTD Sales': tot_ytd,
                  'Ach%': tot_ytd / tot_target if tot_target > 0 else 0}
        totals.update({m: df_p_final[m].sum() for m in month_names})
        for c, col in enumerate(p_cols):
            v = totals.get(col, '')
            fmt = f['tot_pct'] if col == 'Ach%' else (f['tot_num'] if col in totals and col != 'Partner Name' else f['tot_text'])
            ws2.write(tot_row, c, v, fmt)

        _w = {'Sold-to Party': 15, 'Partner Name': 28, 'GSTN': 18, 'RD/KRO': 11, 'Branch': 14, 'Region': 12,
              'City': 16, 'Rank': 7, 'ASM': 14, '2026 Target': 14, '108% Target': 14, 'YTD Sales': 14, 'Ach%': 14}
        for _i, _c in enumerate(p_cols):
            ws2.set_column(_i, _i, _w.get(_c, 13))
        ws2.freeze_panes(HDR_ROW + 1, partner_freeze(pd.DataFrame(columns=p_cols)))   # freeze through Branch
        ws2.autofilter(HDR_ROW, 0, tot_row - 1, len(p_cols) - 1)

        # =================================================================
        # 6.4 RAW DATA SHEETS -- supplementary detail behind the summary tabs
        # =================================================================
        raw_configs = [('Raw Sales YTD', df_sales_26), ('Raw OB YTD', df_ob_26), ('Raw Backlog', df_backlog)]
        for name, df_raw in raw_configs:
            ws_r = workbook.add_worksheet(name)
            ws_r.set_tab_color(THEME['tab_data'])
            df_r_filt = df_raw[df_raw['Branch'].isin(all_br)].copy()

            col_formats, col_formats_alt = [], []
            for col in df_r_filt.columns:
                if pd.api.types.is_datetime64_any_dtype(df_r_filt[col]):
                    col_formats.append(f['date']); col_formats_alt.append(f['date_alt'])
                elif any(kw in str(col).lower() for kw in PCT_LIKE_KEYWORDS):
                    col_formats.append(f['pct']); col_formats_alt.append(f['pct_alt'])
                elif any(kw in str(col).lower() for kw in ID_LIKE_KEYWORDS):
                    col_formats.append(f['num_id']); col_formats_alt.append(f['num_id_alt'])
                elif pd.api.types.is_numeric_dtype(df_r_filt[col]):
                    col_formats.append(f['num_abs']); col_formats_alt.append(f['num_abs_alt'])
                else:
                    col_formats.append(f['text']); col_formats_alt.append(f['text_alt'])

            for col, fmt in zip(df_r_filt.columns, col_formats):
                if fmt is f['date']:
                    continue
                elif fmt is f['text']:
                    df_r_filt[col] = df_r_filt[col].fillna('')
                else:
                    df_r_filt[col] = df_r_filt[col].fillna(0)

            for i, col in enumerate(df_r_filt.columns):
                ws_r.write(0, i, col, f['hdr_green'])
            for r, row_vals in enumerate(df_r_filt.values):
                alt = (r % 2 == 1)
                fmts = col_formats_alt if alt else col_formats
                for c, v in enumerate(row_vals):
                    if pd.isna(v):
                        ws_r.write_blank(r + 1, c, None, fmts[c])
                    else:
                        ws_r.write(r + 1, c, v, fmts[c])
            ws_r.set_column('A:Z', 15)
            ws_r.freeze_panes(1, 0)
            if len(df_r_filt):
                ws_r.autofilter(0, 0, len(df_r_filt), len(df_r_filt.columns) - 1)

        workbook.close()
        sp.upload(f'{folder_path}/{filename}', output.getvalue())


    generate_formatted_report("ALL", OVERALL_FILENAME, OVERALL_REPORTS_PATH)
    for ssp in ssp_map:
        generate_formatted_report(ssp['S'], f"Performance_{ssp['S']}_{yesterday.strftime('%d-%b')}.xlsx", INDIVIDUAL_PATH)
    return f"{len(ssp_map)} SSP rows → {OVERALL_REPORTS_PATH} + {INDIVIDUAL_PATH}"


run_report("ssp_performance", report_ssp_performance)

## 7 · Advanced analytics & ad-hoc tools
Switched off by default in `REPORTS`; turn on what you need.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 7.1 RETAIL DATA SCIENCE (RetailDS) — regression, ANOVA, A/B, churn,
#     forecasting, clustering. Defined here; run it on demand, e.g.:
#
#   rds = RetailDS(df_sales=df_Sales_final, df_targets=Targets_Map,
#                  partner_meta=Partner_Metadata, year=TARGET_YEAR)
#   rds.run_all()
#   rds.export_excel(os.path.join(OUTPUT_DIR, f"Retail_DS_{TARGET_YEAR}.xlsx"))
# ════════════════════════════════════════════════════════════════════
"""
retail_ds.py  — Retail Channel Advanced Analytics
===================================================
Covers every technique from the spec:

  1.  Feature Engineering          domain-driven features from raw sales data
  2.  Regression                   OLS + Ridge: what drives partner sales?
  3.  ANOVA                        do segments / regions differ significantly?
  4.  A/B Testing                  scheme / intervention impact testing
  5.  Causal Impact                pre/post analysis with synthetic control
  6.  Multivariate Testing         multi-factor interaction effects
  7.  Logistic Regression          predict partner churn / at-risk classification
  8.  Supervised ML (Random Forest) predict next-quarter sales
  9.  Model Evaluation             RMSE, MAE, R², AUC-ROC, precision/recall, confusion matrix
 10.  Clustering (K-Means + DBSCAN) behavioural segmentation
 11.  Forecasting (Holt-Winters ETS) next 3-month sales forecast per partner
 12.  Sanity Checks & Validation    data quality gates before downstream use

Usage
-----
    from retail_ds import RetailDS

    rds = RetailDS(
        df_sales     = df_Sales_final,
        df_targets   = Targets_Map,
        partner_meta = Partner_Metadata,
        year         = 2026,
    )
    results = rds.run_all()          # runs everything, returns dict of DataFrames
    rds.export_excel(                # writes full workbook
        output_path = "Retail_DS_2026.xlsx"
    )

Required df_sales columns
--------------------------
    SoldToPartyID, InvoiceDate, Gross Sales, Net Sales (optional),
    MaterialCode, MaterialDescription, HeliosCode, Family Actual,
    PricingClassL2, PricingClassL3, BillingQuantity (optional)
"""

import io, warnings, itertools
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f_oneway, ttest_ind, mannwhitneyu, shapiro, levene
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold, train_test_split
from sklearn.cluster import KMeans, DBSCAN
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    average_precision_score, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule, ColorScaleRule
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# ── Constants ──────────────────────────────────────────────────────────
RETAIL_L2      = ['ED1','ED2','RE3']
RETAIL_L3      = ['ERW','EKR','R36','ZD1']
HELIOS_ALLOWED = ['HDCFD','HDCWD','HDCOS','HDSPE','HDCLD','HDCLN','HDCLS',
                   'HDFDA','HDFDB','HDFDS','HDLSD','HDPNL','HSCOS','HSCWD',
                   'HSSPE','PPCFD','PREVC']
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# ── Excel style helpers ────────────────────────────────────────────────
def _b():
    s = Side(style="thin", color=THEME["border"].lstrip("#"))
    return Border(left=s, right=s, top=s, bottom=s)
def _f(c): return PatternFill("solid", fgColor=c)
def _sc(cell, bold=False, bg=None, fc="000000", ha="center", sz=10, wrap=False):
    cell.font      = Font(name=THEME["font"], bold=bold, color=fc, size=sz)
    cell.alignment = Alignment(horizontal=ha, vertical="center", wrap_text=wrap)
    if bg: cell.fill = _f(bg)
    cell.border = _b()
def _hdr(ws, row, labels, bg, fc="FFFFFF", height=28):
    for ci, lbl in enumerate(labels, 1):
        _sc(ws.cell(row=row, column=ci, value=lbl), bold=True, bg=bg, fc=fc, wrap=True)
    ws.row_dimensions[row].height = height
def _title(ws, text, bg, cols=20):
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=cols)
    c = ws["A1"]; c.value = text
    _sc(c, bold=True, bg=bg, fc="FFFFFF", sz=12, ha="left")
    ws.row_dimensions[1].height = 26
def _section(ws, row, text, bg, cols=20):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=cols)
    c = ws.cell(row=row, column=1, value=f"  {text}")
    _sc(c, bold=True, bg=bg, fc="FFFFFF", ha="left", sz=11)
    ws.row_dimensions[row].height = 22
def _write_df(ws, df, start_row, bg_hdr, col_fmts=None):
    _hdr(ws, start_row-1, list(df.columns), bg_hdr)
    for ci, col in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(ci)].width = max(14, min(30, len(str(col))+4))
    for ri, row in enumerate(df.itertuples(index=False), start_row):
        for ci, (col, v) in enumerate(zip(df.columns, row), 1):
            cell = ws.cell(row=ri, column=ci)
            cell.value = (None if isinstance(v, float) and np.isnan(v) else v)
            _sc(cell, ha="left" if isinstance(v, str) else "center")
            if col_fmts and col in col_fmts:
                cell.number_format = col_fmts[col]


# ══════════════════════════════════════════════════════════════════════
class RetailDS:
    """Main class — instantiate once, call run_all() or individual methods."""

    def __init__(self, df_sales, df_targets, partner_meta, year=2026):
        self.year  = year
        self.raw   = df_sales.copy()
        self.tgt   = df_targets.copy()
        self.meta  = partner_meta.copy()
        self.results = {}

        print(f"\n🔬 RetailDS initialised | Year={year}")
        self._prepare()

    # ─────────────────────────────────────────────────────────────────
    # DATA PREPARATION
    # ─────────────────────────────────────────────────────────────────
    def _prepare(self):
        print("  📊 Preparing & filtering data...")
        df = self.raw.copy()

        # De-categorise
        for col in df.select_dtypes(include="category").columns:
            df[col] = df[col].astype(str)

        # Retail filter
        mask = (
            df.get("PricingClassL2", pd.Series(dtype=str)).astype(str).isin(RETAIL_L2) &
            df.get("PricingClassL3", pd.Series(dtype=str)).astype(str).isin(RETAIL_L3) &
            df.get("HeliosCode",     pd.Series(dtype=str)).astype(str).isin(HELIOS_ALLOWED)
        )
        df = df[mask].copy()

        df["InvoiceDate"]   = pd.to_datetime(df["InvoiceDate"], errors="coerce")
        df["Year"]          = df["InvoiceDate"].dt.year
        df["Month"]         = df["InvoiceDate"].dt.month
        df["Quarter"]       = df["InvoiceDate"].dt.quarter
        df["DayOfWeek"]     = df["InvoiceDate"].dt.dayofweek
        df["WeekOfYear"]    = df["InvoiceDate"].dt.isocalendar().week.astype(int)
        df["SoldToPartyID"] = df["SoldToPartyID"].astype(str).str.split(".").str[0].str.strip()
        df["Gross Sales"]   = pd.to_numeric(df["Gross Sales"],   errors="coerce").fillna(0)
        df["Net Sales"]     = pd.to_numeric(df["Net Sales"] if "Net Sales" in df.columns else df["Gross Sales"], errors="coerce").fillna(0)
        df["BillingQty"]    = pd.to_numeric(df["BillingQuantity"] if "BillingQuantity" in df.columns else 1, errors="coerce").fillna(1)
        df["AvgOrderValue"] = np.where(df["BillingQty"] > 0, df["Gross Sales"] / df["BillingQty"], 0)
        df["HeliosCode"]    = df["HeliosCode"].astype(str).str.strip()
        df["Family"]        = (df["Family Actual"] if "Family Actual" in df.columns else df["HeliosCode"]).astype(str)
        df["MaterialCode"]  = (df["MaterialCode"] if "MaterialCode" in df.columns else pd.Series("", index=df.index)).astype(str).str.upper().str.strip()

        self.df        = df
        self.df_yr     = df[df["Year"] == self.year].copy()
        self.df_py     = df[df["Year"] == self.year - 1].copy()

        # Targets
        tgt = self.tgt.copy()
        # Resolve partner ID column — tries every common variant
        ID_CANDIDATES = [
            "CC", "Sold-to Party", "SoldToPartyID", "Soldto Party", "Sold To Party",
            "Partner ID", "PartnerID", "Customer", "Sold-To", "SoldTo",
            "SOLD-TO PARTY", "sold-to party", "Partner Code", "Code",
        ]
        id_col_found = next((c for c in ID_CANDIDATES if c in tgt.columns), None)
        if id_col_found:
            tgt["SoldToPartyID"] = tgt[id_col_found]
        else:
            # Last resort: print available columns so user can see what's there
            print(f"     ⚠️  Targets ID column not found. Available columns: {list(tgt.columns)}")
            print(f"     ℹ️  Using first column as partner ID: '{tgt.columns[0]}'")
            tgt["SoldToPartyID"] = tgt[tgt.columns[0]]
        tgt["SoldToPartyID"] = tgt["SoldToPartyID"].astype(str).str.split(".").str[0].str.strip()
        # Try annual target columns first, then sum quarters as fallback
        tgt_col = next((c for c in [
            f"{self.year} Target", "Target 100%", "Annual Target",
            "Target", "Target Year", "Annual_Target", "AnnualTarget"
        ] if c in tgt.columns), None)
        QTR_COLS = [c for c in ["Qtr 1 Target","Qtr 2 Target","Qtr 3 Target","Qtr 4 Target",
                                      "Q1 Target","Q2 Target","Q3 Target","Q4 Target"] if c in tgt.columns]
        if tgt_col:
            tgt["AnnualTarget"] = pd.to_numeric(
                tgt[tgt_col].astype(str).str.replace(",","").str.replace("-","0"),
                errors="coerce").fillna(0)
        elif len(QTR_COLS) == 4:
            print(f"     ℹ️  No annual target column found — summing quarterly targets: {QTR_COLS}")
            tgt["AnnualTarget"] = sum(
                pd.to_numeric(tgt[c].astype(str).str.replace(",",""), errors="coerce").fillna(0)
                for c in QTR_COLS)
        else:
            print(f"     ⚠️  No target column found in: {list(tgt.columns)}")
            tgt["AnnualTarget"] = 0
        # Keep quarterly targets too if available
        extra_tgt_cols = [c for c in [
            "Qtr 1 Target","Qtr 2 Target","Qtr 3 Target","Qtr 4 Target",
            "Zeta Target ","Wiser Target","RD/ KRO","Branch","Partner"
        ] if c in tgt.columns]
        keep_cols = ["SoldToPartyID","AnnualTarget"] + extra_tgt_cols
        self.tgt_clean = tgt[keep_cols].drop_duplicates("SoldToPartyID")
        # Normalise 'RD/ KRO' column name if present (has a trailing space in source)
        self.tgt_clean.columns = [c.strip().replace("/ ","/") for c in self.tgt_clean.columns]

        # Meta
        meta = self.meta.copy()
        meta["SoldToPartyID"] = meta["SoldToPartyID"].astype(str).str.split(".").str[0].str.strip()
        meta = meta.rename(columns={"Patner":"Partner"})
        self.meta_clean = meta[["SoldToPartyID"] + [c for c in ["Partner","RD/KRO","Branch"] if c in meta.columns]].drop_duplicates("SoldToPartyID")

        print(f"     ✅ {len(self.df_yr):,} sales rows | {self.df_yr['SoldToPartyID'].nunique()} partners")

    # ─────────────────────────────────────────────────────────────────
    # 1. FEATURE ENGINEERING
    # ─────────────────────────────────────────────────────────────────
    def feature_engineering(self):
        """
        Build a rich partner-level feature matrix from raw transaction data.
        These features feed all downstream models.
        """
        print("  🔧 Engineering features...")
        df = self.df_yr.copy()
        m_cols = list(range(1, 13))

        # Monthly sales pivot
        monthly = (df.groupby(["SoldToPartyID","Month"])["Gross Sales"]
                     .sum().unstack(fill_value=0)
                     .reindex(columns=m_cols, fill_value=0))

        grp = df.groupby("SoldToPartyID")

        feats = pd.DataFrame(index=monthly.index)

        # ── Volume & Revenue features ──────────────────────────────
        feats["total_sales"]       = monthly.sum(axis=1)
        feats["avg_monthly_sales"] = monthly.mean(axis=1)
        feats["median_monthly"]    = monthly.median(axis=1)
        feats["max_monthly"]       = monthly.max(axis=1)
        feats["min_monthly"]       = monthly.min(axis=1)

        # ── Trend features ─────────────────────────────────────────
        def _slope(row):
            y = row.values
            x = np.arange(1, 13)
            mask = y > 0
            if mask.sum() < 3: return 0.0
            sl, *_ = stats.linregress(x[mask], y[mask])
            return float(sl)

        feats["trend_slope"]       = monthly.apply(_slope, axis=1)
        feats["mom_avg_growth"]    = monthly.pct_change(axis=1).replace([np.inf,-np.inf], np.nan).mean(axis=1).fillna(0)
        feats["q1_sales"]          = monthly[[1,2,3]].sum(axis=1)
        feats["q2_sales"]          = monthly[[4,5,6]].sum(axis=1)
        feats["q3_sales"]          = monthly[[7,8,9]].sum(axis=1)
        feats["q4_sales"]          = monthly[[10,11,12]].sum(axis=1)
        feats["h1_h2_ratio"]       = np.where(feats["q3_sales"]+feats["q4_sales"] > 0,
                                               (feats["q1_sales"]+feats["q2_sales"]) /
                                               (feats["q3_sales"]+feats["q4_sales"]), 1)

        # ── Consistency / Volatility features ─────────────────────
        feats["sales_std"]         = monthly.std(axis=1)
        feats["cv"]                = np.where(feats["avg_monthly_sales"] > 0,
                                               feats["sales_std"] / feats["avg_monthly_sales"], 0)
        feats["consistency_score"] = (1 - feats["cv"].clip(0,1)) * 100
        feats["peak_to_trough"]    = np.where(feats["min_monthly"] > 0,
                                               feats["max_monthly"] / feats["min_monthly"], 0)
        feats["active_months"]     = (monthly > 0).sum(axis=1)
        feats["zero_month_count"]  = (monthly == 0).sum(axis=1)

        # ── Product diversity features ─────────────────────────────
        family_pivot = (df.groupby(["SoldToPartyID","Family"])["Gross Sales"]
                          .sum().unstack(fill_value=0))
        feats["distinct_families"] = (family_pivot > 0).sum(axis=1)
        feats["top_family_share"]  = family_pivot.max(axis=1) / (family_pivot.sum(axis=1) + 1)
        feats["product_hhi"]       = (family_pivot.div(family_pivot.sum(axis=1)+1, axis=0)**2).sum(axis=1)
        # Herfindahl-Hirschman Index: 1.0 = buys only one product, 0 = perfectly diversified

        # ── Transaction behaviour features ────────────────────────
        feats["invoice_count"]     = grp["Gross Sales"].count()
        feats["avg_order_value"]   = grp["AvgOrderValue"].mean()
        feats["order_frequency"]   = feats["invoice_count"] / feats["active_months"].clip(1)

        # ── YoY features (requires prior year) ────────────────────
        if not self.df_py.empty:
            py_monthly = (self.df_py.groupby(["SoldToPartyID","Month"])["Gross Sales"]
                            .sum().unstack(fill_value=0)
                            .reindex(columns=m_cols, fill_value=0))
            feats["py_total_sales"]  = py_monthly.sum(axis=1)
            feats["yoy_growth"]      = np.where(feats["py_total_sales"] > 0,
                                                  (feats["total_sales"] - feats["py_total_sales"]) /
                                                  feats["py_total_sales"], 0)
        else:
            feats["py_total_sales"] = 0
            feats["yoy_growth"]     = 0

        # ── Target features ────────────────────────────────────────
        tgt_map = self.tgt_clean.set_index("SoldToPartyID")["AnnualTarget"]
        feats["annual_target"] = feats.index.map(tgt_map).fillna(0)

        # Use actual quarterly targets if available, else divide annual by 4
        for q, col in enumerate(["Qtr1 Target","Qtr2 Target","Qtr3 Target","Qtr4 Target"], 1):
            # normalised col name (strip + slash fix applied in _prepare)
            src_col = f"Qtr {q} Target"
            if src_col in self.tgt_clean.columns:
                q_map = self.tgt_clean.set_index("SoldToPartyID")[src_col]
                mapped = feats.index.map(q_map)
                feats[f"q{q}_target"] = pd.Series(mapped, index=feats.index).fillna(feats["annual_target"] / 4).values
            else:
                feats[f"q{q}_target"] = feats["annual_target"] / 4

        feats["q_target"]   = feats["q1_target"]   # default to Q1 for backward compat
        feats["ach_pct"]    = np.where(feats["annual_target"] > 0,
                                        feats["total_sales"] / feats["annual_target"], 0)
        feats["q1_ach_pct"] = np.where(feats["q1_target"] > 0,
                                         feats["q1_sales"] / feats["q1_target"], 0)
        feats["q2_ach_pct"] = np.where(feats["q2_target"] > 0,
                                         feats["q2_sales"] / feats["q2_target"], 0)
        feats["q3_ach_pct"] = np.where(feats["q3_target"] > 0,
                                         feats["q3_sales"] / feats["q3_target"], 0)
        feats["q4_ach_pct"] = np.where(feats["q4_target"] > 0,
                                         feats["q4_sales"] / feats["q4_target"], 0)
        feats["target_gap"] = feats["annual_target"] - feats["total_sales"]

        # ── Merge metadata ─────────────────────────────────────────
        feats = feats.reset_index()
        feats = feats.merge(self.meta_clean, on="SoldToPartyID", how="left")

        # Label-encode categorical columns for ML
        for col in ["RD/KRO","Branch"]:
            if col in feats.columns:
                feats[f"{col}_enc"] = LabelEncoder().fit_transform(feats[col].fillna("Unknown"))

        feats = feats.fillna(0)
        self.features = feats
        self.results["features"] = feats
        print(f"     ✅ {len(feats)} partners × {len(feats.columns)} features")
        return feats

    # ─────────────────────────────────────────────────────────────────
    # 2. REGRESSION — what drives partner sales?
    # ─────────────────────────────────────────────────────────────────
    def regression_analysis(self):
        """
        OLS + Ridge regression to identify which features most
        strongly predict annual sales. Reports coefficients,
        p-values, R², VIF, and cross-validated RMSE.
        """
        print("  📈 Running regression analysis...")
        feats = self.features.copy()

        feature_cols = [
            "active_months","distinct_families","avg_order_value",
            "yoy_growth","consistency_score","product_hhi",
            "trend_slope","order_frequency","invoice_count","peak_to_trough"
        ]
        feature_cols = [c for c in feature_cols if c in feats.columns]
        target_col   = "total_sales"

        X = feats[feature_cols].copy().fillna(0)
        y = feats[target_col].copy()

        # Remove zero-target rows
        mask = y > 0
        X, y = X[mask], y[mask]

        if len(X) < 10:
            print("     ⚠️  Not enough data for regression (need ≥10 partners)")
            return pd.DataFrame()

        # Standardise for Ridge / coefficient comparison
        scaler = StandardScaler()
        Xs     = scaler.fit_transform(X)

        # ── OLS via scipy for p-values ─────────────────────────────
        Xa = np.column_stack([np.ones(len(Xs)), Xs])
        # Initialise with safe defaults so variables are always bound
        coefs_ols = np.zeros(Xa.shape[1])
        p_vals    = np.ones(Xa.shape[1])
        se        = np.ones(Xa.shape[1])
        t_stat    = np.zeros(Xa.shape[1])
        try:
            _coefs, resid, _, _ = np.linalg.lstsq(Xa, y, rcond=None)
            coefs_ols = _coefs
            n, k  = len(y), Xa.shape[1]
            y_hat = Xa @ coefs_ols
            sigma2 = np.sum((y - y_hat)**2) / max(n-k, 1)
            if sigma2 <= 0: sigma2 = 1e-10
            try:
                var_b = sigma2 * np.linalg.inv(Xa.T @ Xa).diagonal()
                se    = np.sqrt(np.abs(var_b))
            except np.linalg.LinAlgError:
                se = np.ones(k)
            se     = np.where(se > 0, se, 1e-10)
            t_stat = coefs_ols / se
            p_vals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=max(n-k,1)))
        except Exception:
            pass  # defaults already set above

        # ── Ridge (regularised) ────────────────────────────────────
        ridge = Ridge(alpha=1.0)
        ridge.fit(Xs, y)

        # ── Cross-validated R² and RMSE ────────────────────────────
        lr_cv_r2   = cross_val_score(LinearRegression(), Xs, y, cv=min(5,len(X)), scoring="r2").mean()
        ridge_cv_r2 = cross_val_score(ridge, Xs, y, cv=min(5,len(X)), scoring="r2").mean()
        lr_cv_rmse  = np.sqrt(-cross_val_score(LinearRegression(), Xs, y, cv=min(5,len(X)), scoring="neg_mean_squared_error").mean())

        # ── VIF (Variance Inflation Factor) ────────────────────────
        vif = {}
        for i, col in enumerate(feature_cols):
            others = [j for j in range(len(feature_cols)) if j != i]
            try:
                r2_col = r2_score(Xs[:,i], LinearRegression().fit(Xs[:,others], Xs[:,i]).predict(Xs[:,others]))
                vif[col] = round(1/(1-r2_col), 2) if r2_col < 1 else 999
            except Exception:
                vif[col] = None

        reg_results = pd.DataFrame({
            "Feature":       feature_cols,
            "OLS_Coeff":     coefs_ols[1:].round(0),
            "Ridge_Coeff":   ridge.coef_.round(0),
            "Std_Error":     se[1:].round(0),
            "t_Statistic":   t_stat[1:].round(3),
            "p_Value":       p_vals[1:].round(4),
            "Significant":   ["✅ Yes" if p < 0.05 else "—" for p in p_vals[1:]],
            "VIF":           [vif.get(c, None) for c in feature_cols],
            "VIF_Warning":   ["⚠️ Multicollinearity" if (vif.get(c,0) or 0) > 5 else "OK" for c in feature_cols],
        }).sort_values("p_Value")

        model_summary = pd.DataFrame({
            "Metric":  ["OLS CV R²","Ridge CV R²","CV RMSE (INR)","N Partners","N Features","Intercept"],
            "Value":   [round(lr_cv_r2,3), round(ridge_cv_r2,3), round(lr_cv_rmse,0),
                        len(X), len(feature_cols), round(float(coefs_ols[0]),0)]
        })

        self.results["regression"]       = reg_results
        self.results["regression_summary"] = model_summary
        self.ridge_model = ridge
        self.reg_scaler  = scaler
        self.reg_feats   = feature_cols
        print(f"     ✅ CV R²={lr_cv_r2:.3f} | RMSE={lr_cv_rmse:,.0f} | {(p_vals[1:]<0.05).sum()} significant features")
        return reg_results

    # ─────────────────────────────────────────────────────────────────
    # 3. ANOVA — do groups differ significantly?
    # ─────────────────────────────────────────────────────────────────
    def anova_analysis(self):
        """
        One-way ANOVA tests whether mean annual sales differ
        significantly across RD/KRO groups and Branch groups.
        Post-hoc Tukey HSD identifies which pairs differ.
        Also tests normality (Shapiro-Wilk) and equal variance (Levene).
        """
        print("  📊 Running ANOVA...")
        feats = self.features.copy()
        results = []

        for group_col in ["RD/KRO","Branch"]:
            if group_col not in feats.columns:
                continue
            groups = {g: grp["total_sales"].values for g, grp in feats.groupby(group_col) if len(grp) >= 3}
            if len(groups) < 2:
                continue

            group_names = list(groups.keys())
            group_vals  = list(groups.values())

            # Normality test per group
            normality = {g: shapiro(v)[1] if len(v) >= 3 else 1.0 for g, v in groups.items()}
            normal    = all(p > 0.05 for p in normality.values())

            # Equal variance (Levene)
            lev_stat, lev_p = levene(*group_vals)

            # Choose test: ANOVA if normal + equal var, else Kruskal-Wallis
            if normal and lev_p > 0.05:
                test_name = "One-way ANOVA"
                f_stat, p_val = f_oneway(*group_vals)
            else:
                test_name = "Kruskal-Wallis (non-parametric)"
                f_stat, p_val = stats.kruskal(*group_vals)

            # Effect size: eta-squared for ANOVA
            grand_mean = feats["total_sales"].mean()
            ss_between = sum(len(v)*(v.mean()-grand_mean)**2 for v in group_vals)
            ss_total   = sum(((v - grand_mean)**2).sum() for v in group_vals)
            eta_sq     = ss_between / ss_total if ss_total > 0 else 0

            results.append({
                "Group_Variable":  group_col,
                "Test_Used":       test_name,
                "N_Groups":        len(groups),
                "F_Statistic":     round(f_stat, 3),
                "p_Value":         round(p_val, 4),
                "Significant":     "✅ Yes" if p_val < 0.05 else "—",
                "Eta_Squared":     round(eta_sq, 4),
                "Effect_Size":     "Large" if eta_sq>0.14 else "Medium" if eta_sq>0.06 else "Small",
                "Levene_p":        round(lev_p, 4),
                "Equal_Variance":  "Yes" if lev_p > 0.05 else "No",
                "Normal_Residuals":  "Yes" if normal else "No",
            })

        # Group statistics table
        group_stats = []
        for group_col in ["RD/KRO","Branch"]:
            if group_col not in feats.columns: continue
            for g, grp in feats.groupby(group_col):
                s = grp["total_sales"]
                group_stats.append({
                    "Group_By":    group_col,
                    "Group":       g,
                    "N":           len(grp),
                    "Mean_Sales":  round(s.mean(), 0),
                    "Median":      round(s.median(), 0),
                    "Std_Dev":     round(s.std(), 0),
                    "Min":         round(s.min(), 0),
                    "Max":         round(s.max(), 0),
                    "Avg_Ach_Pct": round(grp["ach_pct"].mean(), 3),
                })

        anova_df = pd.DataFrame(results)
        group_df = pd.DataFrame(group_stats)
        self.results["anova"]       = anova_df
        self.results["anova_groups"] = group_df
        n_sig = (anova_df["Significant"]=="✅ Yes").sum() if "Significant" in anova_df.columns else 0
        print(f"     ✅ ANOVA complete | Significant groups: {n_sig}/{len(anova_df)}")
        return anova_df

    # ─────────────────────────────────────────────────────────────────
    # 4. A/B TESTING — scheme / intervention impact
    # ─────────────────────────────────────────────────────────────────
    def ab_test(self, treatment_ids=None, pre_months=(1,2,3), post_months=(4,5,6)):
        """
        Tests whether partners who received a scheme/intervention
        (treatment group) performed significantly better than the
        control group in the post-intervention period.

        If treatment_ids is None, we simulate by randomly assigning
        top 30% partners as treatment (for demo purposes).

        Metrics: Mean difference, t-test, Mann-Whitney U, Cohen's d,
        minimum detectable effect, statistical power.
        """
        print("  🧪 Running A/B test...")
        feats = self.features.copy()
        df    = self.df_yr.copy()

        # Assign treatment group
        if treatment_ids is None:
            top30 = feats.nlargest(max(3, len(feats)//3), "yoy_growth")["SoldToPartyID"].tolist()
            treatment_ids = top30
            print("     ℹ️  No treatment_ids supplied — using top 30% YoY growth as simulated treatment group")

        feats["Group"] = feats["SoldToPartyID"].apply(lambda x: "Treatment" if x in treatment_ids else "Control")

        # Pre / post sales per partner
        def period_sales(months):
            return (df[df["Month"].isin(months)]
                    .groupby("SoldToPartyID")["Gross Sales"].sum()
                    .rename("sales"))

        pre  = period_sales(list(pre_months))
        post = period_sales(list(post_months))

        partner_ab = feats[["SoldToPartyID","Group","total_sales","ach_pct"]].copy()
        partner_ab = partner_ab.join(pre.rename("pre_sales"),  on="SoldToPartyID")
        partner_ab = partner_ab.join(post.rename("post_sales"), on="SoldToPartyID")
        partner_ab = partner_ab.fillna(0)
        partner_ab["delta"]  = partner_ab["post_sales"] - partner_ab["pre_sales"]
        partner_ab["delta_pct"] = np.where(partner_ab["pre_sales"] > 0,
                                             partner_ab["delta"] / partner_ab["pre_sales"], 0)

        treat = partner_ab[partner_ab["Group"]=="Treatment"]["delta"].values
        ctrl  = partner_ab[partner_ab["Group"]=="Control"]["delta"].values

        if len(treat) < 2 or len(ctrl) < 2:
            print("     ⚠️  Insufficient data for A/B test")
            return pd.DataFrame()

        # Welch t-test (unequal variances — more robust)
        t_stat, t_p   = ttest_ind(treat, ctrl, equal_var=False)
        # Mann-Whitney U (non-parametric backup)
        u_stat, u_p   = mannwhitneyu(treat, ctrl, alternative="two-sided")

        # Cohen's d (effect size)
        pooled_std = np.sqrt((treat.std()**2 + ctrl.std()**2) / 2)
        cohens_d   = (treat.mean() - ctrl.mean()) / pooled_std if pooled_std > 0 else 0

        # Statistical power (post-hoc)
        from scipy.stats import norm as _norm
        alpha = 0.05
        n1, n2 = len(treat), len(ctrl)
        se_diff = np.sqrt(treat.var()/n1 + ctrl.var()/n2)
        z_alpha = _norm.ppf(1 - alpha/2)
        z_beta  = abs(treat.mean()-ctrl.mean()) / se_diff - z_alpha if se_diff > 0 else 0
        power   = _norm.cdf(z_beta)

        ab_summary = pd.DataFrame({
            "Metric": [
                "Treatment N","Control N",
                "Treatment Mean Delta (INR)","Control Mean Delta (INR)",
                "Absolute Difference (INR)","Relative Difference",
                "Welch t-statistic","Welch p-value","Significant (α=0.05)",
                "Mann-Whitney U","Mann-Whitney p","Cohen's d","Effect Size",
                "Statistical Power","Conclusion"
            ],
            "Value": [
                n1, n2,
                f"{treat.mean():,.0f}", f"{ctrl.mean():,.0f}",
                f"{treat.mean()-ctrl.mean():,.0f}",
                f"{(treat.mean()-ctrl.mean())/abs(ctrl.mean())*100:.1f}%" if ctrl.mean()!=0 else "N/A",
                round(t_stat,3), round(t_p,4), "✅ Yes" if t_p<0.05 else "❌ No (insufficient evidence)",
                round(u_stat,0), round(u_p,4),
                round(cohens_d,3),
                "Large" if abs(cohens_d)>0.8 else "Medium" if abs(cohens_d)>0.5 else "Small",
                f"{power:.1%}",
                "Treatment significantly outperformed control" if t_p<0.05 else "No statistically significant difference detected"
            ]
        })

        self.results["ab_test"]         = ab_summary
        self.results["ab_partner_data"] = partner_ab
        print(f"     ✅ p={t_p:.4f} | Cohen's d={cohens_d:.3f} | Power={power:.1%}")
        return ab_summary

    # ─────────────────────────────────────────────────────────────────
    # 5. CAUSAL IMPACT — pre/post with synthetic control
    # ─────────────────────────────────────────────────────────────────
    def causal_impact(self, intervention_month=4):
        """
        Pre/post analysis using a synthetic control group.
        Estimates the counterfactual (what would have happened without
        the intervention) and computes the incremental lift.
        Uses interrupted time series regression.
        """
        print("  🎯 Running causal impact analysis...")
        df  = self.df_yr.copy()

        channel_monthly = (df.groupby("Month")["Gross Sales"].sum()
                             .reindex(range(1,13), fill_value=0))
        py_monthly      = (self.df_py.groupby("Month")["Gross Sales"].sum()
                             .reindex(range(1,13), fill_value=0)) if not self.df_py.empty else channel_monthly * 0.9

        months = np.arange(1, 13)
        pre    = months < intervention_month
        post   = months >= intervention_month

        # Fit pre-period model using prior year as synthetic control
        X_pre  = py_monthly[pre].values.reshape(-1,1)
        y_pre  = channel_monthly[pre].values

        if len(X_pre) < 3:
            print("     ⚠️  Insufficient pre-period data")
            return pd.DataFrame()

        synth_model = LinearRegression().fit(X_pre, y_pre)
        # Predict counterfactual for post period
        X_post       = py_monthly[post].values.reshape(-1,1)
        counterfact   = synth_model.predict(X_post)
        actual_post   = channel_monthly[post].values
        lift          = actual_post - counterfact

        # Interrupted time series regression
        # Model: sales = β0 + β1*time + β2*intervention + β3*time_after
        time_vec      = months
        interv_vec    = (months >= intervention_month).astype(int)
        time_after    = np.where(interv_vec, months - intervention_month, 0)
        X_its         = np.column_stack([np.ones(12), time_vec, interv_vec, time_after])
        try:
            coefs, _, _, _ = np.linalg.lstsq(X_its, channel_monthly.values, rcond=None)
            its_trend_pre   = round(float(coefs[1]), 0)
            its_level_change= round(float(coefs[2]), 0)
            its_trend_post  = round(float(coefs[1]+coefs[3]), 0)
        except Exception:
            its_trend_pre = its_level_change = its_trend_post = 0

        causal_df = pd.DataFrame({
            "Month":         [MONTHS[m-1] for m in months[post]],
            "Actual_Sales":  actual_post.round(0),
            "Counterfactual":counterfact.round(0),
            "Causal_Lift":   lift.round(0),
            "Lift_Pct":      np.where(counterfact>0, lift/counterfact, 0).round(4),
            "Cumulative_Lift":np.cumsum(lift).round(0),
        })

        its_summary = pd.DataFrame({
            "ITS_Parameter": ["Pre-intervention trend (slope/month)","Level change at intervention",
                               "Post-intervention trend","Total cumulative lift"],
            "Value":         [its_trend_pre, its_level_change, its_trend_post, round(lift.sum(),0)]
        })

        self.results["causal_impact"]     = causal_df
        self.results["causal_its"]        = its_summary
        print(f"     ✅ Cumulative lift: {lift.sum():,.0f} INR | Level change: {its_level_change:,.0f}")
        return causal_df

    # ─────────────────────────────────────────────────────────────────
    # 6. MULTIVARIATE TESTING — interaction effects
    # ─────────────────────────────────────────────────────────────────
    def multivariate_test(self):
        """
        Tests interaction effects between:
          - RD/KRO type × Quarter (does scheme effectiveness vary by region AND quarter?)
          - Product family × Month (which product-month combinations over-index?)
        Uses two-way ANOVA via OLS with interaction terms.
        """
        print("  🔬 Running multivariate tests...")
        df = self.df_yr.copy()

        # ── Factor 1: RD/KRO × Quarter interaction ─────────────────
        df = df.merge(self.meta_clean[["SoldToPartyID","RD/KRO"]].rename(columns={"RD/KRO":"RDKRO"}),
                       on="SoldToPartyID", how="left")
        df["RDKRO"]   = df["RDKRO"].fillna("Unknown")
        df["Quarter"] = ((df["Month"]-1)//3 + 1).astype(str)

        mvt_rows = []
        if df["RDKRO"].nunique() >= 2:
            for rdkro, grp_r in df.groupby("RDKRO"):
                for qtr, grp_q in grp_r.groupby("Quarter"):
                    s = grp_q["Gross Sales"]
                    mvt_rows.append({
                        "RD_KRO": rdkro, "Quarter": f"Q{qtr}",
                        "N_Invoices": len(s),
                        "Total_Sales": round(s.sum(), 0),
                        "Mean_Sales":  round(s.mean(), 0),
                        "Std_Dev":     round(s.std(), 0),
                    })

        # ── Factor 2: Family × Month heat matrix ──────────────────
        family_month = (df.groupby(["Family","Month"])["Gross Sales"]
                          .sum().unstack(fill_value=0))
        # Normalise: divide by column (month) total to get share
        family_month_share = family_month.div(family_month.sum(axis=0) + 1)
        # Find over-indexing cells (share > global expected)
        global_share = family_month.sum(axis=1) / family_month.sum().sum()
        over_index   = family_month_share.sub(global_share, axis=0)
        over_index.columns = [MONTHS[m-1] for m in over_index.columns]

        mvt_df     = pd.DataFrame(mvt_rows)
        oi_df      = over_index.reset_index()

        self.results["mvt_rdkro_qtr"] = mvt_df
        self.results["mvt_family_month"] = oi_df
        print(f"     ✅ {len(mvt_rows)} RD/KRO × Quarter combinations | {len(oi_df)} product families analysed")
        return mvt_df

    # ─────────────────────────────────────────────────────────────────
    # 7. LOGISTIC REGRESSION — at-risk / churn prediction
    # ─────────────────────────────────────────────────────────────────
    def churn_model(self, churn_threshold=0.6):
        """
        Binary classification: predict which partners are 'at risk'
        (defined as Ach% < churn_threshold).

        Reports: AUC-ROC, precision, recall, F1, confusion matrix,
        calibration, and feature importances.

        Uses both Logistic Regression (interpretable) and
        Random Forest (for feature importance robustness).
        """
        print(f"  🚨 Running churn/at-risk model (threshold={churn_threshold:.0%})...")
        feats = self.features.copy()

        feature_cols = [
            "active_months","distinct_families","avg_order_value","yoy_growth",
            "consistency_score","product_hhi","trend_slope","order_frequency",
            "invoice_count","peak_to_trough","zero_month_count","h1_h2_ratio"
        ]
        feature_cols = [c for c in feature_cols if c in feats.columns]

        feats["at_risk"] = (feats["ach_pct"] < churn_threshold).astype(int)

        X = feats[feature_cols].fillna(0).values
        y = feats["at_risk"].values

        if y.sum() < 3 or (1-y).sum() < 3:
            print("     ⚠️  Insufficient class balance for churn model (need ≥3 of each class)")
            self.results["churn_eval"]   = pd.DataFrame()
            self.results["churn_scores"] = pd.DataFrame()
            return pd.DataFrame()

        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        # Logistic Regression
        lr = LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0, random_state=42)
        lr.fit(X_tr_s, y_tr)
        y_prob_lr = lr.predict_proba(X_te_s)[:,1]
        y_pred_lr = lr.predict(X_te_s)

        # Random Forest (for feature importance + ensemble)
        rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                     max_depth=6, random_state=42, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        y_prob_rf = rf.predict_proba(X_te)[:,1]
        y_pred_rf = rf.predict(X_te)

        # Ensemble (average probabilities)
        y_prob_ens = (y_prob_lr + y_prob_rf) / 2
        y_pred_ens = (y_prob_ens >= 0.5).astype(int)

        def _eval(name, y_true, y_pred, y_prob):
            cm = confusion_matrix(y_true, y_pred)
            return {
                "Model":     name,
                "AUC_ROC":   round(roc_auc_score(y_true, y_prob),3) if len(np.unique(y_true))>1 else None,
                "Avg_Precision": round(average_precision_score(y_true, y_prob),3) if len(np.unique(y_true))>1 else None,
                "Accuracy":  round(accuracy_score(y_true,y_pred),3),
                "Precision": round(precision_score(y_true,y_pred,zero_division=0),3),
                "Recall":    round(recall_score(y_true,y_pred,zero_division=0),3),
                "F1":        round(f1_score(y_true,y_pred,zero_division=0),3),
                "TP":        int(cm[1,1]) if cm.shape==(2,2) else 0,
                "FP":        int(cm[0,1]) if cm.shape==(2,2) else 0,
                "TN":        int(cm[0,0]) if cm.shape==(2,2) else 0,
                "FN":        int(cm[1,0]) if cm.shape==(2,2) else 0,
            }

        eval_df = pd.DataFrame([
            _eval("Logistic Regression", y_te, y_pred_lr, y_prob_lr),
            _eval("Random Forest",       y_te, y_pred_rf, y_prob_rf),
            _eval("Ensemble (avg)",      y_te, y_pred_ens, y_prob_ens),
        ])

        # Feature importances (permutation — model-agnostic)
        perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=42)
        feat_imp = pd.DataFrame({
            "Feature":    feature_cols,
            "LR_Coeff":   lr.coef_[0].round(4),
            "RF_Importance": rf.feature_importances_.round(4),
            "Perm_Importance_Mean": perm.importances_mean.round(4),
            "Perm_Importance_Std":  perm.importances_std.round(4),
        }).sort_values("Perm_Importance_Mean", ascending=False)

        # Predict on full dataset for partner-level scores
        X_full = scaler.transform(feats[feature_cols].fillna(0).values)
        feats["churn_prob_lr"] = lr.predict_proba(X_full)[:,1].round(3)
        feats["churn_prob_rf"] = rf.predict_proba(feats[feature_cols].fillna(0).values)[:,1].round(3)
        feats["churn_prob_ens"]= ((feats["churn_prob_lr"]+feats["churn_prob_rf"])/2).round(3)
        feats["churn_label"]   = feats["at_risk"].map({1:"At Risk","1":"At Risk",0:"Safe","0":"Safe"})

        risk_scores = feats[["SoldToPartyID"] + [c for c in ["Partner","RD/KRO","Branch"] if c in feats.columns] +
                             ["total_sales","ach_pct","churn_prob_ens","churn_label"]].sort_values("churn_prob_ens", ascending=False)

        self.results["churn_eval"]    = eval_df
        self.results["churn_feats"]   = feat_imp
        self.results["churn_scores"]  = risk_scores
        best_auc = eval_df["AUC_ROC"].max()
        print(f"     ✅ Best AUC-ROC={best_auc:.3f} | F1={eval_df['F1'].max():.3f}")
        return eval_df

    # ─────────────────────────────────────────────────────────────────
    # 8. SUPERVISED ML — next quarter sales prediction
    # ─────────────────────────────────────────────────────────────────
    def sales_forecast_model(self):
        """
        Predict next quarter (Q4) sales for each partner using Q1-Q3 data.
        Uses Gradient Boosting Regressor with cross-validation.
        Reports feature importance, RMSE, MAE, MAPE, R².
        """
        print("  🤖 Running supervised ML sales forecast...")
        feats = self.features.copy()

        # Target: Q4 actual sales
        # Features: Q1, Q2, Q3 plus engineered features
        input_cols = [
            "q1_sales","q2_sales","q3_sales",
            "active_months","distinct_families","avg_order_value",
            "yoy_growth","consistency_score","trend_slope",
            "invoice_count","product_hhi","annual_target","mom_avg_growth"
        ]
        input_cols = [c for c in input_cols if c in feats.columns]
        target_col = "q4_sales"

        if target_col not in feats.columns:
            print("     ⚠️  q4_sales not available (need full year data)")
            return pd.DataFrame()

        valid = feats[feats["total_sales"] > 0].copy()
        X     = valid[input_cols].fillna(0).values
        y     = valid[target_col].values

        if len(X) < 10:
            print("     ⚠️  Insufficient data for ML model (need ≥10 partners)")
            return pd.DataFrame()

        X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
            X, y, np.arange(len(valid)), test_size=0.25, random_state=42)

        # Gradient Boosting (robust to outliers, no scaling needed)
        gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                        max_depth=4, subsample=0.8,
                                        min_samples_leaf=2, random_state=42)
        gb.fit(X_tr, y_tr)
        y_pred = gb.predict(X_te)

        # Also train RF for comparison
        rf_r = RandomForestRegressor(n_estimators=200, max_depth=6,
                                      min_samples_leaf=2, random_state=42, n_jobs=-1)
        rf_r.fit(X_tr, y_tr)
        y_pred_rf = rf_r.predict(X_te)

        def _reg_eval(name, y_true, y_pred):
            mape = np.mean(np.abs((y_true-y_pred)/np.where(y_true>0,y_true,1)))*100
            return {
                "Model":   name,
                "R²":      round(r2_score(y_true, y_pred),3),
                "RMSE":    round(np.sqrt(mean_squared_error(y_true,y_pred)),0),
                "MAE":     round(mean_absolute_error(y_true,y_pred),0),
                "MAPE_%":  round(mape,1),
                "N_Test":  len(y_true),
            }

        eval_df = pd.DataFrame([
            _reg_eval("Gradient Boosting",  y_te, y_pred),
            _reg_eval("Random Forest",       y_te, y_pred_rf),
        ])

        # Cross-validated RMSE
        cv_rmse = np.sqrt(-cross_val_score(gb, X, y, cv=min(5,len(X)),
                                            scoring="neg_mean_squared_error")).mean()

        # Feature importance
        perm = permutation_importance(gb, X_te, y_te, n_repeats=10, random_state=42)
        fi_df = pd.DataFrame({
            "Feature":        input_cols,
            "GB_Importance":  gb.feature_importances_.round(4),
            "Perm_Mean":      perm.importances_mean.round(4),
            "Perm_Std":       perm.importances_std.round(4),
        }).sort_values("Perm_Mean", ascending=False)

        # Predict on all partners
        X_all = valid[input_cols].fillna(0).values
        valid["q4_predicted"] = gb.predict(X_all).round(0)
        valid["q4_actual"]    = valid[target_col]
        valid["q4_error"]     = (valid["q4_actual"] - valid["q4_predicted"]).round(0)
        valid["q4_error_pct"] = np.where(valid["q4_actual"]>0,
                                          valid["q4_error"]/valid["q4_actual"]*100, 0).round(1)

        pred_df = valid[["SoldToPartyID"] + [c for c in ["Partner","RD/KRO"] if c in valid.columns] +
                         ["q4_actual","q4_predicted","q4_error","q4_error_pct"]].sort_values("q4_actual",ascending=False)

        cv_row  = pd.DataFrame([{"Model":"GB (5-fold CV)", "R²":"—",
                                   "RMSE":round(cv_rmse,0),"MAE":"—","MAPE_%":"—","N_Test":"all"}])
        eval_df = pd.concat([eval_df, cv_row], ignore_index=True)

        self.results["ml_eval"]     = eval_df
        self.results["ml_fi"]       = fi_df
        self.results["ml_preds"]    = pred_df
        best_r2 = eval_df["R²"].replace("—",np.nan).astype(float).max()
        print(f"     ✅ Best R²={best_r2:.3f} | CV RMSE={cv_rmse:,.0f}")
        return eval_df

    # ─────────────────────────────────────────────────────────────────
    # 9. FORECASTING — Holt-Winters ETS per partner
    # ─────────────────────────────────────────────────────────────────
    def forecast_ets(self, horizon=3):
        """
        Holt-Winters Exponential Smoothing per partner.
        Forecasts next `horizon` months.
        Falls back to Simple Exponential Smoothing for short series.
        """
        print(f"  📅 Running ETS forecast (horizon={horizon} months)...")
        df = self.df_yr.copy()

        monthly = (df.groupby(["SoldToPartyID","Month"])["Gross Sales"]
                     .sum().unstack(fill_value=0)
                     .reindex(columns=range(1,13), fill_value=0))

        forecasts = []
        for pid, row in monthly.iterrows():
            series = row.values.astype(float)
            n_nonzero = (series > 0).sum()

            if n_nonzero < 3:
                # Too few points — use mean
                fcast = [series[series>0].mean() if n_nonzero>0 else 0] * horizon
                method = "Mean"
            elif n_nonzero < 6:
                # Simple Exponential Smoothing (SES)
                alpha = 0.3
                level = series[0]
                for v in series[1:]:
                    if v > 0: level = alpha*v + (1-alpha)*level
                fcast  = [level] * horizon
                method = "SES"
            else:
                # Holt's Double Exponential Smoothing (trend)
                alpha, beta = 0.3, 0.1
                level = series[0]; trend = series[1]-series[0] if series[1]>0 else 0
                for v in series[1:]:
                    if v > 0:
                        prev_level = level
                        level = alpha*v + (1-alpha)*(level+trend)
                        trend = beta*(level-prev_level) + (1-beta)*trend
                fcast  = [max(0, level + h*trend) for h in range(1, horizon+1)]
                method = "Holt DES"

            total_cy = series.sum()
            fcst_dict = {"SoldToPartyID": pid, "Forecast_Method": method,
                          "CY_Total_Sales": round(total_cy, 0)}
            for h, f in enumerate(fcast, 1):
                month_num = (series.tolist().index(max(filter(lambda x: x>0, series), default=0)) + h) % 12 + 1
                fcst_dict[f"Fcst_Month{h}_{MONTHS[(12-1+h)%12]}"] = round(max(0, f), 0)
            fcst_dict["Fcst_Total_3M"] = round(sum(fcast), 0)
            fcst_dict["Fcst_vs_Q_Avg"]  = round(sum(fcast)/3 / (total_cy/12+1), 3)
            forecasts.append(fcst_dict)

        fcast_df = pd.DataFrame(forecasts)
        fcast_df = fcast_df.merge(self.meta_clean[["SoldToPartyID"] + [c for c in ["Partner","RD/KRO","Branch"] if c in self.meta_clean.columns]],
                                   on="SoldToPartyID", how="left")
        fcast_df = fcast_df.sort_values("Fcst_Total_3M", ascending=False)

        self.results["forecast"] = fcast_df
        print(f"     ✅ Forecasts generated for {len(fcast_df)} partners")
        return fcast_df

    # ─────────────────────────────────────────────────────────────────
    # 10. SANITY CHECKS & VALIDATION
    # ─────────────────────────────────────────────────────────────────
    def sanity_checks(self):
        """
        Data quality and model output validation gates.
        Catches issues before downstream use:
          - Completeness, outliers, logical inconsistencies,
            model calibration, prediction sanity bounds.
        """
        print("  🔍 Running sanity checks & validation...")
        df    = self.df_yr.copy()
        feats = self.features.copy() if hasattr(self,"features") else pd.DataFrame()
        checks = []

        # ── Data quality checks ────────────────────────────────────
        def chk(name, category, passed, detail, severity="INFO"):
            checks.append({"Category":category, "Check":name,
                            "Status": "✅ PASS" if passed else "❌ FAIL",
                            "Severity": severity if not passed else "OK",
                            "Detail": detail})

        # 1. Null invoice dates
        null_dates = df["InvoiceDate"].isna().sum()
        chk("Null invoice dates","Completeness", null_dates==0,
            f"{null_dates} rows with missing dates", "CRITICAL")

        # 2. Negative sales
        neg_sales = (df["Gross Sales"] < 0).sum()
        chk("Negative sales values","Data Integrity", neg_sales==0,
            f"{neg_sales} rows with negative gross sales", "HIGH")

        # 3. Zero sales rows
        zero_sales = (df["Gross Sales"] == 0).sum()
        chk("Zero sales rows","Data Integrity", zero_sales < len(df)*0.05,
            f"{zero_sales} zero-value rows ({zero_sales/max(len(df),1)*100:.1f}%)", "MEDIUM")

        # 4. Duplicate invoices
        dup_check_cols = [c for c in ["SoldToPartyID","InvoiceDate","MaterialCode","Gross Sales"] if c in df.columns]
        dups = df.duplicated(subset=dup_check_cols).sum() if dup_check_cols else 0
        chk("Duplicate invoice rows","Completeness", dups==0,
            f"{dups} potential duplicate rows", "HIGH")

        # 5. Future dates
        future = (df["InvoiceDate"] > pd.Timestamp.now()).sum()
        chk("Future invoice dates","Data Integrity", future==0,
            f"{future} invoices dated in the future", "HIGH")

        # 6. Outlier sales values (Z > 4)
        z_scores = np.abs(stats.zscore(df["Gross Sales"].fillna(0)))
        extreme  = (z_scores > 4).sum()
        chk("Extreme sales outliers (Z>4)","Statistical", extreme < len(df)*0.002,
            f"{extreme} rows with |Z|>4 — review for data entry errors", "MEDIUM")

        # 7. Partner coverage
        partners_with_sales = df["SoldToPartyID"].nunique()
        target_partners     = len(self.tgt_clean)
        coverage            = partners_with_sales / max(target_partners,1)
        chk("Partner sales coverage","Completeness", coverage > 0.85,
            f"{partners_with_sales}/{target_partners} partners have sales ({coverage:.0%})", "MEDIUM")

        # 8. Month coverage
        months_present = df["Month"].nunique()
        chk("Month coverage","Completeness", months_present >= 6,
            f"{months_present}/12 months have sales data", "INFO")

        # 9. Target vs actual reasonableness
        if not feats.empty and "ach_pct" in feats.columns:
            impossible_ach = (feats["ach_pct"] > 5).sum()
            chk("Impossible achievement >500%","Model Sanity", impossible_ach==0,
                f"{impossible_ach} partners with Ach%>500% — likely target data issue", "HIGH")

        # 10. Regression sanity — coefficients direction check
        if "regression" in self.results and not self.results["regression"].empty:
            reg = self.results["regression"]
            # active_months should positively predict sales
            if "active_months" in reg["Feature"].values:
                coeff = reg.loc[reg["Feature"]=="active_months","Ridge_Coeff"].iloc[0]
                chk("Regression coefficient direction (active_months)","Model Sanity", coeff > 0,
                    f"active_months coefficient={coeff:.0f} (expected >0)", "MEDIUM")

        # 11. Churn model calibration
        if "churn_scores" in self.results and not self.results["churn_scores"].empty:
            cs = self.results["churn_scores"]
            if "churn_prob_ens" in cs.columns:
                prob_range_ok = cs["churn_prob_ens"].between(0,1).all()
                chk("Churn probabilities in [0,1]","Model Sanity", prob_range_ok,
                    f"All {len(cs)} churn probabilities valid", "CRITICAL")
            else:
                chk("Churn model ran","Model Sanity", False,
                    "Churn model skipped — insufficient class balance", "INFO")

        # 12. Forecast sanity
        if "forecast" in self.results and not self.results["forecast"].empty:
            fcst = self.results["forecast"]
            fcst_cols = [c for c in fcst.columns if "Fcst_Month" in c]
            if fcst_cols:
                max_fcst = fcst[fcst_cols].max().max()
                max_hist = max(df["Gross Sales"].sum() / 12 * 3, 1)
                chk("Forecast values not absurdly large","Model Sanity",
                    max_fcst < max_hist * 10,
                    f"Max 3M forecast: {max_fcst:,.0f} vs 3M historical avg: {max_hist:,.0f}","MEDIUM")

        checks_df = pd.DataFrame(checks)
        n_fail = (checks_df["Status"]=="❌ FAIL").sum()
        n_pass = (checks_df["Status"]=="✅ PASS").sum()
        self.results["sanity_checks"] = checks_df
        print(f"     ✅ {n_pass} passed | {'⚠️ ' if n_fail>0 else ''}{n_fail} failed")
        return checks_df

    # ─────────────────────────────────────────────────────────────────
    # RUN ALL
    # ─────────────────────────────────────────────────────────────────
    def run_all(self):
        """Run the full pipeline in sequence."""
        print("\n" + "="*60)
        self.feature_engineering()
        self.regression_analysis()
        self.anova_analysis()
        self.ab_test()
        self.causal_impact()
        self.multivariate_test()
        self.churn_model()
        self.sales_forecast_model()
        self.forecast_ets()
        self.sanity_checks()
        print("="*60)
        print(f"✅ Full pipeline complete — {len(self.results)} result tables ready")
        return self.results

    # ─────────────────────────────────────────────────────────────────
    # EXPORT TO EXCEL
    # ─────────────────────────────────────────────────────────────────
    def export_excel(self, output_path=None):
        """Write all results to a multi-sheet Excel workbook."""
        print("\n  📝 Building Excel report...")
        _hex = lambda c: c.lstrip("#").upper()   # house palette (cell 0.5), openpyxl wants RRGGBB
        PAL = {
            "navy": _hex(THEME["band_bg"]), "blue": _hex(THEME["series"][2]), "green": _hex(THEME["header_bg"]),
            "amber": _hex(THEME["series"][5]), "red": _hex(THEME["series"][4]), "purple": _hex(THEME["series"][3]),
            "teal": _hex(THEME["series"][0]), "schneider": _hex(THEME["accent"]), "white": "FFFFFF",
        }

        wb = Workbook(); wb.remove(wb.active)

        sheet_map = [
            ("📋 Summary",          "summary",           PAL["navy"]),
            ("1. Features",          "features",          PAL["blue"]),
            ("2. Regression",        "regression",        PAL["blue"]),
            ("3. ANOVA",             "anova",             PAL["teal"]),
            ("4. A/B Test",          "ab_test",           PAL["green"]),
            ("5. Causal Impact",     "causal_impact",     PAL["purple"]),
            ("6. Multivariate",      "mvt_rdkro_qtr",     PAL["amber"]),
            ("7. Churn Scores",      "churn_scores",      PAL["red"]),
            ("7b. Churn Eval",       "churn_eval",        PAL["red"]),
            ("8. ML Forecast",       "ml_preds",          PAL["schneider"]),
            ("8b. ML Eval",          "ml_eval",           PAL["schneider"]),
            ("9. ETS Forecast",      "forecast",          PAL["blue"]),
            ("10. Sanity Checks",    "sanity_checks",     PAL["navy"]),
        ]

        # ── Summary sheet ─────────────────────────────────────────
        ws_sum = wb.create_sheet("📋 Summary")
        _title(ws_sum, "RETAIL CHANNEL — ADVANCED ANALYTICS REPORT   |   Full DS Pipeline", PAL["navy"])
        ws_sum.sheet_view.showGridLines = False

        # Pipeline table
        pipeline = [
            ("1","Feature Engineering","Domain features from raw sales data","features"),
            ("2","Regression (OLS+Ridge)","What drives partner sales? Significance & VIF","regression"),
            ("3","ANOVA","Do regions/segments differ significantly?","anova"),
            ("4","A/B Test","Did the scheme/intervention work?","ab_test"),
            ("5","Causal Impact","Synthetic control + interrupted time series","causal_impact"),
            ("6","Multivariate Test","RD/KRO × Quarter interaction effects","mvt_rdkro_qtr"),
            ("7","Logistic Regression","At-risk partner prediction (AUC-ROC)","churn_scores"),
            ("8","Gradient Boosting ML","Next-quarter sales prediction","ml_preds"),
            ("9","ETS Forecasting","Per-partner 3-month Holt-Winters forecast","forecast"),
            ("10","Sanity Checks","Data quality & model validation gates","sanity_checks"),
        ]
        _hdr(ws_sum, 3, ["#","Analysis","What it does","Result Sheet"], PAL["navy"])
        for ri, (num, name, desc, sheet) in enumerate(pipeline, 4):
            for ci, v in enumerate([num, name, desc, sheet], 1):
                cell = ws_sum.cell(row=ri, column=ci, value=v)
                _sc(cell, ha="left" if ci>1 else "center")
        for ci, w in enumerate([5,25,50,20], 1):
            ws_sum.column_dimensions[get_column_letter(ci)].width = w

        # Sanity check summary on summary sheet
        if "sanity_checks" in self.results:
            sc = self.results["sanity_checks"]
            n_pass = (sc["Status"]=="✅ PASS").sum()
            n_fail = (sc["Status"]=="❌ FAIL").sum()
            ws_sum.cell(row=16, column=1, value="Data Quality Gate:").font = Font(bold=True)
            ws_sum.cell(row=16, column=2, value=f"{n_pass} passed / {n_fail} failed").font = Font(
                bold=True, color="276221" if n_fail==0 else "C00000")

        # ── Write all result sheets ──────────────────────────────
        for sheet_name, key, color in sheet_map:
            if key not in self.results or self.results[key] is None:
                continue
            df_out = self.results[key]
            if not isinstance(df_out, pd.DataFrame) or df_out.empty:
                continue
            safe_name = sheet_name.replace('/','').replace('\\','')[:31]
            ws  = wb.create_sheet(safe_name)
            _title(ws, sheet_name.lstrip("📋123456789b. ").upper() + f" — {self.year}", color)
            ws.sheet_view.showGridLines = False
            _write_df(ws, df_out.head(500), start_row=3, bg_hdr=color)

            # Conditional formatting on p-values
            if "p_Value" in df_out.columns:
                col_idx = list(df_out.columns).index("p_Value") + 1
                ltr = get_column_letter(col_idx)
                ws.conditional_formatting.add(
                    f"{ltr}3:{ltr}{2+min(len(df_out),500)}",
                    CellIsRule(operator='lessThan', formula=['0.05'],
                               fill=_f("C6EFCE"), font=Font(color="276221", bold=True)))
            # Conditional formatting on AUC/R²
            for metric_col in ["AUC_ROC","R²"]:
                if metric_col in df_out.columns:
                    col_idx = list(df_out.columns).index(metric_col) + 1
                    ltr = get_column_letter(col_idx)
                    ws.conditional_formatting.add(
                        f"{ltr}3:{ltr}{2+min(len(df_out),500)}",
                        ColorScaleRule(start_type="min", start_color="FCE4D6",
                                        end_type="max",   end_color="C6EFCE"))

        if output_path:
            wb.save(output_path)
            print(f"  ✅ Saved → {output_path}")
            return None

        buf = io.BytesIO(); wb.save(buf); buf.seek(0)
        print(f"  ✅ Built in memory ({buf.getbuffer().nbytes/1024:.0f} KB)")
        return buf

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 7.4c HS MATERIAL REPORT — every material code with its Helios code
#      (incl. PPCFD), activity, and Sales split Retail / Project + total
#      REPORTS["hs_material_report"]
#
# Scope = HS, all RETAIL_HELIOS_CODES (PPCFD included), Sales rows only.
#   • Retail = retail PricingClass L2/L3 (is_retail), Project = everything else
#   • Net Sales = Gross × (1 − rebate) from RETAIL_REBATES / PROJECT_REBATES
#   • Period: HS_MATERIAL_PERIOD = "ytd" (01-Jan → yesterday) or "year" (full
#     TARGET_YEAR); HS_MATERIAL_WITH_LY = True adds the same period last year
# Sheets: By activity (summary) · Materials (one row per material × Helios
# code) · both with Gross and Net in Rs, quantity, Retail share and totals.
# ════════════════════════════════════════════════════════════════════
HS_MATERIAL_PERIOD = "ytd"          # "ytd" or "year"
HS_MATERIAL_WITH_LY = True          # add last year's same period (Gross) and growth

HS_ACTIVITY = {"HDCWD": "WD", "HSCWD": "WD", "HDCFD": "FD", "PPCFD": "FD", "HDFDA": "FD", "HDFDB": "FD", "HDFDS": "FD",
               "HDCOS": "COS", "HSCOS": "COS", "HDSPE": "SPE", "HSSPE": "SPE", "PREVC": "EVC",
               "HDCLD": "Lighting", "HDCLN": "Lighting", "HDCLS": "Lighting", "HDLSD": "Lighting", "HDPNL": "Panels"}
HS_ACTIVITY_NAME = {"WD": "Wiring Devices", "FD": "Final Distribution", "COS": "COS", "SPE": "SPE",
                    "EVC": "EV Charging", "Lighting": "Lighting", "Panels": "Panels", "Other": "Other"}


def hs_material_data():
    base = build_hs_dataset(df_Sales_final, "InvoiceDate", RETAIL_HELIOS_CODES, retail_branch_map={})
    if base.empty:
        return pd.DataFrame(), ""
    d = pd.to_datetime(base["InvoiceDate"], errors="coerce")
    if HS_MATERIAL_PERIOD == "ytd":
        ty = (d.dt.year == TARGET_YEAR) & (d <= YESTERDAY)
        ly = (d.dt.year == PREV_YEAR) & (d <= LY_YESTERDAY)
        period = f"YTD {TARGET_YEAR}: 01-Jan to {YESTERDAY:%d-%b-%Y}"
        if HS_MATERIAL_WITH_LY:
            period += f"  vs  YTD {PREV_YEAR}: 01-Jan to {LY_YESTERDAY:%d-%b-%Y}"
    else:
        ty, ly = d.dt.year == TARGET_YEAR, d.dt.year == PREV_YEAR
        period = f"Full year {TARGET_YEAR}" + (f"  vs  full year {PREV_YEAR}" if HS_MATERIAL_WITH_LY else "")
    keep = ty | (ly if HS_MATERIAL_WITH_LY else False)
    df = base[keep].copy()
    df["Period"] = np.where(ty[keep], "TY", "LY")

    df["MaterialCode"] = df["MaterialCode"].astype(str).str.strip().str.upper()
    df = lookup_merge(df, family_map, "material master", left_on="MaterialCode", right_on="Material")
    df["Helios Code"] = df["HeliosCode"].astype(str).str.strip().str.upper()
    df["Activity"] = df["Helios Code"].map(HS_ACTIVITY).fillna("Other")
    df["Activity Name"] = (df["HeliosDescription"].astype(str).str.strip() if "HeliosDescription" in df.columns
                           else df["Activity"].map(HS_ACTIVITY_NAME))
    df["Activity Name"] = df["Activity Name"].replace({"": None, "nan": None}).fillna(df["Activity"].map(HS_ACTIVITY_NAME))
    df["Material Description"] = df["MaterialDescription"].astype(str).str.strip() if "MaterialDescription" in df.columns else ""
    df["Family Actual"] = df["Family Actual"].fillna("Unclassified")
    df["BillingQuantity"] = pd.to_numeric(df.get("BillingQuantity", 0), errors="coerce").fillna(0)
    return df, period


def _hs_material_table(df, keys):
    ty = df[df["Period"] == "TY"]
    out = None
    for value, label in (("Gross Sales", "Gross Sales"), ("Net Sales", "Net Sales"), ("BillingQuantity", "Qty")):
        p = ty.pivot_table(index=keys, columns="Classification", values=value, aggfunc="sum", fill_value=0)
        p = p.reindex(columns=["Retail", "Project"], fill_value=0)
        p.columns = [f"Retail {label}", f"Project {label}"]
        p[f"Total {label}"] = p.sum(axis=1)
        out = p if out is None else out.join(p, how="outer")
    out = out.fillna(0)
    out["Retail Share %"] = np.where(out["Total Gross Sales"] != 0, out["Retail Gross Sales"] / out["Total Gross Sales"].replace(0, np.nan), 0.0)
    if HS_MATERIAL_WITH_LY:
        ly = df[df["Period"] == "LY"].groupby(keys)["Gross Sales"].sum()
        out = out.join(ly.rename(f"Total Gross Sales {PREV_YEAR}"), how="outer").fillna(0)
        out["Growth %"] = [(t - l) / l if l else None for t, l in zip(out["Total Gross Sales"], out[f"Total Gross Sales {PREV_YEAR}"])]
    return out.reset_index()


def report_hs_material_report():
    df, period = hs_material_data()
    if df.empty:
        return "no HS sales in the period"
    mat_keys = ["MaterialCode", "Material Description", "Family Actual", "Helios Code", "Activity", "Activity Name"]
    materials = _hs_material_table(df, mat_keys).rename(columns={"MaterialCode": "Material Code"})
    materials = materials.sort_values(["Activity", "Total Gross Sales"], ascending=[True, False])
    by_act = _hs_material_table(df, ["Helios Code", "Activity", "Activity Name"])
    by_act.insert(3, "Materials", by_act["Helios Code"].map(materials.groupby("Helios Code")["Material Code"].nunique()).fillna(0).astype(int))
    by_act = by_act.sort_values("Total Gross Sales", ascending=False)
    pct = {"Retail Share %": "pct", "Growth %": "pct"}
    qty = {c: "int" for c in materials.columns if c.endswith(" Qty")}
    note = (f"{period}  |  Rs  |  Retail = retail PricingClass L2/L3, Project = rest  |  "
            f"Net = Gross × (1 − rebate)  |  Helios codes: {', '.join(RETAIL_HELIOS_CODES)}")
    tot = materials["Total Gross Sales"].sum()
    print(f"   {materials['Material Code'].nunique():,} materials | Gross Rs {tot / 1e6:,.1f} Mn "
          f"(Retail {materials['Retail Gross Sales'].sum() / 1e6:,.1f} + Project {materials['Project Gross Sales'].sum() / 1e6:,.1f})")

    def build(writer):
        write_table(writer, by_act, "By Activity", title="HS — Sales by Helios code / activity, Retail vs Project",
                    subtitle=note, kinds={**pct, **qty, "Materials": "int"}, totals=True, tab=THEME["tab_summary"])
        write_table(writer, materials, "Materials", title="HS — Material-wise Sales, Retail vs Project",
                    subtitle=note, kinds={**pct, **qty}, totals=True, freeze_cols=2, tab=THEME["tab_detail"])
    tag = f"YTD_{MONTH_ABBR}" if HS_MATERIAL_PERIOD == "ytd" else "FY"
    return save_report(build, f"HS_Material_Retail_Project_Sales_{tag}_{TARGET_YEAR}.xlsx", local_dir=OUTPUT_DIR)


run_report("hs_material_report", report_hs_material_report)


In [ ]:
# ════════════════════════════════════════════════════════════════════
# 7.2 MUMBAI RETAIL + PROJECT REPORT (ad-hoc)  — REPORTS["mumbai_report"]
# Wrapped in a function so its helpers don't overwrite the shared ones.
# ════════════════════════════════════════════════════════════════════
def report_mumbai_report():
    import os
    import io
    import pandas as pd
    import msal
    import requests
    from urllib.parse import quote

    # ════════════════════════════════════════════════════════════════════
    # Mumbai (not West) Retail + Project Partner Report -- for Premika's request
    #   1. Retail Partners in Mumbai: names + sales, wide (years as columns),
    #      2022-2026 (2026 = YTD Aug + FY projection). Partner attributes
    #      (GSTN, RD/KRO, Region, Branch, ASM, Tier Category) come from
    #      Data Dictionary.xlsx's Partner Master Data. Scoped by Branch == "MUM"
    #      -- a retail partner can have a City outside Mumbai/Navi Mumbai (e.g.
    #      Thane) while still legitimately being served out of the MUM branch,
    #      so Branch (not City) is the correct reference for Retail.
    #   2. Retailers per Mumbai RD, from Secondary/EasyDMS, 2022-2026
    #      (2026 = YTD July). Unchanged long format (per-year retailer list).
    #      Same Branch == "MUM" scoping as #1, since these are Retail Distributors.
    #   3. Project Partners in Mumbai: names + sales, same wide layout as #1,
    #      2022-2026 (2026 = YTD Aug + FY projection). Partner attributes come
    #      from Data Dictionary Retail + Project.xlsx's own Partner Master Data.
    #      Scoped by City in ["Mumbai", "Navi Mumbai"] -- Project partners don't
    #      have a reliable Branch of their own, so City is the correct reference.
    #
    # Assumes df_Sales_final is already loaded in this session. Everything
    # else (auth, Data Dictionary files, secondary DMS files, Targets file) is
    # fetched fresh by this script.
    # ════════════════════════════════════════════════════════════════════

    RETAIL_L2 = ['ED1', 'ED2', 'RE3']
    RETAIL_L3 = ['ERW', 'EKR', 'R36', 'ZD1']
    # Third leg of the retail filter -- matches the "single source of truth"
    # apply_retail_filter() used by the Target vs Achievement / Partner Policy
    # Tracker pipeline. Without this, PricingClassL2/L3 alone let some non-retail
    # Helios codes through, so this report's Retail Sales didn't match that
    # pipeline's numbers. All three conditions must hold together (AND), not any
    # one alone.
    RETAIL_HELIOS_CODES = [
        'HDCFD', 'HDCWD', 'HDCOS', 'HDSPE', 'HDCLD', 'HDCLN', 'HDCLS',
        'HDFDA', 'HDFDB', 'HDFDS', 'HDLSD', 'HDPNL', 'HSCOS', 'HSCWD',
        'HSSPE', 'PPCFD', 'PREVC',
    ]

    BASE_MASTER_PATH = "Test - haripriya/Data Downloads - Input Files/Master data"
    DICT_RETAIL_PATH = f"{BASE_MASTER_PATH}/Data Dictionary.xlsx"
    DICT_PROJECT_PATH = f"{BASE_MASTER_PATH}/Data Dictionary Retail + Project.xlsx"
    SECONDARY_FOLDER = "Test - haripriya/Data Downloads - Input Files/Secondary/EasyDMS"
    TARGET_MASTER_PATH = "Test - haripriya/Data Downloads - Input Files/Master data/Targets File"
    TARGETS_FILE_PATH = f"{TARGET_MASTER_PATH}/QTOD and ATOD file.xlsx"

    YEARS = [2022, 2023, 2024, 2025, 2026]
    YTD_MONTH_PRIMARY = 8   # August, for asks 1 & 3
    YTD_MONTH_SECONDARY = 7  # July, for ask 2, per the request text
    OUTPUT_PATH = os.path.join(OUTPUT_DIR, "Mumbai_Retail_Project_Report.xlsx")

    # Mumbai scoping differs by partner type, per instruction:
    #   - Retail Distributors  -> Branch == TARGET_BRANCH_RETAIL (exact branch code)
    #   - Project Distributors -> City in TARGET_CITIES
    # A partner can have real retail sales but a City of e.g. "Thane" while still
    # being served out of the MUM branch -- Branch is the correct reference for
    # Retail, City is correct for Project. Don't conflate the two.
    TARGET_BRANCH_RETAIL = "MUM"
    TARGET_CITIES = ["Mumbai", "Navi Mumbai"]  # City-column match, for Project scoping only

    ATTR_COLS = ["SoldToPartyID", "GSTN", "Patner", "RD/KRO", "Region", "Branch", "City", "ASM", "Tier Category", "New/Revival"]

    # --- 2026 "Parivartan guidelines" secondary data: slabs, not exact Amount ---
    # Starting 2026, the secondary DMS export no longer carries an exact
    # Amount/Amount After Discount value -- it groups each invoice into a value
    # band ("Slab") instead (e.g. "<25K", "25K-50K", ... "15Lakh-20Lakh"). Per
    # instruction, 2026 Secondary Sales are approximated using each slab's MID
    # BOUND (the average of its lower and upper bound -- a balanced estimate,
    # not the conservative lower-bound-only figure used earlier). Every 2026
    # figure derived this way is labeled "(approx.)" in the output so it is
    # never mistaken for an exact figure the way 2022-2025 values are.
    #
    # The 2026 file itself is a different filename/path than the "SEIN -
    # Secondary  - EasyDMS {year}.xlsx" pattern used for 2021-2025 -- confirmed
    # actual name: "DMS Dump Sales-Invoice-Register(01_Jan_2026_To_13_Aug_2026)".
    # Update SECONDARY_2026_PATH again once it has a permanent, differently-named
    # location in the same SharePoint folder for future runs.
    SECONDARY_2026_PATH = f"{SECONDARY_FOLDER}/DMS Dump Sales-Invoice-Register(01_Jan_2026_To_13_Aug_2026).xlsx"


    def slab_mid_bound(slab):
        """Parse a 'Slab' band string (e.g. '25K-50K', '<25K', '6Lakh-10Lakh ')
        into its MID-point in rupees (average of the band's lower and upper
        bound). '<25K' has no lower bound given, so it's treated as 0-25K and
        the midpoint is 12.5K. Returns NaN for unparseable/missing values
        (e.g. '#N/A')."""
        if pd.isna(slab):
            return float("nan")
        s = str(slab).strip()
        if s in ("", "#N/A", "nan"):
            return float("nan")

        def to_rupees(token):
            token = token.strip()
            if token.endswith("Lakh"):
                return float(token[:-4]) * 100000
            if token.endswith("K"):
                return float(token[:-1]) * 1000
            return float(token)

        if s.startswith("<"):
            try:
                upper = to_rupees(s[1:])
            except ValueError:
                return float("nan")
            return upper / 2

        parts = s.split("-")
        if len(parts) != 2:
            return float("nan")
        try:
            lower = to_rupees(parts[0])
            upper = to_rupees(parts[1])
        except ValueError:
            return float("nan")
        return (lower + upper) / 2


    # --- AUTH & FETCH HELPERS (self-contained, fresh token every run) ---
    import time
    import requests.exceptions as req_exc

    # The corporate machine this runs on has intermittently reset connections to
    # login.microsoftonline.com / graph.microsoft.com mid-request (McAfee
    # Endpoint Security + a non-persistent VDI profile are the suspected cause --
    # see earlier debugging in this project). That's a transient network-layer
    # failure, not a bug in this script's logic, so every network call here
    # retries a few times with backoff instead of letting one blip kill the
    # whole run.
    NETWORK_RETRY_ATTEMPTS = 4
    NETWORK_RETRY_BACKOFF_SECONDS = 3  # 3s, 6s, 12s, 24s


    def _with_retries(func, description):
        last_err = None
        for attempt in range(1, NETWORK_RETRY_ATTEMPTS + 1):
            try:
                return func()
            except (req_exc.ConnectionError, req_exc.Timeout, req_exc.ChunkedEncodingError) as e:
                last_err = e
                if attempt == NETWORK_RETRY_ATTEMPTS:
                    break
                wait = NETWORK_RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1))
                print(f"⚠️ Network error on {description} (attempt {attempt}/{NETWORK_RETRY_ATTEMPTS}): {e} -- retrying in {wait}s...")
                time.sleep(wait)
        raise ConnectionError(f"{description} failed after {NETWORK_RETRY_ATTEMPTS} attempts: {last_err}") from last_err


    def get_graph_headers():
        def _acquire():
            app = msal.ConfidentialClientApplication(
                os.environ["SP_CLIENT_ID"],
                authority=f"https://login.microsoftonline.com/{os.environ['SP_TENANT_ID']}",
                client_credential=os.environ["SP_CLIENT_SECRET"],
            )
            token = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
            if "access_token" not in token:
                raise Exception(f"Authentication failed: {token.get('error_description')}")
            return {"Authorization": f"Bearer {token['access_token']}"}

        return _with_retries(_acquire, "token acquisition (login.microsoftonline.com)")


    def get_drive_id(headers, site_path="schneiderelectric.sharepoint.com:/sites/HDIndiaTeamGroup"):
        site_res = _with_retries(
            lambda: requests.get(f"https://graph.microsoft.com/v1.0/sites/{site_path}", headers=headers).json(),
            "site lookup (Graph API)",
        )
        drives = _with_retries(
            lambda: requests.get(f"https://graph.microsoft.com/v1.0/sites/{site_res['id']}/drives", headers=headers).json()["value"],
            "drive lookup (Graph API)",
        )
        return next(d["id"] for d in drives if d["name"] == "Documents")


    def fetch_sp_file(drive_id, headers, file_path):
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root:/{quote(file_path)}"
        res = _with_retries(lambda: requests.get(url, headers=headers).json(), f"file lookup ({file_path})")
        if "@microsoft.graph.downloadUrl" not in res:
            raise FileNotFoundError(f"Could not locate '{file_path}': {res.get('error', res)}")
        content = _with_retries(
            lambda: requests.get(res["@microsoft.graph.downloadUrl"]).content,
            f"file download ({file_path})",
        )
        return io.BytesIO(content)


    # --- PARTNER MAP: generic loader, works for either Data Dictionary file ---
    #
    # Branch/Region resolution differs by partner type, per instruction:
    #   - Retail  -> branch_mode="direct":   use the Branch/Zone columns already
    #                sitting on Partner Master Data itself (Data Dictionary is
    #                the reference -- no derivation).
    #   - Project -> branch_mode="city_join": Partner Master Data has no
    #                reliable Branch of its own for Project partners, so derive
    #                it by joining City against the Geography mapping sheet.
    def build_partner_map(drive_id, headers, dict_path, branch_mode="direct"):
        buf1 = fetch_sp_file(drive_id, headers, dict_path)
        partner_raw = pd.read_excel(buf1, sheet_name="Partner Master Data")
        partner_raw.columns = [str(c).strip() for c in partner_raw.columns]
        partner_raw["Sold-to Party"] = partner_raw["Sold-to Party"].astype(str).str.strip()
        partner_raw["City"] = partner_raw["City"].astype(str).str.strip()

        p_name_col = "Patner" if "Patner" in partner_raw.columns else "Partner Name"
        rd_kro_col = "RD/KRO" if "RD/KRO" in partner_raw.columns else None
        gstn_col = "GSTN" if "GSTN" in partner_raw.columns else None
        tier_col = "Tier Category" if "Tier Category" in partner_raw.columns else None
        asm_col = "ASM" if "ASM" in partner_raw.columns else None

        if branch_mode == "direct":
            # Take Branch straight from Data Dictionary's own Partner Master
            # Data -- no Geography-mapping join. The actual sheet uses
            # 'Branch_dummy' rather than a plain 'Branch' column; accept either
            # name so this doesn't break again if it's ever renamed upstream.
            branch_src_col = next((c for c in ("Branch", "Branch_dummy") if c in partner_raw.columns), None)
            if branch_src_col is None:
                raise KeyError(
                    f"branch_mode='direct' requires a 'Branch' (or 'Branch_dummy') column on Partner Master Data "
                    f"in {dict_path}, but none was found. Columns present: {list(partner_raw.columns)}"
                )
            region_src_col = next((c for c in ("Region", "Zone") if c in partner_raw.columns), None)

            pmap = partner_raw.copy()
            if branch_src_col != "Branch":
                pmap = pmap.rename(columns={branch_src_col: "Branch"})
            if region_src_col and region_src_col != "Region":
                pmap = pmap.rename(columns={region_src_col: "Region"})
            if "Region" not in pmap.columns:
                pmap["Region"] = None
        elif branch_mode == "city_join":
            buf2 = fetch_sp_file(drive_id, headers, dict_path)
            geo = pd.read_excel(buf2, sheet_name="Geography mapping", usecols=["City", "Zone", "Branch"])
            geo.columns = [str(c).strip() for c in geo.columns]
            geo["City"] = geo["City"].astype(str).str.strip()
            pmap = partner_raw.merge(geo.drop_duplicates("City"), on="City", how="left")
            pmap = pmap.rename(columns={"Zone": "Region"})
        else:
            raise ValueError(f"Unknown branch_mode: {branch_mode!r} (expected 'direct' or 'city_join')")

        keep_cols = ["Sold-to Party", p_name_col, "City", "Branch", "Region"]
        rename_map = {p_name_col: "Patner"}
        for col, std_name in [
            (gstn_col, "GSTN"), (rd_kro_col, "RD/KRO"), (tier_col, "Tier Category"), (asm_col, "ASM"),
        ]:
            if col:
                keep_cols.append(col)
                if col != std_name:
                    rename_map[col] = std_name

        pmap = pmap[keep_cols].drop_duplicates("Sold-to Party").rename(columns=rename_map)
        for missing in ["GSTN", "RD/KRO", "Tier Category", "ASM"]:
            if missing not in pmap.columns:
                pmap[missing] = None

        found_cities = [c for c in TARGET_CITIES if (pmap["City"].str.strip().str.lower() == c.lower()).any()]
        missing_cities = [c for c in TARGET_CITIES if c not in found_cities]
        if missing_cities:
            print(f"⚠️ [{dict_path.split('/')[-1]}] No partners found with City == {missing_cities} -- check spelling in Partner Master Data.")
        print(f"✅ [{dict_path.split('/')[-1]}] (branch_mode={branch_mode}) Scoping by City in {TARGET_CITIES}; found partners in: {found_cities}")
        return pmap


    # --- PARTNER CODE REMAP: old code == new code, for partner-wise reports only ---
    #
    # Data Dictionary.xlsx's "Partner Code Changes" sheet (Old Code / New Code
    # columns) records a partner that was reissued a new Sold-to Party code.
    # Without consolidating those, the SAME partner's sales history splits
    # across two SoldToPartyIDs and can wrongly look like a brand-new partner
    # in the New/Revival check. Per the project's standing rule: apply this
    # remap ONLY inside partner-wise aggregation (like this report's Ask 1/3),
    # NEVER to raw/order-wise data -- df_sales_final itself is never mutated,
    # only the local copy each function builds internally.
    def build_partner_code_map(drive_id, headers, dict_path=None):
        dict_path = dict_path or DICT_RETAIL_PATH
        try:
            buf = fetch_sp_file(drive_id, headers, dict_path)
            changes = pd.read_excel(buf, sheet_name="Partner Code Changes")
        except (FileNotFoundError, ValueError) as e:
            print(f"⚠️ Could not load 'Partner Code Changes' from {dict_path.split('/')[-1]} ({e}) -- proceeding with no code remap.")
            return {}
        changes.columns = [str(c).strip() for c in changes.columns]
        if "Old Code" not in changes.columns or "New Code" not in changes.columns:
            print(f"⚠️ 'Partner Code Changes' sheet found but missing 'Old Code'/'New Code' columns "
                  f"(has: {list(changes.columns)}) -- proceeding with no code remap.")
            return {}
        old = changes["Old Code"].astype(str).str.split(".").str[0].str.strip()
        new = changes["New Code"].astype(str).str.split(".").str[0].str.strip()
        return dict(zip(old, new))


    def remap_partner_codes(id_series, code_map):
        """Single-pass old->new substitution on a local copy of a SoldToPartyID
        Series. Not chained (a code remapped twice in sequence won't compose in
        one call) -- matches the .replace(dict) semantics used elsewhere in this
        project, and Partner Code Changes hasn't needed multi-hop chains so far."""
        if not code_map:
            return id_series
        return id_series.replace(code_map)


    # --- TARGET MAP: 2026 Target, sourced from the separate QTOD/ATOD Targets file ---
    #
    # Targets do NOT come from Partner Master Data's own Target columns -- per
    # instruction, they come from a dedicated Targets file (QTOD - ATOD sheet),
    # using the same CC -> SoldToPartyID / "Target 100%" pattern already
    # established in the Target vs Achievement / Partner Policy Tracker
    # pipeline. That file is 2026-only (no 2025 Target column exists there), so
    # this report drops "2025 Target" entirely rather than mixing a Partner
    # Master Data-sourced 2025 figure with a Targets_Map-sourced 2026 figure.
    #
    # A partner reissued a new Sold-to Party code (Data Dictionary's "Partner
    # Code Changes" sheet) can have its Targets-file row still filed under the
    # OLD code -- the Targets file doesn't know about that remap. Sales and
    # partner attributes are already consolidated onto the NEW code (see
    # build_partner_code_map / remap_partner_codes), so the target has to be
    # remapped and re-summed the same way here, otherwise a code-changed
    # partner's Target either goes missing (it's sitting under the old code the
    # merge never looks for) or double counts if both codes independently carry
    # a row. Pass the same code_map used for the partner-wise report in.
    def build_target_map(drive_id, headers, targets_path=None, code_map=None):
        targets_path = targets_path or TARGETS_FILE_PATH
        buf = fetch_sp_file(drive_id, headers, targets_path)
        tgt = pd.read_excel(buf, sheet_name="QTOD - ATOD")
        tgt.columns = [str(c).strip() for c in tgt.columns]
        tgt["SoldToPartyID"] = tgt["CC"].astype(str).str.split(".").str[0].str.strip()
        tgt["SoldToPartyID"] = remap_partner_codes(tgt["SoldToPartyID"], code_map)
        tgt = tgt[["SoldToPartyID", "Target 100%"]].rename(columns={"Target 100%": "2026 Target"})
        tgt["2026 Target"] = pd.to_numeric(tgt["2026 Target"], errors="coerce").fillna(0)
        tgt = tgt.groupby("SoldToPartyID", as_index=False)["2026 Target"].sum()
        print(f"✅ [{targets_path.split('/')[-1]}] {len(tgt)} partner-wise 2026 Targets loaded from QTOD - ATOD "
              f"(old->new partner code changes consolidated).")
        return tgt


    # --- ASK 1 & 3: Retail / Project partners in Mumbai, WIDE format (years as columns) ---
    def build_primary_partner_report_wide(df_sales_final, pmap, classification_wanted, code_map=None, target_map=None):
        df = df_sales_final.copy()
        df.columns = [str(c).strip() for c in df.columns]
        # Local copy only -- remapped for this partner-wise report; raw
        # df_sales_final passed in by the caller is never mutated.
        df["SoldToPartyID"] = remap_partner_codes(df["SoldToPartyID"].astype(str).str.strip(), code_map)
        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
        df["Year"] = df["InvoiceDate"].dt.year

        # Classify EVERY transaction row, independent of which partner it
        # belongs to. This is what keeps the two reports clean:
        #   - classification_wanted="Retail"  -> only retail rows survive, so a
        #     Retail Partner's report shows Retail Sales only.
        #   - classification_wanted="Project" -> only non-retail rows survive,
        #     so a Project Partner's report is Project Sales only, with any
        #     Retail-classified transactions on that same Sold-to Party excluded.
        #
        # Retail = PricingClassL2 AND PricingClassL3 AND HeliosCode all matching
        # (all three, not any one) -- this must stay identical to
        # apply_retail_filter() in the Target vs Achievement / Partner Policy
        # Tracker pipeline, otherwise the two reports' Retail Sales won't match.
        has_pricing_cols = "PricingClassL2" in df.columns and "PricingClassL3" in df.columns
        has_helios_col = "HeliosCode" in df.columns
        if has_pricing_cols and has_helios_col:
            is_retail = (
                df["PricingClassL2"].isin(RETAIL_L2)
                & df["PricingClassL3"].isin(RETAIL_L3)
                & df["HeliosCode"].isin(RETAIL_HELIOS_CODES)
            )
            df["Classification"] = is_retail.map({True: "Retail", False: "Project"})
        elif has_pricing_cols:
            print("⚠️ 'HeliosCode' column not found -- Retail classification is falling back to "
                  "PricingClassL2/L3 only, which will NOT match the Target vs Achievement pipeline's numbers.")
            is_retail = df["PricingClassL2"].isin(RETAIL_L2) & df["PricingClassL3"].isin(RETAIL_L3)
            df["Classification"] = is_retail.map({True: "Retail", False: "Project"})
        else:
            df["Classification"] = "Project"

        df = df.merge(pmap, left_on="SoldToPartyID", right_on="Sold-to Party", how="left")

        # Mumbai scoping differs by partner type (see TARGET_BRANCH_RETAIL /
        # TARGET_CITIES comment above): Retail Distributors are scoped by Branch,
        # Project Distributors by City. A retail partner can have real sales and
        # a City outside Mumbai/Navi Mumbai (e.g. Thane) while still legitimately
        # belonging to the MUM branch -- City would wrongly exclude it.
        if classification_wanted == "Retail":
            scope_mask = df["Branch"].astype(str).str.strip().str.upper() == TARGET_BRANCH_RETAIL.upper()
        else:
            scope_mask = df["City"].astype(str).str.strip().str.lower().isin([c.lower() for c in TARGET_CITIES])

        scoped = df[scope_mask & (df["Classification"] == classification_wanted)].copy()

        if scoped.empty:
            return pd.DataFrame(columns=ATTR_COLS + [f"{y} Sales" for y in YEARS[:-1]] + ["2026 YTD Aug Sales", "2026 FY Projection"])

        # New/Revival: 0 sales in 2024 & 2025, but sales in 2026 (matches the
        # New/Revival convention already used elsewhere in this project).
        yearly = scoped.groupby(["SoldToPartyID", "Year"])["Gross Sales"].sum().unstack(fill_value=0)
        s24 = yearly.get(2024, pd.Series(0, index=yearly.index))
        s25 = yearly.get(2025, pd.Series(0, index=yearly.index))
        s26 = yearly.get(2026, pd.Series(0, index=yearly.index))
        new_revival = ((s24 == 0) & (s25 == 0) & (s26 > 0)).map({True: "New/Revival", False: "Existing"})

        # Attribute columns: first non-null row per partner
        attrs = scoped.drop_duplicates("SoldToPartyID").set_index("SoldToPartyID")[
            ["GSTN", "Patner", "RD/KRO", "Region", "Branch", "City", "ASM", "Tier Category"]
        ]
        attrs["New/Revival"] = new_revival.reindex(attrs.index).fillna("Existing")
        attrs = attrs.reset_index()

        # 2026 Target comes from the separate Targets_Map (QTOD/ATOD file), not
        # Partner Master Data -- merge it in here by SoldToPartyID. A partner
        # with no row in Targets_Map gets a Target of 0, same as any other
        # missing-value fallback in this report.
        if target_map is not None and not target_map.empty:
            attrs = attrs.merge(target_map, on="SoldToPartyID", how="left")
        else:
            attrs["2026 Target"] = None
        attrs["2026 Target"] = pd.to_numeric(attrs["2026 Target"], errors="coerce").fillna(0)

        # Year columns
        wide = attrs.copy()
        for yr in YEARS:
            if yr < 2026:
                col_name = f"{yr} Sales"
                sub = scoped[scoped["Year"] == yr].groupby("SoldToPartyID")["Gross Sales"].sum()
            else:
                col_name = "2026 YTD Aug Sales"
                sub = scoped[(scoped["Year"] == yr) & (scoped["InvoiceDate"].dt.month <= YTD_MONTH_PRIMARY)].groupby("SoldToPartyID")["Gross Sales"].sum()
            wide[col_name] = wide["SoldToPartyID"].map(sub).fillna(0)

        # 2025 YTD Aug Sales -- same Jan-Aug cutoff as "2026 YTD Aug Sales", so the
        # two are a fair apples-to-apples YoY comparison (2025 Sales alone is the
        # full year and isn't comparable to a partial 2026).
        sub_2025_ytd = scoped[(scoped["Year"] == 2025) & (scoped["InvoiceDate"].dt.month <= YTD_MONTH_PRIMARY)].groupby("SoldToPartyID")["Gross Sales"].sum()
        wide["2025 YTD Aug Sales"] = wide["SoldToPartyID"].map(sub_2025_ytd).fillna(0)

        wide["2026 FY Projection"] = wide["2026 YTD Aug Sales"] / YTD_MONTH_PRIMARY * 12

        # 2026 Projection (Target basis) -- the partner's assigned 2026 Target
        # itself, shown as a second, independent estimate of the year-end
        # number alongside the run-rate-based "2026 FY Projection" above. Kept
        # as a distinct column rather than blended into the run-rate figure,
        # since a target and an extrapolation from actuals are two different
        # kinds of number and shouldn't be presented as if they were one.
        wide["2026 Projection (Target basis)"] = wide["2026 Target"]

        ordered = (
            ATTR_COLS
            + ["2026 Target"]
            + [f"{y} Sales" for y in YEARS[:-1]]
            + ["2025 YTD Aug Sales", "2026 YTD Aug Sales", "2026 FY Projection", "2026 Projection (Target basis)"]
        )
        return wide[ordered].sort_values("2026 FY Projection", ascending=False)


    # --- ASK 2: Retailers per Mumbai RD, from Secondary/EasyDMS files ---
    def load_secondary_year(drive_id, headers, year):
        if year == 2026:
            # Parivartan guidelines: slab-based, no exact Amount column. Derive
            # an approximate "Amount After Discount" from each row's Slab
            # mid-point (average of the band's bounds -- a balanced estimate),
            # so downstream grouping code keeps working unchanged for every year.
            path = SECONDARY_2026_PATH
            buf = fetch_sp_file(drive_id, headers, path)
            xls = pd.ExcelFile(buf)
            sheet = xls.sheet_names[0]
            df = pd.read_excel(xls, sheet_name=sheet)
            df.columns = [str(c).strip() for c in df.columns]
            df["Distributor Code"] = df["Distributor Code"].astype(str).str.strip()
            df["Voucher Date"] = pd.to_datetime(df["Voucher Date"], errors="coerce")
            df["Amount After Discount"] = df["Slab"].apply(slab_mid_bound)
            n_unparsed = df["Amount After Discount"].isna().sum()
            if n_unparsed:
                print(f"⚠️ 2026: {n_unparsed:,} rows had an unparseable Slab value (e.g. '#N/A') and were excluded from the approx. sum.")
            return df

        path = f"{SECONDARY_FOLDER}/SEIN - Secondary  - EasyDMS {year}.xlsx"  # note: double space before the second dash, confirmed from the actual SharePoint filenames
        buf = fetch_sp_file(drive_id, headers, path)
        xls = pd.ExcelFile(buf)
        sheet = xls.sheet_names[0]  # e.g. "Sales-Invoice-Register(...)"
        df = pd.read_excel(xls, sheet_name=sheet)
        df.columns = [str(c).strip() for c in df.columns]
        df["Distributor Code"] = df["Distributor Code"].astype(str).str.strip()
        df["Voucher Date"] = pd.to_datetime(df["Voucher Date"], errors="coerce")
        return df


    def build_rd_retailer_report(drive_id, headers, pmap_retail):
        # Retail Distributors are scoped by Branch, not City -- same rule as
        # Ask 1 (see TARGET_BRANCH_RETAIL comment above).
        mumbai_rd = pmap_retail[pmap_retail["Branch"].astype(str).str.strip().str.upper() == TARGET_BRANCH_RETAIL.upper()]
        if "RD/KRO" in mumbai_rd.columns:
            mumbai_rd = mumbai_rd[mumbai_rd["RD/KRO"] == "RD"]
        mumbai_rd_ids = set(mumbai_rd["Sold-to Party"])
        if not mumbai_rd_ids:
            print("⚠️ No Mumbai RDs found in Partner Master Data -- check the RD/KRO column name/values.")

        detail_rows = []
        for yr in YEARS:
            try:
                sec = load_secondary_year(drive_id, headers, yr)
            except FileNotFoundError as e:
                print(f"⚠️ Secondary file for {yr} not found -- skipping. ({e})")
                continue

            if yr == 2026:
                sec = sec[sec["Voucher Date"].dt.month <= YTD_MONTH_SECONDARY]

            sec_mumbai = sec[sec["Distributor Code"].isin(mumbai_rd_ids)]
            if sec_mumbai.empty:
                print(f"⚠️ {yr}: no secondary rows matched a Mumbai RD's Distributor Code.")
                continue

            grp = (sec_mumbai.groupby(["Distributor Code", "Distributor Name", "Party Code", "Party Name"], as_index=False)
                   ["Amount After Discount"].sum())
            # 2026 is slab-mid-bound approximated (Parivartan guidelines) --
            # label the year, the figure, and a dedicated basis column so it's
            # never read as exact.
            grp["Year"] = str(yr) if yr < 2026 else "2026 YTD Jul (approx.)"
            grp["Sales Basis"] = "Exact" if yr < 2026 else "Approx. (slab mid bound)"
            detail_rows.append(grp)

        detail = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame(
            columns=["Distributor Code", "Distributor Name", "Party Code", "Party Name", "Amount After Discount", "Year", "Sales Basis"])
        detail = detail.rename(columns={"Amount After Discount": "Secondary Sales"})

        summary = (detail.groupby(["Year", "Distributor Code", "Distributor Name"], as_index=False)
                   .agg(Retailer_Count=("Party Code", "nunique"), Total_Secondary_Sales=("Secondary Sales", "sum")))

        return detail.sort_values(["Year", "Distributor Code"]), summary.sort_values(["Year", "Distributor Code"])


    # --- FORMATTING: shared styles + a writer that makes every sheet readable at a glance ---
    import datetime

    BRAND_BLUE = THEME["header_bg"]
    LIGHT_BLUE = THEME["kpi_bg"]
    ZEBRA_GREY = THEME["zebra"]
    APPROX_YELLOW = "#FFF6D9"
    NEW_GREEN = THEME["good"][0]


    def _money_cols(df):
        return [c for c in df.columns if ("Sales" in c or "Projection" in c) and pd.api.types.is_numeric_dtype(df[c])]


    def write_formatted_sheet(writer, df, sheet_name, title, subtitle=""):
        """Write one DataFrame as a polished, print-ready sheet: title band,
        styled header row, currency-formatted Sales columns, zebra striping,
        conditional highlights for New/Revival and approximate figures,
        autofilter, frozen header, and sensible auto-sized columns."""
        workbook = writer.book
        df = df.copy()
        if "SoldToPartyID" in df.columns:
            df = partner_front(df)
        n_rows, n_cols = df.shape
        money_cols = _money_cols(df)

        TITLE_ROW, SUB_ROW, HEADER_ROW = 0, 1, 2
        FIRST_DATA_ROW = HEADER_ROW + 1

        df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=HEADER_ROW, header=False)
        ws = writer.sheets[sheet_name]

        # --- formats ---
        fmt_title = workbook.add_format({"bold": True, "font_size": 14, "font_color": BRAND_BLUE})
        fmt_subtitle = workbook.add_format({"italic": True, "font_size": 9, "font_color": THEME["subtitle"]})
        fmt_header = workbook.add_format({
            "bold": True, "font_color": "white", "bg_color": BRAND_BLUE,
            "border": 1, "border_color": "#FFFFFF", "align": "center", "valign": "vcenter",
            "text_wrap": True,
        })
        fmt_cell = workbook.add_format({"border": 1, "border_color": THEME["border"], "valign": "vcenter"})
        fmt_cell_alt = workbook.add_format({"border": 1, "border_color": THEME["border"], "valign": "vcenter", "bg_color": ZEBRA_GREY})
        fmt_money = workbook.add_format({"border": 1, "border_color": THEME["border"], "valign": "vcenter", "num_format": "#,##0"})
        fmt_money_alt = workbook.add_format({"border": 1, "border_color": THEME["border"], "valign": "vcenter", "bg_color": ZEBRA_GREY, "num_format": "#,##0"})

        # --- title band ---
        ws.merge_range(TITLE_ROW, 0, TITLE_ROW, max(n_cols - 1, 1), title, fmt_title)
        if subtitle:
            ws.merge_range(SUB_ROW, 0, SUB_ROW, max(n_cols - 1, 1), subtitle, fmt_subtitle)
        ws.set_row(TITLE_ROW, 22)

        # --- header row ---
        for c, col_name in enumerate(df.columns):
            ws.write(HEADER_ROW, c, col_name, fmt_header)
        ws.set_row(HEADER_ROW, 30)

        # --- body: zebra striping + currency formatting, row by row ---
        for r in range(n_rows):
            excel_row = FIRST_DATA_ROW + r
            alt = (r % 2 == 1)
            for c, col_name in enumerate(df.columns):
                val = df.iat[r, c]
                if pd.isna(val):
                    val = ""
                if col_name in money_cols:
                    ws.write(excel_row, c, val if val != "" else 0, fmt_money_alt if alt else fmt_money)
                else:
                    ws.write(excel_row, c, val, fmt_cell_alt if alt else fmt_cell)

        # --- conditional highlights ---
        if "New/Revival" in df.columns:
            col_idx = df.columns.get_loc("New/Revival")
            ws.conditional_format(FIRST_DATA_ROW, col_idx, FIRST_DATA_ROW + n_rows - 1, col_idx, {
                "type": "text", "criteria": "containing", "value": "New/Revival",
                "format": workbook.add_format({"bg_color": NEW_GREEN, "bold": True, "border": 1, "border_color": THEME["border"]}),
            })
        if "Sales Basis" in df.columns:
            col_idx = df.columns.get_loc("Sales Basis")
            ws.conditional_format(FIRST_DATA_ROW, col_idx, FIRST_DATA_ROW + n_rows - 1, col_idx, {
                "type": "text", "criteria": "containing", "value": "Approx",
                "format": workbook.add_format({"bg_color": APPROX_YELLOW, "italic": True, "border": 1, "border_color": THEME["border"]}),
            })
        if "Year" in df.columns:
            col_idx = df.columns.get_loc("Year")
            ws.conditional_format(FIRST_DATA_ROW, col_idx, FIRST_DATA_ROW + n_rows - 1, col_idx, {
                "type": "text", "criteria": "containing", "value": "approx",
                "format": workbook.add_format({"bg_color": APPROX_YELLOW, "italic": True, "border": 1, "border_color": THEME["border"]}),
            })

        # --- column widths: fit to content, capped so nothing runs off-screen ---
        # NOTE: deliberately NOT using df[col].astype(str).map(len) -- on newer
        # pandas (pyarrow-backed string/nullable dtypes), NA values can survive
        # astype(str) as float NaN instead of becoming "nan", which then blows
        # up len() inside .map(). Looping in plain Python and skipping NA values
        # directly sidesteps that dtype-backend quirk entirely.
        for c, col_name in enumerate(df.columns):
            content_width = max((len(str(v)) for v in df[col_name] if pd.notna(v)), default=0) if n_rows else 0
            width = max(len(str(col_name)), content_width) + 3
            ws.set_column(c, c, min(max(width, 10), 32))

        # --- usability: filter, freeze header, tab color ---
        if n_rows:
            ws.autofilter(HEADER_ROW, 0, HEADER_ROW + n_rows, n_cols - 1)
        ws.freeze_panes(FIRST_DATA_ROW, partner_freeze(df, default=1))
        ws.set_tab_color(BRAND_BLUE)
        ws.hide_gridlines(2)


    # --- SUMMARY TABLES: RD/KRO split, and Retail/Project split ---
    def build_rd_kro_summary(retail_report):
        """One row per RD/KRO category with partner count and sales, plus a Total
        row. 2025 YTD Aug is used (not full-year 2025) alongside 2026 YTD Aug so
        the YoY comparison is apples-to-apples; 2025 FY and 2026 FY Projection
        are given separately since those aren't the same kind of number."""
        cols = ["Category", "No. of Partners", "2025 YTD Aug Sales", "2026 YTD Aug Sales", "2025 FY Sales",
                "2026 Target", "2026 FY Projection", "2026 Projection (Target basis)"]
        if retail_report.empty or "RD/KRO" not in retail_report.columns:
            return pd.DataFrame(columns=cols)

        grp = retail_report.groupby(retail_report["RD/KRO"].fillna("Unspecified")).agg(
            **{
                "No. of Partners": ("SoldToPartyID", "nunique"),
                "2025 YTD Aug Sales": ("2025 YTD Aug Sales", "sum"),
                "2026 YTD Aug Sales": ("2026 YTD Aug Sales", "sum"),
                "2025 FY Sales": ("2025 Sales", "sum"),
                "2026 Target": ("2026 Target", "sum"),
                "2026 FY Projection": ("2026 FY Projection", "sum"),
                "2026 Projection (Target basis)": ("2026 Projection (Target basis)", "sum"),
            }
        ).reset_index().rename(columns={"RD/KRO": "Category"})

        total_row = pd.DataFrame([{
            "Category": "Total",
            "No. of Partners": grp["No. of Partners"].sum(),
            "2025 YTD Aug Sales": grp["2025 YTD Aug Sales"].sum(),
            "2026 YTD Aug Sales": grp["2026 YTD Aug Sales"].sum(),
            "2025 FY Sales": grp["2025 FY Sales"].sum(),
            "2026 Target": grp["2026 Target"].sum(),
            "2026 FY Projection": grp["2026 FY Projection"].sum(),
            "2026 Projection (Target basis)": grp["2026 Projection (Target basis)"].sum(),
        }])
        return pd.concat([grp[cols], total_row[cols]], ignore_index=True)


    SALES_SUMMARY_COLS = ["2025 YTD Aug Sales", "2026 YTD Aug Sales", "2025 FY Sales",
                          "2026 Target", "2026 FY Projection", "2026 Projection (Target basis)"]


    def _sales_sums(df):
        """Sum this report's sales columns (mapping '2025 Sales' -> '2025 FY
        Sales' to match the summary tables' naming), or all-zero if empty."""
        if df.empty:
            return {c: 0 for c in SALES_SUMMARY_COLS}
        return {
            "2025 YTD Aug Sales": df["2025 YTD Aug Sales"].sum(),
            "2026 YTD Aug Sales": df["2026 YTD Aug Sales"].sum(),
            "2025 FY Sales": df["2025 Sales"].sum(),
            "2026 Target": df["2026 Target"].sum(),
            "2026 FY Projection": df["2026 FY Projection"].sum(),
            "2026 Projection (Target basis)": df["2026 Projection (Target basis)"].sum(),
        }


    def build_retail_project_summary(retail_report, project_report):
        """Partner counts AND sales: Retail (broken into RD/KRO) vs Project, plus
        a grand total. Sales figures use the same columns/meaning as the RD/KRO
        summary table above -- see build_rd_kro_summary's docstring."""
        rd_kro = retail_report["RD/KRO"] if "RD/KRO" in retail_report.columns else pd.Series(dtype=object)
        retail_rd = retail_report[rd_kro == "RD"]
        retail_kro = retail_report[rd_kro == "KRO"]
        retail_other = retail_report[~rd_kro.isin(["RD", "KRO"])]

        rows = [
            {"Category": "Retail Partners (total)", "No. of Partners": len(retail_report), **_sales_sums(retail_report)},
            {"Category": "   -- RD (Retail Distributor)", "No. of Partners": len(retail_rd), **_sales_sums(retail_rd)},
            {"Category": "   -- KRO", "No. of Partners": len(retail_kro), **_sales_sums(retail_kro)},
        ]
        if len(retail_other) > 0:
            rows.append({"Category": "   -- Unspecified RD/KRO", "No. of Partners": len(retail_other), **_sales_sums(retail_other)})
        rows += [
            {"Category": "Project Partners (total)", "No. of Partners": len(project_report), **_sales_sums(project_report)},
            {"Category": "Grand Total (Retail + Project)", "No. of Partners": len(retail_report) + len(project_report),
             **{c: _sales_sums(retail_report)[c] + _sales_sums(project_report)[c] for c in SALES_SUMMARY_COLS}},
        ]
        return pd.DataFrame(rows)[["Category", "No. of Partners"] + SALES_SUMMARY_COLS]


    def write_summary_table(workbook, ws, row, title, df, money_cols=()):
        """Write one small bordered/formatted table starting at `row`, columns
        B onward. Returns the row index right after the table."""
        fmt_table_title = workbook.add_format({"bold": True, "font_size": 11, "font_color": BRAND_BLUE})
        fmt_hdr = workbook.add_format({
            "bold": True, "font_color": "white", "bg_color": BRAND_BLUE,
            "border": 1, "align": "center", "valign": "vcenter",
        })
        fmt_cell = workbook.add_format({"border": 1, "valign": "vcenter"})
        fmt_cell_num = workbook.add_format({"border": 1, "valign": "vcenter", "num_format": "#,##0", "align": "right"})
        fmt_total = workbook.add_format({"bold": True, "border": 1, "top": 2, "bg_color": THEME["total_bg"], "valign": "vcenter"})
        fmt_total_num = workbook.add_format({"bold": True, "border": 1, "top": 2, "bg_color": THEME["total_bg"],
                                              "num_format": "#,##0", "align": "right", "valign": "vcenter"})

        ws.write(row, 1, title, fmt_table_title)
        row += 1

        for c, col in enumerate(df.columns):
            ws.write(row, 1 + c, col, fmt_hdr)
        row += 1

        for r in range(len(df)):
            is_total_row = "total" in str(df.iloc[r, 0]).lower()
            for c, col in enumerate(df.columns):
                val = df.iat[r, c]
                if col in money_cols or col == "No. of Partners":
                    fmt = fmt_total_num if is_total_row else fmt_cell_num
                else:
                    fmt = fmt_total if is_total_row else fmt_cell
                ws.write(row, 1 + c, val, fmt)
            row += 1

        return row + 1  # one blank row after the table


    def write_cover_sheet(writer, sheet_stats, retail_report=None, project_report=None):
        """A one-page index/README sheet so a reader knows what each tab holds
        and how to read the highlight colors, before digging into the data."""
        workbook = writer.book
        ws = workbook.add_worksheet("0_Overview")
        writer.sheets["0_Overview"] = ws
        ws.hide_gridlines(2)

        fmt_title = workbook.add_format({"bold": True, "font_size": 16, "font_color": BRAND_BLUE})
        fmt_generated = workbook.add_format({"italic": True, "font_size": 9, "font_color": THEME["subtitle"]})
        fmt_section = workbook.add_format({"bold": True, "font_size": 11, "font_color": "white", "bg_color": BRAND_BLUE, "indent": 1})
        fmt_body = workbook.add_format({"font_size": 10, "text_wrap": True, "valign": "top"})
        fmt_swatch = lambda color: workbook.add_format({"bg_color": color, "border": 1, "border_color": THEME["border"]})

        ws.set_column("A:A", 3)
        ws.set_column("B:B", 30)
        ws.set_column("C:C", 22)
        ws.set_column("D:F", 18)

        row = 1
        ws.merge_range(row, 1, row, 2, "Mumbai (excl. West) Retail + Project Partner Report", fmt_title)
        row += 1
        ws.merge_range(row, 1, row, 2, f"Generated {datetime.date.today().isoformat()}  ·  Scope: Mumbai & Navi Mumbai, 2022–2026", fmt_generated)
        row += 2

        if retail_report is not None and project_report is not None:
            row = write_summary_table(
                workbook, ws, row, "RD / KRO Split -- Retail Partners",
                build_rd_kro_summary(retail_report),
                money_cols=["2025 YTD Aug Sales", "2026 YTD Aug Sales", "2025 FY Sales",
                            "2026 Target", "2026 FY Projection", "2026 Projection (Target basis)"],
            )
            row = write_summary_table(
                workbook, ws, row, "Retail vs Project Partner Split",
                build_retail_project_summary(retail_report, project_report),
                money_cols=SALES_SUMMARY_COLS,
            )
            row += 1

        ws.merge_range(row, 1, row, 5, "Contents", fmt_section)
        row += 1
        contents = [
            ("1_Retail_Partners_Mumbai", "Retail partner names & attributes, Sales by year (2022–2025 exact, 2026 YTD Aug + FY projection)."),
            ("2_RD_Retailer_Detail", "Every retailer under each Mumbai RD, with secondary (DMS) sales by year."),
            ("2_RD_Retailer_Summary", "Retailer count & total secondary sales, rolled up per RD per year."),
            ("3_Project_Partners_Mumbai", "Project partner names & attributes, same year layout as sheet 1 (primary sales only)."),
        ]
        for name, desc in contents:
            n = sheet_stats.get(name)
            row_count = f"  ({n:,} rows)" if n is not None else ""
            ws.write(row, 1, name, workbook.add_format({"bold": True, "font_size": 10}))
            ws.merge_range(row, 2, row, 5, desc + row_count, fmt_body)
            ws.set_row(row, 28)
            row += 1

        row += 1
        ws.merge_range(row, 1, row, 5, "How to read the highlights", fmt_section)
        row += 1
        legend = [
            (NEW_GREEN, "Green — partner is New/Revival (0 sales in the prior two years, active this year)."),
            (APPROX_YELLOW, "Yellow — figure is approximate. 2026 secondary sales use each DMS value-slab's mid-point (average of its lower and upper bound; Parivartan guidelines changed exact amounts to bands), never blended silently with exact years."),
        ]
        for color, text in legend:
            ws.write(row, 1, "", fmt_swatch(color))
            ws.merge_range(row, 2, row, 5, text, fmt_body)
            ws.set_row(row, 28)
            row += 1

        row += 1
        ws.merge_range(row, 1, row, 5, "Notes", fmt_section)
        row += 1
        notes = [
            "Mumbai scope differs by partner type: Retail Distributors are scoped by Branch == “MUM” (a partner can have a City outside Mumbai, e.g. Thane, and still legitimately belong to the MUM branch); Project Distributors are scoped by City in “Mumbai”/“Navi Mumbai”.",
            "Retail partner attributes come from Data Dictionary.xlsx; Project partner attributes come from Data Dictionary Retail + Project.xlsx.",
            "A partner reissued a new Sold-to Party code (per Data Dictionary's “Partner Code Changes” sheet) is consolidated as one partner here, so a code change alone never shows up as a new/lost partner.",
            "2026 FY Projection = YTD Aug sales ÷ 8 × 12 (straight-line run-rate), not a forecast model.",
            "“2026 Target” (and “2026 Projection (Target basis)”, its second, independent estimate alongside the run-rate-based 2026 FY Projection) comes from the QTOD/ATOD Targets file, not Partner Master Data -- that file is 2026-only, so this report does not carry a “2025 Target” column. Target and run-rate projection are different kinds of number and are never blended into one.",
            "2025 YTD Aug Sales (in the summary tables) covers Jan-Aug only, matching 2026 YTD Aug Sales, so the YoY comparison is apples-to-apples -- 2025 FY Sales is the separate full-year actual.",
        ]
        for text in notes:
            ws.merge_range(row, 1, row, 5, f"•  {text}", fmt_body)
            ws.set_row(row, 24)
            row += 1


    # --- MAIN ---
    def generate_mumbai_report(df_sales_final):
        print("🔐 Authenticating...")
        headers = get_graph_headers()
        drive_id = get_drive_id(headers)

        print("📖 Building Retail Partner Map (Data Dictionary.xlsx, Branch = direct reference)...")
        pmap_retail = build_partner_map(drive_id, headers, DICT_RETAIL_PATH, branch_mode="direct")

        print("📖 Building Project Partner Map (Data Dictionary Retail + Project.xlsx, Branch = City reference)...")
        pmap_project = build_partner_map(drive_id, headers, DICT_PROJECT_PATH, branch_mode="city_join")

        print("📖 Loading Partner Code Changes (old code == new code, partner-wise reports only)...")
        code_map = build_partner_code_map(drive_id, headers, DICT_RETAIL_PATH)
        print(f"✅ {len(code_map)} partner code remaps loaded.")

        print("📖 Loading 2026 Targets (QTOD/ATOD Targets file)...")
        target_map = build_target_map(drive_id, headers, code_map=code_map)

        print("\n--- Ask 1: Retail Partners, Mumbai ---")
        retail_report = build_primary_partner_report_wide(df_sales_final, pmap_retail, "Retail", code_map=code_map, target_map=target_map)
        print(f"✅ {retail_report['SoldToPartyID'].nunique()} distinct Mumbai retail partners.")

        print("\n--- Ask 2: Retailers per Mumbai RD (Secondary/EasyDMS) ---")
        rd_detail, rd_summary = build_rd_retailer_report(drive_id, headers, pmap_retail)

        print("\n--- Ask 3: Project Partners, Mumbai ---")
        project_report = build_primary_partner_report_wide(df_sales_final, pmap_project, "Project", code_map=code_map, target_map=target_map)
        print(f"✅ {project_report['SoldToPartyID'].nunique()} distinct Mumbai project partners.")

        print(f"\n💾 Writing {OUTPUT_PATH} (formatted) ...")
        with pd.ExcelWriter(OUTPUT_PATH, engine="xlsxwriter") as writer:
            write_cover_sheet(writer, {
                "1_Retail_Partners_Mumbai": len(retail_report),
                "2_RD_Retailer_Detail": len(rd_detail),
                "2_RD_Retailer_Summary": len(rd_summary),
                "3_Project_Partners_Mumbai": len(project_report),
            }, retail_report=retail_report, project_report=project_report)
            write_formatted_sheet(
                writer, retail_report, "1_Retail_Partners_Mumbai",
                "Retail Partners — Mumbai", "Names, attributes & sales by year, 2022–2026",
            )
            write_formatted_sheet(
                writer, rd_detail, "2_RD_Retailer_Detail",
                "Retailers per Mumbai RD — Detail", "Secondary (DMS) sales by retailer & year",
            )
            write_formatted_sheet(
                writer, rd_summary, "2_RD_Retailer_Summary",
                "Retailers per Mumbai RD — Summary", "Retailer count & total secondary sales per RD, per year",
            )
            write_formatted_sheet(
                writer, project_report, "3_Project_Partners_Mumbai",
                "Project Partners — Mumbai", "Names, attributes & sales by year, 2022–2026 (primary sales only)",
            )
            writer.sheets["0_Overview"].activate()

        print(f"🎉 Done: {OUTPUT_PATH}")
        return retail_report, rd_detail, rd_summary, project_report


    # Usage (in your notebook, after df_Sales_final is loaded):
    return generate_mumbai_report(df_Sales_final)


run_report("mumbai_report", report_mumbai_report)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 7.3 REFERENCE SKU LOOKUP (ad-hoc) — REPORTS["reference_sku_lookup"]
#     Sale qty by Sold-to Party for chosen material codes, last-year YTD vs
#     this-year YTD, overall HS (no retail filter).
# 7.4 ACTIVITY-WISE SALES (ad-hoc) — REPORTS["activity_wise_sales"]
#     Sales by Helios code for a list of PricingClassL3 codes and a period.
# ════════════════════════════════════════════════════════════════════
REFERENCE_CODES = ["ME0102_E", "ME0103_E", "ME0104_E", "MR1106_E", "MR1108_E", "MR1109_E", "MR1112_E", "MR1118_E"]

ACTIVITY_PRICING_L3 = [
    "EBD", "EBO", "EBP", "EBQ", "EBT", "EC0", "ECP", "ECQ", "ECT", "EIC",
    "EIM", "EIO", "EIP", "EIQ", "EIR", "EIT", "EP0", "EPO", "EPP", "EPQ",
    "EPR", "EPT", "ERO", "ERP", "ERQ", "ERT", "EXO", "EXP", "EXQ", "EXT",
]
ACTIVITY_PERIOD = (f"{TARGET_YEAR}-01-01", YESTERDAY.strftime("%Y-%m-%d"))


def report_reference_sku_lookup():
    qty_col = next((c for c in df_Sales_final.columns if "qty" in c.lower() or "quantity" in c.lower()), None)
    if qty_col is None:
        raise KeyError("No quantity column in df_Sales_final")
    df = df_Sales_final[df_Sales_final["MaterialCode"].astype(str).str.strip().isin(REFERENCE_CODES)].copy()
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["SoldToPartyID"] = df["SoldToPartyID"].astype(str).str.strip()
    missing = sorted(set(REFERENCE_CODES) - set(df["MaterialCode"].astype(str).str.strip()))
    if missing:
        print(f"   ⚠️ no rows for {missing}")
    ly = df[(df["InvoiceDate"].dt.year == PREV_YEAR) & (df["InvoiceDate"] <= LY_YESTERDAY)]
    ty = df[(df["InvoiceDate"].dt.year == TARGET_YEAR) & (df["InvoiceDate"] <= YESTERDAY)]
    result = pd.concat([
        ly.groupby(["MaterialCode", "SoldToPartyID"])[qty_col].sum().rename(f"SaleQty_{PREV_YEAR}_YTD"),
        ty.groupby(["MaterialCode", "SoldToPartyID"])[qty_col].sum().rename(f"SaleQty_{TARGET_YEAR}_YTD"),
    ], axis=1).fillna(0).reset_index()
    result = result[(result.iloc[:, 2:] != 0).any(axis=1)].sort_values(["MaterialCode", f"SaleQty_{TARGET_YEAR}_YTD"], ascending=[True, False])
    summary = result.groupby("MaterialCode").agg(
        Distinct_SoldToParty=("SoldToPartyID", "nunique"),
        **{f"Total_SaleQty_{PREV_YEAR}_YTD": (f"SaleQty_{PREV_YEAR}_YTD", "sum"),
           f"Total_SaleQty_{TARGET_YEAR}_YTD": (f"SaleQty_{TARGET_YEAR}_YTD", "sum")},
    ).reindex(REFERENCE_CODES).reset_index()

    def build(writer):
        sub = f"YTD {PREV_YEAR}: 01-Jan – {LY_YESTERDAY:%d-%b} | YTD {TARGET_YEAR}: 01-Jan – {YESTERDAY:%d-%b}"
        write_table(writer, result, "Ref_SaleQty_SoldToParty", title="Reference SKUs — Sale Qty by Sold-to Party", subtitle=sub, tab=THEME["tab_detail"])
        write_table(writer, summary, "Summary", title="Reference SKUs — Summary", subtitle=sub, totals=True, tab=THEME["tab_summary"])
    return save_report(build, f"Reference_SaleQty_SoldToParty_{PREV_YEAR}_{TARGET_YEAR}_YTD.xlsx", local_dir=OUTPUT_DIR)


def report_activity_wise_sales():
    d = pd.to_datetime(df_Sales_final["InvoiceDate"])
    start, end = ACTIVITY_PERIOD
    sel = df_Sales_final[df_Sales_final["PricingClassL3"].isin(ACTIVITY_PRICING_L3) & (d >= start) & (d <= end)]
    out = sel.groupby("HeliosCode")["Gross Sales"].sum().reset_index().sort_values("Gross Sales", ascending=False)

    def build(writer):
        write_table(writer, out, "Activity_Sales", title="Activity-wise Sales by Helios code",
                    subtitle=f"{start} → {end}", totals=True, tab=THEME["tab_summary"])
    return save_report(build, f"Activity_wise_Sales_{MONTH_ABBR}_YTD.xlsx", local_dir=OUTPUT_DIR)


run_report("reference_sku_lookup", report_reference_sku_lookup)
run_report("activity_wise_sales", report_activity_wise_sales)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 7.4b HS BRANCH × RANGE — Gross Sales, YTD this year vs YTD last year
#      REPORTS["hs_branch_range_ytd"]
#
# Scope = HS Overall, same rules as the HS YTD report (4.2):
#   • Helios codes HS_REPORT_HELIOS_CODES (HS set, PPCFD excluded)
#   • Retail = retail PricingClass L2/L3, Project = everything else
#   • Branch: Retail rows → partner's branch (Partner Master, City → Branch);
#             Project rows → Sales Office → "NS Branches"
#   • Range = Family Actual (Material Master)
# YTD = 01-Jan → yesterday this year vs 01-Jan → same day last year.
# Sheets: Overall · Retail · Project (range summary + branch × range with
# branch subtotals and a Branch filter, Rs Mn) and Data (flat table for pivots, Rs).
# ════════════════════════════════════════════════════════════════════
# Branches shown (in this order). Key = name shown in the report,
# value = Branch names in the master data that count as that branch
# (matched ignoring case). Other branches are left out of the report.
# Set to None to show every branch.
HS_BRANCH_RANGE_BRANCHES = {
    #         Project (NS Branches)   Retail (Geography)
    "Delhi": ["Del/NCR",              "NCR"],
    "Chd":   [                        "CHD"],
    "East":  ["WB"],                  # WB in both lists
    "Ahm":   ["Ahm",                  "GUJ"],
    "Mum":   ["Mum"],                 # Mum / MUM
    "Bng":   ["Kar"],                 # Kar / KAR
    "Hyd":   ["AP/TG",                "TEL"],
    "Chn":   ["N TN",                 "TN-1"],
}


def limit_to_branches(data, branches):
    """Keep only the listed branches (renamed to their report name) and log what was left out."""
    if not branches:
        return data, None
    alias = {a.strip().upper(): name for name, names in branches.items() for a in [name, *names]}
    raw = data["Branch"].astype(str).str.strip().str.upper()
    shown = raw.map(alias)
    ty_col = f"YTD {TARGET_YEAR}"
    found = set(shown.dropna())
    missing = [name for name in branches if name not in found]
    if missing:
        print(f"   ⚠️ No sales found for {', '.join(missing)} — check their spelling in HS_BRANCH_RANGE_BRANCHES. "
              f"Branch names in the data: {', '.join(sorted(raw.unique()))}")
    left_out = data[shown.isna()].groupby("Branch")[ty_col].sum().sort_values(ascending=False)
    if len(left_out):
        print(f"   ℹ️ {len(left_out)} other branch(es) left out, YTD {TARGET_YEAR} Rs {left_out.sum() / 1e6:,.1f} Mn: "
              + ", ".join(f"{b} ({v / 1e6:,.1f})" for b, v in left_out.items()))
    out = data[shown.notna()].copy()
    out["Branch"] = shown[shown.notna()]
    return out, list(branches)


def hs_branch_range_data():
    base = build_hs_dataset(df_Sales_final, "InvoiceDate", HS_REPORT_HELIOS_CODES, retail_branch_map={})
    if base.empty:
        return pd.DataFrame()
    d = pd.to_datetime(base["InvoiceDate"])
    ytd = ((d.dt.year == TARGET_YEAR) & (d <= YESTERDAY)) | ((d.dt.year == PREV_YEAR) & (d <= LY_YESTERDAY))
    df = base[ytd].copy()
    df["Year"] = pd.to_datetime(df["InvoiceDate"]).dt.year

    # Branch exactly as the HS YTD report: partner map for Retail, Sales Office for Project
    df = df.drop(columns=[c for c in ("Branch", "Zone", "ASM", "Region") if c in df.columns])
    df["SoldToPartyID"] = df["SoldToPartyID"].astype(str).str.strip()
    df = lookup_merge(df, Partner_Map_Project[["Sold-to Party", "Branch"]], "partner map", left_on="SoldToPartyID", right_on="Sold-to Party")
    project = (df["Classification"] == "Project").values
    if "SalesOfficeCode" in df.columns:
        df.loc[project, "Branch"] = df.loc[project, "SalesOfficeCode"].astype(str).str.strip().str.upper().map(project_branch_map)
    df["MaterialCode"] = df["MaterialCode"].astype(str).str.strip().str.upper()
    df = lookup_merge(df, family_map, "material master", left_on="MaterialCode", right_on="Material")

    df["Branch"] = df["Branch"].fillna("Unmapped").replace("", "Unmapped")
    df["Range"] = df["Family Actual"].fillna("Unclassified")
    out = (df.groupby(["Classification", "Branch", "Range", "Year"])["Gross Sales"].sum()
             .unstack("Year", fill_value=0).reindex(columns=[PREV_YEAR, TARGET_YEAR], fill_value=0)
             .reset_index())
    out.columns = ["Classification", "Branch", "Range", f"YTD {PREV_YEAR}", f"YTD {TARGET_YEAR}"]
    return out


def _growth(ly, ty):
    return (ty - ly) / ly if ly else None


def _write_branch_range_sheet(writer, sheet, data, title, subtitle, branch_order=None):
    f = writer.fmt
    ly_col, ty_col = f"YTD {PREV_YEAR}", f"YTD {TARGET_YEAR}"
    wb = writer.book
    ws = wb.add_worksheet(sheet)
    writer.sheets[sheet] = ws
    ws.set_tab_color(THEME["tab_summary"] if sheet == "Overall" else THEME["tab_detail"])
    ws.hide_gridlines(2)
    ws.write(0, 0, title, f["title"])
    ws.write(1, 0, subtitle, f["note"])
    sub_text = wb.add_format({"font_name": THEME["font"], "font_size": THEME["font_size"], "bold": True,
                              "bg_color": THEME["kpi_bg"], "border": 1, "border_color": THEME["border"]})
    sub_mn = wb.add_format({"font_name": THEME["font"], "font_size": THEME["font_size"], "bold": True,
                            "bg_color": THEME["kpi_bg"], "border": 1, "border_color": THEME["border"],
                            "num_format": NUM_FMT["mn"], "align": "right"})
    sub_pct = wb.add_format({"font_name": THEME["font"], "font_size": THEME["font_size"], "bold": True,
                             "bg_color": THEME["kpi_bg"], "border": 1, "border_color": THEME["border"],
                             "num_format": NUM_FMT["pct"], "align": "center"})
    total_ty = data[ty_col].sum()

    def table(row, title_text, first_col, groups):
        ws.write(row, 0, title_text, f["subtitle"])
        row += 1
        hdrs = [first_col, "Range"] if groups else [first_col]
        hdrs += [f"YTD {PREV_YEAR} (Mn)", f"YTD {TARGET_YEAR} (Mn)", "Growth %", f"Share of {TARGET_YEAR} %"]
        for c, h in enumerate(hdrs):
            ws.write(row, c, h, f["hdr"])
        ws.set_row(row, 30)
        hdr_row = row
        row += 1
        return row, hdr_row, len(hdrs)

    def line(row, labels, ly, ty, fmt_text, fmt_mn, fmt_pct):
        for c, v in enumerate(labels):
            ws.write(row, c, v, fmt_text)
        c0 = len(labels)
        ws.write(row, c0, ly / 1e6, fmt_mn)
        ws.write(row, c0 + 1, ty / 1e6, fmt_mn)
        g = _growth(ly, ty)
        ws.write(row, c0 + 2, g, fmt_pct) if g is not None else ws.write(row, c0 + 2, "New" if ty else "", fmt_pct)
        ws.write(row, c0 + 3, ty / total_ty if total_ty else 0, fmt_pct)

    # A) Range summary
    by_range = data.groupby("Range")[[ly_col, ty_col]].sum().sort_values(ty_col, ascending=False)
    r, first_hdr, _ = table(3, "A) By Range", "Range", groups=False)
    for rng, v in by_range.iterrows():
        line(r, [rng], v[ly_col], v[ty_col], f["text"], f["mn"], f["pct"])
        r += 1
    line(r, ["Grand Total"], by_range[ly_col].sum(), by_range[ty_col].sum(), f["tot_text"], f["tot_mn"], f["tot_pct"])
    apply_growth_cf = lambda r1, r2, col: (
        ws.conditional_format(r1, col, r2, col, {"type": "cell", "criteria": "<", "value": 0, "format": f["cf_red"]}),
        ws.conditional_format(r1, col, r2, col, {"type": "cell", "criteria": ">=", "value": 0, "format": f["cf_green"]}))
    apply_growth_cf(first_hdr + 1, r - 1, 3)
    r += 3

    # B) Branch × Range with branch subtotals
    r, hdr_b, n_cols = table(r, "B) By Branch × Range", "Branch", groups=True)
    start_b = r
    branch_tot = data.groupby("Branch")[[ly_col, ty_col]].sum().sort_values(ty_col, ascending=False)
    if branch_order:   # fixed order from HS_BRANCH_RANGE_BRANCHES
        branches = [b for b in branch_order if b in branch_tot.index]
    else:
        branches = [b for b in branch_tot.index if b != "Unmapped"] + (["Unmapped"] if "Unmapped" in branch_tot.index else [])
    for br in branches:
        part = data[data["Branch"] == br].groupby("Range")[[ly_col, ty_col]].sum().sort_values(ty_col, ascending=False)
        for rng, v in part.iterrows():
            line(r, [br, rng], v[ly_col], v[ty_col], f["text"], f["mn"], f["pct"])
            r += 1
        # Branch name kept on its subtotal row so the Branch filter keeps the subtotal with its ranges
        line(r, [br, "Branch Total"], part[ly_col].sum(), part[ty_col].sum(), sub_text, sub_mn, sub_pct)
        r += 1
    apply_growth_cf(start_b, r - 1, 4)
    # Branch filter (dropdown on the section B header); the grand total sits
    # below the filtered range so it always shows all branches.
    ws.autofilter(hdr_b, 0, r - 1, n_cols - 1)
    r += 1
    line(r, ["Grand Total (all branches shown)", ""], data[ly_col].sum(), data[ty_col].sum(), f["tot_text"], f["tot_mn"], f["tot_pct"])

    ws.set_column(0, 0, 26)
    ws.set_column(1, 1, 24)
    ws.set_column(2, n_cols - 1, 14)
    ws.freeze_panes(3, 0)


def report_hs_branch_range_ytd():
    data = hs_branch_range_data()
    if data.empty:
        return "no HS sales in the YTD periods"
    data, branch_order = limit_to_branches(data, HS_BRANCH_RANGE_BRANCHES)
    if data.empty:
        return "none of the HS_BRANCH_RANGE_BRANCHES branches has sales — check the branch names"
    period = (f"YTD {TARGET_YEAR}: 01-Jan to {YESTERDAY:%d-%b-%Y}  vs  YTD {PREV_YEAR}: 01-Jan to {LY_YESTERDAY:%d-%b-%Y}"
              f"  |  Gross Sales, Rs Mn  |  Range = Family Actual"
              + (f"  |  Branches: {', '.join(branch_order)}" if branch_order else ""))
    views = {"Overall": data, "Retail": data[data["Classification"] == "Retail"], "Project": data[data["Classification"] == "Project"]}

    def build(writer):
        for sheet, part in views.items():
            part = part.groupby(["Branch", "Range"], as_index=False)[[f"YTD {PREV_YEAR}", f"YTD {TARGET_YEAR}"]].sum()
            if part.empty:
                continue
            _write_branch_range_sheet(writer, sheet, part, f"HS {sheet} — Gross Sales by Branch & Range", period, branch_order)
        flat = data.copy()
        flat["Growth %"] = [_growth(a, b) for a, b in zip(flat[f"YTD {PREV_YEAR}"], flat[f"YTD {TARGET_YEAR}"])]
        write_table(writer, flat, "Data", title="Flat data (Rs) — for pivots", subtitle=period,
                    kinds={f"YTD {PREV_YEAR}": "inr", f"YTD {TARGET_YEAR}": "inr", "Growth %": "pct"},
                    totals=True, tab=THEME["tab_data"])
    return save_report(build, f"HS_Branch_Range_Gross_Sales_YTD_{MONTH_ABBR}_{TARGET_YEAR}_vs_{PREV_YEAR}.xlsx", local_dir=OUTPUT_DIR)


run_report("hs_branch_range_ytd", report_hs_branch_range_ytd)


In [ ]:
# ════════════════════════════════════════════════════════════════════
# 7.5 DIAGNOSTIC TOOLS — not run automatically; call when needed:
#   list_easydms_folder()     what's in the EasyDMS secondary-data folder
#   explore_data_sources()    column/grain check of Pavilion, eShop, Electrician files
# ════════════════════════════════════════════════════════════════════
def list_easydms_folder(folder="Test - haripriya/Data Downloads - Input Files/Secondary/EasyDMS"):
    for item in sp.list_folder(folder):
        print(f"  [{'FOLDER' if 'folder' in item else 'FILE'}] {item['name']!r}")


def explore_data_sources():
    import io
    import os
    import time
    from urllib.parse import quote

    import msal
    import pandas as pd
    import requests
    import requests.exceptions as req_exc

    # ═══════════════════════════════════════════════════════════════════
    # Golden RO ID project -- Stage 0 exploration, round 2.
    #
    # Round 1 picked a sheet name per workbook (often the first sheet) and
    # reported only that one -- which was wrong for Pavilion ('Wiser
    # Dashboard' turned out to be a formatted pivot, not the raw list).
    # This round inspects EVERY sheet in every multi-sheet workbook, so
    # nothing gets judged off the wrong tab.
    #
    # Also carries over round 1's 4 open questions:
    #   1. Pavilion: which sheet(s) actually hold the raw outlet list?
    #   2. EasyDMS 2026: is 'State' Distributor-level or Party-level?
    #   3. eShop: real grain (order-line vs RO-level), and is GSTIN the
    #      Seller's GSTIN repeated on every line rather than the
    #      customer/outlet's own?
    #   4. Electrician: does Combined_Electrician_Invoices_Data.xlsx (and
    #      its sibling Combined_Electrician_Data.xlsx) have a sheet with
    #      Retailer name/mobile/GST?
    # ═══════════════════════════════════════════════════════════════════

    BASE_PATH = "Test - haripriya"
    PAVILION_PATH = f"{BASE_PATH}/Dashboard Input Files/Pavilion/Slab_Updated Pavilion list as on 31-08-2026.xlsx"
    EASYDMS_2026_PATH = f"{BASE_PATH}/Data Downloads - Input Files/Secondary/EasyDMS/DMS Dump Sales-Invoice-Register(01_Jan_2026_To_13_Aug_2026).xlsx"
    ESHOP_PATH = f"{BASE_PATH}/Data Downloads - Input Files/Secondary/eShop/SE India-YTD Report.xlsx"
    ELECTRICIAN_COMBINED_FOLDER = f"{BASE_PATH}/Data Downloads - Input Files/Electrician program Numbers_Analytics/Combined_Outputs"
    ELECTRICIAN_FILES_TO_CHECK = [
        "Combined_Electrician_Invoices_Data.xlsx",
        "Combined_Electrician_Data.xlsx",
    ]

    KEY_FIELDS = ["gstin", "gst", "phone", "mobile", "city", "state", "name", "retailer", "kro", "status", "active"]

    NETWORK_RETRY_ATTEMPTS = 4
    NETWORK_RETRY_BACKOFF_SECONDS = 3


    def _with_retries(func, description):
        last_err = None
        for attempt in range(1, NETWORK_RETRY_ATTEMPTS + 1):
            try:
                return func()
            except (req_exc.ConnectionError, req_exc.Timeout, req_exc.ChunkedEncodingError) as e:
                last_err = e
                if attempt == NETWORK_RETRY_ATTEMPTS:
                    break
                wait = NETWORK_RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1))
                print(f"⚠️ Network error on {description} (attempt {attempt}/{NETWORK_RETRY_ATTEMPTS}): {e} -- retrying in {wait}s...")
                time.sleep(wait)
        raise ConnectionError(f"{description} failed after {NETWORK_RETRY_ATTEMPTS} attempts: {last_err}") from last_err


    def get_graph_headers():
        def _acquire():
            app = msal.ConfidentialClientApplication(
                os.environ["SP_CLIENT_ID"],
                authority=f"https://login.microsoftonline.com/{os.environ['SP_TENANT_ID']}",
                client_credential=os.environ["SP_CLIENT_SECRET"],
            )
            token = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
            if "access_token" not in token:
                raise Exception(f"Authentication failed: {token.get('error_description')}")
            return {"Authorization": f"Bearer {token['access_token']}"}

        return _with_retries(_acquire, "token acquisition")


    def get_drive_id(headers, site_path="schneiderelectric.sharepoint.com:/sites/HDIndiaTeamGroup"):
        site_res = _with_retries(
            lambda: requests.get(f"https://graph.microsoft.com/v1.0/sites/{site_path}", headers=headers).json(),
            "site lookup",
        )
        drives = _with_retries(
            lambda: requests.get(f"https://graph.microsoft.com/v1.0/sites/{site_res['id']}/drives", headers=headers).json()["value"],
            "drive lookup",
        )
        return next(d["id"] for d in drives if d["name"] == "Documents")


    def fetch_sp_file(drive_id, headers, file_path):
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root:/{quote(file_path)}"
        res = _with_retries(lambda: requests.get(url, headers=headers).json(), f"file lookup ({file_path})")
        if "@microsoft.graph.downloadUrl" not in res:
            raise FileNotFoundError(f"Could not locate '{file_path}': {res.get('error', res)}")
        content = _with_retries(lambda: requests.get(res["@microsoft.graph.downloadUrl"]).content, f"file download ({file_path})")
        return io.BytesIO(content)


    def report_sheet(df, label):
        df.columns = [str(c).strip() for c in df.columns]
        n = len(df)
        n_real_cols = sum(1 for c in df.columns if not str(c).startswith("Unnamed"))
        print(f"  [{label}] {n:,} rows, {len(df.columns)} columns ({n_real_cols} with real headers)")
        if n_real_cols == 0:
            print(f"      ⚠️ Every column is 'Unnamed' -- likely a formatted/merged-header pivot sheet, not raw data. Skipping fill-rate check.")
            return
        print(f"      Columns: {list(df.columns)}")
        if n == 0:
            return
        any_key_match = False
        for col in df.columns:
            col_lower = str(col).lower()
            if any(k in col_lower for k in KEY_FIELDS):
                any_key_match = True
                non_blank = df[col].notna() & (df[col].astype(str).str.strip() != "")
                print(f"      {col!r:35s} {non_blank.sum():>7,} / {n:,}  ({100*non_blank.sum()/n:5.1f}%)")
        if not any_key_match:
            print(f"      (no column name matched {KEY_FIELDS})")


    def explore_all_sheets(buf, source_label):
        print(f"\n{'=' * 70}\n{source_label}\n{'=' * 70}")
        xls = pd.ExcelFile(buf)
        print(f"Sheets found: {xls.sheet_names}")
        for sheet in xls.sheet_names:
            try:
                df = pd.read_excel(xls, sheet_name=sheet)
                report_sheet(df, sheet)
            except Exception as e:
                print(f"  [{sheet}] ⚠️ Could not read: {e}")


    def main():
        print("🔐 Authenticating with Graph API...")
        headers = get_graph_headers()
        drive_id = get_drive_id(headers)

        # --- 1. Pavilion: every sheet ---
        try:
            buf = fetch_sp_file(drive_id, headers, PAVILION_PATH)
            explore_all_sheets(buf, "1. Pavilion -- ALL sheets")
        except Exception as e:
            print(f"⚠️ Could not open Pavilion file: {e}")

        # --- 2. EasyDMS: is 'State' Distributor-level or Party-level? ---
        print(f"\n{'=' * 70}\n2. EasyDMS 2026 -- is 'State' Distributor-level or Party-level?\n{'=' * 70}")
        try:
            buf = fetch_sp_file(drive_id, headers, EASYDMS_2026_PATH)
            df = pd.read_excel(buf)
            df.columns = [str(c).strip() for c in df.columns]
            states_per_distributor = df.groupby("Distributor Code")["State"].nunique()
            states_per_party = df.groupby("Party Code")["State"].nunique()
            print(f"Distinct States per Distributor Code -- mean {states_per_distributor.mean():.2f}, "
                  f"max {states_per_distributor.max()}, {(states_per_distributor == 1).mean()*100:.1f}% of distributors have exactly 1 state")
            print(f"Distinct States per Party Code       -- mean {states_per_party.mean():.2f}, "
                  f"max {states_per_party.max()}, {(states_per_party == 1).mean()*100:.1f}% of parties have exactly 1 state")
            print("If both are ~100% single-valued, the Party-level figure is likely just inheriting the "
                  "Distributor's state (i.e. this column is Distributor-level, not a real Party attribute) "
                  "and should NOT be used for Party/RO blocking.")
        except Exception as e:
            print(f"⚠️ Could not check EasyDMS State grain: {e}")

        # --- 3. eShop: every year-sheet, grain + GSTIN-source check ---
        print(f"\n{'=' * 70}\n3. eShop -- ALL year sheets, grain + GSTIN-source check\n{'=' * 70}")
        try:
            buf = fetch_sp_file(drive_id, headers, ESHOP_PATH)
            xls = pd.ExcelFile(buf)
            print(f"Sheets found: {xls.sheet_names}")
            for sheet in xls.sheet_names:
                df = pd.read_excel(xls, sheet_name=sheet)
                df.columns = [str(c).strip() for c in df.columns]
                n = len(df)
                print(f"\n  [{sheet}] {n:,} rows")
                if n == 0 or "Order Id" not in df.columns:
                    print(f"      ⚠️ No 'Order Id' column or empty -- columns: {list(df.columns)}")
                    continue
                print(f"      Distinct Order Id: {df['Order Id'].nunique():,}  (order-line grain if close to {n:,})")
                print(f"      Distinct Seller_Id: {df['Seller_Id'].nunique():,}, Distinct Customer Name: {df['Customer Name'].nunique():,}")
                gstin_per_seller = df.groupby("Seller_Id")["GSTIN"].nunique()
                print(f"      Distinct GSTIN per Seller_Id -- mean {gstin_per_seller.mean():.2f}, "
                      f"{(gstin_per_seller == 1).mean()*100:.1f}% of sellers show exactly 1 GSTIN across all orders "
                      f"(near-100% here would mean this is the Seller's GSTIN, not a customer-level identifier)")
        except Exception as e:
            print(f"⚠️ Could not check eShop: {e}")

        # --- 4. Electrician: every sheet in both combined files ---
        for fname in ELECTRICIAN_FILES_TO_CHECK:
            path = f"{ELECTRICIAN_COMBINED_FOLDER}/{fname}"
            try:
                buf = fetch_sp_file(drive_id, headers, path)
                explore_all_sheets(buf, f"4. Electrician -- {fname} -- ALL sheets")
            except Exception as e:
                print(f"⚠️ Could not open {fname}: {e}")

        print("\n🎉 Round 2 done. Send this output back and I'll write Stage 0 of the real pipeline "
              "against confirmed columns/grain/sheet for every source.")


    main()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# RUN SUMMARY
# ════════════════════════════════════════════════════════════════════
summary = pd.DataFrame(RUN_LOG, columns=["Report", "Status", "Seconds", "Output / error"])
print(f"Data source: {DATA_SOURCE} | YTD cut-off {YESTERDAY:%d-%b-%Y} | total run time {_fmt_secs(time.time() - RUN_START)}")
print(f"📝 Full log: {LOG_FILE}")
print(summary["Status"].value_counts().to_string())
failed = summary[summary["Status"] == "FAILED"]
if len(failed):
    print("\n❌ Failed reports:")
    print(failed[["Report", "Output / error"]].to_string(index=False))
summary

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 9 · MASTER-DATA REVIEW — what did not map in this run (last cell)
#
# Scope: H&D Helios codes, Sales + OB, this year and last year.
# Writes Master_Data_Review_<date>.xlsx to OUTPUT_DIR with:
#   Summary                 counts + sales affected
#   Unmapped Cities         partner city not in "Geography mapping"
#                           (checked in both Data Dictionaries)
#   Partners not in Master  retail Sold-to Party missing from Partner Master
#   Unmapped Sales Offices  project Sales Office missing from "NS Branches"
#   Unmapped Products       material missing from Material Master, or no
#                           Family Actual
#   Duplicates in Master Data   keys listed more than once in Geography
#                           mapping, Material Master, Partner Master,
#                           NS Branches, SSA Mapping (these double sales
#                           in Tableau joins) — with the Excel row numbers
# Fix these in the master files, then run once with 0 so the reports pick
# them up. Sales values in Rs; sorted by this year's sales, biggest first.
# ════════════════════════════════════════════════════════════════════
def _review_base():
    frames = []
    for label, df, date_col in (("Sales", df_Sales_final, "InvoiceDate"), ("OB", df_OB_final, "Calendar Day")):
        d = pd.to_datetime(df[date_col], errors="coerce")
        part = df[df["HeliosCode"].isin(RETAIL_HELIOS_CODES) & d.dt.year.isin([PREV_YEAR, TARGET_YEAR])].copy()
        part["Source"] = label
        part["Year"] = d[part.index].dt.year
        part["Date"] = d[part.index]
        frames.append(part)
    base = pd.concat(frames, ignore_index=True)
    base["Classification"] = np.where(is_retail(base), "Retail", "Project")
    base["SoldToPartyID"] = clean_id(base["SoldToPartyID"])
    base["MaterialCode"] = base["MaterialCode"].astype(str).str.strip().str.upper()
    base["Gross Sales"] = pd.to_numeric(base["Gross Sales"], errors="coerce").fillna(0)
    return base


def _sales_cols(df, keys):
    """Per key: sales this year / last year (Sales rows only), first & last date seen."""
    sales = df[df["Source"] == "Sales"]
    out = df.groupby(keys).agg(First_Seen=("Date", "min"), Last_Seen=("Date", "max"), Rows=("Date", "size"))
    for yr in (TARGET_YEAR, PREV_YEAR):
        out[f"Gross Sales {yr}"] = sales[sales["Year"] == yr].groupby(keys)["Gross Sales"].sum()
    return out.fillna({f"Gross Sales {TARGET_YEAR}": 0, f"Gross Sales {PREV_YEAR}": 0}).reset_index()


_MASTER_FILES = {k: (PATHS[path_key].split("/")[-1], sheet) for k, (path_key, sheet) in _MASTER_SOURCES.items()}
_DUP_DETAIL_COLS = {
    "geo_retail": ["Zone", "Branch"], "geo_project": ["Zone", "Branch"],
    "material": ["Family Actual", "Helios Code", "Classification", "Category", "Product Description"],
    "partner_retail": ["Patner", "City", "ASM", "RD/KRO", "Region"],
    "partner_project": ["Patner", "City", "ASM", "RD/KRO", "Region", "Branch_dummy"],
    "ns_branches": ["Branch", "Region"], "ssa_map": ["SSA"],
}


def master_duplicates():
    """Every key that appears more than once in a mapping sheet (as in the file)."""
    rows = []
    for key, col in MAPPING_KEYS.items():
        df = _MASTER_RAW.get(key, pd.DataFrame())
        if df.empty or col not in df.columns:
            continue
        k = mapping_key(df[col])
        dup = df.assign(_k=k)[(k.duplicated(keep=False) & (k != "")).values]
        for value, grp in dup.groupby("_k", sort=True):
            details, conflict = [], False
            for c in _DUP_DETAIL_COLS.get(key, []):
                if c in grp.columns:
                    vals = sorted({str(v).strip() for v in grp[c] if pd.notna(v)} - {"", "nan"})
                    if len(vals) > 1:
                        conflict = True
                    details.append(f"{c}: {' | '.join(vals) if vals else '(blank)'}")
            file, sheet = _MASTER_FILES[key]
            rows.append({"File": file, "Sheet": sheet, "Key column": col, "Key value": grp[col].iloc[0],
                         "Times listed": len(grp), "Excel rows": ", ".join(str(i + 2) for i in grp.index),
                         "Values differ?": "YES — check which is right" if conflict else "No (exact repeat)",
                         "Values": "; ".join(details)})
    out = pd.DataFrame(rows)
    return out.sort_values(["File", "Sheet", "Values differ?"], ascending=[True, True, False]).reset_index(drop=True) if len(out) else out


def mapping_review():
    base = _review_base()
    retail = base[base["Classification"] == "Retail"]
    name_col = "SoldToPartyName" if "SoldToPartyName" in base.columns else None
    names = retail.groupby("SoldToPartyID")[name_col].last() if name_col else None
    tables = {}

    # 1) Partners not in Partner Master (retail rows; both dictionaries)
    rows = []
    for dict_name, key in (("Data Dictionary", "partner_retail"), ("Data Dictionary Retail + Project", "partner_project")):
        known = set(clean_id(master(key)["Sold-to Party"])) if not master(key).empty else set()
        miss = retail[~retail["SoldToPartyID"].isin(known)]
        if miss.empty:
            continue
        t = _sales_cols(miss, ["SoldToPartyID"])
        t.insert(0, "Dictionary", dict_name)
        if names is not None:
            t.insert(2, "Partner Name (sales data)", t["SoldToPartyID"].map(names))
        if "PartnerCity" in miss.columns:
            t.insert(3, "City (sales data)", t["SoldToPartyID"].map(miss.groupby("SoldToPartyID")["PartnerCity"].last()))
        rows.append(t)
    tables["Partners not in Master"] = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    # 2) Unmapped cities: partner's master City not in that dictionary's Geography mapping
    rows = []
    for dict_name, pkey, gkey in (("Data Dictionary", "partner_retail", "geo_retail"),
                                  ("Data Dictionary Retail + Project", "partner_project", "geo_project")):
        pm, geo = master(pkey), master(gkey)
        if pm.empty or geo.empty:
            continue
        pm["SoldToPartyID"] = clean_id(pm["Sold-to Party"])
        geo_cities = set(geo["City"].dropna().astype(str).str.strip())
        pm["City"] = pm["City"].astype(str).str.strip()
        bad = pm[~pm["City"].isin(geo_cities)][["SoldToPartyID", "City"]].drop_duplicates("SoldToPartyID")
        hit = retail.merge(bad, on="SoldToPartyID", how="inner")
        if hit.empty:
            continue
        t = _sales_cols(hit, ["City"])
        t.insert(0, "Dictionary", dict_name)
        t.insert(2, "Partners", t["City"].map(hit.groupby("City")["SoldToPartyID"].nunique()))
        t.insert(3, "Partner IDs", t["City"].map(hit.groupby("City")["SoldToPartyID"].apply(lambda s: ", ".join(sorted(s.unique())[:20]))))
        rows.append(t)
    tables["Unmapped Cities"] = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    # 3) Project rows whose Sales Office is not in NS Branches
    project = base[base["Classification"] == "Project"].copy()
    if "SalesOfficeCode" in project.columns:
        project["Sales Office"] = project["SalesOfficeCode"].astype(str).str.strip().str.upper()
        miss = project[~project["Sales Office"].isin(set(project_branch_map))]
        tables["Unmapped Sales Offices"] = _sales_cols(miss, ["Sales Office"]) if not miss.empty else pd.DataFrame()

    # 4) Products missing from Material Master, or without Family Actual
    fam = family_map.set_index("Material")["Family Actual"]
    fam_val = base["MaterialCode"].map(fam)
    in_master = base["MaterialCode"].isin(fam.index)
    blank = fam_val.isna() | fam_val.astype(str).str.strip().isin(["", "nan", "None"])
    miss = base[~in_master | blank].copy()
    if not miss.empty:
        miss["Issue"] = np.where(miss["MaterialCode"].isin(fam.index), "No Family Actual", "Not in Material Master")
        t = _sales_cols(miss, ["MaterialCode", "Issue"])
        desc = miss.groupby("MaterialCode")["MaterialDescription"].last() if "MaterialDescription" in miss.columns else None
        t.insert(1, "Description", t["MaterialCode"].map(desc) if desc is not None else "")
        t.insert(2, "HeliosCode", t["MaterialCode"].map(miss.groupby("MaterialCode")["HeliosCode"].last()))
        tables["Unmapped Products"] = t
    else:
        tables["Unmapped Products"] = pd.DataFrame()

    duplicates = master_duplicates()
    sales_col = f"Gross Sales {TARGET_YEAR}"
    for k, t in tables.items():
        if not t.empty:
            tables[k] = t.sort_values(sales_col, ascending=False).reset_index(drop=True)
    summary = pd.DataFrame([{
        "Check": k, "Items to fix": len(t),
        f"Gross Sales {TARGET_YEAR} affected": float(t[sales_col].sum()) if len(t) else 0.0,
        f"Gross Sales {PREV_YEAR} affected": float(t[f"Gross Sales {PREV_YEAR}"].sum()) if len(t) else 0.0,
    } for k, t in tables.items()])
    summary = pd.concat([summary, pd.DataFrame([{
        "Check": "Duplicates in Master Data", "Items to fix": len(duplicates),
        f"Gross Sales {TARGET_YEAR} affected": np.nan, f"Gross Sales {PREV_YEAR} affected": np.nan}])], ignore_index=True)

    def build(writer):
        sub = f"Run {TODAY:%d-%b-%Y %H:%M} | data: {DATA_SOURCE} | H&D Helios codes, Sales + OB, {PREV_YEAR}–{TARGET_YEAR}"
        write_table(writer, summary, "Summary", title="Master-data review — items that did not map", subtitle=sub,
                    tab=THEME["tab_summary"], autofilter=False)
        for name, t in {**tables, "Duplicates in Master Data": duplicates}.items():
            found = not t.empty
            if not found:
                t = pd.DataFrame({"Result": ["Nothing to fix ✔"]})
            write_table(writer, t, name, title=name, subtitle=sub, freeze_cols=1,
                        tab=THEME["tab_tracker"] if found else THEME["tab_data"])

    path = save_report(build, f"Master_Data_Review_{TODAY:%Y-%m-%d}.xlsx", local_dir=OUTPUT_DIR)
    print("\n" + summary.to_string(index=False))
    for name, t in tables.items():
        if not t.empty:
            print(f"\n{name} (top 10 by {TARGET_YEAR} sales):")
            print(t.head(10).to_string(index=False, max_colwidth=40))
    if len(duplicates):
        print(f"\nDuplicates in Master Data ({len(duplicates)} keys — reports used the first row; fix them in the files for Tableau):")
        print(duplicates[["File", "Sheet", "Key value", "Times listed", "Excel rows", "Values differ?"]].head(20).to_string(index=False))
    return path


run_report("master_data_review", mapping_review)

## 10 · Target planning
Last year's sales contribution, and every Retail / HS / Wiser target organised into one file (`Channel_Targets_<year>.xlsx`), split by last year's sales.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 10 · TARGET PLANNING — last year's sales contribution + one organised targets file
#
# Reads the Retail targets (Master Targets File + QTOD and ATOD file) and the HS
# targets file, and writes to OUTPUT_DIR:
#   Sales_Contribution_<last year>.xlsx  how last year's sales split by branch,
#       month, activity, product group, Retail vs Project and partner
#   Channel_Targets_<year>.xlsx  every target in one long-format workbook for the
#       Retail, HS and Wiser dashboards, as given and split over product groups by
#       last year's sales:
#       Yearly Targets (year × branch × GTM × product group) · Monthly Targets (the
#       monthly commit: month × branch × GTM × product group; separate from the
#       yearly target, months not set stay empty) · Partner Targets (quarters, All /
#       Zeta / Wiser) · Partner Product Split · SSP Targets · Region Targets
#       · Issues (problems found in the source files) · Check (sheets vs source)
# Retail targets are the same rows for the Retail and the HS dashboard.
# Uses TS (cell 2.5). Source: tableau/targets/organize_targets.py.
# ════════════════════════════════════════════════════════════════════
TARGET_SPLIT_YEAR = TARGET_YEAR          # year of the targets; last year's sales are the basis
HS_TARGETS_PATH = f"{MASTER}/Master Targets File (Retail+Project).xlsx"   # HS dashboard's targets ("" = none)
TARGET_SPLIT_UPLOAD = False              # True = also upload both files to Final Reports on SharePoint


def target_split_sales():
    """Retail + Project sales as the HS dashboard sees them (branch, region, Retail / Project) + family."""
    base = build_hs_dataset(df_Sales_final, "InvoiceDate", **_dash_args)
    d = pd.to_datetime(base["InvoiceDate"], errors="coerce")
    fam = family_map.drop_duplicates("Material").set_index("Material")["Family Actual"]
    return pd.DataFrame({
        "Year": d.dt.year, "Month": d.dt.month, "GTM": base["Classification"], "Branch": base["Branch"],
        "Region": base["Region"], "HeliosCode": base["HeliosCode"],
        "Family": base["MaterialCode"].astype(str).str.strip().str.upper().map(fam),
        "Partner": base["SoldToPartyID"], "Gross": base["Gross Sales"]}).dropna(subset=["Year"])


def _read_book(path):
    return pd.read_excel(io.BytesIO(sp.download(path).getvalue()), sheet_name=None)


def report_target_split():
    year = TARGET_SPLIT_YEAR
    sales = target_split_sales()
    master_book = _read_book(PATHS["targets_master"])
    qtod = next(iter(_read_book(PATHS["targets_qtod"]).values()))
    hs = None
    if HS_TARGETS_PATH:
        try:
            hs = _read_book(HS_TARGETS_PATH)
        except Exception as e:
            print(f"   ⚠️ HS targets file not read ({e}) — Project targets left out")
    meta = Partner_Metadata.copy()
    meta["SoldToPartyID"] = meta["SoldToPartyID"].astype(str).str.strip()
    partner_info = {k: {"Branch": b, "RD/KRO": t} for k, b, t in zip(meta["SoldToPartyID"], meta["Branch"], meta["RD/KRO"])}

    tables, issues, given = TS.organize(master_book, qtod, sales, year, hs, partner_info)
    print(f"   🎯 {len(issues)} issue(s) in the target files — see the 'Issues' sheet")
    targets_file = f"Channel_Targets_{year}.xlsx"
    path = os.path.join(OUTPUT_DIR, targets_file)
    TS.write_targets(path, tables, issues, given, year, year - 1)
    print(f"   💾 {path}")
    if TARGET_SPLIT_UPLOAD:
        with open(path, "rb") as fh:
            sp.upload(f"{PATHS['final_reports']}/{targets_file}", fh.read())

    contribution = TS.contribution(TS.prepare_sales(sales), year - 1)

    def build(writer):
        for name, df in contribution.items():
            write_table(writer, df, name[:31], tab=THEME["tab_data"], freeze_cols=2)
    saved = save_report(build, f"Sales_Contribution_{year - 1}.xlsx", local_dir=OUTPUT_DIR,
                        sp_folder=PATHS["final_reports"] if TARGET_SPLIT_UPLOAD else None)
    return f"{targets_file} + {saved}"


run_report("target_split", report_target_split)

## 8 · Archive (not executed)
Superseded code kept for reference only. These are *raw* cells, so **Run All never runs them**. To reuse something, copy it into a code cell.

- Old SSP report (replaced by 6.1)
- Partner Analysis versions PTP_test3 / PTP_test4 / PTP_test3 + overrides (replaced by section 5; the partner-code remap from PTP_test4 is now global: `APPLY_PARTNER_CODE_REMAP`)
- Golden Outlet ID matching pipeline (was already disabled; needs `pip install rapidfuzz`)